# Folder Layout

In [ ]:
import os
os.makedirs("src", exist_ok=True)

In [ ]:
from pathlib import Path
folder = Path("src")
files = [
    "__init__.py",
    "environments.py"   # all environments and gymnasium wrapper
    "tilecoding.py",    # the function approximator
    "exploration.py",   # the nine exploration rules
    "agents.py",        # SARSA (lambda) & Q-Learning
    "runner.py",        # one training run 
    "sweep.py",         # many runs, in parallel, resumable
    "analysis.py",      # statistics
    "plots.py"          # figures
]
sub_folders = [
    "scripts",          
    "results",          # .pkl files 
    "figures",          # .png files
    "paper"             # .tex files
]
f_folder = Path("src/scripts")
script_files = [
    "tune.py",          # phase 1: choose hyperparameters
    "evaluate.py",      # phase 2: the real experiment
    "make_figures.py"   # phase 3: figures and tables
]

for file in files:
    (folder/file).touch(exist_ok=True)

for sub_folder in sub_folders:
    (folder/sub_folder).mkdir(exist_ok=True)

for script_file in script_files:
    (f_folder/script_file).touch(exist_ok=True)

# Building Gymnasium Wrapper & Environments

In [5]:
%%writefile src/environments.py

import numpy as np
import gymnasium as gym


TILE_BOUNDS = {
    "MountainCar-v0": None,

    "CartPole-v1": (
        np.array(
            [-4.8, -5.0, -0.41887903, -5.0],
            dtype=np.float64
        ),
        np.array(
            [4.8, 5.0, 0.41887903, 5.0],
            dtype=np.float64
        )
    ),

    "Acrobot-v1": None,

    "LunarLander-v3": None,
}


class GymEnv:
    """Adapts Gymnasium to a 4-tuple step interface."""

    def __init__(self, env_id, seed=None):
        self.env = gym.make(env_id)
        self.spec_name = env_id
        self.n_actions = int(self.env.action_space.n)

        ms = self.env.spec.max_episode_steps
        self.max_episode_steps = (
            int(ms) if ms is not None else 10_000
        )

        self._next_seed = seed

    def reset(self):
        obs, _ = self.env.reset(seed=self._next_seed)
        self._next_seed = None

        return np.asarray(
            obs,
            dtype=np.float64
        )

    def step(self, action):
        obs, reward, terminated, truncated, _ = (
            self.env.step(int(action))
        )

        return (
            np.asarray(obs, dtype=np.float64),
            float(reward),
            bool(terminated),
            bool(truncated)
        )

    def tile_bounds(self):
        custom = TILE_BOUNDS.get(self.spec_name)

        if custom is not None:
            low, high = custom

            return (
                low.copy(),
                high.copy()
            )

        low = np.asarray(
            self.env.observation_space.low,
            dtype=np.float64
        ).copy()

        high = np.asarray(
            self.env.observation_space.high,
            dtype=np.float64
        ).copy()

        if (
            not np.isfinite(low).all()
            or not np.isfinite(high).all()
        ):
            raise ValueError(
                f"No finite tile-coding bounds defined "
                f"for {self.spec_name}."
            )

        return low, high

Overwriting src/environments.py


# Tile Coder

## The IHT Class
It does not do tile coding itself, rather it manages the mapping of a *which tile it hit* to *where in the weight array represents the tile*.

IHT = Index Hash Table

The object will remember:

* the maximum number of available feature indices,
* which tiles have already been seen,
* which index each tile received,
* how often the table overflowed.

\_\_slots\_\_: This is a Python **feature** which says that objects of the class can only have the attributes passed as its argument. This also helps to reduce memory overhead (as only the passed arguments can be attributed) and protect from naming errors too. 

\_d: Python dictionary, initiated as empty inside \_\_init()\_\_. This will store the tile key and allocated integer index pair. The `\_' in a variable name is a convention to indicate that the variable is not truly private; it can be accessed but should not need to, by other parts of the program. 

\_\_len(self)\_\_: It enables using len() to get the size of \_d, without having to explicitly address it. For instance, say, iht = IHT(4096), and there have been 3 allocations to `\_d'. Then, len(iht) = 3. 

@property: It enables to use the function following it to be used as an attribute. So, fullness can be used as iht.fullness instead of iht.fullness().

fullness(self): This calculates the fraction of the IHT capacity already occupied. 

get_index(): This function returns the weight index associated with the tile indicated by `key'. The workflow of the function can be summarized as: 

```
Have I seen this tile before?
            │
       ┌────┴────┐
      yes        no
       │          │
return old       Is table full?
index             │
             ┌────┴────┐
            no         yes
             │           │
      allocate new    collision/
         index          modulo
```

d.get(): The get() is an useful function for Python dictionaries. It fetches the other attribute's value associated with its argument from the dictionary. For example, let d = {
    2: 5
    3: 29
    7: 11
}
Then d.get(3) returns 29. However, d.get(1) would return `None' as the key 1 is not present in d. 


In [ ]:
%%writefile src/tilecoding.py

class IHT:
    """Index Hash Table: maps tile coordinates to dense indices in [0, size).

    Collision-free until `size` distinct tiles have been seen; afterwards
    new tiles wrap by modulo.
    """

    __slots__ = ("size", "_d", "overfull_count")

    def __init__(self, size):
        self.size = int(size)
        self._d = {}
        self.overfull_count = 0

    def __len__(self):
        return len(self._d)

    @property
    def fullness(self):
        return len(self._d) / self.size

    def get_index(self, key):
        d = self._d
        idx = d.get(key)

        if idx is not None:
            return idx

        n = len(d)

        if n >= self.size:
            self.overfull_count += 1
            return key % self.size

        d[key] = n
        return n

Overwriting src/tilecoding.py


## The Tile Coder

This particular code does not yet encode a state. It is the constructor: it prepares the geometry, scales, offsets, and hash table that the later methods will use.

```class TileCoder:``` An object of this class will know:
* the state-space boundaries,
* the number of state dimensions,
* how many tilings exist,
* how many tiles exist per dimension,
* how continuous values should be scaled,
* how each tiling is offset,
* which IHT maps tile coordinates to feature indices.

memory_size: It is the size of the IHT (the maximum dictionary (_d) size). 

low & high: These are the lower & higher bounds of the dimensions. For instance, in case of Mountaincar, low = [-1.2, 0.07] which is the lowest possible value of position & velocity respectively. `self.low.shape[0]' gives the number of dimensions of the states. 

if np.isscalar()...else: This check enables either assigning the same number of tiles per dimension (if condition) or varying number (else condition).

np.full(): This has the syntax ```np.full(size, value)```. It creates an np array of size ```size``` with all elements having the value ```value```. 

span: One thing to note is that this is an array not a scalar for multidimensional states (since low & high are arrays). Now the span[span==0] = 1.0 is a zero span safety. It turns those entries for which span = 0 to 1, leaving the other dimension spans untouched. 

scale: It converts the physical state dimension values into tile coordinate values. It is constructed by elementwise division of tpd & span. This in words mean *how many tiles per unit of dimension is allocated in the tile coordinate corresponding to that dimension*. 

x[None,:]: This turns the preceding 1D array into a 2D row matrix of size (1,n(x)). Similarly, x[:,None] turns it into a column matrix. 

In [ ]:
%%writefile -a src/tilecoding.py

class TileCoder:
    def __init__(self, low, high, num_tilings=8, tiles_per_dim=8, memory_size=4096):
        self.low = np.asarray(low, dtype=np.float64)
        self.high = np.asarray(high, dtype=np.float64)
        self.dim = int(self.low.shape[0])
        self.num_tilings = int(num_tilings)

        if np.isscalar(tiles_per_dim):
            tpd = np.full(self.dim, int(tiles_per_dim), dtype=np.int64)
        else:
            tpd = np.asarray(tiles_per_dim, dtype=np.int64)
        
        self.tiles_per_dim = tpd

        span = self.high - self.low
        span[span == 0] = 1.0
        self.scale = tpd / span

        self.iht = IHT(memory_size)
        self.memory_size = int(memory_size)

        t_idx = np.arange(self.num_tilings, dtype=np.int64)
        odd = (1 + 2 * np.arange(self.dim, dtype=np.int64))[None, :]
        self._offsets = t_idx[:, None] * odd


Appending to src/tilecoding.py


## The Hot Function

```indices(state)``` takes one continuous state, finds the one active tile in each tiling, and converts those tiles into integer feature indices using the IHT.

```//```: This is integer floor division. So, 62//8 = 7, not 7.75.

```get = self.iht.get_index```: This creates a local variable copy of self.iht.get_index(). Now simply get(key) can be used. Also, since this will be called millions of time during training, this copy helps python to just resolve it once instead of calling it each time. 

```*```: The \* in \*coords[t].tolist() means take the elements out of this iterable and insert them individually here. So, say t = 2 and coords[t] = array([3, 5]). Then (to, \*coords[t].tolist()) becomes (2, \*[3, 5]) = (2, 3, 5).

```hash()```: The hash() function converts its argument tuple into a Python hash integer. This conversion to *integer* is necessary as return key % self.size only works if key is an integer. 

In [10]:
%%writefile -a src/tilecoding.py

    def indices(self, state):
        """Return the `num_tilings` active feature indices for `state`"""

        s = np.clip(np.asarray(state, dtype=np.float64), self.low, self.high)
        q = np.floor((s-self.low) * self.scale * self.num_tilings).astype(np.int64)

        coords = (q[None, :] + self._offsets) // self.num_tilings

        out = np.empty(self.num_tilings, dtype=np.int64)
        get = self.iht.get_index
        for t in range(self.num_tilings):
            out[t] = get(hash((t, *coords[t].tolist())))
        return out

Appending to src/tilecoding.py


## The Linear Action-Value Function

This class is the part that turns the active tile indices from TileCoder.indices(state) into actual action-value estimates $\hat{q}(s, a)$ using linear function approximation. 

```sum(axis=1)```: This sums across columns of the 2D array. 

In [11]:
%%writefile -a src/tilecoding.py

class LinearQ:
    """Q(s,a) = sum of w[a, i] over the active features of s (state)"""

    __slots__ = ("w", "n_actions", "n_features")

    def __init__(self, n_actions, n_features, init=0.0):
        self.n_actions = int(n_actions)
        self.n_features = int(n_features)
        self.w = np.full((n_actions, n_features), float(init), dtype=np.float64)

    def q_all(self, idx):
        """Q(s, .) for every action -> shape (n_actions, )"""
        return self.w[:, idx].sum(axis=1)

Appending to src/tilecoding.py


## The Configuration Dictionary

```cfg```: This is a nested dictionary. So, return dict(cfg[env_id]) returns a dictionary associated with the passed environment name. 

In [12]:
%%writefile -a src/tilecoding.py

def default_tiling_config(env_id):
    cfg = {
        "MountainCar-v0": dict(num_tilings=8, tiles_per_dim=8, memory_size=4096),
        "CartPole-v1": dict(num_tilings=8, tiles_per_dim=6, memory_size=32768),
        "Acrobot-v1": dict(num_tilings=8, tiles_per_dim=6, memory_size=131072),
        "LunarLander-v3": dict(num_tilings=16, tiles_per_dim=4, memory_size=262144),
    }

    return dict(cfg[env_id])

Appending to src/tilecoding.py


## Quick Check

Same states have identical indices

In [ ]:
import src.tilecoding as TC

tc = TC.TileCoder([-1.2, -0.07], [0.6, 0.07], 8, 8, 4096)

i1 = tc.indices([-0.5, 0.01])
i2 = tc.indices([-0.5, 0.01])

assert len(i1) == 8
assert (i1==i2).all()
assert (i1>=0).all() and (i1<4096).all()
print(i1)

[0 1 2 3 4 5 6 7]


Nearby states have some common indices

In [ ]:
import src.tilecoding as TC

tc = TC.TileCoder([-1.2, -0.07], [0.6, 0.07], 8, 8, 4096)

i1 = tc.indices([-0.5, 0.01])
i2 = tc.indices([-0.49, 0.011])

assert len(i1) == 8
assert (i1==i2).any()
assert (i1!=i2).any()

assert (i1>=0).all() and (i1<4096).all()
assert (i2>=0).all() and (i2<4096).all()

print(i1)
print(i2)


[0 1 2 3 4 5 6 7]
[0 8 2 3 4 5 6 9]


Distant states have no common indices

In [ ]:
import src.tilecoding as TC

tc = TC.TileCoder([-1.2, -0.07], [0.6, 0.07], 8, 8, 4096)

i1 = tc.indices([-0.5, 0.01])
i2 = tc.indices([-0.9, -0.06])

assert len(i1) == 8
assert (i1!=i2).all()

assert (i1>=0).all() and (i1<4096).all()
assert (i2>=0).all() and (i2<4096).all()

print(i1)
print(i2)


[0 1 2 3 4 5 6 7]
[ 8  9 10 11 12 13 14 15]


# Exploration

## The Explorer

It is the common base/interface that all nine exploration rules will follow.The main idea is that **the agent asks the explorer for an action, instead of it knowing which exploration strategy it is following**. This enables the agent to remain identical, while only the exploration object is swapped. Every explorer exposes the same interface.
```
Agent
  │
  │ asks for action
  ▼
Explorer
  │
  ├── DecayExplorer
  ├── VDBEExplorer
  ├── BoltzmannExplorer
  ├── ProposedExplorer
  ├── TileUCBExplorer
  └── ...
```
```name``` is a class attribute, while ```self.t``` is an instance attribute, associated with an individual object. 

```def _epsilon_now(self, feat_idx=None)```: This defines the rule for selecting the epsilon to be used now. 

```raise NotImplementedError```: The base Explorer does not know how epsilon should be calculated.Different  subclasses have different answers. Therefore the base class deliberately says: *I cannot implement this. A subclass must tell me what _epsilon_now() means.*

```rng.random()``` produces a random number uniformly in [0,1). 

Till now conceptually, one step looks like:
```
Current state s
       │
       ▼
TileCoder.indices(s)
       │
       ▼
LinearQ.q_all(idx)
       │
       ▼
[Q(s,0), Q(s,1), Q(s,2)]
       │
       ▼
Explorer.select(...)
       │
       ▼
choose action a
       │
       ▼
Environment.step(a)
       │
       ▼
reward r, next state s'
       │
       ▼
compute TD error δ
       │
       ├──────────────► update LinearQ weights
       │
       └──────────────► explorer.update(δ, ...)
```

Summary of this class:
```
| Part              | Purpose                                             |
| ----------------- | --------------------------------------------------- |
| `name`            | identifies the strategy                             |
| `uses_features`   | says whether tile features are required             |
| `n_actions`       | number of available actions                         |
| `t`               | number of action-selection calls                    |
| `current_epsilon` | most recently used \(\epsilon\)                     |
| `_epsilon_now()`  | compute current exploration level                   |
| `update()`        | receive TD/value/feature information after learning |
| `reset_episode()` | reset strategy-specific episode state               |
| `select()`        | perform epsilon-greedy action selection             |
```

## The Tie-breaker

Its job is:

Find the action(s) having the largest Q-value. If there is only one, choose it. If several actions are tied for maximum, randomly choose one of those tied actions.

```q == m```: This performs element by element comparison through an array. Since q is an array and m is the maximum value, each element in q is matched withm. The resulting array ties will have *False* where matches were not found, and *True* elsewhere. 

```np.flatnonzero()``` returns the indices where the input is nonzero/True.

In [ ]:
%%writefile src/exploration.py

import numpy as np

def argmax_random_tie(q, rng):
    """Argmax with ties broken uniformly at random"""

    m = q.max()
    ties = np.flatnonzero(q == m)
    if ties.size == 1:
        return int(ties[0])
    return int(ties[rng.integers(ties.size)])

Overwriting src/exploration.py


In [ ]:
%%writefile -a src/exploration.py

class Explorer:
    name = "base"
    uses_features = False

    def __init__(self, n_actions):
        self.n_actions = int(n_actions)
        self.t = 0
        self.current_epsilon = 0.0

    def _epsilon_now(self, feat_idx=None):
        raise NotImplementedError
    
    def update(self, td_error, value=0.0, feat_idx=None):
        return None
    
    def reset_episode(self):
        return None
    
    def select(self, q_values, rng, feat_idx=None):
        eps = self._epsilon_now(feat_idx)
        self.current_epsilon = eps
        self.t += 1
        if rng.random() < eps:
            return int(rng.integers(self.n_actions))
        return argmax_random_tie(q_values, rng)

Appending to src/exploration.py


## Schedules


### Fixed $\epsilon$

```super().__init__(n_actions)```: As FixedEpsilon has its own ```__init__()```, ```super().__init__()``` is needed to tell Python to run its parent's one (*Explorer's ```__init__()```*). The flow summarized:
```
FixedEpsilon.__init__()
        ↓
super().__init__(3)
        ↓
Explorer.__init__(3)
        ↓
self.n_actions = 3
self.t = 0
self.current_epsilon = 0.0
```

In [ ]:
%%writefile -a src/exploration.py

class FixedEpsilon(Explorer):
    name = "fixed"
    
    def __init__(self, n_actions, epsilon=0.1):
        super().__init__(n_actions)
        self.epsilon = float(epsilon)

    def _epsilon_now(self, feat_idx=None):
        return self.epsilon

Appending to src/exploration.py


### Decay $\epsilon$

The $\epsilon$ decays by the formula: 
\begin{equation}
    \epsilon_{end} = \epsilon_{start}\,e^{rt}
\end{equation}

The max() in the log and numerator are protection mechanisms against unwanted zeros. 

In [ ]:
%%writefile -a src/exploration.py

class DecayEpsilon(Explorer):
    name = "decay"

    def __init__(self, n_actions, eps_start=1, eps_end=0.01, decay_steps=50_000, mode="exponential"):
        super().__init__(n_actions)
        self.eps_start, self.eps_end = float(eps_start), float(eps_end)
        self.decay_steps, self.mode = int(decay_steps), mode
        self._rate = np.log(max(eps_end, 1e-12)/eps_start)/max(decay_steps, 1)

    def _epsilon_now(self, feat_idx=None):
        if self.mode == "linear":
            frac = min(1.0, self.t / max(self.decay_steps, 1))
            return self.eps_start + frac * (self.eps_end - self.eps_start)

        return float(
            max(
                self.eps_end,
                self.eps_start * np.exp(self._rate * self.t)
            )
        )

Appending to src/exploration.py


### Boltzmann

Boltzmann exploration does not make the explore/exploit split. Instead, every action gets a probability based on its Q-value:
\begin{equation}
    P(a|s) = \cfrac{e^{\frac{Q(s,a)-Q_{max}(s,a)}{\tau}}}{\sum_b e^{\frac{Q(s,b)-Q_{max}(s,b)}{\tau}}}
\end{equation}

It then selects the non-greedy actions by probability $1-P(\text{greedy})$. 

```choice()``` selects the action number according to the probabilities specified by `p'.


In [ ]:
%%writefile -a src/exploration.py

class Boltzmann(Explorer):
    name = "boltzmann"

    def __init__(self, n_actions, tau_start=1.0, tau_end=0.05, decay_steps=50_000):
        super().__init__(n_actions)
        self.tau_start, self.tau_end = float(tau_start), float(tau_end)
        self.decay_steps = int(decay_steps)
        self._rate = np.log(max(tau_end, 1e-12)/tau_start)/max(decay_steps, 1)

    def select(self, q_values, rng, feat_idx=None):
        tau = float(max(self.tau_end, self.tau_start * np.exp(self._rate * self.t)))
        self.t += 1
        z = np.clip((q_values - q_values.max()) / max(tau, 1e-8), -50.0, 0.0)
        p = np.exp(z)
        p /= p.sum()
        self.current_epsilon = float(1.0 -p[int(np.argmax(q_values))])
        return int(rng.choice(self.n_actions, p=p))

Appending to src/exploration.py


### VDBE

Unlike FixedEpsilon or DecayEpsilon, its exploration probability is adaptive: it changes according to the magnitude of the TD error.

```delta_param```: This is the $\cfrac{1}{|A|}$ in the $\epsilon$ update formula. 

```@staticmethod```: This tells that the following function belongs conceptually to the VDBE class, but does not require an instance (self). It also does not read variables like self.sigma, self.n_actions etc.

```_f(x)```: min(x, 50.0) is used since exp(-50) is already effectively zero, so exp(-x) for large x is overkill. 

Summarized workflow:
```
                 current state
                      │
                      ▼
                   Q values
                      │
                      ▼
             VDBE current epsilon
                      │
                      ▼
           Explorer.select() chooses
                    action
                      │
                      ▼
                 environment
                      │
                      ▼
              reward + next state
                      │
                      ▼
              compute TD error δ
                      │
                      ▼
                VDBE.update()
                      │
          ┌───────────┴───────────┐
          ▼                       ▼
    compute |αδ|/σ             old epsilon
          │                       │
          ▼                       │
        f(x)                      │
          │                       │
          └───────────┬───────────┘
                      ▼
        ε ← d f + (1-d) ε
                      │
                      ▼
          used on next action
```

In [ ]:
%%writefile -a src/exploration.py

class VDBE(Explorer):
    name = "vdbe"

    def __init__(self, n_actions, sigma=1.0, eps_init=1.0, alpha_scale=1.0, eps_min=0.0):
        super().__init__(n_actions)
        self.sigma = float(sigma)
        self.eps = float(eps_init)
        self.delta_param = 1.0 / self.n_actions
        self.alpha_scale = float(alpha_scale)
        self.eps_min = float(eps_min)

    def _epsilon_now(self, feat_idx=None):
        return max(self.eps_min, self.eps)

    @staticmethod
    def _f(x):
        e = np.exp(-min(x, 50.0))
        return float((1.0 - e) / (1.0 + e))
    
    def update(self, td_error, value=0.0, feat_idx=None):
        f = self._f(abs(self.alpha_scale * float(td_error)) / max(self.sigma, 1e-12))
        d = self.delta_param
        self.eps = d * f + (1.0 - d) * self.eps

Appending to src/exploration.py


### VDBEState

The per-feature variant of VDBE

In [10]:
%%writefile -a src/exploration.py

class VDBEState(VDBE):
    name = "vdbe-state"
    uses_features = True

    def __init__(self, n_actions, n_features, sigma=1.0, eps_init=1.0,
                 alpha_scale=1.0, eps_min=0.0):
        super().__init__(n_actions, sigma, eps_init, alpha_scale, eps_min)

        self.eps_features = np.full(
            int(n_features),
            float(eps_init),
            dtype=np.float64
        )

    def _epsilon_now(self, feat_idx=None):
        if feat_idx is None:
            return max(self.eps_min, self.eps)

        return max(
            self.eps_min,
            float(self.eps_features[feat_idx].mean())
        )

    def update(self, td_error, value=0.0, feat_idx=None):
        f = self._f(
            abs(self.alpha_scale * float(td_error))
            / max(self.sigma, 1e-12)
        )

        d = self.delta_param

        self.eps = d * f + (1.0 - d) * self.eps

        if feat_idx is None:
            return

        self.eps_features[feat_idx] = (
            d * f
            + (1.0 - d) * self.eps_features[feat_idx]
        )

Appending to src/exploration.py


### RATE

RATE compares two moving quantities:

$ m_{\delta} \approx \text{recent average magnitude of TD errors} $

and

$m_v \approx \text{recent average magnitude of the current value estimate}$

Then, $\rho=\cfrac{m_{\delta}}{m_{\delta}+m_v}$

and, $\epsilon=\epsilon_{min} + (\epsilon_{max}-\epsilon_{min})\rho^{\kappa}$

In [11]:
%%writefile -a src/exploration.py

class RATE(Explorer):
    name = "rate"

    def __init__(self, n_actions, eps_min=0.01, eps_max=1.0, beta=0.01, kappa=1.0):
        super().__init__(n_actions)
        self.eps_min, self.eps_max = float(eps_min), float(eps_max)
        self.beta, self.kappa = float(beta), float(kappa)
        self.m_d = 0.0
        self.m_v = 0.0
        self.rho = 1.0

    def _epsilon_now(self, feat_idx=None):
        return self.eps_min + (self.eps_max - self.eps_min) * (self.rho ** self.kappa)
    
    def update(self, td_error, value=0.0, feat_idx=None):
        b = self.beta
        self.m_d += b * (abs(float(td_error)) - self.m_d)
        self.m_v += b * (abs(float(value)) - self.m_v)
        den = self.m_d + self.m_v
        self.rho = (self.m_d / den) if den > 1e-8 else 1.0

Appending to src/exploration.py


### RATEState

This is the per-feature version of RATE, and it is more interesting than the global RATE because it makes exploration depend on which part of the state space the agent is currently in. RATEState adds a separate TD-error statistic for every tile-coded feature.

One thing to note is that the inheritance chain here is different: 
```
Explorer
   ↑
 RATE
   ↑
RATEState
```

Let, ```feat_idx = np.array([13, 82, 205, 377, 612, 901, 1205, 1600])```. Those might be the eight active tile features for the current state. RATEState uses those indices to determine the local exploration level.

We need `\_seen' because, M[i] = 0 is ambiguous. Does it mean: 

This feature has been visited and its error really is approximately zero?

or:

This feature has never been visited and was merely initialized to zero?

Those are completely different situations. So `\_seen' distinguishes them.

```_epsilon_now()```: 
```
feat_idx absent
    ↓
use global RATE

feat_idx provided
    ↓
use feature-specific RATE
```
```seen = self._seen[feat_idx]```: This is a Boolean mask corresponding to the currently active features (e.g. array([True, True, False, True])). Note that `seen' is not the entire `\_seen' array. It only contains the features associated with the current state. If no active features were seen previously, then $\epsilon = \epsilon_{max} = 1$ is set. This makes sense--- 
```
completely unfamiliar region
        ↓
maximum exploration
```

```local```: This is the average stored TD-error magnitude among the currently active, previously seen features. So, it's a local estimate of how uncertain/unsettled learning appears in this part of the state space.

Why ignore unseen features rather than treating them as zero? Because zero would incorrectly mean: *perfect prediction.*

The $m_{\delta}$ in $\rho$ has been replaced by `local' for this case. 

```super().update()```: This calls the RATE's update function. So RATEState still maintains the global statistics too. Why? Because: $m_v$ is still needed for local normalization, global $\rho$ is needed as fallback when no feat_idx is supplied.

```fresh```: This tells us: which currently active features have never been seen before?

Every previously seen active feature is updated according to: 

$M_i \leftarrow M_i + \beta (|\delta| - M_i)$. 

While the fresh features are initialized to $|\delta|$.

In [12]:
%%writefile -a src/exploration.py

class RATEState(RATE):
    name = "rate-state"
    uses_features = True

    def __init__(self, n_actions, n_features, eps_min=0.01, eps_max=1.0, beta=0.01, kappa=1.0):
        super().__init__(n_actions, eps_min, eps_max, beta, kappa)
        self.M = np.zeros(int(n_features), dtype=np.float64)
        self._seen = np.zeros(int(n_features), dtype=bool)

    def _epsilon_now(self, feat_idx=None):
        if feat_idx is None:
            return self.eps_min + (self.eps_max - self.eps_min) * (self.rho ** self.kappa)
        seen = self._seen[feat_idx]
        if not seen.any():
            return self.eps_max
        local = float(self.M[feat_idx][seen].mean())
        den = local + self.m_v
        rho = (local/den) if den > 1e-8 else 1.0
        return self.eps_min + (self.eps_max - self.eps_min) * (rho ** self.kappa)

    def update(self, td_error, value=0.0, feat_idx=None):
        super().update(td_error, value, feat_idx)
        if feat_idx is None:
            return
        a = abs(float(td_error))
        fresh = ~self._seen[feat_idx]
        if fresh.any():
            f = feat_idx[fresh]
            self.M[f] = a
            self._seen[f] = True
        old = feat_idx[~fresh]
        if old.size:
            self.M[old] += self.beta * (a - self.M[old])

Appending to src/exploration.py


### TileUCB
```N[a,i]```: This is the number of times action `a' was taken while feature `i' was active. 

Because UCB bonuses need to be on an appropriate scale relative to your Q-values, ```relative=True``` lets the bonus scale automatically with typical value magnitude.

If ```feat_idx``` is None, action is selected greedily with random tie-braking. 

```self.N[:, feat_idx]```: This selects only the currently active features of the count array. 

TileUCB says:

I don't need TD error to tell me whether something is uncertain. If I have hardly tried an action in this region, I should investigate it.

Workflow:
```
                         current state s
                               │
                               ▼
                       TileCoder.indices(s)
                               │
                               ▼
                         active feat_idx
                               │
                               ▼
                   N[:, feat_idx]
                               │
                               ▼
              average counts for each action
                               │
                               ▼
                     n(s,0), n(s,1), ...
                               │
                               │
         global time t ────────┤
                               │
         value EMA m_v ────────┤
                               ▼
            bonus(a) = c × scale ×
                  sqrt(log(t+1)/(n(s,a)+1))
                               │
                               ▼
                         Q(s,a)+bonus(a)
                               │
                               ▼
                       choose best action
                               │
                               ▼
                N[selected_action, features] += 1
                               │
                               ▼
                  that action's future bonus
                    gradually becomes smaller
```

In [13]:
%%writefile -a src/exploration.py

class TileUCB(Explorer):
    name = "tile-ucb"
    uses_features = True

    def __init__(self, n_actions, n_features, c=0.5, beta=0.01, relative=True, epsilon=0.0):
        super().__init__(n_actions)
        self.N = np.zeros((int(n_actions), int(n_features)), dtype=np.float64)
        self.c, self.beta = float(c), float(beta)
        self.relative = bool(relative)
        self.epsilon = float(epsilon)
        self.m_v = 0.0
    
    def select(self, q_values, rng, feat_idx=None):
        self.t += 1
        if feat_idx is None:
            self.current_epsilon = self.epsilon
            return argmax_random_tie(q_values, rng)
        n_sa = self.N[:, feat_idx].mean(axis=1)
        scale = self.m_v if self.relative else 1.0
        bonus = self.c * scale * np.sqrt(np.log(self.t + 1.0) / (n_sa + 1.0))
        if self.epsilon > 0.0 and rng.random() < self.epsilon:
            a = int(rng.integers(self.n_actions))
        else: 
            a = argmax_random_tie(q_values + bonus, rng)
        self.N[a, feat_idx] += 1.0
        self.current_epsilon = float (a != int(np.argmax(q_values)))
        return a
    
    def update(self, td_error, value=0.0, feat_idx=None):
        self.m_v += self.beta * (abs(float(value)) - self.m_v)

Appending to src/exploration.py


### The Registry

The registry is basically a lookup table + factory function that lets the rest of the project create an exploration strategy just by giving its name as a string.

```make_explorer()```: This is a **factory function**. A factory function is simply: A function whose job is to construct and return an object.

```**kw```: It collects any additional named arguments into a dictionary. For example, 
```
make_explorer(
    "rate",
    n_actions=3,
    eps_min=0.01,
    eps_max=1.0,
    beta=0.02,
    kappa=2.0
)
```

Here, ```kw``` becomes---
```
{
    "eps_min": 0.01,
    "eps_max": 1.0,
    "beta": 0.02,
    "kappa": 2.0
}
```

In [14]:
%%writefile -a src/exploration.py

REGISTRY = {
    "fixed": FixedEpsilon,
    "decay": DecayEpsilon,
    "decay-linear": DecayEpsilon,
    "boltzmann": Boltzmann,
    "vdbe": VDBE,
    "vdbe-state": VDBEState,
    "rate": RATE,
    "rate-state": RATEState,
    "tile-ucb": TileUCB,
}

NEEDS_FEATURES = ("rate-state", "vdbe-state", "tile-ucb")


def make_explorer(kind, n_actions, n_features=None, **kw):
    cls = REGISTRY[kind]

    if kind in NEEDS_FEATURES:
        return cls(n_actions=n_actions, n_features=n_features, **kw)

    return cls(n_actions=n_actions, **kw)

Appending to src/exploration.py


### Quick Check

In [22]:
import numpy as np
import importlib
import src.exploration as EXP

EXP = importlib.reload(EXP)

rng = np.random.default_rng(0)
q = np.array([0.1, 0.3, 0.2])
for k in EXP.REGISTRY:
    kw = {"mode": "linear"} if k == "decay-linear" else {}
    ex = EXP.make_explorer(k, 3, n_features=4096, **kw)
    a = ex.select(q, rng, np.arange(8))
    ex.update(0.5, 1.0, np.arange(8))
    assert 0 <= a < 3, k
print("all", len(EXP.REGISTRY), "explorers OK")

all 9 explorers OK


# Agents

## The Base Agent

This is the base class for the RL agents that use linear function approximation.

```__init__()``` constructor parameters are summarized below:
```
coder         → TileCoder object
n_actions     → number of actions
explorer      → exploration object
alpha_bar     → effective learning-rate parameter
gamma         → discount factor
lam           → eligibility-trace decay parameter λ
trace_cutoff  → threshold for deleting tiny traces
q_init        → initial Q-weight value
```

Another useful flow diagram to understand the role of LinearQ:
```
continuous observation
        ↓
    TileCoder
        ↓
active feature indices
        ↓
    LinearQ
        ↓
    Q-values
```
```self.Q.w``` is originally a 2D matrix ($|A|$, $F$). ```.reshape(-1)``` turns it into a one-dimensional vector of the size ($|A|\times$ n_features). The `-1' tells NumPy to identify the required size automatically.This is needed as later eligibility traces want a single trace vector. Note that, despite the flatten weight vector, any weight of an action can be addressed using the flat indexing method:
\begin{equation}
    i = n_{features} \times N_a + N_{feature}
\end{equation}

So this constructor is basically assembling all the machinery that the actual learning algorithm will use:
```
             observation
                 ↓
              coder
                 ↓
          active features
                 ↓
       ┌─────────┴────────┐
       ↓                  ↓
    LinearQ            traces z
       ↓                  ↓
   Q(s, actions)      credit assignment
       │
       ▼
    explorer
       │
       ▼
      action
       │
       ▼
   environment
   ```

In [3]:
%%writefile src/agents.py

import numpy as np

from dataclasses import dataclass
from .tilecoding import LinearQ
from .exploration import argmax_random_tie


@dataclass
class EpisodeRecord:
    ret: float
    length: int
    epsilon_mean: float
    td_abs_mean: float
    total_steps: int

class LinearAgent:
    def __init__(self, coder, n_actions, explorer, alpha_bar=0.5, gamma=1.0, lam=0.9, trace_cutoff=1e-3, q_init=0.0):
        self.coder = coder
        self.n_actions = int(n_actions)
        self.explorer = explorer
        self.num_tilings = coder.num_tilings
        self.alpha = float(alpha_bar) / self.num_tilings
        self.alpha_bar = float(alpha_bar)
        self.gamma, self.lam = float(gamma), float(lam)
        self.trace_cutoff = float(trace_cutoff)

        self.Q = LinearQ(n_actions, coder.n_features, init=q_init)
        self._wf = self.Q.w.reshape(-1)
        self.n_features = coder.n_features

        self.z = np.zeros(self.n_features * self.n_actions, dtype=np.float64)
        self.active = np.zeros(0, dtype=np.int64)
        self.total_steps = 0

Overwriting src/agents.py


## Sparse Traces
```def _flat(self, a, idx)```: Gives the flat index of the flattened weight vector from last step. 

```_reset_traces()```: This is called at the start of new episodes. Eligibility traces should not carry over between episodes. So we wipe them.

```keeps```: A boolean mask. Say, vals = [0.3, 0.7, 0.1] and trace_cutoff = 0.2. Then, 0.1 does not qualify. Hence, keep = [True, True, False]. 

```new = np.concatanate()```: This gives the indices of the features that were just modified inside ```_set_replacing_traces()```.

```_apply_updat()```: Performs the update on recently visited active features' weights by: 
\begin{equation}
    w = w + \alpha\delta z
\end{equation}

In [4]:
%%writefile -a src/agents.py

    def _flat(self, a, idx):
        return a * self.n_features + idx
    
    def _reset_traces(self):
        if self.active.size:
            self.z[self.active] = 0.0
        self.active = np.zeros(0, dtype=np.int64)
    
    def _decay_traces(self):
        if not self.active.size:
            return
        self.z[self.active] *= self.gamma * self.lam
        vals = self.z[self.active]
        keep = vals >= self.trace_cutoff
        if not keep.all():
            self.z[self.active[~keep]] = 0.0
            self.active = self.active[keep]
    
    def _set_replacing_traces(self, a, idx):
        base = idx
        for other in range(self.n_actions):
            f = other * self.n_features + base
            if other == a:
                self.z[f] = 1.0
            else:
                self.z[f] = 0.0
        new = np.concatenate(
            [np.asarray(o * self.n_features + base, dtype = np.int64)
            for o in range(self.n_actions)])
        self.active = np.union1d(self.active, new)

    def _apply_update(self, delta):
        if self.active.size:
            self._wf[self.active] += self.alpha * delta * self.z[self.active]

Appending to src/agents.py


## The Learning Loop

```SarsaLambdaAgent``` inherits from the class `LinearAgent'. 

This is the actual Sarsa($\lambda$) learning loop. Everything from 9.2 and 9.3 now gets connected:

* TileCoder turns observations into active features.
* LinearQ gives \(Q(s,a)\).
* Explorer chooses actions.
* eligibility traces assign credit backward.
* the TD error updates the weights.
* terminated versus truncated determines whether we bootstrap.

Because the current state-action pair must receive the current TD error with eligibility: 1, traces are replaced before the weight update. 

Complete SARSA ($\lambda$) sequence:
```
START EPISODE
      │
      ▼
 clear traces
reset explorer
      │
      ▼
      S₀
      │
  tile coding
      ▼
   Q(S₀, ·)
      │
   explorer
      ▼
      A₀
      │
      ▼
┌──────────────────────────────────────┐
│                                      │
│    execute Aₜ                         │
│        ↓                             │
│ observe Rₜ₊₁, Sₜ₊₁                    │
│        ↓                             │
│  compute Q(Sₜ,Aₜ)                     │
│        ↓                             │
│ δ = Rₜ₊₁ − Q(Sₜ,Aₜ)                    │
│        ↓                             │
│ set replacing traces for (Sₜ,Aₜ)      │
│        │                             │
│        ├── terminated? ── yes ───────┤
│        │                    ↓        │
│        │              no bootstrap   │
│        │              update weights │
│        │              update explorer│
│        │              END            │
│        │                             │
│        no                            │
│        ↓                             │
│ encode Sₜ₊₁                           │
│ compute Q(Sₜ₊₁,·)                     │
│ choose Aₜ₊₁                           │
│        ↓                             │
│ δ += γQ(Sₜ₊₁,Aₜ₊₁)                    │
│        ↓                             │
│ update weights using δz              │
│ update explorer                      │
│        ↓                             │
│ truncated? ─ yes → END               │
│        │                             │
│        no                            │
│        ↓                             │
│     z ← γλz                          │
│        ↓                             │
│     Sₜ₊₁ → Sₜ                         │
│     Aₜ₊₁ → Aₜ                          │
│        │                             │
└────────┴──────── repeat ──────────────┘
```

In [5]:
%%writefile -a src/agents.py

class SarsaLambdaAgent(LinearAgent):
    algo = "sarsa-lambda"

    def run_episode(self, env, rng):
        self._reset_traces()
        self.explorer.reset_episode()

        obs = env.reset()
        idx = self.coder.indices(obs)
        q_all = self.Q.q_all(idx)
        a = self.explorer.select(q_all, rng, idx)

        ret = 0.0; length = 0; eps_sum = td_sum = 0.0

        while True:
            obs2, reward, terminated, truncated = env.step(a)
            ret += reward
            length += 1
            self.total_steps += 1
            q_sa = float(self._wf[self._flat(a, idx)].sum())

            delta = reward - q_sa
            self._set_replacing_traces(a, idx)

            if terminated:
                self._apply_update(delta)
                self.explorer.update(delta, q_sa, idx)
                eps_sum += self.explorer.current_epsilon
                td_sum += abs(delta)
                break

            idx2 = self.coder.indices(obs2)
            q_all2 = self.Q.q_all(idx2)
            a2 = self.explorer.select(q_all2, rng, idx2)

            delta += self.gamma * float(q_all2[a2])

            self._apply_update(delta)
            self.explorer.update(delta, q_sa, idx)
            eps_sum += self.explorer.current_epsilon
            td_sum += abs(delta)

            if truncated:
                break

            self._decay_traces()
            idx, a, q_all = idx2, a2, q_all2
        
        return EpisodeRecord(ret, length, eps_sum / max(length, 1), td_sum / max(length, 1), self.total_steps)


Appending to src/agents.py


## QLearningAgent

```*a``` collects all extra positional arguments into a tuple.

**Watkin's $Q(\lambda)$:**
It says approximately:

Keep eligibility traces while behavior remains greedy. If a non-greedy action is taken, cut the traces.

**Numerical intuition**

Imagine: $ \lambda=0.9,\qquad\gamma=1$

Without cuts, an earlier trace evolves:

$ 1,\ 0.9,\ 0.81,\ 0.729,\ 0.6561,\ldots $

So a reward five steps later can still affect that old state-action pair.

Now compare two explorers.

*Explorer A*

Non-greedy action only once every 20 steps. Its traces might survive roughly:

10, 15, 20 steps...

before being interrupted.

*Explorer B*

Non-greedy action every 3 steps. Its traces repeatedly become:

1 → 0.9 → 0.81 → CUT

So it receives much shorter credit assignment.

If B learns worse, what caused it? Was it:

$ \text{its exploration strategy} $

or:

$ \text{its traces constantly getting cut}? $

We can't tell. That's bad experimental design. So, we set $\lambda=0$, makes every Q-learning condition one-step. Then no explorer has an advantage/ disadvantage from trace cutting.

**Why does Sarsa get $\lambda=0.9$, then?**

Because Sarsa is on-policy. Suppose Sarsa's explorer selects a weird exploratory action. That's fine. Its target is:

$ R+\gamma Q(S',A') $

where $A'$ is the action actually chosen by that same exploratory policy. So, the multi-step trajectory remains consistent with the policy Sarsa is learning about. No off-policy mismatch occurs. Therefore its traces don't need to be cut merely because an exploratory action happened.



In [6]:
%%writefile -a src/agents.py

class QLearningAgent(LinearAgent):
    algo = "q-learning"

    def __init__(self, *a, **kw):
        kw["lam"] = 0.0
        super().__init__(*a, **kw)

Appending to src/agents.py


## Greedy Evaluation
Its job is to answer:

After training with some exploration rule, how good is the policy the agent actually learned if we turn exploration completely off?

Since exploration itself is what we are comparing, scoring methods using training return would unfairly penalize methods that explore more. Therefore the project evaluates the learned greedy policy separately.

It runs the current learned Q-function for several episodes without changing anything.

It measures:

$$ \boxed{\text{Mean return of the current greedy policy}} $$

Suppose after training our agent has learned:

$$ Q(s,a) $$

for every represented state and action.

During evaluation, we ask:

If I always pick the action with the highest learned Q-value, how well does the resulting policy perform?

So this is not training.

Inside each evaluation episode, repeatedly:

* encode current state,
* calculate Q-values,
* choose greedy action,
* execute it,
* add reward,
* stop on termination/truncation.

In [8]:
from pathlib import Path
import shutil

path = Path(r"C:\RL\src\agents.py")
backup = Path(r"C:\RL\src\agents_before_evaluate.py")

shutil.copy2(path, backup)

text = path.read_text(encoding="utf-8")

method = '''
    def evaluate(self, env, rng, n_episodes=5):
        """Mean return of the GREEDY policy: no learning, no traces."""

        total = 0.0

        for _ in range(n_episodes):
            obs = env.reset()

            while True:
                idx = self.coder.indices(obs)
                a = argmax_random_tie(self.Q.q_all(idx), rng)

                obs, reward, term, trunc = env.step(a)
                total += reward

                if term or trunc:
                    break

        return total / n_episodes

'''

marker = "class SarsaLambdaAgent(LinearAgent):"

linear_start = text.find("class LinearAgent:")
sarsa_start = text.find(marker)

if linear_start == -1:
    print("LinearAgent not found. Nothing changed.")

elif sarsa_start == -1:
    print("SarsaLambdaAgent not found. Nothing changed.")

else:
    linear_section = text[linear_start:sarsa_start]

    if "def evaluate(" in linear_section:
        print("LinearAgent already contains evaluate(). Nothing changed.")

    else:
        try:
            new_text = (
                text[:sarsa_start]
                + method
                + text[sarsa_start:]
            )

            path.write_text(new_text, encoding="utf-8")

            check = path.read_text(encoding="utf-8")

            new_linear_start = check.find("class LinearAgent:")
            new_sarsa_start = check.find(marker)
            new_linear_section = check[new_linear_start:new_sarsa_start]

            if "def evaluate(" not in new_linear_section:
                raise RuntimeError(
                    "Insertion verification failed."
                )

            print("evaluate() inserted into LinearAgent successfully.")

            if backup.exists():
                backup.unlink()
                print("Backup deleted successfully.")

        except Exception as e:
            print("Insertion failed:", e)
            print("Backup preserved at:")
            print(backup)

evaluate() inserted into LinearAgent successfully.
Backup deleted successfully.


## Quick Check

In [3]:
import importlib
import numpy as np

import src.tilecoding as TC
import src.exploration as EXP
import src.agents as AG
import src.runner as RUN

TC = importlib.reload(TC)
EXP = importlib.reload(EXP)
AG = importlib.reload(AG)
RUN = importlib.reload(RUN)

seed = 0
rng = np.random.default_rng(seed)
eval_rng = np.random.default_rng(1000)

env = RUN.GymEnv("MountainCar-v0", seed=seed)
eval_env = RUN.GymEnv("MountainCar-v0", seed=1000)

low = env.env.observation_space.low
high = env.env.observation_space.high

cfg = TC.default_tiling_config("MountainCar-v0")

coder = TC.TileCoder(
    low,
    high,
    **cfg
)

explorer = EXP.make_explorer(
    "decay",
    n_actions=env.n_actions,
    n_features=coder.n_features,
    eps_start=1.0,
    eps_end=0.01,
    decay_steps=50_000
)

agent = AG.SarsaLambdaAgent(
    coder=coder,
    n_actions=env.n_actions,
    explorer=explorer,
    alpha_bar=0.5,
    gamma=1.0,
    lam=0.9,
    trace_cutoff=1e-3,
    q_init=0.0
)

target_steps = 30_000
episode = 0

while agent.total_steps < target_steps:
    rec = agent.run_episode(env, rng)
    episode += 1

    if episode % 25 == 0:
        print(
            f"Episode {episode:4d} | "
            f"steps {agent.total_steps:6d} | "
            f"return {rec.ret:7.1f} | "
            f"eps {rec.epsilon_mean:.3f} | "
            f"|TD| {rec.td_abs_mean:.3f}"
        )

greedy_return = agent.evaluate(
    eval_env,
    eval_rng,
    n_episodes=20
)

print()
print("Training complete")
print("Environment steps :", agent.total_steps)
print("Episodes          :", episode)
print("Greedy return     :", greedy_return)
print("IHT fullness      :", coder.iht.fullness)
print("IHT overfull count:", coder.iht.overfull_count)

if -170 <= greedy_return <= -130:
    print("PASS: learning is in the expected neighborhood.")
elif greedy_return <= -195:
    print("FAIL: still near random-policy performance (-200).")
else:
    print("Learning occurred, but the result is outside the rough checkpoint range.")

Episode   25 | steps   5000 | return  -200.0 | eps 0.635 | |TD| 4.667
Episode   50 | steps  10000 | return  -200.0 | eps 0.400 | |TD| 2.901
Episode   75 | steps  14885 | return  -200.0 | eps 0.255 | |TD| 2.246
Episode  100 | steps  19094 | return  -167.0 | eps 0.172 | |TD| 1.909
Episode  125 | steps  23131 | return  -153.0 | eps 0.119 | |TD| 1.435
Episode  150 | steps  26704 | return  -145.0 | eps 0.085 | |TD| 1.448
Episode  175 | steps  30105 | return  -163.0 | eps 0.063 | |TD| 2.657

Training complete
Environment steps : 30105
Episodes          : 175
Greedy return     : -141.8
IHT fullness      : 0.107421875
IHT overfull count: 0
PASS: learning is in the expected neighborhood.


# Runner

## Step Budgets

Episode count is not a fair resource budget when methods change episode length. If method A averages 195 steps for 500 Mountain Car episodes, it receives 97,500 transitions. If method B averages 140, it receives only 70,000. Method A has 27,500 extra transitions – about 39% more data – solely because its episodes are longer. A 100,000-step budget removes that asymmetry.

Each budget should be long enough that a reasonable decay baseline gets most of the way to its asymptotic performance, but not so long that it has already been flat for half the run.

## The Configuration Object

This section is creating a single object that contains the complete description of one training run.

The main idea is:
```
RunConfig
   │
   ├── Which environment?
   ├── Which RL algorithm?
   ├── Which exploration method?
   ├── What hyperparameters?
   ├── What random seed?
   ├── How long to train?
   ├── What learning rate?
   ├── What γ and λ?
   ├── How should rewards be scaled?
   └── How often/how much should we evaluate?
```

```dataclass``: This is a Python standard-library module.Its purpose is to make classes whose primary job is holding data much easier to write. For instance, suppose we write:
```
@dataclass
class RunConfig:
    env_id: str
    seed: int = 0
```
This automatically gives us:
```
class RunConfig:
    def __init__(self, env_id, seed=0):
        self.env_id = env_id
        self.seed = seed
```

```explorer_kwargs```: This means keyword arguments that will be passed specifically to the exploration rule. `kwargs' is standard Python shorthand for: keyword arguments. 

```field(default_factory=dict)```: This means--- Whenever a new RunConfig object is created without explicit explorer_kwargs, call dict() to make a brand-new dictionary for that object. This is to prevent the unwanted effects of mutation characteristics of a dictionary. 

```reward_scale: float = 1.0```: Means: *Don't change environment rewards*. 

```n_bins: int = 100```: This controls how many bins are used when converting irregular episode data into a common step-based grid. This is needed since episodes can finish at different number of steps. Hence, we cannot directly compare epsiode averages. Instead, we divide the step budget into ranges. For example, say $n_{bins}=100$ and $n_{steps}=100,000$. Thus, we compare two runs across step ranges like $0-999$, $1000-1999$ etc.

```n_eval_points: int = 20```: This says, during training, evaluate the greedy policy at approximately 20 locations. Let, $n_{steps}=100,000$. Then, the greedy policy will be evaluated every $\frac{100,000}{20}=5000$ steps.

## Periodic Evaluation Loop
This section is the main outer training loop of the runner. The ```SarsaLambdaAgent.run_episode()``` already knows how to train for one episode; this code repeatedly calls that method until the run has consumed its configured step budget, while periodically freezing learning and asking:

“At this point in training, how good is the greedy policy?”

So there are three nested levels:
```
RUN
│
├── Episode 1
│      ├── step 1
│      ├── step 2
│      ├── ...
│
├── Episode 2
│      ├── step 1
│      ├── ...
│
└── ...
```
This code is controlling the outermost level.

Remember that, ```cfg.n_steps```=budget for the whole training run

Because this outer loop trains one whole episode at a time, it does not stop in the middle of an episode exactly when the counter reaches $100,000$. Therefore a $100,000$-step budget may actually finish at something like:

$$ 100{,}087. $$

That's expected with this implementation. The check happens between episodes, not between individual transitions.

```
TRAINING LOGS
──────────────────────────────
ep_returns  behavior return
ep_steps    training progress
ep_eps      exploration amount
ep_td       TD-error magnitude


GREEDY EVALUATION LOGS
──────────────────────────────
eval_steps    evaluation timing
eval_returns  learned policy quality
```

Full workflow:
```
Read RunConfig
      │
      ▼
Are periodic evaluations enabled?
      │
      ├── no ─────────────────────┐
      │                           │
      └── yes                     │
           │                      │
           ▼                      │
 determine eval interval          │
 set first eval boundary          │
 initialize eval logs             │
           │                      │
           └──────────┬───────────┘
                      ▼
        while training budget remains
                      │
                      ▼
               run one episode
                      │
                      ▼
        receive EpisodeRecord
                      │
          ┌───────────┼─────────────┐
          ▼           ▼             ▼
      log return   log ε        log |δ|
          │
          ▼
    evaluation due?
          │
      ┌───┴────┐
      │        │
      no      yes
      │        │
      │        ▼
      │   greedy evaluation
      │        │
      │        ▼
      │   save step + score
      │        │
      │        ▼
      │   advance next_eval
      │   past all crossed
      │   boundaries
      │
      └────────┐
               ▼
        next training episode
```

## Binning onto a Common Grid
Its job is:

Different training runs finish episodes at different cumulative step counts. Convert those irregular episode-end measurements onto the same fixed step grid so that runs can later be averaged point-by-point.

```which = np.clip(np.digitize(steps, edges) - 1, 0, n_bins - 1)```: This line determines: which bin does each step belong to? `np.digitize()' takes values and tells which interval/bucket each value falls into. 
Suppose: edges = [0,100,200,300,...,1000]

and: steps = [50, 150, 250]

then roughly: np.digitize(steps, edges) returns: [1, 2, 3]

```m = which == b```: Creates a boolean mask of which bin numbers of `which' are 'b'. 

```last``` means: Most recent non-missing bin value we've encountered while scanning left-to-right.

## Run Single
```
GymEnv.step()
    ↓
ONE environment transition


agent.run_episode()
    ↓
ONE complete training episode


run_single(cfg)
    ↓
ONE complete experimental run
    ↓
hundreds/thousands of episodes
    ↓
until step budget is exhausted
```

```
FULL STUDY
│
├── Configuration 1
│      └── run_single(cfg1)
│             ├── episode
│             ├── episode
│             ├── episode
│             └── ...
│
├── Configuration 2
│      └── run_single(cfg2)
│
└── Configuration 3
       └── run_single(cfg3)
```

If both environment randomness and action randomness came from the same generator:
```
RATE
uses lots of random draws for actions
       ↓
moves RNG forward rapidly

Decay
uses fewer random draws
       ↓
RNG is at a different location
```
Then future reset states could differ merely because one explorer consumed more random numbers. We'd no longer be comparing methods on the same stochastic conditions.

By separating them:
```
environment randomness   → GymEnv RNG

action randomness        → rng
```
one doesn't disturb the other.

Why use different seed for evaluation environment? If we used the training env, evaluation would consume its random stream. Then merely measuring the policy would change what states training sees later.
So, 
```
   env
    ↓
training only

 eval_env
    ↓
evaluation only
```
Summary:
```
TRAINING
────────────────────────
env       environment randomness
rng       action/exploration randomness


EVALUATION
────────────────────────
eval_env  evaluation environment randomness
eval_rng  evaluation tie-breaking randomness
```
```hasattr(object, "name")``` asks: Does this object have an attribute/method with this name?

Full workflow:
```
1. Read config
       │
       ▼
2. Create Mountain Car training environment
       │
       ├── environment seed = 7
       │
       └── action RNG seed = 100007
       │
       ▼
3. Create separate evaluation environment
       │
       ├── evaluation env seed = 500007
       │
       └── evaluation RNG seed = 300007
       │
       ▼
4. Read observation bounds
       │
       ▼
5. Build Mountain Car tile coder
       │
       ▼
6. make_explorer("rate")
       │
       ▼
7. RATE(n_actions=3, beta=.01, kappa=1)
       │
       ▼
8. Build SarsaLambdaAgent
       │
       ├── Q weights
       ├── traces
       ├── α = ᾱ/T
       └── explorer
       │
       ▼
9. Wrap training environment for reward scaling
       │
       ▼
10. Start training
       │
       ├── episode
       ├── log return
       ├── log ε
       ├── log |δ|
       │
       ├── episode
       │
       ├── ...
       │
       └── until ≈100,000 steps
       │
       ▼
11. Every ≈5,000 steps
       │
       └── run 5 GREEDY evaluation episodes
       │
       ▼
12. Convert logs to NumPy arrays
       │
       ▼
13. Bin irregular episode data onto 100-step-grid bins
       │
       ├── return curve
       ├── exploration curve
       └── TD-error curve
       │
       ▼
14. Record final greedy score
       │
       ▼
15. Check tile-hash-table health
       │
       ▼
16. Return one result dictionary
```

## Full Code

In [22]:
%%writefile src/runner.py

import numpy as np

from dataclasses import dataclass, field

from .environments import GymEnv
from .tilecoding import TileCoder, default_tiling_config
from .exploration import make_explorer
from .agents import SarsaLambdaAgent, QLearningAgent


STEP_BUDGET = {
    "MountainCar-v0": 100_000,
    "CartPole-v1": 100_000,
    "Acrobot-v1": 150_000,
    "LunarLander-v3": 300_000,
}


@dataclass
class RunConfig:
    env_id: str
    algo: str = "sarsa-lambda"
    explorer: str = "decay"
    explorer_kwargs: dict = field(default_factory=dict)
    seed: int = 0
    n_steps: int = 100_000
    alpha_bar: float = 0.5
    gamma: float = 1.0
    lam: float = 0.9
    q_init: float = 0.0
    reward_scale: float = 1.0
    n_bins: int = 100
    n_eval_points: int = 20
    n_eval_episodes: int = 10


def _bin_by_step(steps, values, n_steps, n_bins):
    out = np.full(
        n_bins,
        np.nan,
        dtype=np.float64
    )

    if steps.size == 0:
        return out

    edges = np.linspace(
        0,
        n_steps,
        n_bins + 1
    )

    which = np.clip(
        np.digitize(steps, edges) - 1,
        0,
        n_bins - 1
    )

    for b in range(n_bins):
        m = (
            (which == b)
            & np.isfinite(values)
        )

        if m.any():
            out[b] = values[m].mean()

    last = np.nan

    for i in range(n_bins):
        if np.isnan(out[i]):
            out[i] = last
        else:
            last = out[i]

    return out


def run_single(cfg):
    if cfg.n_steps <= 0:
        raise ValueError(
            "n_steps must be positive."
        )

    if cfg.reward_scale <= 0:
        raise ValueError(
            "reward_scale must be positive."
        )

    if cfg.n_bins <= 0:
        raise ValueError(
            "n_bins must be positive."
        )

    if cfg.n_eval_points < 0:
        raise ValueError(
            "n_eval_points cannot be negative."
        )

    if cfg.n_eval_episodes <= 0:
        raise ValueError(
            "n_eval_episodes must be positive."
        )

    env = GymEnv(
        cfg.env_id,
        seed=cfg.seed
    )

    rng = np.random.default_rng(
        100_000 + cfg.seed
    )

    eval_env = GymEnv(
        cfg.env_id,
        seed=500_000 + cfg.seed
    )

    eval_rng = np.random.default_rng(
        300_000 + cfg.seed
    )

    low, high = env.tile_bounds()

    tiling_cfg = default_tiling_config(
        cfg.env_id
    )

    coder = TileCoder(
        low=low,
        high=high,
        **tiling_cfg
    )

    explorer = make_explorer(
        cfg.explorer,
        n_actions=env.n_actions,
        n_features=coder.n_features,
        **cfg.explorer_kwargs
    )

    if cfg.algo == "sarsa-lambda":
        agent_cls = SarsaLambdaAgent

    elif cfg.algo == "q-learning":
        agent_cls = QLearningAgent

    else:
        raise ValueError(
            f"Unknown algorithm: {cfg.algo}"
        )

    agent = agent_cls(
        coder=coder,
        n_actions=env.n_actions,
        explorer=explorer,
        alpha_bar=cfg.alpha_bar,
        gamma=cfg.gamma,
        lam=cfg.lam,
        q_init=cfg.q_init
    )

    if not hasattr(agent, "run_episode"):
        raise NotImplementedError(
            f"{type(agent).__name__} does not define run_episode()."
        )

    class TrainEnv:
        def __init__(
            self,
            base_env,
            scale,
            max_steps
        ):
            self.base_env = base_env
            self.scale = float(scale)
            self.max_steps = int(max_steps)
            self.steps = 0
            self.last_budget_cut = False

        def reset(self):
            self.last_budget_cut = False
            return self.base_env.reset()

        def step(self, action):
            if self.steps >= self.max_steps:
                raise RuntimeError(
                    "Training environment step budget exhausted."
                )

            (
                obs,
                reward,
                terminated,
                truncated
            ) = self.base_env.step(action)

            self.steps += 1

            budget_cut = (
                self.steps >= self.max_steps
                and not terminated
                and not truncated
            )

            self.last_budget_cut = budget_cut

            return (
                obs,
                reward * self.scale,
                terminated,
                truncated or budget_cut
            )

    train_env = TrainEnv(
        base_env=env,
        scale=cfg.reward_scale,
        max_steps=cfg.n_steps
    )

    ep_returns = []
    ep_steps = []
    ep_eps = []
    ep_td = []
    ep_complete = []

    eval_steps = []
    eval_returns = []

    do_eval = cfg.n_eval_points > 0

    if do_eval:
        eval_every = max(
            1,
            cfg.n_steps // cfg.n_eval_points
        )

        next_eval = eval_every

    while agent.total_steps < cfg.n_steps:
        rec = agent.run_episode(
            train_env,
            rng
        )

        ep_returns.append(
            rec.ret / cfg.reward_scale
        )

        ep_steps.append(
            rec.total_steps
        )

        ep_eps.append(
            rec.epsilon_mean
        )

        ep_td.append(
            rec.td_abs_mean
        )

        ep_complete.append(
            not train_env.last_budget_cut
        )

        if (
            do_eval
            and agent.total_steps >= next_eval
        ):
            eval_steps.append(
                agent.total_steps
            )

            eval_returns.append(
                agent.evaluate(
                    eval_env,
                    eval_rng,
                    cfg.n_eval_episodes
                )
            )

            while (
                next_eval
                <= agent.total_steps
            ):
                next_eval += eval_every

    if agent.total_steps != cfg.n_steps:
        raise RuntimeError(
            f"Expected exactly {cfg.n_steps} training steps, "
            f"but got {agent.total_steps}."
        )

    if train_env.steps != cfg.n_steps:
        raise RuntimeError(
            f"Training environment recorded {train_env.steps} steps, "
            f"expected {cfg.n_steps}."
        )

    ep_steps = np.asarray(
        ep_steps,
        dtype=np.int64
    )

    ep_returns = np.asarray(
        ep_returns,
        dtype=np.float64
    )

    ep_eps = np.asarray(
        ep_eps,
        dtype=np.float64
    )

    ep_td = np.asarray(
        ep_td,
        dtype=np.float64
    )

    ep_complete = np.asarray(
        ep_complete,
        dtype=bool
    )

    eval_steps = np.asarray(
        eval_steps,
        dtype=np.int64
    )

    eval_returns = np.asarray(
        eval_returns,
        dtype=np.float64
    )

    complete_returns = ep_returns.copy()
    complete_returns[~ep_complete] = np.nan

    return_curve = _bin_by_step(
        ep_steps,
        complete_returns,
        cfg.n_steps,
        cfg.n_bins
    )

    epsilon_curve = _bin_by_step(
        ep_steps,
        ep_eps,
        cfg.n_steps,
        cfg.n_bins
    )

    td_curve = _bin_by_step(
        ep_steps,
        ep_td,
        cfg.n_steps,
        cfg.n_bins
    )

    final_eval = (
        float(eval_returns[-1])
        if eval_returns.size > 0
        else float("nan")
    )

    result = {
        "env_id": cfg.env_id,
        "algo": cfg.algo,
        "explorer": cfg.explorer,
        "explorer_kwargs": dict(
            cfg.explorer_kwargs
        ),
        "seed": cfg.seed,
        "n_steps": cfg.n_steps,
        "alpha_bar": cfg.alpha_bar,
        "gamma": cfg.gamma,
        "lam": cfg.lam,
        "q_init": cfg.q_init,
        "reward_scale": cfg.reward_scale,
        "n_bins": cfg.n_bins,
        "n_eval_points": cfg.n_eval_points,
        "n_eval_episodes": cfg.n_eval_episodes,
        "ep_steps": ep_steps,
        "ep_returns": ep_returns,
        "ep_eps": ep_eps,
        "ep_td": ep_td,
        "ep_complete": ep_complete,
        "return_curve": return_curve,
        "epsilon_curve": epsilon_curve,
        "td_curve": td_curve,
        "eval_steps": eval_steps,
        "eval_returns": eval_returns,
        "final_eval": final_eval,
        "iht_fullness": coder.iht.fullness,
        "iht_overfull_count": (
            coder.iht.overfull_count
        ),
        "total_steps": agent.total_steps,
        "episodes_completed": int(
            ep_complete.sum()
        ),
    }

    env.env.close()
    eval_env.env.close()

    return result

Overwriting src/runner.py


## Sample Run

In [3]:
from src.runner import RunConfig, run_single

cfg = RunConfig(
    env_id="MountainCar-v0",
    algo="sarsa-lambda",
    explorer="decay",
    seed=0,
    n_steps=30_000
)

result = run_single(cfg)

print("Final greedy return:", result["final_eval"])
print("Total training steps:", result["total_steps"])
print("IHT fullness:", result["iht_fullness"])
print("IHT overfull count:", result["iht_overfull_count"])

Final greedy return: -122.8
Total training steps: 30096
IHT fullness: 0.109130859375
IHT overfull count: 0


# DQN

## The Network and Replay

```
              SAME exploration methods
                       │
             ┌─────────┴─────────┐
             ▼                   ▼
       Tile coding         Neural network
       + Linear Q              DQN
```

So we're moving from:

“I manually define the features using tile coding.”

to:

“The neural network learns internal features from data.”

```class QNetwork(nn.Module)```: This means: Create a class called QNetwork that inherits from PyTorch's nn.Module class. `nn.Module' is the base class for essentially every PyTorch neural network.

```nn.Sequential``` means:

Take these layers and execute them one after another in the order listed. If we wrote:

```
nn.Sequential(
    A,
    B,
    C
)
```

then input $x$ flows as: $$ x\rightarrow A(x)\rightarrow B(A(x)) \rightarrow C(B(A(x))). $$

So our network is literally a pipeline.

In [1]:
from pathlib import Path
folder = Path("src")

(folder/"dqn.py").touch(exist_ok=True)

In [15]:
%%writefile src/dqn.py

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .environments import GymEnv
from .exploration import (
    make_explorer,
    argmax_random_tie,
    NEEDS_FEATURES,
)


DEVICE = torch.device("cpu")
torch.set_num_threads(1)

class QNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(), 
            nn.Linear(hidden, hidden), nn.ReLU(), 
            nn.Linear(hidden, n_actions),
        )
    
    def forward(self, x):
        return self.net(x)

Overwriting src/dqn.py


## Replay Buffer

DQN says:

Don't immediately throw away old experiences. Store them in memory.

So:
```
ENVIRONMENT STEP
       │
       ▼
(s, a, r, s', terminated)
       │
       ▼
ReplayBuffer.add(...)
       │
       ▼
┌─────────────────────────┐
│      ReplayBuffer       │
│                         │
│ transition              │
│ transition              │
│ transition              │
│ ...                     │
└────────────┬────────────┘
             │
             │ sample(64)
             ▼
       random indices
             │
             ▼
       NumPy arrays
             │
             ▼
      PyTorch tensors
             │
             ▼
     o, a, r, o2, d
             │
             ▼
   Double-DQN target code
```

In RL, adjacent observations are highly correlated.But neural-network gradient methods work much better when batches contain a diverse mixture of examples. Replay lets us take random observations. 

This stores a Boolean-like termination indicator.

Conceptually:

$$ done = \begin{cases} 1 & \text{if genuinely terminal}\\ 0 & \text{otherwise} \end{cases} $$

Although it's conceptually Boolean, it's stored as float: 0.0 or 1.0

```pos``` means: Position where the next transition should be written.

Note: PyTorch neural networks use float32 by default.

Methods like: ```__len__``` are Python special methods. We don't normally call: buffer.```__len__()``` directly. Instead Python automatically invokes it when we write: len(buffer)


The ReplayBuffer knows:

I store experiences and convert samples to tensors.

It does not decide:

Should the experiment use CPU or GPU?

That belongs elsewhere. So later:
```
o, a, r, o2, d = buffer.sample(
    cfg.batch_size,
    DEVICE
)
```
works regardless of what DEVICE is.

In [16]:
%%writefile -a src/dqn.py

class ReplayBuffer:
    def __init__(self, capacity, obs_dim, rng):
        self.capacity = int(capacity)
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=np.float32)
        self.pos = 0
        self.full = False
        self.rng = rng

    def __len__(self):
        return self.capacity if self.full else self.pos

    def add(self, obs, action, reward, next_obs, terminated):
        i = self.pos

        self.obs[i] = obs
        self.actions[i] = int(action)
        self.rewards[i] = float(reward)
        self.next_obs[i] = next_obs
        self.dones[i] = float(terminated)

        self.pos = (self.pos + 1) % self.capacity

        if self.pos == 0:
            self.full = True

    def sample(self, batch_size, device):
        batch_size = int(batch_size)
        n = len(self)

        if n < batch_size:
            raise ValueError(
                f"Cannot sample {batch_size} transitions "
                f"from a buffer containing {n}."
            )

        idx = self.rng.choice(
            n,
            size=batch_size,
            replace=False
        )

        o = torch.as_tensor(
            self.obs[idx],
            dtype=torch.float32,
            device=device
        )

        a = torch.as_tensor(
            self.actions[idx],
            dtype=torch.int64,
            device=device
        )

        r = torch.as_tensor(
            self.rewards[idx],
            dtype=torch.float32,
            device=device
        )

        o2 = torch.as_tensor(
            self.next_obs[idx],
            dtype=torch.float32,
            device=device
        )

        d = torch.as_tensor(
            self.dones[idx],
            dtype=torch.float32,
            device=device
        )

        return o, a, r, o2, d

Appending to src/dqn.py


## DQNConfig
```
DQNConfig
│
├── Which environment?
├── Which explorer?
├── Which seed?
├── How many training steps?
│
├── What neural-network size?
├── How large is replay?
├── How large is a minibatch?
├── What learning rate?
├── When does learning start?
├── How frequently do we optimize?
├── How frequently do we update target network?
│
├── DQN or Double DQN?
│
└── How should evaluation/logging work?
```

In [17]:
%%writefile -a src/dqn.py

from dataclasses import dataclass, field


@dataclass
class DQNConfig:
    env_id: str

    explorer: str = "decay"
    explorer_kwargs: dict = field(default_factory=dict)

    seed: int = 0
    n_steps: int = 100_000

    gamma: float = 1.0
    reward_scale: float = 1.0

    hidden: int = 128
    replay_capacity: int = 50_000
    batch_size: int = 64

    learning_rate: float = 1e-3
    learning_starts: int = 1_000
    train_every: int = 1
    target_update_every: int = 1_000

    double: bool = True

    n_bins: int = 100
    n_eval_points: int = 20
    n_eval_episodes: int = 10

Appending to src/dqn.py


## DQN Update

```
run_dqn(cfg)
│
├── interact with environment
├── interact with environment
├── interact with environment
│
├── _dqn_update(...)
│       ↓
│   ONE gradient update
│
├── interact with environment
├── _dqn_update(...)
│
├── ...
│
└── finish entire experiment
```

So: ```_dqn_update()```=one minibatch learning operation

Its full workflow:
```
_dqn_update(...)
       │
       ▼
ReplayBuffer.sample(64)
       │
       ▼
o, a, r, o2, d
       │
       │
       ├──────────────────────────┐
       │                          │
       ▼                          ▼
 online q(o2)               target(o2)
       │                          │
       ▼                          │
 choose next action               │
       │                          │
       └──────────────┬───────────┘
                      ▼
             Double-DQN next_q
                      │
                      ▼
         target = r + γ(1-d)next_q
                      │
                      ▼
                  fixed target
                      

current o
   │
   ▼
online q(o)
   │
   ▼
gather actual action a
   │
   ▼
Q(s,a)
   │
   ├───────────────┐
   │               │
   ▼               ▼
TD error         Huber loss
                   │
                   ▼
              zero_grad()
                   │
                   ▼
               backward()
                   │
                   ▼
             optimizer.step()
                   │
                   ▼
             online q updated


TD error + Q magnitude
       │
       ▼
 minibatch means
       │
       ▼
explorer.update(...)
       │
       ▼
return diagnostics
	​
```

**Why Huber instead of plain MSE?**

The reason is to keep---

* quadratic for small errors;
* linear for large errors.

That prevents one huge transition from generating a huge gradient.

Suppose:

$$ |\delta|=100. $$

Squared error would scale like:

$$ 100^2=10,000. $$

Huber grows approximately linearly in that large-error regime.

That is generally more stable for noisy bootstrapped RL targets.

```optimizer.zero_grad()```: PyTorch gradients accumulate by default. That is, after ```loss.backward()```, parameter gradients are added into: ```parameter.grad``` rather than replacing whatever was already there.

Imagine:

*update 1*

gradient: $$ g_1=3. $$

*update 2*

gradient: $$ g_2=2. $$

If we don't clear gradients first, PyTorch can accumulate: $$ 3+2=5. $$

But for ordinary DQN we want update 2 to use: $$ 2, $$. Not stale gradient information from the previous minibatch.

So every update cycle is:
```
zero_grad
   ↓
backward
   ↓
step
```

```loss.backward()```: This asks PyTorch:

Calculate the gradient of the loss with respect to every trainable parameter involved in producing q_sa.

Mathematically:

$$ \nabla_\theta L. $$

Remember our network contains thousands of parameters.

Mountain Car's network:

$$ 2\rightarrow128\rightarrow128\rightarrow3. $$

We previously calculated around seventeen thousand parameters.

We certainly don't want to manually derive:

$$ \frac{\partial L}{\partial w_{1,1}}, \frac{\partial L}{\partial w_{1,2}}, \dots $$

PyTorch autograd does it automatically.

Here, 
```
target network        ❌ not gradient-updated
next-state q branch   ❌ not gradient-updated

current q(o) branch   ✅ gradient-updated
```
```optimizer.step()```: Now the optimizer looks at every:

parameter.grad

computed by backward() and changes the online-network parameters.

Very roughly, ordinary gradient descent would do:

$θ←θ−η∇_θL$

Extremely important: optimizer.step() changes q, not target. 

Explorers like:

```
rate-state
vdbe-state
tile-ucb
```
were built around tile-feature indices.

DQN cannot naturally provide those. So not every exploration rule that depends specifically on tile features transfers meaningfully to DQN.

Global rules such as:
```
fixed
decay
boltzmann
vdbe
rate
```
transfer cleanly because they don't require feat_idx.


This block is the heart of how DQN turns a replayed transition into a learning target, and the key new idea is the difference between ordinary DQN and Double DQN.
```
q
    ↓
online Q-network
    ↓
the one currently being learned


target
    ↓
target Q-network
    ↓
used to construct stable TD targets
```

As the target in semi-gradient TD is not differentiated w.r.t. weight, it is kept free from gradient through ```with torch.no_grad()```. 

```argmax(dim=1)``` means: for each row/state, find which action column has the largest Q-value.

```keepdim=True```: Normally argmax(dim=1) on shape (3,3) would give (3,) like [1,0,1]. But: `keepdim=True' keeps the action dimension. So, we get:
```
[
 [1],
 [0],
 [1]
]
```
shape: (3,1)

Why? Because gather() on the next line wants the index tensor aligned with the matrix dimension it's indexing.

```gather(dim, indices)``` means: select values from a tensor at specified indices along a dimension.

```.squeeze(1)``` removes dimension 1 if its size is 1. So: (3,1) becomes: (3,). 

Overall workflow:
```
next state s'
       │
       ├──────────────► online q network
       │                    │
       │                    ▼
       │              all action values
       │                    │
       │                    ▼
       │                 argmax
       │                    │
       │                    ▼
       │              chosen action a*
       │                    │
       ▼                    │
target network ◄────────────┘
       │
       ▼
Q_target(s', a*)
```

Mathematically:

$$ a^* = \arg\max_a Q_{\text{online}}(s',a) $$

and:

$$ next\_q = Q_{\text{target}}(s',a^*). $$

```
loss = F.smooth_l1_loss(
    q_sa,
    tgt

)
```
: PyTorch's smooth L1 loss is essentially the Huber loss.It combines: squared error for small errors with absolute-like error for large errors. 

For an error:

$$ e=q-tgt, $$

smooth L1 behaves roughly as:

$$ L(e) = \begin{cases} \frac12e^2, & |e|<1\\[4pt] |e|-\frac12, & |e|\ge1. \end{cases} $$

In [18]:
%%writefile -a src/dqn.py

def _dqn_update(q, target, optimizer, replay, explorer, cfg):
    o, a, r, o2, d = replay.sample(
        cfg.batch_size,
        DEVICE
    )

    with torch.no_grad():
        if cfg.double:
            next_a = q(o2).argmax(
                dim=1,
                keepdim=True
            )

            next_q = target(o2).gather(
                1,
                next_a
            ).squeeze(1)

        else:
            next_q = target(o2).max(
                dim=1
            ).values

        tgt = (
            r
            + cfg.gamma
            * (1.0 - d)
            * next_q
        )

    q_sa = q(o).gather(
        1,
        a.unsqueeze(1)
    ).squeeze(1)

    td = tgt - q_sa

    loss = F.smooth_l1_loss(
        q_sa,
        tgt
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    td_abs_mean = float(
        td.abs().mean().item()
    )

    q_abs_mean = float(
        q_sa.abs().mean().item()
    )

    explorer.update(
        td_abs_mean,
        q_abs_mean,
        None
    )

    return (
        float(loss.item()),
        td_abs_mean,
        q_abs_mean
    )

Appending to src/dqn.py


## Evaluate DQN

Its purpose is:

Measure how good the policy currently learned by the online Q-network is, without exploration and without learning.

During training: 
```
QNetwork
   ↓
explorer
   ↓
action
   ↓
environment
   ↓
replay
   ↓
gradient updates
```

During evaluation:
```
QNetwork
   ↓
GREEDY action
   ↓
separate environment
   ↓
return only
```

**Why was_training first before q.eval()?**

This makes _evaluate_dqn() non-destructive.

It borrows the network, evaluates it, and leaves it in the same mode it found it.

```q.eval()```: This tells PyTorch:

This model is being used for inference/evaluation, not training.

Here, ```with torch.no_grad():``` is used as evaluation does not involve any training. 

So:

Linear
$$ s \rightarrow \phi(s) \rightarrow Q(s,\cdot) \rightarrow greedy\ action $$
DQN
$$ s \rightarrow QNetwork(s) \rightarrow Q(s,\cdot) \rightarrow greedy\ action. $$

The evaluation principle stays identical.

In [19]:
%%writefile -a src/dqn.py

def _evaluate_dqn(q, env, rng, n_episodes):
    n_episodes = int(n_episodes)

    if n_episodes <= 0:
        raise ValueError(
            "n_episodes must be positive."
        )

    total_return = 0.0

    was_training = q.training
    q.eval()

    with torch.no_grad():
        for _ in range(int(n_episodes)):
            obs = env.reset()
            ep_return = 0.0

            while True:
                x = torch.as_tensor(
                    obs,
                    dtype=torch.float32,
                    device=DEVICE
                ).unsqueeze(0)

                q_values = (
                    q(x)
                    .squeeze(0)
                    .cpu()
                    .numpy()
                )

                action = argmax_random_tie(
                    q_values,
                    rng
                )

                obs, reward, terminated, truncated = (
                    env.step(action)
                )

                ep_return += reward

                if terminated or truncated:
                    break

            total_return += ep_return

    if was_training:
        q.train()

    return total_return / int(n_episodes)

Appending to src/dqn.py


## Run DQN

```
run_dqn(cfg)
       │
       ├── interacts with environment
       │
       ├── selects actions
       │
       ├── stores replay
       │
       │
       ├── _dqn_update()
       │       ↓
       │   ONE gradient update
       │
       ├── synchronizes target network
       │
       ├── _evaluate_dqn()
       │       ↓
       │   measure greedy policy
       │
       └── returns all experiment data
```
```
target.load_state_dict(
    q.state_dict()
)
```
This line is extremely important. Immediately after construction, q and target were initialized independently. Suppose:

q weights       = random set A
target weights  = random set B

We don't want that. Initially the target network should be a copy of the online network:

$$ \theta^-_0=\theta_0. $$

So: q.state_dict() gives the online network's parameters. And: target.load_state_dict(...) copies them.

Afterward:

q      = A
target = A

```
                    ┌─────────────────────┐
                    │      run_dqn()      │
                    └──────────┬──────────┘
                               │
                               ▼
                         current state
                               │
                               ▼
                         online QNetwork
                               │
                               ▼
                        exploration.py
                               │
                               ▼
                            action
                               │
                               ▼
                         training env
                               │
                               ▼
                    (s,a,r,s',terminated)
                               │
                               ▼
                         ReplayBuffer
                               │
                         random batch 64
                               │
                               ▼
                    ┌────────────────────┐
                    │   _dqn_update()    │
                    └─────────┬──────────┘
                              │
               ┌──────────────┴──────────────┐
               ▼                             ▼
          online network                target network
          selects action                evaluates action
               │                             │
               └──────────────┬──────────────┘
                              ▼
                       Double-DQN target
                              │
                              ▼
                           TD error
                              │
                              ▼
                         Huber loss
                              │
                              ▼
                          backward
                              │
                              ▼
                        optimizer.step
                              │
                              ▼
                      updated online q
                              │
                              ▼
                      explorer.update()
                              │
                              ▼
                  periodically target ← q


            periodically, completely separately:

                         online q
                            │
                            ▼
                   _evaluate_dqn()
                            │
                         greedy
                            │
                            ▼
                        eval_env
                            │
                            ▼
                    mean native return
```

In [21]:
%%writefile -a src/dqn.py

def run_dqn(cfg):
    if cfg.n_steps <= 0:
        raise ValueError("n_steps must be positive.")

    if cfg.reward_scale <= 0:
        raise ValueError("reward_scale must be positive.")

    if cfg.hidden <= 0:
        raise ValueError("hidden must be positive.")

    if cfg.batch_size <= 0:
        raise ValueError("batch_size must be positive.")

    if cfg.replay_capacity < cfg.batch_size:
        raise ValueError(
            "replay_capacity must be at least batch_size."
        )

    if cfg.learning_starts < cfg.batch_size:
        raise ValueError(
            "learning_starts must be at least batch_size."
        )

    if cfg.train_every <= 0:
        raise ValueError("train_every must be positive.")

    if cfg.target_update_every <= 0:
        raise ValueError(
            "target_update_every must be positive."
        )

    if cfg.learning_rate <= 0:
        raise ValueError("learning_rate must be positive.")

    if not 0.0 <= cfg.gamma <= 1.0:
        raise ValueError("gamma must be in [0, 1].")

    if cfg.n_bins <= 0:
        raise ValueError("n_bins must be positive.")

    if cfg.n_eval_points < 0:
        raise ValueError(
            "n_eval_points cannot be negative."
        )

    if cfg.n_eval_episodes <= 0:
        raise ValueError(
            "n_eval_episodes must be positive."
        )

    if cfg.explorer in NEEDS_FEATURES:
        raise ValueError(
            f"{cfg.explorer} requires tile features "
            "and cannot be used with DQN."
        )

    env = GymEnv(
        cfg.env_id,
        seed=cfg.seed
    )

    action_rng = np.random.default_rng(
        100_000 + cfg.seed
    )

    replay_rng = np.random.default_rng(
        200_000 + cfg.seed
    )

    eval_rng = np.random.default_rng(
        300_000 + cfg.seed
    )

    torch.manual_seed(
        400_000 + cfg.seed
    )

    eval_env = GymEnv(
        cfg.env_id,
        seed=500_000 + cfg.seed
    )

    obs_shape = env.env.observation_space.shape

    if obs_shape is None or len(obs_shape) != 1:
        raise ValueError(
            "DQN expects a one-dimensional observation vector."
        )

    obs_dim = int(obs_shape[0])

    q = QNetwork(
        obs_dim=obs_dim,
        n_actions=env.n_actions,
        hidden=cfg.hidden
    ).to(DEVICE)

    target = QNetwork(
        obs_dim=obs_dim,
        n_actions=env.n_actions,
        hidden=cfg.hidden
    ).to(DEVICE)

    target.load_state_dict(
        q.state_dict()
    )

    q.train()
    target.eval()

    optimizer = torch.optim.Adam(
        q.parameters(),
        lr=cfg.learning_rate
    )

    replay = ReplayBuffer(
        capacity=cfg.replay_capacity,
        obs_dim=obs_dim,
        rng=replay_rng
    )

    explorer = make_explorer(
        cfg.explorer,
        n_actions=env.n_actions,
        **cfg.explorer_kwargs
    )

    ep_steps = []
    ep_returns = []
    ep_lengths = []
    ep_eps = []
    ep_td = []
    ep_loss = []
    ep_q = []
    ep_complete = []

    eval_steps = []
    eval_returns = []

    do_eval = cfg.n_eval_points > 0

    if do_eval:
        eval_every = max(
            1,
            cfg.n_steps // cfg.n_eval_points
        )
        next_eval = eval_every

    total_steps = 0
    n_updates = 0
    episodes_completed = 0

    explorer.reset_episode()
    obs = env.reset()

    ep_return = 0.0
    ep_length = 0
    ep_eps_sum = 0.0
    ep_td_sum = 0.0
    ep_loss_sum = 0.0
    ep_q_sum = 0.0
    ep_update_count = 0

    while total_steps < cfg.n_steps:
        with torch.no_grad():
            x = torch.as_tensor(
                obs,
                dtype=torch.float32,
                device=DEVICE
            ).unsqueeze(0)

            q_values = (
                q(x)
                .squeeze(0)
                .cpu()
                .numpy()
            )

        action = explorer.select(
            q_values,
            action_rng,
            None
        )

        obs2, reward, terminated, truncated = (
            env.step(action)
        )

        replay.add(
            obs,
            action,
            reward * cfg.reward_scale,
            obs2,
            terminated
        )

        total_steps += 1

        ep_return += reward
        ep_length += 1
        ep_eps_sum += explorer.current_epsilon

        if (
            total_steps >= cfg.learning_starts
            and len(replay) >= cfg.batch_size
            and total_steps % cfg.train_every == 0
        ):
            loss_value, td_mean, q_mean = _dqn_update(
                q=q,
                target=target,
                optimizer=optimizer,
                replay=replay,
                explorer=explorer,
                cfg=cfg
            )

            n_updates += 1

            ep_loss_sum += loss_value
            ep_td_sum += td_mean
            ep_q_sum += q_mean
            ep_update_count += 1

        if (
            total_steps
            % cfg.target_update_every
            == 0
        ):
            target.load_state_dict(
                q.state_dict()
            )
            target.eval()

        if terminated or truncated:
            episodes_completed += 1

            ep_steps.append(total_steps)
            ep_returns.append(ep_return)
            ep_lengths.append(ep_length)
            ep_complete.append(True)

            ep_eps.append(
                ep_eps_sum / max(ep_length, 1)
            )

            if ep_update_count > 0:
                ep_td.append(
                    ep_td_sum / ep_update_count
                )

                ep_loss.append(
                    ep_loss_sum / ep_update_count
                )

                ep_q.append(
                    ep_q_sum / ep_update_count
                )
            else:
                ep_td.append(float("nan"))
                ep_loss.append(float("nan"))
                ep_q.append(float("nan"))

            if total_steps < cfg.n_steps:
                explorer.reset_episode()
                obs = env.reset()

                ep_return = 0.0
                ep_length = 0
                ep_eps_sum = 0.0
                ep_td_sum = 0.0
                ep_loss_sum = 0.0
                ep_q_sum = 0.0
                ep_update_count = 0

        else:
            obs = obs2

        if (
            do_eval
            and total_steps >= next_eval
        ):
            eval_steps.append(total_steps)

            eval_returns.append(
                _evaluate_dqn(
                    q,
                    eval_env,
                    eval_rng,
                    cfg.n_eval_episodes
                )
            )

            while next_eval <= total_steps:
                next_eval += eval_every

    if (
        ep_length > 0
        and (
            len(ep_steps) == 0
            or ep_steps[-1] != total_steps
        )
    ):
        ep_steps.append(total_steps)
        ep_returns.append(ep_return)
        ep_lengths.append(ep_length)
        ep_complete.append(False)

        ep_eps.append(
            ep_eps_sum / max(ep_length, 1)
        )

        if ep_update_count > 0:
            ep_td.append(
                ep_td_sum / ep_update_count
            )

            ep_loss.append(
                ep_loss_sum / ep_update_count
            )

            ep_q.append(
                ep_q_sum / ep_update_count
            )
        else:
            ep_td.append(float("nan"))
            ep_loss.append(float("nan"))
            ep_q.append(float("nan"))

    ep_steps = np.asarray(
        ep_steps,
        dtype=np.int64
    )

    ep_returns = np.asarray(
        ep_returns,
        dtype=np.float64
    )

    ep_lengths = np.asarray(
        ep_lengths,
        dtype=np.int64
    )

    ep_eps = np.asarray(
        ep_eps,
        dtype=np.float64
    )

    ep_td = np.asarray(
        ep_td,
        dtype=np.float64
    )

    ep_loss = np.asarray(
        ep_loss,
        dtype=np.float64
    )

    ep_q = np.asarray(
        ep_q,
        dtype=np.float64
    )

    ep_complete = np.asarray(
        ep_complete,
        dtype=bool
    )

    eval_steps = np.asarray(
        eval_steps,
        dtype=np.int64
    )

    eval_returns = np.asarray(
        eval_returns,
        dtype=np.float64
    )

    def bin_by_step(steps, values):
        out = np.full(
            cfg.n_bins,
            np.nan,
            dtype=np.float64
        )

        if steps.size == 0:
            return out

        edges = np.linspace(
            0,
            cfg.n_steps,
            cfg.n_bins + 1
        )

        which = np.clip(
            np.digitize(steps, edges) - 1,
            0,
            cfg.n_bins - 1
        )

        for b in range(cfg.n_bins):
            m = (
                (which == b)
                & np.isfinite(values)
            )

            if m.any():
                out[b] = values[m].mean()

        last = np.nan

        for i in range(cfg.n_bins):
            if np.isnan(out[i]):
                out[i] = last
            else:
                last = out[i]

        return out

    complete_returns = ep_returns.copy()
    complete_returns[~ep_complete] = np.nan

    return_curve = bin_by_step(
        ep_steps,
        complete_returns
    )

    epsilon_curve = bin_by_step(
        ep_steps,
        ep_eps
    )

    td_curve = bin_by_step(
        ep_steps,
        ep_td
    )

    loss_curve = bin_by_step(
        ep_steps,
        ep_loss
    )

    q_curve = bin_by_step(
        ep_steps,
        ep_q
    )

    final_eval = (
        float(eval_returns[-1])
        if eval_returns.size > 0
        else float("nan")
    )

    result = {
        "env_id": cfg.env_id,
        "algo": (
            "double-dqn"
            if cfg.double
            else "dqn"
        ),
        "explorer": cfg.explorer,
        "explorer_kwargs": dict(
            cfg.explorer_kwargs
        ),
        "seed": cfg.seed,
        "n_steps": cfg.n_steps,
        "gamma": cfg.gamma,
        "reward_scale": cfg.reward_scale,
        "hidden": cfg.hidden,
        "replay_capacity": cfg.replay_capacity,
        "batch_size": cfg.batch_size,
        "learning_rate": cfg.learning_rate,
        "learning_starts": cfg.learning_starts,
        "train_every": cfg.train_every,
        "target_update_every": (
            cfg.target_update_every
        ),
        "double": cfg.double,
        "n_eval_points": cfg.n_eval_points,
        "n_eval_episodes": cfg.n_eval_episodes,
        "ep_steps": ep_steps,
        "ep_returns": ep_returns,
        "ep_lengths": ep_lengths,
        "ep_eps": ep_eps,
        "ep_td": ep_td,
        "ep_loss": ep_loss,
        "ep_q": ep_q,
        "ep_complete": ep_complete,
        "return_curve": return_curve,
        "epsilon_curve": epsilon_curve,
        "td_curve": td_curve,
        "loss_curve": loss_curve,
        "q_curve": q_curve,
        "eval_steps": eval_steps,
        "eval_returns": eval_returns,
        "final_eval": final_eval,
        "total_steps": total_steps,
        "n_updates": n_updates,
        "replay_size": len(replay),
        "episodes_completed": episodes_completed,
    }

    env.env.close()
    eval_env.env.close()

    return result

Appending to src/dqn.py


## Quick Check

In [3]:
import importlib
import numpy as np

import src.dqn as DQN

DQN = importlib.reload(DQN)

cfg = DQN.DQNConfig(
    env_id="MountainCar-v0",
    explorer="decay",
    seed=0,
    n_steps=5_000,
    learning_starts=1_000,
    replay_capacity=50_000,
    batch_size=64,
    learning_rate=1e-3,
    train_every=1,
    target_update_every=1_000,
    double=True,
    n_bins=100,
    n_eval_points=5,
    n_eval_episodes=10
)

result = DQN.run_dqn(cfg)

print("Algorithm:", result["algo"])
print("Environment:", result["env_id"])
print("Explorer:", result["explorer"])
print("Total training steps:", result["total_steps"])
print("Gradient updates:", result["n_updates"])
print("Replay size:", result["replay_size"])
print("Completed episodes:", result["episodes_completed"])
print("Evaluation steps:", result["eval_steps"])
print("Evaluation returns:", result["eval_returns"])
print("Final greedy return:", result["final_eval"])

assert result["total_steps"] == cfg.n_steps
assert result["replay_size"] == min(
    cfg.n_steps,
    cfg.replay_capacity
)
assert result["n_updates"] > 0
assert len(result["eval_steps"]) > 0
assert len(result["eval_steps"]) == len(
    result["eval_returns"]
)
assert np.isfinite(result["final_eval"])
assert result["ep_steps"].shape == result["ep_returns"].shape
assert result["ep_steps"].shape == result["ep_lengths"].shape
assert result["ep_steps"].shape == result["ep_complete"].shape
assert result["td_curve"].shape == (cfg.n_bins,)
assert result["loss_curve"].shape == (cfg.n_bins,)
assert result["q_curve"].shape == (cfg.n_bins,)
assert result["epsilon_curve"].shape == (cfg.n_bins,)
assert result["return_curve"].shape == (cfg.n_bins,)

print("DQN SMOKE TEST PASSED")

Algorithm: double-dqn
Environment: MountainCar-v0
Explorer: decay
Total training steps: 5000
Gradient updates: 4001
Replay size: 5000
Completed episodes: 25
Evaluation steps: [1000 2000 3000 4000 5000]
Evaluation returns: [-200. -200. -200. -200. -200.]
Final greedy return: -200.0
DQN SMOKE TEST PASSED


In [4]:
import importlib
import numpy as np

import src.dqn as DQN

DQN = importlib.reload(DQN)

cfg = DQN.DQNConfig(
    env_id="MountainCar-v0",
    explorer="decay",
    seed=0,
    n_steps=30_000,
    learning_starts=1_000,
    replay_capacity=50_000,
    batch_size=64,
    learning_rate=1e-3,
    train_every=1,
    target_update_every=1_000,
    double=True,
    n_bins=100,
    n_eval_points=20,
    n_eval_episodes=10
)

result = DQN.run_dqn(cfg)

print("Total steps:", result["total_steps"])
print("Updates:", result["n_updates"])
print("Replay size:", result["replay_size"])
print("Completed episodes:", result["episodes_completed"])
print("Evaluation steps:", result["eval_steps"])
print("Evaluation returns:", result["eval_returns"])
print("Final greedy return:", result["final_eval"])

assert result["total_steps"] == 30_000
assert result["n_updates"] == 29_001
assert result["replay_size"] == 30_000
assert len(result["eval_steps"]) == 20
assert len(result["eval_returns"]) == 20
assert np.isfinite(result["final_eval"])

print("30K DQN TEST PASSED")

Total steps: 30000
Updates: 29001
Replay size: 30000
Completed episodes: 150
Evaluation steps: [ 1500  3000  4500  6000  7500  9000 10500 12000 13500 15000 16500 18000
 19500 21000 22500 24000 25500 27000 28500 30000]
Evaluation returns: [-200. -200. -200. -200. -200. -200. -200. -200. -200. -200. -200. -200.
 -200. -200. -200. -200. -200. -200. -200. -200.]
Final greedy return: -200.0
30K DQN TEST PASSED


# Sweep

## Imports
This first block creates src/sweep.py and puts in all imports that the completed sweep module will need, so later blocks can simply be appended with %%writefile -a.

In [ ]:
%%writefile src/sweep.py
import json
import multiprocessing as mp
import pickle
import traceback

from dataclasses import asdict, is_dataclass
from pathlib import Path

from .runner import RunConfig, run_single
from .dqn import DQNConfig, run_dqn

## CFG Tag

Its job is:

Given one experimental configuration, generate one deterministic string that uniquely identifies that run.

```json.dumps()``` means: Convert a Python object into a JSON string. Example: 
```data = {
    "seed": 3,
    "env_id": "MountainCar-v0"
}
```
becomes something like:

{"env_id":"MountainCar-v0","seed":3}

In [5]:
%%writefile -a src/sweep.py


def cfg_tag(cfg):
    if not is_dataclass(cfg):
        raise TypeError(
            "cfg_tag() expects a dataclass configuration."
        )

    data = asdict(cfg)

    payload = json.dumps(
        data,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True
    )

    return (
        f"{type(cfg).__name__}|"
        f"{payload}"
    )

Appending to src/sweep.py


## Result Tag

We now have: cfg_tag(cfg) which answers:

What is the unique identity of a configuration that we are about to run?

But after a configuration has been executed, we no longer have just: configuration— we have a result dictionary.

For example, DQN currently returns something conceptually like:
```
{
    "env_id": "MountainCar-v0",
    "seed": 0,
    "eval_returns": ...,
    "final_eval": -150.2,
    ...
}
```
The sweep needs to ask:

Which configuration produced this saved result?

That is what: result_tag(result) answers.
```
configuration
     │
     └── cfg_tag()
             ↓
            KEY
             ↓
     stored inside result
             ↓
        result_tag()
             ↓
            KEY
```

In [6]:
%%writefile -a src/sweep.py


def result_tag(result):
    if not isinstance(result, dict):
        raise TypeError(
            "result_tag() expects a result dictionary."
        )

    key = result.get("key")

    if not isinstance(key, str) or not key:
        raise KeyError(
            "Result record does not contain a valid 'key'."
        )

    return key

Appending to src/sweep.py


## Run Dispatcher

```
RunConfig
   ↓
run_single()
   ↓
linear tile coding


DQNConfig
   ↓
run_dqn()
   ↓
Double DQN
```

In [7]:
%%writefile -a src/sweep.py


def _run_config(cfg):
    if isinstance(cfg, RunConfig):
        return run_single(cfg)

    if isinstance(cfg, DQNConfig):
        return run_dqn(cfg)

    raise TypeError(
        f"Unsupported configuration type: "
        f"{type(cfg).__name__}"
    )

Appending to src/sweep.py


## Worker
**What exactly is a worker?**

Eventually we'll have a multiprocessing pool.

Imagine:

Main process
    │
    ├── Worker 1 → configuration A
    ├── Worker 2 → configuration B
    ├── Worker 3 → configuration C
    └── Worker 4 → configuration D

Each worker process receives one cfg and needs to return one result record.

That's _worker():

```
configuration
      ↓
  _worker()
      ↓
┌──────────────────────┐
│ calculate identity   │
│ run experiment       │
│ attach metadata      │
│ catch failures       │
└──────────────────────┘
      ↓
result dictionary
```

Notice the distinction between the two functions we have now:

```
_run_config()
    = run the actual RL experiment

_worker()
    = manage one sweep job
```

```except Exception as e:```

This catches ordinary Python exceptions deriving from Exception.

Examples:

ValueError
RuntimeError
TypeError
FloatingPointError
Memory-related Python errors in many cases
Gymnasium errors
PyTorch RuntimeError

It deliberately does not generally catch things like:

KeyboardInterrupt
SystemExit

because those derive from BaseException, not ordinary Exception.

```
cfg
 │
 ├──── cfg_tag(cfg)
 │          ↓
 │      stable key
 │
 ▼
_worker(cfg)
 │
 ├── _run_config(cfg)
 │      │
 │      ├── RunConfig → run_single()
 │      │
 │      └── DQNConfig → run_dqn()
 │
 ├── success
 │      ↓
 │   attach key/config
 │
 └── failure
        ↓
    convert exception
    into result record
```

In [8]:
%%writefile -a src/sweep.py


def _worker(cfg):
    key = cfg_tag(cfg)
    config_type = type(cfg).__name__
    config_snapshot = asdict(cfg)

    try:
        result = _run_config(cfg)

        if not isinstance(result, dict):
            raise TypeError(
                "Runner must return a result dictionary."
            )

        result = dict(result)

        result["key"] = key
        result["config_type"] = config_type
        result["config"] = config_snapshot

        return result

    except Exception as e:
        return {
            "key": key,
            "config_type": config_type,
            "config": config_snapshot,
            "error_type": type(e).__name__,
            "error": f"{type(e).__name__}: {e}",
            "traceback": traceback.format_exc(),
        }

Appending to src/sweep.py


## Atomic Save-Helper

Full flow

Suppose initially:

results.pkl

contains results from 100 completed experiments.

We want to save results 1–110.

The helper does:

Step 1
results.pkl
    contains 1–100

Step 2
write complete 1–110 into
results.pkl.tmp

Step 3
only after writing finishes:
results.pkl.tmp → results.pkl

If something goes wrong during step 2:

results.pkl

still contains the intact 1–100.

That is the protection.
```
                    run_stage()
                         │
            ┌────────────┴────────────┐
            │                         │
        resume logic           multiprocessing
                                      │
                             many _worker(cfg)
                                      │
                                  _run_config
                                 /           \
                                /             \
                         run_single()       run_dqn()
                                \             /
                                 \           /
                                  result
                                     │
                              attach key/config
                                     │
                                  parent
                                     │
                              _save_atomic()
```

In [9]:
%%writefile -a src/sweep.py


def _save_atomic(obj, path):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")

    with open(tmp, "wb") as f:
        pickle.dump(obj, f)

    tmp.replace(path)

Appending to src/sweep.py


## Run Stage

This is the main engine of sweep.py. It combines the pieces we have written so far: resume detection, multiprocessing, _worker(), error handling, progress reporting, retry behavior, checkpointing, and atomic saving.

Suppose we eventually generate: 300 experimental configurations.

```
                 300 configurations
                         │
                         ▼
                  load old results
                         │
                         ▼
               which ones are done?
                         │
                         ▼
                 remove completed
                         │
                         ▼
              distribute remaining
                across processes
                         │
          ┌──────────────┼──────────────┐
          ▼              ▼              ▼
       worker 1       worker 2       worker 3 ...
          │              │              │
          ▼              ▼              ▼
       result         result         result
          └──────────────┼──────────────┘
                         ▼
                 parent process
                         │
                         ▼
                  append results
                         │
                         ▼
                checkpoint safely
                         │
                         ▼
                   results.pkl
```

```label```: This is purely for progress output.

workers=8 means approximately:

8 experiments can be executing simultaneously

assuming enough jobs are waiting.

This is not the same as eight threads inside one DQN.

Recall that we already made PyTorch use one thread:

torch.set_num_threads(1)

```retry_errors=False```: means: No. A recorded failure counts as an attempted/completed experimental datum.

```imap_unordered```:

Suppose jobs take:

A → 20 seconds
B → 90 seconds
C → 30 seconds
D → 15 seconds

If we insisted on receiving results in submission order:

A
B
C
D

then after A finished we'd wait 90 seconds for B even though C and D had already finished.

imap_unordered instead gives results approximately:

D
A
C
B

as soon as each completes.

In [10]:
%%writefile -a src/sweep.py


def run_stage(
    configs,
    out_path,
    label,
    workers,
    save_every=10,
    retry_errors=False
):
    configs = list(configs)
    out_path = Path(out_path)
    label = str(label).strip() or "sweep"
    workers = int(workers)
    save_every = int(save_every)

    if workers <= 0:
        raise ValueError("workers must be positive.")

    if save_every <= 0:
        raise ValueError("save_every must be positive.")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    results = []

    if out_path.exists():
        with open(out_path, "rb") as f:
            results = pickle.load(f)

        if not isinstance(results, list):
            raise TypeError(
                "Existing results file must contain a list."
            )

    result_index = {}

    for i, result in enumerate(results):
        key = result_tag(result)

        if key in result_index:
            raise ValueError(
                "Existing results file contains duplicate "
                f"configuration key: {key}"
            )

        result_index[key] = i

    if retry_errors:
        done = {
            result_tag(result)
            for result in results
            if "error" not in result
        }
    else:
        done = set(result_index)

    pending = []
    queued = set()

    for cfg in configs:
        key = cfg_tag(cfg)

        if key in done or key in queued:
            continue

        pending.append(cfg)
        queued.add(key)

    total = len(pending)

    print(
        f"[{label}] "
        f"loaded={len(results)} "
        f"pending={total} "
        f"workers={workers}"
    )

    if total == 0:
        return results

    ctx = mp.get_context("spawn")

    try:
        with ctx.Pool(processes=workers) as pool:
            iterator = pool.imap_unordered(
                _worker,
                pending,
                chunksize=1
            )

            for i, result in enumerate(iterator, 1):
                key = result_tag(result)

                if key in result_index:
                    results[result_index[key]] = result
                else:
                    result_index[key] = len(results)
                    results.append(result)

                status = (
                    "ERROR"
                    if "error" in result
                    else "OK"
                )

                print(
                    f"[{label}] "
                    f"{i}/{total} "
                    f"{status}"
                )

                if i % save_every == 0 or i == total:
                    _save_atomic(results, out_path)

    except KeyboardInterrupt:
        _save_atomic(results, out_path)
        print(
            f"[{label}] interrupted; "
            f"checkpoint saved to {out_path}"
        )
        raise

    except Exception:
        _save_atomic(results, out_path)
        raise

    return results

Appending to src/sweep.py


## Configuration Builders

Builds the following configs:
```
ENVIRONMENTS
    MountainCar-v0
    CartPole-v1
    Acrobot-v1
    LunarLander-v3


STEP BUDGETS
    MountainCar    150,000
    CartPole       100,000
    Acrobot        100,000
    LunarLander    300,000


TUNING SEEDS
    1000, 1001, 1002


FINAL SEEDS
    0 ... 29


GREEDY EVALUATION
    10 episodes
    separate evaluation environment
    separate random stream


LINEAR ᾱ
    candidates:
        0.1
        0.25
        0.5

    selected once per environment
    using decay baseline

    then fixed across all
    exploration methods in that env


LINEAR λ
    0.9 fixed


FIXED ε
    0.05
    0.10
    0.20


DECAY
    linear
    εstart = 1.0
    εend   = 0.01

    horizon:
        0.2 budget
        0.4 budget
        0.6 budget


BOLTZMANN
    exponential temperature annealing
    τstart = 1.0
    τend   = 0.05

    horizon:
        0.2 budget
        0.4 budget
        0.6 budget


VDBE σ
    0.5
    1
    5
    20
    100
    500
    2000
    10000


RATE
    β:
        0.01
        0.05

    κ:
        1
        2

    tuning environments:
        MountainCar
        LunarLander

    one global winner
    used unchanged everywhere


DQN
    hidden = 128
    replay = 50,000
    batch = 64
    learning rate = 0.001
    learning starts = 1,000
    train every = 1 step
    target update every = 1,000 steps
    Double DQN = True
    CPU / one thread
```

In [12]:
%%writefile src/sweep_configs.py
from .runner import RunConfig
from .dqn import DQNConfig
from .exploration import NEEDS_FEATURES


ENV_IDS = (
    "MountainCar-v0",
    "CartPole-v1",
    "Acrobot-v1",
    "LunarLander-v3",
)

TUNING_SEEDS = tuple(range(1000, 1003))
FINAL_SEEDS = tuple(range(30))

STEP_BUDGET = {
    "MountainCar-v0": 150_000,
    "CartPole-v1": 100_000,
    "Acrobot-v1": 100_000,
    "LunarLander-v3": 300_000,
}

ALPHA_BAR_GRID = (
    0.1,
    0.25,
    0.5,
)

ALPHA_SELECTION_DECAY_FRACTION = 0.4

LAMBDA_FIXED = 0.9

FIXED_EPS_GRID = (
    0.05,
    0.1,
    0.2,
)

DECAY_MODE = "linear"
DECAY_EPS_START = 1.0
DECAY_EPS_END = 0.01

DECAY_HORIZON_FRACTIONS = (
    0.2,
    0.4,
    0.6,
)

BOLTZMANN_TAU_START = 1.0
BOLTZMANN_TAU_END = 0.05

BOLTZMANN_HORIZON_FRACTIONS = (
    0.2,
    0.4,
    0.6,
)

VDBE_SIGMA_GRID = (
    0.5,
    1.0,
    5.0,
    20.0,
    100.0,
    500.0,
    2000.0,
    10_000.0,
)

RATE_BETA_GRID = (
    0.01,
    0.05,
)

RATE_KAPPA_GRID = (
    1.0,
    2.0,
)

RATE_TUNING_ENVS = (
    "MountainCar-v0",
    "LunarLander-v3",
)

HEADLINE_EXPLORERS = (
    "fixed",
    "decay",
    "boltzmann",
    "vdbe",
    "rate",
)

LINEAR_FEATURE_EXPLORERS = (
    "vdbe-state",
    "rate-state",
    "tile-ucb",
)

DQN_SETTINGS = {
    "gamma": 1.0,
    "reward_scale": 1.0,
    "hidden": 128,
    "replay_capacity": 50_000,
    "batch_size": 64,
    "learning_rate": 1e-3,
    "learning_starts": 1_000,
    "train_every": 1,
    "target_update_every": 1_000,
    "double": True,
    "n_bins": 100,
    "n_eval_points": 20,
    "n_eval_episodes": 10,
}


def fixed_specs():
    return [
        (
            "fixed",
            {
                "epsilon": epsilon,
            },
        )
        for epsilon in FIXED_EPS_GRID
    ]


def alpha_selection_specs(n_steps):
    n_steps = int(n_steps)

    if n_steps <= 0:
        raise ValueError(
            "n_steps must be positive."
        )

    return [
        (
            "decay",
            {
                "eps_start": DECAY_EPS_START,
                "eps_end": DECAY_EPS_END,
                "decay_steps": int(
                    round(
                        ALPHA_SELECTION_DECAY_FRACTION
                        * n_steps
                    )
                ),
                "mode": DECAY_MODE,
            },
        )
    ]


def decay_specs(n_steps):
    n_steps = int(n_steps)

    if n_steps <= 0:
        raise ValueError(
            "n_steps must be positive."
        )

    return [
        (
            "decay",
            {
                "eps_start": DECAY_EPS_START,
                "eps_end": DECAY_EPS_END,
                "decay_steps": int(
                    round(
                        fraction * n_steps
                    )
                ),
                "mode": DECAY_MODE,
            },
        )
        for fraction
        in DECAY_HORIZON_FRACTIONS
    ]


def boltzmann_specs(n_steps):
    n_steps = int(n_steps)

    if n_steps <= 0:
        raise ValueError(
            "n_steps must be positive."
        )

    return [
        (
            "boltzmann",
            {
                "tau_start": BOLTZMANN_TAU_START,
                "tau_end": BOLTZMANN_TAU_END,
                "decay_steps": int(
                    round(
                        fraction * n_steps
                    )
                ),
            },
        )
        for fraction
        in BOLTZMANN_HORIZON_FRACTIONS
    ]


def vdbe_specs():
    return [
        (
            "vdbe",
            {
                "sigma": sigma,
                "eps_init": 1.0,
                "eps_min": 0.0,
            },
        )
        for sigma in VDBE_SIGMA_GRID
    ]


def rate_specs():
    return [
        (
            "rate",
            {
                "eps_min": 0.01,
                "eps_max": 1.0,
                "beta": beta,
                "kappa": kappa,
            },
        )
        for beta in RATE_BETA_GRID
        for kappa in RATE_KAPPA_GRID
    ]


def _normalise_explorer_specs(
    explorer_specs
):
    out = []

    for spec in explorer_specs:
        if (
            not isinstance(spec, tuple)
            or len(spec) != 2
        ):
            raise TypeError(
                "Each explorer specification must be "
                "(explorer_name, kwargs)."
            )

        kind, kwargs = spec

        if (
            not isinstance(kind, str)
            or not kind
        ):
            raise TypeError(
                "Explorer name must be a non-empty "
                "string."
            )

        if kwargs is None:
            kwargs = {}

        if not isinstance(
            kwargs,
            dict
        ):
            raise TypeError(
                "Explorer kwargs must be a dictionary."
            )

        out.append(
            (
                kind,
                dict(kwargs),
            )
        )

    return out


def _resolve_explorer_specs(
    explorer_specs,
    n_steps
):
    if callable(explorer_specs):
        explorer_specs = explorer_specs(
            n_steps
        )

    return _normalise_explorer_specs(
        explorer_specs
    )


def _validate_step_budget(
    step_budget,
    env_ids
):
    if not isinstance(
        step_budget,
        dict
    ):
        raise TypeError(
            "step_budget must be a dictionary."
        )

    budgets = {}

    for env_id in env_ids:
        if env_id not in step_budget:
            raise KeyError(
                f"No step budget defined for "
                f"{env_id}."
            )

        value = int(
            step_budget[env_id]
        )

        if value <= 0:
            raise ValueError(
                f"Step budget for {env_id} "
                "must be positive."
            )

        budgets[env_id] = value

    return budgets


def build_linear_configs(
    env_ids,
    seeds,
    explorer_specs,
    step_budget=STEP_BUDGET,
    alpha_bars=(0.5,),
    algo="sarsa-lambda",
    gamma=1.0,
    lam=LAMBDA_FIXED,
    q_init=0.0,
    reward_scale=1.0,
    n_bins=100,
    n_eval_points=20,
    n_eval_episodes=10
):
    env_ids = tuple(env_ids)

    seeds = tuple(
        int(seed)
        for seed in seeds
    )

    alpha_bars = tuple(
        float(alpha_bar)
        for alpha_bar in alpha_bars
    )

    if not env_ids:
        raise ValueError(
            "env_ids must not be empty."
        )

    if not seeds:
        raise ValueError(
            "seeds must not be empty."
        )

    if not alpha_bars:
        raise ValueError(
            "alpha_bars must not be empty."
        )

    if any(
        alpha_bar <= 0
        for alpha_bar in alpha_bars
    ):
        raise ValueError(
            "Every alpha_bar must be positive."
        )

    budgets = _validate_step_budget(
        step_budget,
        env_ids
    )

    configs = []

    for env_id in env_ids:
        n_steps = budgets[env_id]

        specs = _resolve_explorer_specs(
            explorer_specs,
            n_steps
        )

        for (
            explorer,
            explorer_kwargs,
        ) in specs:
            for alpha_bar in alpha_bars:
                for seed in seeds:
                    configs.append(
                        RunConfig(
                            env_id=env_id,
                            algo=algo,
                            explorer=explorer,
                            explorer_kwargs=dict(
                                explorer_kwargs
                            ),
                            seed=seed,
                            n_steps=n_steps,
                            alpha_bar=alpha_bar,
                            gamma=gamma,
                            lam=lam,
                            q_init=q_init,
                            reward_scale=reward_scale,
                            n_bins=n_bins,
                            n_eval_points=(
                                n_eval_points
                            ),
                            n_eval_episodes=(
                                n_eval_episodes
                            ),
                        )
                    )

    return configs


def build_dqn_configs(
    env_ids,
    seeds,
    explorer_specs,
    step_budget=STEP_BUDGET,
    dqn_overrides=None
):
    env_ids = tuple(env_ids)

    seeds = tuple(
        int(seed)
        for seed in seeds
    )

    if not env_ids:
        raise ValueError(
            "env_ids must not be empty."
        )

    if not seeds:
        raise ValueError(
            "seeds must not be empty."
        )

    budgets = _validate_step_budget(
        step_budget,
        env_ids
    )

    settings = dict(
        DQN_SETTINGS
    )

    if dqn_overrides is not None:
        if not isinstance(
            dqn_overrides,
            dict
        ):
            raise TypeError(
                "dqn_overrides must be a dictionary."
            )

        reserved = {
            "env_id",
            "explorer",
            "explorer_kwargs",
            "seed",
            "n_steps",
        }

        overlap = (
            reserved.intersection(
                dqn_overrides
            )
        )

        if overlap:
            raise ValueError(
                "dqn_overrides may not replace "
                "builder-owned fields: "
                f"{sorted(overlap)}"
            )

        settings.update(
            dqn_overrides
        )

    configs = []

    for env_id in env_ids:
        n_steps = budgets[env_id]

        specs = _resolve_explorer_specs(
            explorer_specs,
            n_steps
        )

        feature_rules = {
            explorer
            for explorer, _ in specs
            if explorer in NEEDS_FEATURES
        }

        if feature_rules:
            raise ValueError(
                "DQN cannot use feature-dependent "
                "explorers: "
                f"{sorted(feature_rules)}"
            )

        for (
            explorer,
            explorer_kwargs,
        ) in specs:
            for seed in seeds:
                configs.append(
                    DQNConfig(
                        env_id=env_id,
                        explorer=explorer,
                        explorer_kwargs=dict(
                            explorer_kwargs
                        ),
                        seed=seed,
                        n_steps=n_steps,
                        **settings,
                    )
                )

    return configs

Overwriting src/sweep_configs.py


## Smoke Sweep()

In [1]:
%%writefile src/scripts/smoke_sweep.py
from pathlib import Path

from src.sweep import run_stage
from src.sweep_configs import (
    build_linear_configs,
    build_dqn_configs,
)


def main():
    out_path = Path(
        "src/results/smoke_sweep.pkl"
    )

    smoke_budget = {
        "MountainCar-v0": 40,
    }

    explorer_specs = [
        (
            "fixed",
            {
                "epsilon": 0.1,
            },
        )
    ]

    linear_configs = build_linear_configs(
        env_ids=(
            "MountainCar-v0",
        ),
        seeds=(
            910001,
            910002,
        ),
        explorer_specs=explorer_specs,
        step_budget=smoke_budget,
        alpha_bars=(
            0.5,
        ),
        algo="sarsa-lambda",
        gamma=1.0,
        lam=0.9,
        q_init=0.0,
        reward_scale=1.0,
        n_bins=4,
        n_eval_points=1,
        n_eval_episodes=2,
    )

    dqn_configs = build_dqn_configs(
        env_ids=(
            "MountainCar-v0",
        ),
        seeds=(
            920001,
            920002,
        ),
        explorer_specs=explorer_specs,
        step_budget=smoke_budget,
        dqn_overrides={
            "replay_capacity": 128,
            "batch_size": 8,
            "learning_starts": 8,
            "train_every": 1,
            "target_update_every": 10,
            "n_bins": 4,
            "n_eval_points": 1,
            "n_eval_episodes": 2,
        },
    )

    configs = (
        linear_configs
        + dqn_configs
    )

    results = run_stage(
        configs=configs,
        out_path=out_path,
        label="smoke sweep",
        workers=2,
        save_every=1,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    config_types = {
        result["config_type"]
        for result in results
    }

    print(
        f"Stored results: {len(results)}"
    )

    print(
        f"Errors: {len(errors)}"
    )

    print(
        f"Config types: "
        f"{sorted(config_types)}"
    )

    if errors:
        first = errors[0]

        raise RuntimeError(
            "Smoke sweep produced an error: "
            f"{first['error']}"
        )

    if len(results) != 4:
        raise RuntimeError(
            "Expected exactly 4 smoke results, "
            f"but found {len(results)}."
        )

    if config_types != {
        "RunConfig",
        "DQNConfig",
    }:
        raise RuntimeError(
            "Smoke sweep did not exercise both "
            "configuration types."
        )

    print(
        "SWEEP SMOKE TEST PASSED"
    )

Writing src/scripts/smoke_sweep.py


In [2]:
%%writefile -a src/scripts/smoke_sweep.py


if __name__ == "__main__":
    main()

Appending to src/scripts/smoke_sweep.py


In [3]:
from pathlib import Path
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.smoke_sweep",
    ],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)

print("\nSTDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

RETURN CODE: 0

STDOUT:
[smoke sweep] loaded=0 pending=4 workers=2
[smoke sweep] 1/4 OK
[smoke sweep] 2/4 OK
[smoke sweep] 3/4 OK
[smoke sweep] 4/4 OK
Stored results: 4
Errors: 0
Config types: ['DQNConfig', 'RunConfig']
SWEEP SMOKE TEST PASSED


STDERR:



In [4]:
from pathlib import Path
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.smoke_sweep",
    ],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)

print("\nSTDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

RETURN CODE: 0

STDOUT:
[smoke sweep] loaded=4 pending=0 workers=2
Stored results: 4
Errors: 0
Config types: ['DQNConfig', 'RunConfig']
SWEEP SMOKE TEST PASSED


STDERR:



In [6]:
from pathlib import Path

for path in (
    Path("src/results/smoke_sweep.pkl"),
    Path("src/results/smoke_sweep.pkl.tmp"),
):
    if path.exists():
        path.unlink()

print("Smoke-test result files cleared.")

Smoke-test result files cleared.


# Tune Alpha

Our alpha_selection_specs() gives one fixed decaying-\$epsilon$ setup.

We chose the middle decay horizon:

$$ 40\% $$

of each environment's training budget.

That means while choosing $\bar{\alpha}$, we deliberately do not also tune the exploration schedule.

Otherwise we would have two variables changing:

$$ \bar{\alpha} \quad\text{and}\quad \text{decay horizon}. $$

Instead:

exploration schedule = fixed
λ = fixed
algorithm = fixed
γ = fixed
everything else = fixed

only ᾱ varies

In [7]:
%%writefile src/scripts/tune_alpha.py
import os

from pathlib import Path

from src.sweep import run_stage
from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    STEP_BUDGET,
    ALPHA_BAR_GRID,
    LAMBDA_FIXED,
    alpha_selection_specs,
    build_linear_configs,
)


MAX_WORKERS = 8


def main():
    out_path = Path(
        "src/results/tuning/alpha_selection.pkl"
    )

    configs = build_linear_configs(
        env_ids=ENV_IDS,
        seeds=TUNING_SEEDS,
        explorer_specs=alpha_selection_specs,
        step_budget=STEP_BUDGET,
        alpha_bars=ALPHA_BAR_GRID,
        algo="sarsa-lambda",
        gamma=1.0,
        lam=LAMBDA_FIXED,
        q_init=0.0,
        reward_scale=1.0,
        n_bins=100,
        n_eval_points=20,
        n_eval_episodes=10,
    )

    expected = (
        len(ENV_IDS)
        * len(TUNING_SEEDS)
        * len(ALPHA_BAR_GRID)
    )

    if len(configs) != expected:
        raise RuntimeError(
            "Unexpected number of alpha-selection "
            f"configs: expected {expected}, "
            f"got {len(configs)}."
        )

    cpu_count = os.cpu_count() or 2

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(configs),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "LINEAR STEP-SIZE SELECTION"
    )

    print(
        f"Environments: {len(ENV_IDS)}"
    )

    print(
        f"Alpha values: {len(ALPHA_BAR_GRID)}"
    )

    print(
        f"Seeds per setting: {len(TUNING_SEEDS)}"
    )

    print(
        f"Total runs: {len(configs)}"
    )

    print(
        f"Detected CPUs: {cpu_count}"
    )

    print(
        f"Workers: {workers}"
    )

    print(
        f"Output: {out_path}"
    )

    results = run_stage(
        configs=configs,
        out_path=out_path,
        label="alpha selection",
        workers=workers,
        save_every=5,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    successful = [
        result
        for result in results
        if "error" not in result
    ]

    print()

    print(
        f"Stored results: {len(results)}"
    )

    print(
        f"Successful: {len(successful)}"
    )

    print(
        f"Errors: {len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED CONFIGURATIONS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            config = result.get(
                "config",
                {}
            )

            print(
                f"{i}. "
                f"env={config.get('env_id')} "
                f"alpha_bar={config.get('alpha_bar')} "
                f"seed={config.get('seed')}"
            )

            print(
                f"   {result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} alpha-selection "
            "runs failed."
        )

    if len(successful) != expected:
        raise RuntimeError(
            "Successful result count is "
            f"{len(successful)}, expected "
            f"{expected}."
        )

    print()

    print(
        "ALPHA-SELECTION SWEEP COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/tune_alpha.py


In [12]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.tune_alpha",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()
print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Alpha-selection sweep failed."
    )

LINEAR STEP-SIZE SELECTION
Environments: 4
Alpha values: 3
Seeds per setting: 3
Total runs: 36
Detected CPUs: 20
Workers: 8
Output: src\results\tuning\alpha_selection.pkl
[alpha selection] loaded=36 pending=9 workers=8
[alpha selection] 1/9 OK
[alpha selection] 2/9 OK
[alpha selection] 3/9 OK
[alpha selection] 4/9 OK
[alpha selection] 5/9 OK
[alpha selection] 6/9 OK
[alpha selection] 7/9 OK
[alpha selection] 8/9 OK
[alpha selection] 9/9 OK

Stored results: 36
Successful: 36
Errors: 0

ALPHA-SELECTION SWEEP COMPLETED

RETURN CODE: 0


## Select Alpha

It loads:

src/results/tuning/alpha_selection.pkl

and reconstructs the \(3\times3\) result table for every environment:

alpha_bar = 0.10
    seed 1000
    seed 1001
    seed 1002

alpha_bar = 0.25
    seed 1000
    seed 1001
    seed 1002

alpha_bar = 0.50
    seed 1000
    seed 1001
    seed 1002

For each \(\bar\alpha\), it calculates:

$$ \text{mean final greedy return} = \frac{ G_{1000}+G_{1001}+G_{1002} }{3} $$

and chooses the largest mean.

That applies even when returns are negative.

For example:

$$ -120 > -150 $$

so \(-120\) is better.

The final_eval field in each runner result is the final greedy-evaluation score, so that is precisely the quantity we want to aggregate here.

Why the edge check is important

Our grid is:

$$ \{0.1,\;0.25,\;0.5\} $$

Suppose the winner is:

$$ 0.25. $$

Great—it is internal to the grid.

But suppose the winner is:

$$ 0.5. $$

Then we do not know whether:

$$ 0.75 $$

or:

$$ 1.0 $$

would have been even better.

Likewise, if \(0.1\) wins, perhaps a smaller value would improve things.

So the script deliberately refuses to create the final:

selected_alpha.json

when a winner lies on the boundary

In [13]:
%%writefile src/scripts/select_alpha.py
import json
import pickle

from pathlib import Path

import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    ALPHA_BAR_GRID,
)


RESULTS_PATH = Path(
    "src/results/tuning/alpha_selection.pkl"
)

SUMMARY_PATH = Path(
    "src/results/tuning/alpha_selection_summary.json"
)

SELECTED_PATH = Path(
    "src/results/tuning/selected_alpha.json"
)


def _write_json_atomic(obj, path):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(path)


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"Results file not found: "
            f"{RESULTS_PATH}"
        )

    if SELECTED_PATH.exists():
        SELECTED_PATH.unlink()

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(f)

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Alpha-selection results must "
            "be stored as a list."
        )

    expected = (
        len(ENV_IDS)
        * len(TUNING_SEEDS)
        * len(ALPHA_BAR_GRID)
    )

    if len(results) != expected:
        raise RuntimeError(
            "Unexpected number of results: "
            f"expected {expected}, "
            f"found {len(results)}."
        )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    if errors:
        raise RuntimeError(
            "Alpha-selection results still "
            f"contain {len(errors)} errors."
        )

    grouped = {}

    for result in results:
        config = result.get(
            "config",
            {}
        )

        env_id = config.get(
            "env_id",
            result.get("env_id"),
        )

        alpha_bar = config.get(
            "alpha_bar",
            result.get("alpha_bar"),
        )

        seed = config.get(
            "seed",
            result.get("seed"),
        )

        if env_id is None:
            raise KeyError(
                "A result is missing env_id."
            )

        if alpha_bar is None:
            raise KeyError(
                "A result is missing alpha_bar."
            )

        if seed is None:
            raise KeyError(
                "A result is missing seed."
            )

        if "final_eval" not in result:
            raise KeyError(
                "A result is missing final_eval."
            )

        alpha_bar = float(
            alpha_bar
        )

        seed = int(
            seed
        )

        final_eval = float(
            result["final_eval"]
        )

        if not np.isfinite(
            final_eval
        ):
            raise ValueError(
                "Non-finite final evaluation "
                f"for env={env_id}, "
                f"alpha_bar={alpha_bar}, "
                f"seed={seed}."
            )

        key = (
            env_id,
            alpha_bar,
        )

        grouped.setdefault(
            key,
            {}
        )

        if seed in grouped[key]:
            raise RuntimeError(
                "Duplicate result for "
                f"env={env_id}, "
                f"alpha_bar={alpha_bar}, "
                f"seed={seed}."
            )

        grouped[key][seed] = (
            final_eval
        )

    summary = {}
    selected = {}
    edge_winners = []

    alpha_min = float(
        min(ALPHA_BAR_GRID)
    )

    alpha_max = float(
        max(ALPHA_BAR_GRID)
    )

    expected_seeds = set(
        int(seed)
        for seed in TUNING_SEEDS
    )

    print(
        "LINEAR STEP-SIZE SELECTION RESULTS"
    )

    print()

    for env_id in ENV_IDS:
        print(
            f"Environment: {env_id}"
        )

        print(
            "-" * (
                len(env_id) + 13
            )
        )

        environment_rows = []

        for alpha_bar in ALPHA_BAR_GRID:
            alpha_bar = float(
                alpha_bar
            )

            key = (
                env_id,
                alpha_bar,
            )

            if key not in grouped:
                raise RuntimeError(
                    "Missing result group for "
                    f"env={env_id}, "
                    f"alpha_bar={alpha_bar}."
                )

            seed_scores = (
                grouped[key]
            )

            actual_seeds = set(
                seed_scores
            )

            if (
                actual_seeds
                != expected_seeds
            ):
                raise RuntimeError(
                    "Seed mismatch for "
                    f"env={env_id}, "
                    f"alpha_bar={alpha_bar}. "
                    f"Expected "
                    f"{sorted(expected_seeds)}, "
                    f"found "
                    f"{sorted(actual_seeds)}."
                )

            ordered_scores = [
                float(
                    seed_scores[
                        int(seed)
                    ]
                )
                for seed
                in TUNING_SEEDS
            ]

            mean_score = float(
                np.mean(
                    ordered_scores
                )
            )

            std_score = float(
                np.std(
                    ordered_scores,
                    ddof=1,
                )
            )

            environment_rows.append(
                {
                    "alpha_bar": (
                        alpha_bar
                    ),
                    "seed_scores": (
                        ordered_scores
                    ),
                    "mean_final_eval": (
                        mean_score
                    ),
                    "std_final_eval": (
                        std_score
                    ),
                }
            )

            score_text = "  ".join(
                f"{int(seed)}="
                f"{score:.3f}"
                for seed, score
                in zip(
                    TUNING_SEEDS,
                    ordered_scores,
                )
            )

            print(
                f"alpha_bar="
                f"{alpha_bar:<4g}  "
                f"{score_text}  "
                f"mean="
                f"{mean_score:.3f}  "
                f"std="
                f"{std_score:.3f}"
            )

        means = np.asarray(
            [
                row[
                    "mean_final_eval"
                ]
                for row
                in environment_rows
            ],
            dtype=np.float64,
        )

        best_mean = float(
            np.max(means)
        )

        winner_indices = np.flatnonzero(
            np.isclose(
                means,
                best_mean,
                rtol=1e-12,
                atol=1e-12,
            )
        )

        if len(
            winner_indices
        ) != 1:
            tied = [
                environment_rows[i][
                    "alpha_bar"
                ]
                for i
                in winner_indices
            ]

            raise RuntimeError(
                "Alpha selection produced "
                f"a tie for {env_id}: "
                f"{tied}."
            )

        winner = environment_rows[
            int(
                winner_indices[0]
            )
        ]

        winner_alpha = float(
            winner["alpha_bar"]
        )

        selected[
            env_id
        ] = winner_alpha

        summary[
            env_id
        ] = {
            "candidates": (
                environment_rows
            ),
            "selected_alpha_bar": (
                winner_alpha
            ),
            "selected_mean_final_eval": (
                float(
                    winner[
                        "mean_final_eval"
                    ]
                )
            ),
        }

        print()

        print(
            f"WINNER: alpha_bar="
            f"{winner_alpha:g}"
        )

        print(
            "Mean final greedy return: "
            f"{winner['mean_final_eval']:.3f}"
        )

        if (
            np.isclose(
                winner_alpha,
                alpha_min,
            )
            or np.isclose(
                winner_alpha,
                alpha_max,
            )
        ):
            edge_winners.append(
                env_id
            )

            print(
                "WARNING: winner is at "
                "the edge of the alpha grid."
            )

        print()
        print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    print(
        "Summary written to:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    if edge_winners:
        print(
            "EDGE-WINNER CHECK FAILED"
        )

        print()

        print(
            "The following environments "
            "selected an alpha value at "
            "the boundary of the tested grid:"
        )

        for env_id in edge_winners:
            print(
                f"  {env_id}: "
                f"alpha_bar="
                f"{selected[env_id]:g}"
            )

        print()

        print(
            "Do not freeze the selected "
            "alpha values yet."
        )

        print(
            "The alpha grid must be "
            "extended in the indicated "
            "direction and those "
            "environment(s) rerun."
        )

        raise RuntimeError(
            "One or more alpha winners "
            "are at the grid boundary."
        )

    _write_json_atomic(
        selected,
        SELECTED_PATH,
    )

    print(
        "SELECTED ALPHA VALUES"
    )

    print()

    for env_id in ENV_IDS:
        print(
            f"{env_id}: "
            f"{selected[env_id]:g}"
        )

    print()

    print(
        "Selected values written to:"
    )

    print(
        SELECTED_PATH
    )

    print()

    print(
        "ALPHA SELECTION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/select_alpha.py


In [14]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.select_alpha",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 1

STDOUT:
LINEAR STEP-SIZE SELECTION RESULTS

Environment: MountainCar-v0
---------------------------
alpha_bar=0.1   1000=-137.600  1001=-110.100  1002=-106.700  mean=-118.133  std=16.944
alpha_bar=0.25  1000=-124.100  1001=-110.600  1002=-100.700  mean=-111.800  std=11.746
alpha_bar=0.5   1000=-101.800  1001=-107.200  1002=-119.700  mean=-109.567  std=9.182

WINNER: alpha_bar=0.5
Mean final greedy return: -109.567


Environment: CartPole-v1
------------------------
alpha_bar=0.1   1000=257.200  1001=396.200  1002=308.000  mean=320.467  std=70.334
alpha_bar=0.25  1000=417.900  1001=452.500  1002=277.300  mean=382.567  std=92.791
alpha_bar=0.5   1000=420.100  1001=500.000  1002=372.600  mean=430.900  std=64.383

WINNER: alpha_bar=0.5
Mean final greedy return: 430.900


Environment: Acrobot-v1
-----------------------
alpha_bar=0.1   1000=-215.800  1001=-246.300  1002=-290.300  mean=-250.800  std=37.453
alpha_bar=0.25  1000=-110.500  1001=-156.300  1002=-129.800  mean=-132.

In [15]:
%%writefile src/scripts/extend_alpha.py
import os

from pathlib import Path

from src.sweep import run_stage
from src.sweep_configs import (
    TUNING_SEEDS,
    STEP_BUDGET,
    LAMBDA_FIXED,
    alpha_selection_specs,
    build_linear_configs,
)


MAX_WORKERS = 8

UPPER_EXTENSION_ENVS = (
    "MountainCar-v0",
    "CartPole-v1",
    "Acrobot-v1",
)

UPPER_EXTENSION_ALPHA = (
    0.75,
    1.0,
)

LOWER_EXTENSION_ENVS = (
    "LunarLander-v3",
)

LOWER_EXTENSION_ALPHA = (
    0.025,
    0.05,
)


def main():
    out_path = Path(
        "src/results/tuning/alpha_selection.pkl"
    )

    if not out_path.exists():
        raise FileNotFoundError(
            f"Existing alpha-selection file "
            f"not found: {out_path}"
        )

    upper_configs = build_linear_configs(
        env_ids=UPPER_EXTENSION_ENVS,
        seeds=TUNING_SEEDS,
        explorer_specs=alpha_selection_specs,
        step_budget=STEP_BUDGET,
        alpha_bars=UPPER_EXTENSION_ALPHA,
        algo="sarsa-lambda",
        gamma=1.0,
        lam=LAMBDA_FIXED,
        q_init=0.0,
        reward_scale=1.0,
        n_bins=100,
        n_eval_points=20,
        n_eval_episodes=10,
    )

    lower_configs = build_linear_configs(
        env_ids=LOWER_EXTENSION_ENVS,
        seeds=TUNING_SEEDS,
        explorer_specs=alpha_selection_specs,
        step_budget=STEP_BUDGET,
        alpha_bars=LOWER_EXTENSION_ALPHA,
        algo="sarsa-lambda",
        gamma=1.0,
        lam=LAMBDA_FIXED,
        q_init=0.0,
        reward_scale=1.0,
        n_bins=100,
        n_eval_points=20,
        n_eval_episodes=10,
    )

    configs = (
        upper_configs
        + lower_configs
    )

    expected_new = (
        len(UPPER_EXTENSION_ENVS)
        * len(UPPER_EXTENSION_ALPHA)
        * len(TUNING_SEEDS)
        +
        len(LOWER_EXTENSION_ENVS)
        * len(LOWER_EXTENSION_ALPHA)
        * len(TUNING_SEEDS)
    )

    if len(configs) != expected_new:
        raise RuntimeError(
            "Unexpected number of extension "
            f"configs: expected {expected_new}, "
            f"got {len(configs)}."
        )

    cpu_count = os.cpu_count() or 2

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(configs),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "LINEAR STEP-SIZE GRID EXTENSION"
    )

    print()

    print(
        "Higher-alpha environments:"
    )

    for env_id in UPPER_EXTENSION_ENVS:
        print(
            f"  {env_id}"
        )

    print(
        f"Higher alpha values: "
        f"{UPPER_EXTENSION_ALPHA}"
    )

    print()

    print(
        "Lower-alpha environments:"
    )

    for env_id in LOWER_EXTENSION_ENVS:
        print(
            f"  {env_id}"
        )

    print(
        f"Lower alpha values: "
        f"{LOWER_EXTENSION_ALPHA}"
    )

    print()

    print(
        f"New runs: {len(configs)}"
    )

    print(
        f"Detected CPUs: {cpu_count}"
    )

    print(
        f"Workers: {workers}"
    )

    print(
        f"Output: {out_path}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=out_path,
        label="alpha extension",
        workers=workers,
        save_every=5,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    successful = [
        result
        for result in results
        if "error" not in result
    ]

    expected_total = (
        36
        + expected_new
    )

    print()

    print(
        f"Stored results: {len(results)}"
    )

    print(
        f"Successful: {len(successful)}"
    )

    print(
        f"Errors: {len(errors)}"
    )

    if errors:
        print()
        print(
            "FAILED CONFIGURATIONS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            config = result.get(
                "config",
                {}
            )

            print(
                f"{i}. "
                f"env={config.get('env_id')} "
                f"alpha_bar="
                f"{config.get('alpha_bar')} "
                f"seed={config.get('seed')}"
            )

            print(
                f"   {result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} alpha-extension "
            "runs failed."
        )

    if len(results) != expected_total:
        raise RuntimeError(
            "Unexpected total result count: "
            f"expected {expected_total}, "
            f"found {len(results)}."
        )

    if len(successful) != expected_total:
        raise RuntimeError(
            "Unexpected successful result count: "
            f"expected {expected_total}, "
            f"found {len(successful)}."
        )

    print()

    print(
        "ALPHA GRID EXTENSION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/extend_alpha.py


In [16]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.extend_alpha",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Alpha-grid extension failed."
    )

LINEAR STEP-SIZE GRID EXTENSION

Higher-alpha environments:
  MountainCar-v0
  CartPole-v1
  Acrobot-v1
Higher alpha values: (0.75, 1.0)

Lower-alpha environments:
  LunarLander-v3
Lower alpha values: (0.025, 0.05)

New runs: 24
Detected CPUs: 20
Workers: 8
Output: src\results\tuning\alpha_selection.pkl

[alpha extension] loaded=36 pending=24 workers=8
[alpha extension] 1/24 OK
[alpha extension] 2/24 OK
[alpha extension] 3/24 OK
[alpha extension] 4/24 OK
[alpha extension] 5/24 OK
[alpha extension] 6/24 OK
[alpha extension] 7/24 OK
[alpha extension] 8/24 OK
[alpha extension] 9/24 OK
[alpha extension] 10/24 OK
[alpha extension] 11/24 OK
[alpha extension] 12/24 OK
[alpha extension] 13/24 OK
[alpha extension] 14/24 OK
[alpha extension] 15/24 OK
[alpha extension] 16/24 OK
[alpha extension] 17/24 OK
[alpha extension] 18/24 OK
[alpha extension] 19/24 OK
[alpha extension] 20/24 OK
[alpha extension] 21/24 OK
[alpha extension] 22/24 OK
[alpha extension] 23/24 OK
[alpha extension] 24/24 OK

Store

In [17]:
%%writefile src/scripts/select_alpha.py
import json
import pickle

from pathlib import Path

import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    ALPHA_BAR_GRID_BY_ENV,
)


RESULTS_PATH = Path(
    "src/results/tuning/alpha_selection.pkl"
)

SUMMARY_PATH = Path(
    "src/results/tuning/alpha_selection_summary.json"
)

SELECTED_PATH = Path(
    "src/results/tuning/selected_alpha.json"
)


def _write_json_atomic(obj, path):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(path)


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"Results file not found: "
            f"{RESULTS_PATH}"
        )

    if SELECTED_PATH.exists():
        SELECTED_PATH.unlink()

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(f)

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Alpha-selection results must "
            "be stored as a list."
        )

    expected = sum(
        len(
            ALPHA_BAR_GRID_BY_ENV[
                env_id
            ]
        )
        * len(TUNING_SEEDS)
        for env_id in ENV_IDS
    )

    if len(results) != expected:
        raise RuntimeError(
            "Unexpected number of results: "
            f"expected {expected}, "
            f"found {len(results)}."
        )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    if errors:
        raise RuntimeError(
            "Alpha-selection results still "
            f"contain {len(errors)} errors."
        )

    grouped = {}

    for result in results:
        config = result.get(
            "config",
            {}
        )

        env_id = config.get(
            "env_id",
            result.get("env_id"),
        )

        alpha_bar = config.get(
            "alpha_bar",
            result.get("alpha_bar"),
        )

        seed = config.get(
            "seed",
            result.get("seed"),
        )

        if env_id is None:
            raise KeyError(
                "A result is missing env_id."
            )

        if alpha_bar is None:
            raise KeyError(
                "A result is missing alpha_bar."
            )

        if seed is None:
            raise KeyError(
                "A result is missing seed."
            )

        if "final_eval" not in result:
            raise KeyError(
                "A result is missing final_eval."
            )

        if env_id not in ENV_IDS:
            raise ValueError(
                f"Unexpected environment "
                f"in results: {env_id}"
            )

        alpha_bar = float(
            alpha_bar
        )

        seed = int(
            seed
        )

        final_eval = float(
            result["final_eval"]
        )

        if not np.isfinite(
            final_eval
        ):
            raise ValueError(
                "Non-finite final evaluation "
                f"for env={env_id}, "
                f"alpha_bar={alpha_bar}, "
                f"seed={seed}."
            )

        valid_alpha = {
            float(value)
            for value
            in ALPHA_BAR_GRID_BY_ENV[
                env_id
            ]
        }

        if alpha_bar not in valid_alpha:
            raise ValueError(
                "Unexpected alpha_bar "
                f"for {env_id}: "
                f"{alpha_bar}"
            )

        key = (
            env_id,
            alpha_bar,
        )

        grouped.setdefault(
            key,
            {}
        )

        if seed in grouped[key]:
            raise RuntimeError(
                "Duplicate result for "
                f"env={env_id}, "
                f"alpha_bar={alpha_bar}, "
                f"seed={seed}."
            )

        grouped[key][seed] = (
            final_eval
        )

    expected_seeds = {
        int(seed)
        for seed in TUNING_SEEDS
    }

    summary = {}
    selected = {}
    edge_winners = []

    print(
        "LINEAR STEP-SIZE SELECTION RESULTS"
    )

    print()

    for env_id in ENV_IDS:
        alpha_grid = tuple(
            float(alpha_bar)
            for alpha_bar
            in ALPHA_BAR_GRID_BY_ENV[
                env_id
            ]
        )

        if len(alpha_grid) < 3:
            raise RuntimeError(
                f"Alpha grid for {env_id} "
                "must contain at least "
                "three values."
            )

        if tuple(
            sorted(alpha_grid)
        ) != alpha_grid:
            raise RuntimeError(
                f"Alpha grid for {env_id} "
                "must be sorted in "
                "ascending order."
            )

        alpha_min = float(
            alpha_grid[0]
        )

        alpha_max = float(
            alpha_grid[-1]
        )

        print(
            f"Environment: {env_id}"
        )

        print(
            "-" * (
                len(env_id) + 13
            )
        )

        environment_rows = []

        for alpha_bar in alpha_grid:
            key = (
                env_id,
                alpha_bar,
            )

            if key not in grouped:
                raise RuntimeError(
                    "Missing result group for "
                    f"env={env_id}, "
                    f"alpha_bar={alpha_bar}."
                )

            seed_scores = (
                grouped[key]
            )

            actual_seeds = set(
                seed_scores
            )

            if (
                actual_seeds
                != expected_seeds
            ):
                raise RuntimeError(
                    "Seed mismatch for "
                    f"env={env_id}, "
                    f"alpha_bar={alpha_bar}. "
                    f"Expected "
                    f"{sorted(expected_seeds)}, "
                    f"found "
                    f"{sorted(actual_seeds)}."
                )

            ordered_scores = [
                float(
                    seed_scores[
                        int(seed)
                    ]
                )
                for seed
                in TUNING_SEEDS
            ]

            mean_score = float(
                np.mean(
                    ordered_scores
                )
            )

            std_score = float(
                np.std(
                    ordered_scores,
                    ddof=1,
                )
            )

            min_score = float(
                np.min(
                    ordered_scores
                )
            )

            max_score = float(
                np.max(
                    ordered_scores
                )
            )

            environment_rows.append(
                {
                    "alpha_bar": (
                        alpha_bar
                    ),
                    "seed_scores": (
                        ordered_scores
                    ),
                    "mean_final_eval": (
                        mean_score
                    ),
                    "std_final_eval": (
                        std_score
                    ),
                    "min_final_eval": (
                        min_score
                    ),
                    "max_final_eval": (
                        max_score
                    ),
                }
            )

            score_text = "  ".join(
                f"{int(seed)}="
                f"{score:.3f}"
                for seed, score
                in zip(
                    TUNING_SEEDS,
                    ordered_scores,
                )
            )

            print(
                f"alpha_bar="
                f"{alpha_bar:<5g}  "
                f"{score_text}  "
                f"mean="
                f"{mean_score:.3f}  "
                f"std="
                f"{std_score:.3f}"
            )

        means = np.asarray(
            [
                row[
                    "mean_final_eval"
                ]
                for row
                in environment_rows
            ],
            dtype=np.float64,
        )

        best_mean = float(
            np.max(
                means
            )
        )

        winner_indices = (
            np.flatnonzero(
                np.isclose(
                    means,
                    best_mean,
                    rtol=1e-12,
                    atol=1e-12,
                )
            )
        )

        if len(
            winner_indices
        ) != 1:
            tied = [
                environment_rows[i][
                    "alpha_bar"
                ]
                for i
                in winner_indices
            ]

            raise RuntimeError(
                "Alpha selection produced "
                f"a tie for {env_id}: "
                f"{tied}."
            )

        winner_index = int(
            winner_indices[0]
        )

        winner = (
            environment_rows[
                winner_index
            ]
        )

        winner_alpha = float(
            winner[
                "alpha_bar"
            ]
        )

        is_lower_edge = bool(
            np.isclose(
                winner_alpha,
                alpha_min,
                rtol=1e-12,
                atol=1e-12,
            )
        )

        is_upper_edge = bool(
            np.isclose(
                winner_alpha,
                alpha_max,
                rtol=1e-12,
                atol=1e-12,
            )
        )

        is_edge = (
            is_lower_edge
            or is_upper_edge
        )

        selected[
            env_id
        ] = winner_alpha

        summary[
            env_id
        ] = {
            "alpha_grid": [
                float(value)
                for value
                in alpha_grid
            ],
            "candidates": (
                environment_rows
            ),
            "selected_alpha_bar": (
                winner_alpha
            ),
            "selected_mean_final_eval": (
                float(
                    winner[
                        "mean_final_eval"
                    ]
                )
            ),
            "selected_std_final_eval": (
                float(
                    winner[
                        "std_final_eval"
                    ]
                )
            ),
            "winner_index": (
                winner_index
            ),
            "winner_at_edge": (
                is_edge
            ),
            "winner_at_lower_edge": (
                is_lower_edge
            ),
            "winner_at_upper_edge": (
                is_upper_edge
            ),
        }

        print()

        print(
            f"WINNER: alpha_bar="
            f"{winner_alpha:g}"
        )

        print(
            "Mean final greedy return: "
            f"{winner['mean_final_eval']:.3f}"
        )

        print(
            "Standard deviation: "
            f"{winner['std_final_eval']:.3f}"
        )

        if is_edge:
            direction = (
                "LOWER"
                if is_lower_edge
                else "UPPER"
            )

            edge_winners.append(
                {
                    "env_id": env_id,
                    "alpha_bar": (
                        winner_alpha
                    ),
                    "direction": (
                        direction
                    ),
                }
            )

            print(
                f"WARNING: winner is at "
                f"the {direction.lower()} "
                "edge of the alpha grid."
            )

        else:
            left_alpha = (
                environment_rows[
                    winner_index - 1
                ][
                    "alpha_bar"
                ]
            )

            right_alpha = (
                environment_rows[
                    winner_index + 1
                ][
                    "alpha_bar"
                ]
            )

            print(
                "Winner is bracketed by "
                f"alpha_bar={left_alpha:g} "
                "and "
                f"alpha_bar={right_alpha:g}."
            )

        print()
        print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    print(
        "Summary written to:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    if edge_winners:
        print(
            "EDGE-WINNER CHECK FAILED"
        )

        print()

        print(
            "The following environments "
            "still select an alpha value "
            "at a grid boundary:"
        )

        print()

        for item in edge_winners:
            print(
                f"  {item['env_id']}: "
                f"alpha_bar="
                f"{item['alpha_bar']:g} "
                f"({item['direction']} edge)"
            )

        print()

        print(
            "Do not freeze the selected "
            "alpha values yet."
        )

        print(
            "Only the affected "
            "environment(s) need another "
            "grid extension."
        )

        raise RuntimeError(
            "One or more alpha winners "
            "are still at a grid boundary."
        )

    _write_json_atomic(
        selected,
        SELECTED_PATH,
    )

    print(
        "SELECTED ALPHA VALUES"
    )

    print()

    for env_id in ENV_IDS:
        print(
            f"{env_id}: "
            f"{selected[env_id]:g}"
        )

    print()

    print(
        "Selected values written to:"
    )

    print(
        SELECTED_PATH
    )

    print()

    print(
        "ALPHA SELECTION COMPLETED"
    )


if __name__ == "__main__":
    main()

Overwriting src/scripts/select_alpha.py


In [18]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.select_alpha",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 1

STDOUT:
LINEAR STEP-SIZE SELECTION RESULTS

Environment: MountainCar-v0
---------------------------
alpha_bar=0.1    1000=-137.600  1001=-110.100  1002=-106.700  mean=-118.133  std=16.944
alpha_bar=0.25   1000=-124.100  1001=-110.600  1002=-100.700  mean=-111.800  std=11.746
alpha_bar=0.5    1000=-101.800  1001=-107.200  1002=-119.700  mean=-109.567  std=9.182
alpha_bar=0.75   1000=-110.900  1001=-107.400  1002=-105.700  mean=-108.000  std=2.651
alpha_bar=1      1000=-200.000  1001=-195.000  1002=-144.500  mean=-179.833  std=30.702

WINNER: alpha_bar=0.75
Mean final greedy return: -108.000
Standard deviation: 2.651
Winner is bracketed by alpha_bar=0.5 and alpha_bar=1.


Environment: CartPole-v1
------------------------
alpha_bar=0.1    1000=257.200  1001=396.200  1002=308.000  mean=320.467  std=70.334
alpha_bar=0.25   1000=417.900  1001=452.500  1002=277.300  mean=382.567  std=92.791
alpha_bar=0.5    1000=420.100  1001=500.000  1002=372.600  mean=430.900  std=64.383
alp

In [19]:
%%writefile src/scripts/extend_alpha_cartpole.py
import os

from pathlib import Path

from src.sweep import run_stage
from src.sweep_configs import (
    TUNING_SEEDS,
    STEP_BUDGET,
    LAMBDA_FIXED,
    alpha_selection_specs,
    build_linear_configs,
)


MAX_WORKERS = 6

ENV_IDS = (
    "CartPole-v1",
)

ALPHA_VALUES = (
    1.25,
    1.5,
)


def main():
    out_path = Path(
        "src/results/tuning/alpha_selection.pkl"
    )

    if not out_path.exists():
        raise FileNotFoundError(
            f"Existing alpha-selection file "
            f"not found: {out_path}"
        )

    configs = build_linear_configs(
        env_ids=ENV_IDS,
        seeds=TUNING_SEEDS,
        explorer_specs=alpha_selection_specs,
        step_budget=STEP_BUDGET,
        alpha_bars=ALPHA_VALUES,
        algo="sarsa-lambda",
        gamma=1.0,
        lam=LAMBDA_FIXED,
        q_init=0.0,
        reward_scale=1.0,
        n_bins=100,
        n_eval_points=20,
        n_eval_episodes=10,
    )

    expected_new = (
        len(ENV_IDS)
        * len(ALPHA_VALUES)
        * len(TUNING_SEEDS)
    )

    if len(configs) != expected_new:
        raise RuntimeError(
            "Unexpected number of CartPole "
            f"extension configs: expected "
            f"{expected_new}, got "
            f"{len(configs)}."
        )

    cpu_count = os.cpu_count() or 2

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(configs),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "CARTPOLE STEP-SIZE GRID EXTENSION"
    )

    print()

    print(
        f"Environment: {ENV_IDS[0]}"
    )

    print(
        f"New alpha values: {ALPHA_VALUES}"
    )

    print(
        f"Seeds: {TUNING_SEEDS}"
    )

    print(
        f"New runs: {len(configs)}"
    )

    print(
        f"Detected CPUs: {cpu_count}"
    )

    print(
        f"Workers: {workers}"
    )

    print(
        f"Output: {out_path}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=out_path,
        label="CartPole alpha extension",
        workers=workers,
        save_every=2,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    successful = [
        result
        for result in results
        if "error" not in result
    ]

    expected_total = 66

    print()

    print(
        f"Stored results: {len(results)}"
    )

    print(
        f"Successful: {len(successful)}"
    )

    print(
        f"Errors: {len(errors)}"
    )

    if errors:
        print()
        print(
            "FAILED CONFIGURATIONS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            config = result.get(
                "config",
                {}
            )

            print(
                f"{i}. "
                f"env={config.get('env_id')} "
                f"alpha_bar="
                f"{config.get('alpha_bar')} "
                f"seed={config.get('seed')}"
            )

            print(
                f"   {result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} CartPole "
            "alpha-extension runs failed."
        )

    if len(results) != expected_total:
        raise RuntimeError(
            "Unexpected total result count: "
            f"expected {expected_total}, "
            f"found {len(results)}."
        )

    if len(successful) != expected_total:
        raise RuntimeError(
            "Unexpected successful result count: "
            f"expected {expected_total}, "
            f"found {len(successful)}."
        )

    print()

    print(
        "CARTPOLE ALPHA GRID EXTENSION "
        "COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/extend_alpha_cartpole.py


In [20]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.extend_alpha_cartpole",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "CartPole alpha-grid extension failed."
    )

CARTPOLE STEP-SIZE GRID EXTENSION

Environment: CartPole-v1
New alpha values: (1.25, 1.5)
Seeds: (1000, 1001, 1002)
New runs: 6
Detected CPUs: 20
Workers: 6
Output: src\results\tuning\alpha_selection.pkl

[CartPole alpha extension] loaded=60 pending=6 workers=6
[CartPole alpha extension] 1/6 OK
[CartPole alpha extension] 2/6 OK
[CartPole alpha extension] 3/6 OK
[CartPole alpha extension] 4/6 OK
[CartPole alpha extension] 5/6 OK
[CartPole alpha extension] 6/6 OK

Stored results: 66
Successful: 66
Errors: 0

CARTPOLE ALPHA GRID EXTENSION COMPLETED

RETURN CODE: 0


In [21]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.select_alpha",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
LINEAR STEP-SIZE SELECTION RESULTS

Environment: MountainCar-v0
---------------------------
alpha_bar=0.1    1000=-137.600  1001=-110.100  1002=-106.700  mean=-118.133  std=16.944
alpha_bar=0.25   1000=-124.100  1001=-110.600  1002=-100.700  mean=-111.800  std=11.746
alpha_bar=0.5    1000=-101.800  1001=-107.200  1002=-119.700  mean=-109.567  std=9.182
alpha_bar=0.75   1000=-110.900  1001=-107.400  1002=-105.700  mean=-108.000  std=2.651
alpha_bar=1      1000=-200.000  1001=-195.000  1002=-144.500  mean=-179.833  std=30.702

WINNER: alpha_bar=0.75
Mean final greedy return: -108.000
Standard deviation: 2.651
Winner is bracketed by alpha_bar=0.5 and alpha_bar=1.


Environment: CartPole-v1
------------------------
alpha_bar=0.1    1000=257.200  1001=396.200  1002=308.000  mean=320.467  std=70.334
alpha_bar=0.25   1000=417.900  1001=452.500  1002=277.300  mean=382.567  std=92.791
alpha_bar=0.5    1000=420.100  1001=500.000  1002=372.600  mean=430.900  std=64.383
alp

**Final Frozen Alphas**
```
MountainCar-v0  → ᾱ = 0.75
CartPole-v1     → ᾱ = 1.00
Acrobot-v1      → ᾱ = 0.50
LunarLander-v3  → ᾱ = 0.10
```

# Baseline Tuning

We tune four baseline exploration methods:

Fixed ε
Decay ε
Boltzmann
VDBE

RATE is not included yet.

For every environment, we now hold the selected $\bar\alpha$ fixed:

MountainCar: 0.75
CartPole:    1.00
Acrobot:     0.50
LunarLander: 0.10

and vary only the exploration hyperparameter.

The grids are:
```
Fixed ε
    0.05
    0.10
    0.20

Decay
    horizon = 20% budget
    horizon = 40% budget
    horizon = 60% budget

Boltzmann
    horizon = 20% budget
    horizon = 40% budget
    horizon = 60% budget

VDBE σ
    0.5
    1
    5
    20
    100
    500
    2000
    10000
```

In [22]:
%%writefile src/scripts/tune_linear_baselines.py
import json
import os

from pathlib import Path

import numpy as np

from src.sweep import run_stage
from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    STEP_BUDGET,
    LAMBDA_FIXED,
    fixed_specs,
    decay_specs,
    boltzmann_specs,
    vdbe_specs,
    build_linear_configs,
)


MAX_WORKERS = 8

SELECTED_ALPHA_PATH = Path(
    "src/results/tuning/selected_alpha.json"
)

OUT_PATH = Path(
    "src/results/tuning/linear_baselines.pkl"
)


def _load_selected_alpha():
    if not SELECTED_ALPHA_PATH.exists():
        raise FileNotFoundError(
            "Selected-alpha file not found: "
            f"{SELECTED_ALPHA_PATH}"
        )

    with open(
        SELECTED_ALPHA_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        selected = json.load(f)

    if not isinstance(
        selected,
        dict,
    ):
        raise TypeError(
            "selected_alpha.json must contain "
            "a dictionary."
        )

    expected_envs = set(
        ENV_IDS
    )

    actual_envs = set(
        selected
    )

    if actual_envs != expected_envs:
        raise RuntimeError(
            "Environment mismatch in "
            "selected_alpha.json. "
            f"Expected {sorted(expected_envs)}, "
            f"found {sorted(actual_envs)}."
        )

    cleaned = {}

    for env_id in ENV_IDS:
        alpha_bar = float(
            selected[env_id]
        )

        if (
            not np.isfinite(alpha_bar)
            or alpha_bar <= 0
        ):
            raise ValueError(
                "Invalid selected alpha for "
                f"{env_id}: {alpha_bar}"
            )

        cleaned[env_id] = (
            alpha_bar
        )

    return cleaned


def _build_configs(
    selected_alpha
):
    configs = []

    counts = {
        "fixed": 0,
        "decay": 0,
        "boltzmann": 0,
        "vdbe": 0,
    }

    for env_id in ENV_IDS:
        alpha_bar = (
            selected_alpha[
                env_id
            ]
        )

        fixed_configs = (
            build_linear_configs(
                env_ids=(
                    env_id,
                ),
                seeds=TUNING_SEEDS,
                explorer_specs=(
                    fixed_specs()
                ),
                step_budget=STEP_BUDGET,
                alpha_bars=(
                    alpha_bar,
                ),
                algo="sarsa-lambda",
                gamma=1.0,
                lam=LAMBDA_FIXED,
                q_init=0.0,
                reward_scale=1.0,
                n_bins=100,
                n_eval_points=20,
                n_eval_episodes=10,
            )
        )

        decay_configs = (
            build_linear_configs(
                env_ids=(
                    env_id,
                ),
                seeds=TUNING_SEEDS,
                explorer_specs=(
                    decay_specs
                ),
                step_budget=STEP_BUDGET,
                alpha_bars=(
                    alpha_bar,
                ),
                algo="sarsa-lambda",
                gamma=1.0,
                lam=LAMBDA_FIXED,
                q_init=0.0,
                reward_scale=1.0,
                n_bins=100,
                n_eval_points=20,
                n_eval_episodes=10,
            )
        )

        boltzmann_configs = (
            build_linear_configs(
                env_ids=(
                    env_id,
                ),
                seeds=TUNING_SEEDS,
                explorer_specs=(
                    boltzmann_specs
                ),
                step_budget=STEP_BUDGET,
                alpha_bars=(
                    alpha_bar,
                ),
                algo="sarsa-lambda",
                gamma=1.0,
                lam=LAMBDA_FIXED,
                q_init=0.0,
                reward_scale=1.0,
                n_bins=100,
                n_eval_points=20,
                n_eval_episodes=10,
            )
        )

        vdbe_configs = (
            build_linear_configs(
                env_ids=(
                    env_id,
                ),
                seeds=TUNING_SEEDS,
                explorer_specs=(
                    vdbe_specs()
                ),
                step_budget=STEP_BUDGET,
                alpha_bars=(
                    alpha_bar,
                ),
                algo="sarsa-lambda",
                gamma=1.0,
                lam=LAMBDA_FIXED,
                q_init=0.0,
                reward_scale=1.0,
                n_bins=100,
                n_eval_points=20,
                n_eval_episodes=10,
            )
        )

        counts[
            "fixed"
        ] += len(
            fixed_configs
        )

        counts[
            "decay"
        ] += len(
            decay_configs
        )

        counts[
            "boltzmann"
        ] += len(
            boltzmann_configs
        )

        counts[
            "vdbe"
        ] += len(
            vdbe_configs
        )

        configs.extend(
            fixed_configs
        )

        configs.extend(
            decay_configs
        )

        configs.extend(
            boltzmann_configs
        )

        configs.extend(
            vdbe_configs
        )

    return (
        configs,
        counts,
    )


def main():
    selected_alpha = (
        _load_selected_alpha()
    )

    configs, counts = (
        _build_configs(
            selected_alpha
        )
    )

    expected_fixed = (
        len(ENV_IDS)
        * 3
        * len(TUNING_SEEDS)
    )

    expected_decay = (
        len(ENV_IDS)
        * 3
        * len(TUNING_SEEDS)
    )

    expected_boltzmann = (
        len(ENV_IDS)
        * 3
        * len(TUNING_SEEDS)
    )

    expected_vdbe = (
        len(ENV_IDS)
        * 8
        * len(TUNING_SEEDS)
    )

    expected_counts = {
        "fixed": (
            expected_fixed
        ),
        "decay": (
            expected_decay
        ),
        "boltzmann": (
            expected_boltzmann
        ),
        "vdbe": (
            expected_vdbe
        ),
    }

    if counts != expected_counts:
        raise RuntimeError(
            "Unexpected configuration "
            "counts. "
            f"Expected "
            f"{expected_counts}, "
            f"found {counts}."
        )

    expected_total = sum(
        expected_counts.values()
    )

    if (
        len(configs)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected total number "
            "of baseline configs: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(configs)}."
        )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(configs),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "LINEAR BASELINE EXPLORATION TUNING"
    )

    print()

    print(
        "Selected alpha values:"
    )

    for env_id in ENV_IDS:
        print(
            f"  {env_id}: "
            f"{selected_alpha[env_id]:g}"
        )

    print()

    print(
        f"Fixed-epsilon runs: "
        f"{counts['fixed']}"
    )

    print(
        f"Decay runs: "
        f"{counts['decay']}"
    )

    print(
        f"Boltzmann runs: "
        f"{counts['boltzmann']}"
    )

    print(
        f"VDBE runs: "
        f"{counts['vdbe']}"
    )

    print()

    print(
        f"Total runs: "
        f"{len(configs)}"
    )

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=OUT_PATH,
        label=(
            "linear baseline tuning"
        ),
        workers=workers,
        save_every=5,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    successful = [
        result
        for result in results
        if "error" not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED CONFIGURATIONS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            config = result.get(
                "config",
                {}
            )

            print(
                f"{i}. "
                f"env="
                f"{config.get('env_id')} "
                f"explorer="
                f"{config.get('explorer')} "
                f"kwargs="
                f"{config.get('explorer_kwargs')} "
                f"seed="
                f"{config.get('seed')}"
            )

            print(
                f"   "
                f"{result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} linear "
            "baseline tuning runs "
            "failed."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Successful result count "
            f"is {len(successful)}, "
            f"expected "
            f"{expected_total}."
        )

    print()

    print(
        "LINEAR BASELINE TUNING COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/tune_linear_baselines.py


In [23]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.tune_linear_baselines",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Linear baseline tuning failed."
    )

LINEAR BASELINE EXPLORATION TUNING

Selected alpha values:
  MountainCar-v0: 0.75
  CartPole-v1: 1
  Acrobot-v1: 0.5
  LunarLander-v3: 0.1

Fixed-epsilon runs: 36
Decay runs: 36
Boltzmann runs: 36
VDBE runs: 96

Total runs: 204
Detected CPUs: 20
Workers: 8
Output: src\results\tuning\linear_baselines.pkl

[linear baseline tuning] loaded=0 pending=204 workers=8
[linear baseline tuning] 1/204 OK
[linear baseline tuning] 2/204 OK
[linear baseline tuning] 3/204 OK
[linear baseline tuning] 4/204 OK
[linear baseline tuning] 5/204 OK
[linear baseline tuning] 6/204 OK
[linear baseline tuning] 7/204 OK
[linear baseline tuning] 8/204 OK
[linear baseline tuning] 9/204 OK
[linear baseline tuning] 10/204 OK
[linear baseline tuning] 11/204 OK
[linear baseline tuning] 12/204 OK
[linear baseline tuning] 13/204 OK
[linear baseline tuning] 14/204 OK
[linear baseline tuning] 15/204 OK
[linear baseline tuning] 16/204 OK
[linear baseline tuning] 17/204 OK
[linear baseline tuning] 18/204 OK
[linear baseline 

In [24]:
%%writefile src/scripts/select_linear_baselines.py
import json
import pickle

from pathlib import Path

import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    STEP_BUDGET,
    FIXED_EPS_GRID,
    DECAY_HORIZON_FRACTIONS,
    BOLTZMANN_HORIZON_FRACTIONS,
    VDBE_SIGMA_GRID,
)


RESULTS_PATH = Path(
    "src/results/tuning/linear_baselines.pkl"
)

SUMMARY_PATH = Path(
    "src/results/tuning/"
    "linear_baseline_selection_summary.json"
)

SELECTED_PATH = Path(
    "src/results/tuning/"
    "selected_linear_baselines.json"
)

METHODS = (
    "fixed",
    "decay",
    "boltzmann",
    "vdbe",
)


def _write_json_atomic(
    obj,
    path
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(path)


def _candidate_value(
    env_id,
    explorer,
    explorer_kwargs
):
    if explorer == "fixed":
        return float(
            explorer_kwargs[
                "epsilon"
            ]
        )

    if explorer == "decay":
        decay_steps = float(
            explorer_kwargs[
                "decay_steps"
            ]
        )

        budget = float(
            STEP_BUDGET[
                env_id
            ]
        )

        return (
            decay_steps
            / budget
        )

    if explorer == "boltzmann":
        decay_steps = float(
            explorer_kwargs[
                "decay_steps"
            ]
        )

        budget = float(
            STEP_BUDGET[
                env_id
            ]
        )

        return (
            decay_steps
            / budget
        )

    if explorer == "vdbe":
        return float(
            explorer_kwargs[
                "sigma"
            ]
        )

    raise ValueError(
        f"Unsupported explorer: "
        f"{explorer}"
    )


def _expected_grid(
    explorer
):
    if explorer == "fixed":
        return tuple(
            float(value)
            for value
            in FIXED_EPS_GRID
        )

    if explorer == "decay":
        return tuple(
            float(value)
            for value
            in DECAY_HORIZON_FRACTIONS
        )

    if explorer == "boltzmann":
        return tuple(
            float(value)
            for value
            in BOLTZMANN_HORIZON_FRACTIONS
        )

    if explorer == "vdbe":
        return tuple(
            float(value)
            for value
            in VDBE_SIGMA_GRID
        )

    raise ValueError(
        f"Unsupported explorer: "
        f"{explorer}"
    )


def _candidate_label(
    explorer,
    value
):
    if explorer == "fixed":
        return (
            f"epsilon={value:g}"
        )

    if explorer == "decay":
        return (
            "horizon="
            f"{value:g}*budget"
        )

    if explorer == "boltzmann":
        return (
            "horizon="
            f"{value:g}*budget"
        )

    if explorer == "vdbe":
        return (
            f"sigma={value:g}"
        )

    return str(value)


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            "Baseline tuning results "
            "not found: "
            f"{RESULTS_PATH}"
        )

    if SELECTED_PATH.exists():
        SELECTED_PATH.unlink()

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(f)

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Baseline results must "
            "be stored as a list."
        )

    expected_total = (
        len(ENV_IDS)
        * len(TUNING_SEEDS)
        * (
            len(FIXED_EPS_GRID)
            + len(
                DECAY_HORIZON_FRACTIONS
            )
            + len(
                BOLTZMANN_HORIZON_FRACTIONS
            )
            + len(
                VDBE_SIGMA_GRID
            )
        )
    )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected result count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    if errors:
        raise RuntimeError(
            "Baseline tuning results "
            f"contain {len(errors)} "
            "error records."
        )

    grouped = {}

    for result in results:
        config = result.get(
            "config",
            {}
        )

        env_id = config.get(
            "env_id",
            result.get("env_id"),
        )

        explorer = config.get(
            "explorer",
            result.get("explorer"),
        )

        explorer_kwargs = config.get(
            "explorer_kwargs",
            result.get(
                "explorer_kwargs"
            ),
        )

        seed = config.get(
            "seed",
            result.get("seed"),
        )

        alpha_bar = config.get(
            "alpha_bar",
            result.get("alpha_bar"),
        )

        if env_id not in ENV_IDS:
            raise ValueError(
                "Unexpected environment: "
                f"{env_id}"
            )

        if explorer not in METHODS:
            raise ValueError(
                "Unexpected explorer: "
                f"{explorer}"
            )

        if not isinstance(
            explorer_kwargs,
            dict,
        ):
            raise TypeError(
                "explorer_kwargs must "
                "be a dictionary."
            )

        if seed is None:
            raise KeyError(
                "Result missing seed."
            )

        if alpha_bar is None:
            raise KeyError(
                "Result missing "
                "alpha_bar."
            )

        if "final_eval" not in result:
            raise KeyError(
                "Result missing "
                "final_eval."
            )

        seed = int(seed)

        alpha_bar = float(
            alpha_bar
        )

        final_eval = float(
            result[
                "final_eval"
            ]
        )

        if not np.isfinite(
            final_eval
        ):
            raise ValueError(
                "Non-finite final "
                "evaluation for "
                f"env={env_id}, "
                f"explorer={explorer}, "
                f"seed={seed}."
            )

        value = _candidate_value(
            env_id,
            explorer,
            explorer_kwargs,
        )

        key = (
            env_id,
            explorer,
            float(value),
        )

        grouped.setdefault(
            key,
            {}
        )

        if seed in grouped[key]:
            raise RuntimeError(
                "Duplicate tuning result "
                "for "
                f"env={env_id}, "
                f"explorer={explorer}, "
                f"value={value}, "
                f"seed={seed}."
            )

        grouped[key][seed] = {
            "final_eval": (
                final_eval
            ),
            "alpha_bar": (
                alpha_bar
            ),
            "explorer_kwargs": (
                dict(
                    explorer_kwargs
                )
            ),
        }

    expected_seeds = {
        int(seed)
        for seed in TUNING_SEEDS
    }

    summary = {}
    selected = {}

    edge_winners = []

    print(
        "LINEAR BASELINE "
        "TUNING RESULTS"
    )

    print()

    for env_id in ENV_IDS:
        summary[
            env_id
        ] = {}

        selected[
            env_id
        ] = {}

        print(
            "=" * 70
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 70
        )

        print()

        for explorer in METHODS:
            grid = (
                _expected_grid(
                    explorer
                )
            )

            if tuple(
                sorted(grid)
            ) != grid:
                raise RuntimeError(
                    f"Grid for "
                    f"{explorer} "
                    "must be sorted."
                )

            rows = []

            print(
                f"Method: {explorer}"
            )

            print(
                "-" * 50
            )

            for value in grid:
                value = float(
                    value
                )

                key = (
                    env_id,
                    explorer,
                    value,
                )

                if key not in grouped:
                    raise RuntimeError(
                        "Missing tuning "
                        "group for "
                        f"env={env_id}, "
                        f"explorer="
                        f"{explorer}, "
                        f"value={value}."
                    )

                seed_records = (
                    grouped[key]
                )

                actual_seeds = set(
                    seed_records
                )

                if (
                    actual_seeds
                    != expected_seeds
                ):
                    raise RuntimeError(
                        "Seed mismatch for "
                        f"env={env_id}, "
                        f"explorer="
                        f"{explorer}, "
                        f"value={value}. "
                        f"Expected "
                        f"{sorted(expected_seeds)}, "
                        f"found "
                        f"{sorted(actual_seeds)}."
                    )

                scores = [
                    float(
                        seed_records[
                            int(seed)
                        ][
                            "final_eval"
                        ]
                    )
                    for seed
                    in TUNING_SEEDS
                ]

                alpha_values = {
                    float(
                        seed_records[
                            int(seed)
                        ][
                            "alpha_bar"
                        ]
                    )
                    for seed
                    in TUNING_SEEDS
                }

                if len(
                    alpha_values
                ) != 1:
                    raise RuntimeError(
                        "Alpha mismatch "
                        "between seeds for "
                        f"env={env_id}, "
                        f"explorer="
                        f"{explorer}, "
                        f"value={value}."
                    )

                mean_score = float(
                    np.mean(
                        scores
                    )
                )

                std_score = float(
                    np.std(
                        scores,
                        ddof=1,
                    )
                )

                representative = (
                    seed_records[
                        int(
                            TUNING_SEEDS[
                                0
                            ]
                        )
                    ]
                )

                row = {
                    "candidate_value": (
                        value
                    ),
                    "candidate_label": (
                        _candidate_label(
                            explorer,
                            value,
                        )
                    ),
                    "seed_scores": (
                        scores
                    ),
                    "mean_final_eval": (
                        mean_score
                    ),
                    "std_final_eval": (
                        std_score
                    ),
                    "alpha_bar": (
                        float(
                            next(
                                iter(
                                    alpha_values
                                )
                            )
                        )
                    ),
                    "explorer_kwargs": (
                        representative[
                            "explorer_kwargs"
                        ]
                    ),
                }

                rows.append(
                    row
                )

                score_text = (
                    "  ".join(
                        f"{int(seed)}="
                        f"{score:.3f}"
                        for seed, score
                        in zip(
                            TUNING_SEEDS,
                            scores,
                        )
                    )
                )

                print(
                    f"{_candidate_label(explorer, value):<24} "
                    f"{score_text}  "
                    f"mean="
                    f"{mean_score:.3f}  "
                    f"std="
                    f"{std_score:.3f}"
                )

            means = np.asarray(
                [
                    row[
                        "mean_final_eval"
                    ]
                    for row
                    in rows
                ],
                dtype=np.float64,
            )

            best_mean = float(
                np.max(
                    means
                )
            )

            winner_indices = (
                np.flatnonzero(
                    np.isclose(
                        means,
                        best_mean,
                        rtol=1e-12,
                        atol=1e-12,
                    )
                )
            )

            if (
                len(
                    winner_indices
                )
                != 1
            ):
                tied = [
                    rows[i][
                        "candidate_label"
                    ]
                    for i
                    in winner_indices
                ]

                raise RuntimeError(
                    "Tuning produced a "
                    f"tie for "
                    f"{env_id}, "
                    f"{explorer}: "
                    f"{tied}"
                )

            winner_index = int(
                winner_indices[
                    0
                ]
            )

            winner = rows[
                winner_index
            ]

            is_lower_edge = (
                winner_index == 0
            )

            is_upper_edge = (
                winner_index
                == len(rows) - 1
            )

            is_edge = (
                is_lower_edge
                or is_upper_edge
            )

            summary[
                env_id
            ][
                explorer
            ] = {
                "candidates": rows,
                "selected": (
                    winner
                ),
                "winner_at_edge": (
                    is_edge
                ),
                "winner_at_lower_edge": (
                    is_lower_edge
                ),
                "winner_at_upper_edge": (
                    is_upper_edge
                ),
            }

            selected[
                env_id
            ][
                explorer
            ] = {
                "explorer": (
                    explorer
                ),
                "explorer_kwargs": (
                    winner[
                        "explorer_kwargs"
                    ]
                ),
                "alpha_bar": (
                    winner[
                        "alpha_bar"
                    ]
                ),
                "candidate_value": (
                    winner[
                        "candidate_value"
                    ]
                ),
                "mean_final_eval": (
                    winner[
                        "mean_final_eval"
                    ]
                ),
            }

            print()

            print(
                "WINNER: "
                f"{winner['candidate_label']}"
            )

            print(
                "Mean final greedy "
                "return: "
                f"{winner['mean_final_eval']:.3f}"
            )

            print(
                "Standard deviation: "
                f"{winner['std_final_eval']:.3f}"
            )

            if is_edge:
                direction = (
                    "LOWER"
                    if is_lower_edge
                    else "UPPER"
                )

                edge_winners.append(
                    {
                        "env_id": (
                            env_id
                        ),
                        "explorer": (
                            explorer
                        ),
                        "value": (
                            winner[
                                "candidate_value"
                            ]
                        ),
                        "label": (
                            winner[
                                "candidate_label"
                            ]
                        ),
                        "direction": (
                            direction
                        ),
                    }
                )

                print(
                    "WARNING: winner "
                    f"is at the "
                    f"{direction.lower()} "
                    "edge of the grid."
                )

            else:
                left = rows[
                    winner_index - 1
                ][
                    "candidate_label"
                ]

                right = rows[
                    winner_index + 1
                ][
                    "candidate_label"
                ]

                print(
                    "Winner is bracketed "
                    f"by {left} and "
                    f"{right}."
                )

            print()
            print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    print(
        "=" * 70
    )

    print(
        "SUMMARY WRITTEN TO"
    )

    print(
        SUMMARY_PATH
    )

    print(
        "=" * 70
    )

    print()

    if edge_winners:
        print(
            "EDGE-WINNER CHECK FAILED"
        )

        print()

        print(
            "The following baseline "
            "tuning winners lie at "
            "a grid boundary:"
        )

        print()

        for item in edge_winners:
            print(
                f"  {item['env_id']} | "
                f"{item['explorer']} | "
                f"{item['label']} | "
                f"{item['direction']} edge"
            )

        print()

        print(
            "Do not freeze all "
            "baseline configurations yet."
        )

        print(
            "Only the affected "
            "method/environment grids "
            "need extension."
        )

        raise RuntimeError(
            "One or more baseline "
            "winners are at a "
            "grid boundary."
        )

    _write_json_atomic(
        selected,
        SELECTED_PATH,
    )

    print(
        "SELECTED LINEAR BASELINES"
    )

    print()

    for env_id in ENV_IDS:
        print(
            f"{env_id}"
        )

        for explorer in METHODS:
            item = (
                selected[
                    env_id
                ][
                    explorer
                ]
            )

            print(
                f"  {explorer:<10} "
                f"{_candidate_label(explorer, item['candidate_value'])}"
            )

        print()

    print(
        "Selected configurations "
        "written to:"
    )

    print(
        SELECTED_PATH
    )

    print()

    print(
        "LINEAR BASELINE "
        "SELECTION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/select_linear_baselines.py


In [25]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.select_linear_baselines",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 1

STDOUT:
LINEAR BASELINE TUNING RESULTS

Environment: MountainCar-v0

Method: fixed
--------------------------------------------------
epsilon=0.05             1000=-143.600  1001=-114.000  1002=-180.200  mean=-145.933  std=33.162
epsilon=0.1              1000=-155.100  1001=-134.600  1002=-141.700  mean=-143.800  std=10.410
epsilon=0.2              1000=-141.100  1001=-151.900  1002=-180.400  mean=-157.800  std=20.303

WINNER: epsilon=0.1
Mean final greedy return: -143.800
Standard deviation: 10.410
Winner is bracketed by epsilon=0.05 and epsilon=0.2.


Method: decay
--------------------------------------------------
horizon=0.2*budget       1000=-98.100  1001=-105.500  1002=-113.100  mean=-105.567  std=7.500
horizon=0.4*budget       1000=-110.900  1001=-107.400  1002=-105.700  mean=-108.000  std=2.651
horizon=0.6*budget       1000=-99.800  1001=-107.100  1002=-121.300  mean=-109.400  std=10.933

WINNER: horizon=0.2*budget
Mean final greedy return: -105.567
Standard dev

In [26]:
%%writefile src/scripts/extend_linear_baselines.py
import json
import os

from pathlib import Path

from src.sweep import run_stage
from src.sweep_configs import (
    TUNING_SEEDS,
    STEP_BUDGET,
    LAMBDA_FIXED,
    DECAY_EPS_START,
    DECAY_EPS_END,
    DECAY_MODE,
    BOLTZMANN_TAU_START,
    BOLTZMANN_TAU_END,
    build_linear_configs,
)


MAX_WORKERS = 8

SELECTED_ALPHA_PATH = Path(
    "src/results/tuning/selected_alpha.json"
)

OUT_PATH = Path(
    "src/results/tuning/linear_baselines.pkl"
)


def _load_selected_alpha():
    if not SELECTED_ALPHA_PATH.exists():
        raise FileNotFoundError(
            f"Selected-alpha file not found: "
            f"{SELECTED_ALPHA_PATH}"
        )

    with open(
        SELECTED_ALPHA_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        selected = json.load(f)

    return {
        env_id: float(alpha_bar)
        for env_id, alpha_bar
        in selected.items()
    }


def _fixed_specs(values):
    return [
        (
            "fixed",
            {
                "epsilon": float(
                    epsilon
                ),
            },
        )
        for epsilon in values
    ]


def _decay_specs(
    n_steps,
    fractions
):
    return [
        (
            "decay",
            {
                "eps_start": (
                    DECAY_EPS_START
                ),
                "eps_end": (
                    DECAY_EPS_END
                ),
                "decay_steps": int(
                    round(
                        float(fraction)
                        * int(n_steps)
                    )
                ),
                "mode": (
                    DECAY_MODE
                ),
            },
        )
        for fraction in fractions
    ]


def _boltzmann_specs(
    n_steps,
    fractions
):
    return [
        (
            "boltzmann",
            {
                "tau_start": (
                    BOLTZMANN_TAU_START
                ),
                "tau_end": (
                    BOLTZMANN_TAU_END
                ),
                "decay_steps": int(
                    round(
                        float(fraction)
                        * int(n_steps)
                    )
                ),
            },
        )
        for fraction in fractions
    ]


def _vdbe_specs(values):
    return [
        (
            "vdbe",
            {
                "sigma": float(
                    sigma
                ),
                "eps_init": 1.0,
                "eps_min": 0.0,
            },
        )
        for sigma in values
    ]


def _build_one(
    env_id,
    alpha_bar,
    explorer_specs
):
    return build_linear_configs(
        env_ids=(
            env_id,
        ),
        seeds=TUNING_SEEDS,
        explorer_specs=(
            explorer_specs
        ),
        step_budget=STEP_BUDGET,
        alpha_bars=(
            alpha_bar,
        ),
        algo="sarsa-lambda",
        gamma=1.0,
        lam=LAMBDA_FIXED,
        q_init=0.0,
        reward_scale=1.0,
        n_bins=100,
        n_eval_points=20,
        n_eval_episodes=10,
    )


def main():
    if not OUT_PATH.exists():
        raise FileNotFoundError(
            "Existing baseline result "
            f"file not found: {OUT_PATH}"
        )

    selected_alpha = (
        _load_selected_alpha()
    )

    configs = []

    configs.extend(
        _build_one(
            "MountainCar-v0",
            selected_alpha[
                "MountainCar-v0"
            ],
            lambda n_steps: (
                _decay_specs(
                    n_steps,
                    (
                        0.05,
                        0.1,
                    ),
                )
            ),
        )
    )

    configs.extend(
        _build_one(
            "MountainCar-v0",
            selected_alpha[
                "MountainCar-v0"
            ],
            lambda n_steps: (
                _boltzmann_specs(
                    n_steps,
                    (
                        0.8,
                        1.0,
                    ),
                )
            ),
        )
    )

    configs.extend(
        _build_one(
            "CartPole-v1",
            selected_alpha[
                "CartPole-v1"
            ],
            _fixed_specs(
                (
                    0.3,
                    0.4,
                )
            ),
        )
    )

    configs.extend(
        _build_one(
            "CartPole-v1",
            selected_alpha[
                "CartPole-v1"
            ],
            lambda n_steps: (
                _boltzmann_specs(
                    n_steps,
                    (
                        0.05,
                        0.1,
                    ),
                )
            ),
        )
    )

    configs.extend(
        _build_one(
            "Acrobot-v1",
            selected_alpha[
                "Acrobot-v1"
            ],
            _fixed_specs(
                (
                    0.01,
                    0.025,
                )
            ),
        )
    )

    configs.extend(
        _build_one(
            "Acrobot-v1",
            selected_alpha[
                "Acrobot-v1"
            ],
            lambda n_steps: (
                _decay_specs(
                    n_steps,
                    (
                        0.8,
                        1.0,
                    ),
                )
            ),
        )
    )

    configs.extend(
        _build_one(
            "Acrobot-v1",
            selected_alpha[
                "Acrobot-v1"
            ],
            _vdbe_specs(
                (
                    50_000.0,
                    100_000.0,
                )
            ),
        )
    )

    configs.extend(
        _build_one(
            "LunarLander-v3",
            selected_alpha[
                "LunarLander-v3"
            ],
            _fixed_specs(
                (
                    0.01,
                    0.025,
                )
            ),
        )
    )

    expected_new = (
        8
        * 2
        * len(TUNING_SEEDS)
    )

    if len(configs) != expected_new:
        raise RuntimeError(
            "Unexpected number of "
            "extension configs: "
            f"expected {expected_new}, "
            f"found {len(configs)}."
        )

    expected_total = (
        204
        + expected_new
    )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(configs),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "LINEAR BASELINE GRID EXTENSION"
    )

    print()

    print(
        "Extensions:"
    )

    print(
        "  MountainCar decay: "
        "0.05, 0.1"
    )

    print(
        "  MountainCar Boltzmann: "
        "0.8, 1.0"
    )

    print(
        "  CartPole fixed epsilon: "
        "0.3, 0.4"
    )

    print(
        "  CartPole Boltzmann: "
        "0.05, 0.1"
    )

    print(
        "  Acrobot fixed epsilon: "
        "0.01, 0.025"
    )

    print(
        "  Acrobot decay: "
        "0.8, 1.0"
    )

    print(
        "  Acrobot VDBE sigma: "
        "50000, 100000"
    )

    print(
        "  LunarLander fixed epsilon: "
        "0.01, 0.025"
    )

    print()

    print(
        f"New runs: "
        f"{len(configs)}"
    )

    print(
        f"Expected total results: "
        f"{expected_total}"
    )

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=OUT_PATH,
        label=(
            "linear baseline extension"
        ),
        workers=workers,
        save_every=5,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    successful = [
        result
        for result in results
        if "error" not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED CONFIGURATIONS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            config = result.get(
                "config",
                {}
            )

            print(
                f"{i}. "
                f"env="
                f"{config.get('env_id')} "
                f"explorer="
                f"{config.get('explorer')} "
                f"kwargs="
                f"{config.get('explorer_kwargs')} "
                f"seed="
                f"{config.get('seed')}"
            )

            print(
                f"   "
                f"{result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} extension "
            "runs failed."
        )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected total result "
            f"count: expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected successful "
            f"result count: expected "
            f"{expected_total}, "
            f"found "
            f"{len(successful)}."
        )

    print()

    print(
        "LINEAR BASELINE GRID "
        "EXTENSION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/extend_linear_baselines.py


In [27]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.extend_linear_baselines",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Linear baseline "
        "grid extension failed."
    )

LINEAR BASELINE GRID EXTENSION

Extensions:
  MountainCar decay: 0.05, 0.1
  MountainCar Boltzmann: 0.8, 1.0
  CartPole fixed epsilon: 0.3, 0.4
  CartPole Boltzmann: 0.05, 0.1
  Acrobot fixed epsilon: 0.01, 0.025
  Acrobot decay: 0.8, 1.0
  Acrobot VDBE sigma: 50000, 100000
  LunarLander fixed epsilon: 0.01, 0.025

New runs: 48
Expected total results: 252
Detected CPUs: 20
Workers: 8
Output: src\results\tuning\linear_baselines.pkl

[linear baseline extension] loaded=204 pending=48 workers=8
[linear baseline extension] 1/48 OK
[linear baseline extension] 2/48 OK
[linear baseline extension] 3/48 OK
[linear baseline extension] 4/48 OK
[linear baseline extension] 5/48 OK
[linear baseline extension] 6/48 OK
[linear baseline extension] 7/48 OK
[linear baseline extension] 8/48 OK
[linear baseline extension] 9/48 OK
[linear baseline extension] 10/48 OK
[linear baseline extension] 11/48 OK
[linear baseline extension] 12/48 OK
[linear baseline extension] 13/48 OK
[linear baseline extension] 14/4

In [28]:
%%writefile src/scripts/select_linear_baselines.py
import json
import pickle

from pathlib import Path

import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    STEP_BUDGET,
    FIXED_EPS_GRID_BY_ENV,
    DECAY_HORIZON_GRID_BY_ENV,
    BOLTZMANN_HORIZON_GRID_BY_ENV,
    VDBE_SIGMA_GRID_BY_ENV,
)


RESULTS_PATH = Path(
    "src/results/tuning/linear_baselines.pkl"
)

SUMMARY_PATH = Path(
    "src/results/tuning/"
    "linear_baseline_selection_summary.json"
)

SELECTED_PATH = Path(
    "src/results/tuning/"
    "selected_linear_baselines.json"
)

METHODS = (
    "fixed",
    "decay",
    "boltzmann",
    "vdbe",
)


def _write_json_atomic(
    obj,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(path)


def _expected_grid(
    env_id,
    explorer,
):
    if explorer == "fixed":
        source = (
            FIXED_EPS_GRID_BY_ENV
        )

    elif explorer == "decay":
        source = (
            DECAY_HORIZON_GRID_BY_ENV
        )

    elif explorer == "boltzmann":
        source = (
            BOLTZMANN_HORIZON_GRID_BY_ENV
        )

    elif explorer == "vdbe":
        source = (
            VDBE_SIGMA_GRID_BY_ENV
        )

    else:
        raise ValueError(
            f"Unsupported explorer: "
            f"{explorer}"
        )

    if env_id not in source:
        raise KeyError(
            f"No grid defined for "
            f"{env_id}, {explorer}."
        )

    return tuple(
        float(value)
        for value
        in source[env_id]
    )


def _candidate_value(
    env_id,
    explorer,
    explorer_kwargs,
):
    if explorer == "fixed":
        return float(
            explorer_kwargs[
                "epsilon"
            ]
        )

    if explorer in (
        "decay",
        "boltzmann",
    ):
        decay_steps = float(
            explorer_kwargs[
                "decay_steps"
            ]
        )

        budget = float(
            STEP_BUDGET[
                env_id
            ]
        )

        return (
            decay_steps
            / budget
        )

    if explorer == "vdbe":
        return float(
            explorer_kwargs[
                "sigma"
            ]
        )

    raise ValueError(
        f"Unsupported explorer: "
        f"{explorer}"
    )


def _candidate_label(
    explorer,
    value,
):
    if explorer == "fixed":
        return (
            f"epsilon={value:g}"
        )

    if explorer == "decay":
        return (
            "horizon="
            f"{value:g}*budget"
        )

    if explorer == "boltzmann":
        return (
            "horizon="
            f"{value:g}*budget"
        )

    if explorer == "vdbe":
        return (
            f"sigma={value:g}"
        )

    raise ValueError(
        f"Unsupported explorer: "
        f"{explorer}"
    )


def _expected_total():
    total = 0

    for env_id in ENV_IDS:
        for explorer in METHODS:
            total += (
                len(
                    _expected_grid(
                        env_id,
                        explorer,
                    )
                )
                * len(
                    TUNING_SEEDS
                )
            )

    return total


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            "Baseline tuning results "
            "not found: "
            f"{RESULTS_PATH}"
        )

    if SELECTED_PATH.exists():
        SELECTED_PATH.unlink()

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(f)

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Baseline results must "
            "be stored as a list."
        )

    expected_total = (
        _expected_total()
    )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected result count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    if errors:
        raise RuntimeError(
            "Baseline tuning results "
            f"contain {len(errors)} "
            "error records."
        )

    grouped = {}

    for result in results:
        config = result.get(
            "config",
            {}
        )

        env_id = config.get(
            "env_id",
            result.get(
                "env_id"
            ),
        )

        explorer = config.get(
            "explorer",
            result.get(
                "explorer"
            ),
        )

        explorer_kwargs = (
            config.get(
                "explorer_kwargs",
                result.get(
                    "explorer_kwargs"
                ),
            )
        )

        seed = config.get(
            "seed",
            result.get(
                "seed"
            ),
        )

        alpha_bar = config.get(
            "alpha_bar",
            result.get(
                "alpha_bar"
            ),
        )

        if env_id not in ENV_IDS:
            raise ValueError(
                "Unexpected environment: "
                f"{env_id}"
            )

        if explorer not in METHODS:
            raise ValueError(
                "Unexpected explorer: "
                f"{explorer}"
            )

        if not isinstance(
            explorer_kwargs,
            dict,
        ):
            raise TypeError(
                "explorer_kwargs must "
                "be a dictionary."
            )

        if seed is None:
            raise KeyError(
                "Result missing seed."
            )

        if alpha_bar is None:
            raise KeyError(
                "Result missing "
                "alpha_bar."
            )

        if "final_eval" not in result:
            raise KeyError(
                "Result missing "
                "final_eval."
            )

        seed = int(
            seed
        )

        alpha_bar = float(
            alpha_bar
        )

        final_eval = float(
            result[
                "final_eval"
            ]
        )

        if not np.isfinite(
            final_eval
        ):
            raise ValueError(
                "Non-finite final "
                "evaluation for "
                f"env={env_id}, "
                f"explorer="
                f"{explorer}, "
                f"seed={seed}."
            )

        value = float(
            _candidate_value(
                env_id,
                explorer,
                explorer_kwargs,
            )
        )

        valid_values = (
            _expected_grid(
                env_id,
                explorer,
            )
        )

        if not any(
            np.isclose(
                value,
                valid,
                rtol=1e-12,
                atol=1e-12,
            )
            for valid in valid_values
        ):
            raise ValueError(
                "Unexpected candidate "
                f"value for "
                f"{env_id}, "
                f"{explorer}: "
                f"{value}"
            )

        canonical_value = next(
            valid
            for valid
            in valid_values
            if np.isclose(
                value,
                valid,
                rtol=1e-12,
                atol=1e-12,
            )
        )

        key = (
            env_id,
            explorer,
            canonical_value,
        )

        grouped.setdefault(
            key,
            {}
        )

        if seed in grouped[key]:
            raise RuntimeError(
                "Duplicate tuning "
                "result for "
                f"env={env_id}, "
                f"explorer="
                f"{explorer}, "
                f"value="
                f"{canonical_value}, "
                f"seed={seed}."
            )

        grouped[key][seed] = {
            "final_eval": (
                final_eval
            ),
            "alpha_bar": (
                alpha_bar
            ),
            "explorer_kwargs": (
                dict(
                    explorer_kwargs
                )
            ),
        }

    expected_seeds = {
        int(seed)
        for seed
        in TUNING_SEEDS
    }

    summary = {}
    selected = {}
    edge_winners = []

    print(
        "LINEAR BASELINE "
        "TUNING RESULTS"
    )

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Expected results: "
        f"{expected_total}"
    )

    print()

    for env_id in ENV_IDS:
        summary[
            env_id
        ] = {}

        selected[
            env_id
        ] = {}

        print(
            "=" * 72
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 72
        )

        print()

        for explorer in METHODS:
            grid = (
                _expected_grid(
                    env_id,
                    explorer,
                )
            )

            if (
                tuple(
                    sorted(grid)
                )
                != grid
            ):
                raise RuntimeError(
                    "Grid must be sorted "
                    "in ascending order "
                    f"for {env_id}, "
                    f"{explorer}."
                )

            if len(grid) < 3:
                raise RuntimeError(
                    "Grid must contain "
                    "at least three "
                    f"values for "
                    f"{env_id}, "
                    f"{explorer}."
                )

            rows = []

            print(
                f"Method: "
                f"{explorer}"
            )

            print(
                "-" * 54
            )

            for value in grid:
                key = (
                    env_id,
                    explorer,
                    value,
                )

                if key not in grouped:
                    raise RuntimeError(
                        "Missing tuning "
                        "group for "
                        f"env={env_id}, "
                        f"explorer="
                        f"{explorer}, "
                        f"value={value}."
                    )

                seed_records = (
                    grouped[key]
                )

                actual_seeds = set(
                    seed_records
                )

                if (
                    actual_seeds
                    != expected_seeds
                ):
                    raise RuntimeError(
                        "Seed mismatch "
                        f"for {env_id}, "
                        f"{explorer}, "
                        f"value={value}. "
                        f"Expected "
                        f"{sorted(expected_seeds)}, "
                        f"found "
                        f"{sorted(actual_seeds)}."
                    )

                scores = [
                    float(
                        seed_records[
                            int(seed)
                        ][
                            "final_eval"
                        ]
                    )
                    for seed
                    in TUNING_SEEDS
                ]

                alpha_values = {
                    float(
                        seed_records[
                            int(seed)
                        ][
                            "alpha_bar"
                        ]
                    )
                    for seed
                    in TUNING_SEEDS
                }

                if len(
                    alpha_values
                ) != 1:
                    raise RuntimeError(
                        "Alpha mismatch "
                        "between seeds "
                        f"for {env_id}, "
                        f"{explorer}, "
                        f"value={value}."
                    )

                mean_score = float(
                    np.mean(
                        scores
                    )
                )

                std_score = float(
                    np.std(
                        scores,
                        ddof=1,
                    )
                )

                representative = (
                    seed_records[
                        int(
                            TUNING_SEEDS[
                                0
                            ]
                        )
                    ]
                )

                row = {
                    "candidate_value": (
                        float(
                            value
                        )
                    ),
                    "candidate_label": (
                        _candidate_label(
                            explorer,
                            value,
                        )
                    ),
                    "seed_scores": (
                        scores
                    ),
                    "mean_final_eval": (
                        mean_score
                    ),
                    "std_final_eval": (
                        std_score
                    ),
                    "alpha_bar": (
                        float(
                            next(
                                iter(
                                    alpha_values
                                )
                            )
                        )
                    ),
                    "explorer_kwargs": (
                        representative[
                            "explorer_kwargs"
                        ]
                    ),
                }

                rows.append(
                    row
                )

                score_text = (
                    "  ".join(
                        f"{int(seed)}="
                        f"{score:.3f}"
                        for seed, score
                        in zip(
                            TUNING_SEEDS,
                            scores,
                        )
                    )
                )

                label = (
                    _candidate_label(
                        explorer,
                        value,
                    )
                )

                print(
                    f"{label:<26} "
                    f"{score_text}  "
                    f"mean="
                    f"{mean_score:.3f}  "
                    f"std="
                    f"{std_score:.3f}"
                )

            means = np.asarray(
                [
                    row[
                        "mean_final_eval"
                    ]
                    for row
                    in rows
                ],
                dtype=np.float64,
            )

            best_mean = float(
                np.max(
                    means
                )
            )

            winner_indices = (
                np.flatnonzero(
                    np.isclose(
                        means,
                        best_mean,
                        rtol=1e-12,
                        atol=1e-12,
                    )
                )
            )

            if (
                len(
                    winner_indices
                )
                != 1
            ):
                tied = [
                    rows[i][
                        "candidate_label"
                    ]
                    for i
                    in winner_indices
                ]

                raise RuntimeError(
                    "Tuning produced "
                    f"a tie for "
                    f"{env_id}, "
                    f"{explorer}: "
                    f"{tied}"
                )

            winner_index = int(
                winner_indices[
                    0
                ]
            )

            winner = (
                rows[
                    winner_index
                ]
            )

            is_lower_edge = (
                winner_index
                == 0
            )

            is_upper_edge = (
                winner_index
                == len(rows) - 1
            )

            is_edge = (
                is_lower_edge
                or is_upper_edge
            )

            summary[
                env_id
            ][
                explorer
            ] = {
                "grid": [
                    float(value)
                    for value
                    in grid
                ],
                "candidates": (
                    rows
                ),
                "selected": (
                    winner
                ),
                "winner_index": (
                    winner_index
                ),
                "winner_at_edge": (
                    is_edge
                ),
                "winner_at_lower_edge": (
                    is_lower_edge
                ),
                "winner_at_upper_edge": (
                    is_upper_edge
                ),
            }

            selected[
                env_id
            ][
                explorer
            ] = {
                "explorer": (
                    explorer
                ),
                "explorer_kwargs": (
                    winner[
                        "explorer_kwargs"
                    ]
                ),
                "alpha_bar": (
                    winner[
                        "alpha_bar"
                    ]
                ),
                "candidate_value": (
                    winner[
                        "candidate_value"
                    ]
                ),
                "mean_final_eval": (
                    winner[
                        "mean_final_eval"
                    ]
                ),
                "std_final_eval": (
                    winner[
                        "std_final_eval"
                    ]
                ),
            }

            print()

            print(
                "WINNER: "
                f"{winner['candidate_label']}"
            )

            print(
                "Mean final greedy "
                "return: "
                f"{winner['mean_final_eval']:.3f}"
            )

            print(
                "Standard deviation: "
                f"{winner['std_final_eval']:.3f}"
            )

            if is_edge:
                direction = (
                    "LOWER"
                    if is_lower_edge
                    else "UPPER"
                )

                edge_winners.append(
                    {
                        "env_id": (
                            env_id
                        ),
                        "explorer": (
                            explorer
                        ),
                        "label": (
                            winner[
                                "candidate_label"
                            ]
                        ),
                        "value": (
                            winner[
                                "candidate_value"
                            ]
                        ),
                        "direction": (
                            direction
                        ),
                    }
                )

                print(
                    "WARNING: winner "
                    f"is at the "
                    f"{direction.lower()} "
                    "edge of the grid."
                )

            else:
                left = (
                    rows[
                        winner_index - 1
                    ][
                        "candidate_label"
                    ]
                )

                right = (
                    rows[
                        winner_index + 1
                    ][
                        "candidate_label"
                    ]
                )

                print(
                    "Winner is bracketed "
                    f"by {left} and "
                    f"{right}."
                )

            print()
            print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    print(
        "=" * 72
    )

    print(
        "SUMMARY WRITTEN TO"
    )

    print(
        SUMMARY_PATH
    )

    print(
        "=" * 72
    )

    print()

    if edge_winners:
        print(
            "EDGE-WINNER CHECK FAILED"
        )

        print()

        print(
            "The following winners "
            "still lie at a "
            "grid boundary:"
        )

        print()

        for item in edge_winners:
            print(
                f"  "
                f"{item['env_id']} | "
                f"{item['explorer']} | "
                f"{item['label']} | "
                f"{item['direction']} edge"
            )

        print()

        print(
            "Do not freeze all "
            "baseline configurations yet."
        )

        print(
            "Only these remaining "
            "method/environment pairs "
            "need another extension."
        )

        raise RuntimeError(
            "One or more baseline "
            "winners are still at "
            "a grid boundary."
        )

    _write_json_atomic(
        selected,
        SELECTED_PATH,
    )

    print(
        "SELECTED LINEAR BASELINES"
    )

    print()

    for env_id in ENV_IDS:
        print(
            env_id
        )

        for explorer in METHODS:
            item = (
                selected[
                    env_id
                ][
                    explorer
                ]
            )

            print(
                f"  {explorer:<10} "
                f"{_candidate_label(explorer, item['candidate_value'])}"
                f"   mean="
                f"{item['mean_final_eval']:.3f}"
            )

        print()

    print(
        "Selected configurations "
        "written to:"
    )

    print(
        SELECTED_PATH
    )

    print()

    print(
        "LINEAR BASELINE "
        "SELECTION COMPLETED"
    )


if __name__ == "__main__":
    main()

Overwriting src/scripts/select_linear_baselines.py


In [29]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.select_linear_baselines",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 1

STDOUT:
LINEAR BASELINE TUNING RESULTS

Stored results: 252
Expected results: 252

Environment: MountainCar-v0

Method: fixed
------------------------------------------------------
epsilon=0.05               1000=-143.600  1001=-114.000  1002=-180.200  mean=-145.933  std=33.162
epsilon=0.1                1000=-155.100  1001=-134.600  1002=-141.700  mean=-143.800  std=10.410
epsilon=0.2                1000=-141.100  1001=-151.900  1002=-180.400  mean=-157.800  std=20.303

WINNER: epsilon=0.1
Mean final greedy return: -143.800
Standard deviation: 10.410
Winner is bracketed by epsilon=0.05 and epsilon=0.2.


Method: decay
------------------------------------------------------
horizon=0.05*budget        1000=-99.000  1001=-111.000  1002=-108.800  mean=-106.267  std=6.389
horizon=0.1*budget         1000=-102.500  1001=-136.300  1002=-108.900  mean=-115.900  std=17.954
horizon=0.2*budget         1000=-98.100  1001=-105.500  1002=-113.100  mean=-105.567  std=7.500
horizon=0.4*

In [30]:
%%writefile src/scripts/extend_acrobot_fixed.py
import json
import os

from pathlib import Path

from src.sweep import run_stage
from src.sweep_configs import (
    TUNING_SEEDS,
    STEP_BUDGET,
    LAMBDA_FIXED,
    build_linear_configs,
)


MAX_WORKERS = 6

SELECTED_ALPHA_PATH = Path(
    "src/results/tuning/selected_alpha.json"
)

OUT_PATH = Path(
    "src/results/tuning/linear_baselines.pkl"
)

ENV_ID = "Acrobot-v1"

EPSILON_VALUES = (
    0.0,
    0.005,
)


def main():
    if not OUT_PATH.exists():
        raise FileNotFoundError(
            f"Existing baseline results not found: "
            f"{OUT_PATH}"
        )

    if not SELECTED_ALPHA_PATH.exists():
        raise FileNotFoundError(
            f"Selected-alpha file not found: "
            f"{SELECTED_ALPHA_PATH}"
        )

    with open(
        SELECTED_ALPHA_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        selected_alpha = json.load(f)

    alpha_bar = float(
        selected_alpha[
            ENV_ID
        ]
    )

    explorer_specs = [
        (
            "fixed",
            {
                "epsilon": float(
                    epsilon
                )
            },
        )
        for epsilon
        in EPSILON_VALUES
    ]

    configs = build_linear_configs(
        env_ids=(
            ENV_ID,
        ),
        seeds=TUNING_SEEDS,
        explorer_specs=explorer_specs,
        step_budget=STEP_BUDGET,
        alpha_bars=(
            alpha_bar,
        ),
        algo="sarsa-lambda",
        gamma=1.0,
        lam=LAMBDA_FIXED,
        q_init=0.0,
        reward_scale=1.0,
        n_bins=100,
        n_eval_points=20,
        n_eval_episodes=10,
    )

    expected_new = (
        len(EPSILON_VALUES)
        * len(TUNING_SEEDS)
    )

    if len(configs) != expected_new:
        raise RuntimeError(
            "Unexpected number of "
            "extension configurations: "
            f"expected {expected_new}, "
            f"found {len(configs)}."
        )

    expected_total = (
        252
        + expected_new
    )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(configs),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "ACROBOT FIXED-EPSILON GRID EXTENSION"
    )

    print()

    print(
        f"Environment: {ENV_ID}"
    )

    print(
        f"Selected alpha_bar: "
        f"{alpha_bar:g}"
    )

    print(
        f"New epsilon values: "
        f"{EPSILON_VALUES}"
    )

    print(
        f"Seeds: "
        f"{TUNING_SEEDS}"
    )

    print(
        f"New runs: "
        f"{len(configs)}"
    )

    print(
        f"Expected total results: "
        f"{expected_total}"
    )

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=OUT_PATH,
        label=(
            "Acrobot fixed epsilon extension"
        ),
        workers=workers,
        save_every=2,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    successful = [
        result
        for result in results
        if "error" not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()
        print(
            "FAILED CONFIGURATIONS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            config = result.get(
                "config",
                {}
            )

            print(
                f"{i}. "
                f"epsilon="
                f"{config.get('explorer_kwargs', {}).get('epsilon')} "
                f"seed="
                f"{config.get('seed')}"
            )

            print(
                f"   "
                f"{result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} Acrobot "
            "fixed-epsilon extension "
            "runs failed."
        )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected total result count: "
            f"expected {expected_total}, "
            f"found {len(results)}."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected successful result "
            f"count: expected "
            f"{expected_total}, "
            f"found "
            f"{len(successful)}."
        )

    print()

    print(
        "ACROBOT FIXED-EPSILON "
        "GRID EXTENSION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/extend_acrobot_fixed.py


In [31]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.extend_acrobot_fixed",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Acrobot fixed-epsilon "
        "extension failed."
    )

ACROBOT FIXED-EPSILON GRID EXTENSION

Environment: Acrobot-v1
Selected alpha_bar: 0.5
New epsilon values: (0.0, 0.005)
Seeds: (1000, 1001, 1002)
New runs: 6
Expected total results: 258
Detected CPUs: 20
Workers: 6
Output: src\results\tuning\linear_baselines.pkl

[Acrobot fixed epsilon extension] loaded=252 pending=6 workers=6
[Acrobot fixed epsilon extension] 1/6 OK
[Acrobot fixed epsilon extension] 2/6 OK
[Acrobot fixed epsilon extension] 3/6 OK
[Acrobot fixed epsilon extension] 4/6 OK
[Acrobot fixed epsilon extension] 5/6 OK
[Acrobot fixed epsilon extension] 6/6 OK

Stored results: 258
Successful: 258
Errors: 0

ACROBOT FIXED-EPSILON GRID EXTENSION COMPLETED

RETURN CODE: 0


In [33]:
%%writefile src/scripts/select_linear_baselines.py
import json
import pickle

from pathlib import Path

import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    STEP_BUDGET,
    FIXED_EPS_GRID_BY_ENV,
    DECAY_HORIZON_GRID_BY_ENV,
    BOLTZMANN_HORIZON_GRID_BY_ENV,
    VDBE_SIGMA_GRID_BY_ENV,
)


RESULTS_PATH = Path(
    "src/results/tuning/linear_baselines.pkl"
)

SUMMARY_PATH = Path(
    "src/results/tuning/"
    "linear_baseline_selection_summary.json"
)

SELECTED_PATH = Path(
    "src/results/tuning/"
    "selected_linear_baselines.json"
)

METHODS = (
    "fixed",
    "decay",
    "boltzmann",
    "vdbe",
)


def _write_json_atomic(
    obj,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(path)


def _expected_grid(
    env_id,
    explorer,
):
    if explorer == "fixed":
        source = (
            FIXED_EPS_GRID_BY_ENV
        )

    elif explorer == "decay":
        source = (
            DECAY_HORIZON_GRID_BY_ENV
        )

    elif explorer == "boltzmann":
        source = (
            BOLTZMANN_HORIZON_GRID_BY_ENV
        )

    elif explorer == "vdbe":
        source = (
            VDBE_SIGMA_GRID_BY_ENV
        )

    else:
        raise ValueError(
            f"Unsupported explorer: "
            f"{explorer}"
        )

    if env_id not in source:
        raise KeyError(
            f"No grid defined for "
            f"{env_id}, {explorer}."
        )

    return tuple(
        float(value)
        for value
        in source[env_id]
    )


def _candidate_value(
    env_id,
    explorer,
    explorer_kwargs,
):
    if explorer == "fixed":
        return float(
            explorer_kwargs[
                "epsilon"
            ]
        )

    if explorer in (
        "decay",
        "boltzmann",
    ):
        decay_steps = float(
            explorer_kwargs[
                "decay_steps"
            ]
        )

        budget = float(
            STEP_BUDGET[
                env_id
            ]
        )

        return (
            decay_steps
            / budget
        )

    if explorer == "vdbe":
        return float(
            explorer_kwargs[
                "sigma"
            ]
        )

    raise ValueError(
        f"Unsupported explorer: "
        f"{explorer}"
    )


def _candidate_label(
    explorer,
    value,
):
    if explorer == "fixed":
        return (
            f"epsilon={value:g}"
        )

    if explorer == "decay":
        return (
            "horizon="
            f"{value:g}*budget"
        )

    if explorer == "boltzmann":
        return (
            "horizon="
            f"{value:g}*budget"
        )

    if explorer == "vdbe":
        return (
            f"sigma={value:g}"
        )

    raise ValueError(
        f"Unsupported explorer: "
        f"{explorer}"
    )


def _expected_total():
    total = 0

    for env_id in ENV_IDS:
        for explorer in METHODS:
            total += (
                len(
                    _expected_grid(
                        env_id,
                        explorer,
                    )
                )
                * len(
                    TUNING_SEEDS
                )
            )

    return total


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            "Baseline tuning results "
            "not found: "
            f"{RESULTS_PATH}"
        )

    if SELECTED_PATH.exists():
        SELECTED_PATH.unlink()

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(f)

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Baseline results must "
            "be stored as a list."
        )

    expected_total = (
        _expected_total()
    )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected result count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    if errors:
        raise RuntimeError(
            "Baseline tuning results "
            f"contain {len(errors)} "
            "error records."
        )

    grouped = {}

    for result in results:
        config = result.get(
            "config",
            {}
        )

        env_id = config.get(
            "env_id",
            result.get(
                "env_id"
            ),
        )

        explorer = config.get(
            "explorer",
            result.get(
                "explorer"
            ),
        )

        explorer_kwargs = (
            config.get(
                "explorer_kwargs",
                result.get(
                    "explorer_kwargs"
                ),
            )
        )

        seed = config.get(
            "seed",
            result.get(
                "seed"
            ),
        )

        alpha_bar = config.get(
            "alpha_bar",
            result.get(
                "alpha_bar"
            ),
        )

        if env_id not in ENV_IDS:
            raise ValueError(
                "Unexpected environment: "
                f"{env_id}"
            )

        if explorer not in METHODS:
            raise ValueError(
                "Unexpected explorer: "
                f"{explorer}"
            )

        if not isinstance(
            explorer_kwargs,
            dict,
        ):
            raise TypeError(
                "explorer_kwargs must "
                "be a dictionary."
            )

        if seed is None:
            raise KeyError(
                "Result missing seed."
            )

        if alpha_bar is None:
            raise KeyError(
                "Result missing "
                "alpha_bar."
            )

        if "final_eval" not in result:
            raise KeyError(
                "Result missing "
                "final_eval."
            )

        seed = int(
            seed
        )

        alpha_bar = float(
            alpha_bar
        )

        final_eval = float(
            result[
                "final_eval"
            ]
        )

        if not np.isfinite(
            final_eval
        ):
            raise ValueError(
                "Non-finite final "
                "evaluation for "
                f"env={env_id}, "
                f"explorer="
                f"{explorer}, "
                f"seed={seed}."
            )

        value = float(
            _candidate_value(
                env_id,
                explorer,
                explorer_kwargs,
            )
        )

        valid_values = (
            _expected_grid(
                env_id,
                explorer,
            )
        )

        matches = [
            valid
            for valid
            in valid_values
            if np.isclose(
                value,
                valid,
                rtol=1e-12,
                atol=1e-12,
            )
        ]

        if not matches:
            raise ValueError(
                "Unexpected candidate "
                f"value for "
                f"{env_id}, "
                f"{explorer}: "
                f"{value}"
            )

        if len(matches) != 1:
            raise RuntimeError(
                "Candidate value matches "
                "multiple grid entries for "
                f"{env_id}, "
                f"{explorer}: "
                f"{value}"
            )

        canonical_value = float(
            matches[0]
        )

        key = (
            env_id,
            explorer,
            canonical_value,
        )

        grouped.setdefault(
            key,
            {}
        )

        if seed in grouped[key]:
            raise RuntimeError(
                "Duplicate tuning "
                "result for "
                f"env={env_id}, "
                f"explorer="
                f"{explorer}, "
                f"value="
                f"{canonical_value}, "
                f"seed={seed}."
            )

        grouped[key][seed] = {
            "final_eval": (
                final_eval
            ),
            "alpha_bar": (
                alpha_bar
            ),
            "explorer_kwargs": (
                dict(
                    explorer_kwargs
                )
            ),
        }

    expected_seeds = {
        int(seed)
        for seed
        in TUNING_SEEDS
    }

    summary = {}
    selected = {}
    edge_winners = []

    print(
        "LINEAR BASELINE "
        "TUNING RESULTS"
    )

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Expected results: "
        f"{expected_total}"
    )

    print()

    for env_id in ENV_IDS:
        summary[
            env_id
        ] = {}

        selected[
            env_id
        ] = {}

        print(
            "=" * 72
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 72
        )

        print()

        for explorer in METHODS:
            grid = (
                _expected_grid(
                    env_id,
                    explorer,
                )
            )

            if (
                tuple(
                    sorted(grid)
                )
                != grid
            ):
                raise RuntimeError(
                    "Grid must be sorted "
                    "in ascending order "
                    f"for {env_id}, "
                    f"{explorer}."
                )

            if len(grid) < 3:
                raise RuntimeError(
                    "Grid must contain "
                    "at least three "
                    f"values for "
                    f"{env_id}, "
                    f"{explorer}."
                )

            if len(
                set(grid)
            ) != len(grid):
                raise RuntimeError(
                    "Grid contains duplicate "
                    f"values for "
                    f"{env_id}, "
                    f"{explorer}."
                )

            rows = []

            print(
                f"Method: "
                f"{explorer}"
            )

            print(
                "-" * 54
            )

            for value in grid:
                key = (
                    env_id,
                    explorer,
                    value,
                )

                if key not in grouped:
                    raise RuntimeError(
                        "Missing tuning "
                        "group for "
                        f"env={env_id}, "
                        f"explorer="
                        f"{explorer}, "
                        f"value={value}."
                    )

                seed_records = (
                    grouped[key]
                )

                actual_seeds = set(
                    seed_records
                )

                if (
                    actual_seeds
                    != expected_seeds
                ):
                    raise RuntimeError(
                        "Seed mismatch "
                        f"for {env_id}, "
                        f"{explorer}, "
                        f"value={value}. "
                        f"Expected "
                        f"{sorted(expected_seeds)}, "
                        f"found "
                        f"{sorted(actual_seeds)}."
                    )

                scores = [
                    float(
                        seed_records[
                            int(seed)
                        ][
                            "final_eval"
                        ]
                    )
                    for seed
                    in TUNING_SEEDS
                ]

                alpha_values = {
                    float(
                        seed_records[
                            int(seed)
                        ][
                            "alpha_bar"
                        ]
                    )
                    for seed
                    in TUNING_SEEDS
                }

                if len(
                    alpha_values
                ) != 1:
                    raise RuntimeError(
                        "Alpha mismatch "
                        "between seeds "
                        f"for {env_id}, "
                        f"{explorer}, "
                        f"value={value}."
                    )

                mean_score = float(
                    np.mean(
                        scores
                    )
                )

                std_score = float(
                    np.std(
                        scores,
                        ddof=1,
                    )
                )

                min_score = float(
                    np.min(
                        scores
                    )
                )

                max_score = float(
                    np.max(
                        scores
                    )
                )

                representative = (
                    seed_records[
                        int(
                            TUNING_SEEDS[
                                0
                            ]
                        )
                    ]
                )

                row = {
                    "candidate_value": (
                        float(
                            value
                        )
                    ),
                    "candidate_label": (
                        _candidate_label(
                            explorer,
                            value,
                        )
                    ),
                    "seed_scores": (
                        scores
                    ),
                    "mean_final_eval": (
                        mean_score
                    ),
                    "std_final_eval": (
                        std_score
                    ),
                    "min_final_eval": (
                        min_score
                    ),
                    "max_final_eval": (
                        max_score
                    ),
                    "alpha_bar": (
                        float(
                            next(
                                iter(
                                    alpha_values
                                )
                            )
                        )
                    ),
                    "explorer_kwargs": (
                        representative[
                            "explorer_kwargs"
                        ]
                    ),
                }

                rows.append(
                    row
                )

                score_text = (
                    "  ".join(
                        f"{int(seed)}="
                        f"{score:.3f}"
                        for seed, score
                        in zip(
                            TUNING_SEEDS,
                            scores,
                        )
                    )
                )

                label = (
                    _candidate_label(
                        explorer,
                        value,
                    )
                )

                print(
                    f"{label:<26} "
                    f"{score_text}  "
                    f"mean="
                    f"{mean_score:.3f}  "
                    f"std="
                    f"{std_score:.3f}"
                )

            means = np.asarray(
                [
                    row[
                        "mean_final_eval"
                    ]
                    for row
                    in rows
                ],
                dtype=np.float64,
            )

            best_mean = float(
                np.max(
                    means
                )
            )

            winner_indices = (
                np.flatnonzero(
                    np.isclose(
                        means,
                        best_mean,
                        rtol=1e-12,
                        atol=1e-12,
                    )
                )
            )

            if (
                len(
                    winner_indices
                )
                != 1
            ):
                tied = [
                    rows[i][
                        "candidate_label"
                    ]
                    for i
                    in winner_indices
                ]

                raise RuntimeError(
                    "Tuning produced "
                    f"a tie for "
                    f"{env_id}, "
                    f"{explorer}: "
                    f"{tied}"
                )

            winner_index = int(
                winner_indices[
                    0
                ]
            )

            winner = (
                rows[
                    winner_index
                ]
            )

            is_lower_edge = (
                winner_index
                == 0
            )

            is_upper_edge = (
                winner_index
                == len(rows) - 1
            )

            is_natural_lower_bound = (
                explorer == "fixed"
                and is_lower_edge
                and np.isclose(
                    winner[
                        "candidate_value"
                    ],
                    0.0,
                    rtol=1e-12,
                    atol=1e-12,
                )
            )

            is_unresolved_lower_edge = (
                is_lower_edge
                and not is_natural_lower_bound
            )

            is_edge = (
                is_unresolved_lower_edge
                or is_upper_edge
            )

            summary[
                env_id
            ][
                explorer
            ] = {
                "grid": [
                    float(value)
                    for value
                    in grid
                ],
                "candidates": (
                    rows
                ),
                "selected": (
                    winner
                ),
                "winner_index": (
                    winner_index
                ),
                "winner_at_edge": (
                    is_edge
                ),
                "winner_at_lower_edge": (
                    is_lower_edge
                ),
                "winner_at_upper_edge": (
                    is_upper_edge
                ),
                "winner_at_natural_lower_bound": (
                    is_natural_lower_bound
                ),
                "winner_at_unresolved_lower_edge": (
                    is_unresolved_lower_edge
                ),
            }

            selected[
                env_id
            ][
                explorer
            ] = {
                "explorer": (
                    explorer
                ),
                "explorer_kwargs": (
                    winner[
                        "explorer_kwargs"
                    ]
                ),
                "alpha_bar": (
                    winner[
                        "alpha_bar"
                    ]
                ),
                "candidate_value": (
                    winner[
                        "candidate_value"
                    ]
                ),
                "mean_final_eval": (
                    winner[
                        "mean_final_eval"
                    ]
                ),
                "std_final_eval": (
                    winner[
                        "std_final_eval"
                    ]
                ),
            }

            print()

            print(
                "WINNER: "
                f"{winner['candidate_label']}"
            )

            print(
                "Mean final greedy "
                "return: "
                f"{winner['mean_final_eval']:.3f}"
            )

            print(
                "Standard deviation: "
                f"{winner['std_final_eval']:.3f}"
            )

            if is_edge:
                direction = (
                    "LOWER"
                    if is_unresolved_lower_edge
                    else "UPPER"
                )

                edge_winners.append(
                    {
                        "env_id": (
                            env_id
                        ),
                        "explorer": (
                            explorer
                        ),
                        "label": (
                            winner[
                                "candidate_label"
                            ]
                        ),
                        "value": (
                            winner[
                                "candidate_value"
                            ]
                        ),
                        "direction": (
                            direction
                        ),
                    }
                )

                print(
                    "WARNING: winner "
                    f"is at the "
                    f"{direction.lower()} "
                    "edge of the grid."
                )

            elif is_natural_lower_bound:
                print(
                    "Winner is at the valid "
                    "natural lower boundary "
                    "epsilon=0."
                )

                print(
                    "No lower fixed-epsilon "
                    "value exists, so this "
                    "search is complete."
                )

            else:
                left = (
                    rows[
                        winner_index - 1
                    ][
                        "candidate_label"
                    ]
                )

                right = (
                    rows[
                        winner_index + 1
                    ][
                        "candidate_label"
                    ]
                )

                print(
                    "Winner is bracketed "
                    f"by {left} and "
                    f"{right}."
                )

            print()
            print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    print(
        "=" * 72
    )

    print(
        "SUMMARY WRITTEN TO"
    )

    print(
        SUMMARY_PATH
    )

    print(
        "=" * 72
    )

    print()

    if edge_winners:
        print(
            "EDGE-WINNER CHECK FAILED"
        )

        print()

        print(
            "The following winners "
            "still lie at an "
            "extendable grid boundary:"
        )

        print()

        for item in edge_winners:
            print(
                f"  "
                f"{item['env_id']} | "
                f"{item['explorer']} | "
                f"{item['label']} | "
                f"{item['direction']} edge"
            )

        print()

        print(
            "Do not freeze all "
            "baseline configurations yet."
        )

        print(
            "Only these remaining "
            "method/environment pairs "
            "need another extension."
        )

        raise RuntimeError(
            "One or more baseline "
            "winners are still at "
            "an extendable grid boundary."
        )

    _write_json_atomic(
        selected,
        SELECTED_PATH,
    )

    print(
        "SELECTED LINEAR BASELINES"
    )

    print()

    for env_id in ENV_IDS:
        print(
            env_id
        )

        for explorer in METHODS:
            item = (
                selected[
                    env_id
                ][
                    explorer
                ]
            )

            print(
                f"  {explorer:<10} "
                f"{_candidate_label(explorer, item['candidate_value'])}"
                f"   mean="
                f"{item['mean_final_eval']:.3f}"
                f"   std="
                f"{item['std_final_eval']:.3f}"
            )

        print()

    print(
        "Selected configurations "
        "written to:"
    )

    print(
        SELECTED_PATH
    )

    print()

    print(
        "LINEAR BASELINE "
        "SELECTION COMPLETED"
    )


if __name__ == "__main__":
    main()

Overwriting src/scripts/select_linear_baselines.py


In [34]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.select_linear_baselines",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
LINEAR BASELINE TUNING RESULTS

Stored results: 258
Expected results: 258

Environment: MountainCar-v0

Method: fixed
------------------------------------------------------
epsilon=0.05               1000=-143.600  1001=-114.000  1002=-180.200  mean=-145.933  std=33.162
epsilon=0.1                1000=-155.100  1001=-134.600  1002=-141.700  mean=-143.800  std=10.410
epsilon=0.2                1000=-141.100  1001=-151.900  1002=-180.400  mean=-157.800  std=20.303

WINNER: epsilon=0.1
Mean final greedy return: -143.800
Standard deviation: 10.410
Winner is bracketed by epsilon=0.05 and epsilon=0.2.


Method: decay
------------------------------------------------------
horizon=0.05*budget        1000=-99.000  1001=-111.000  1002=-108.800  mean=-106.267  std=6.389
horizon=0.1*budget         1000=-102.500  1001=-136.300  1002=-108.900  mean=-115.900  std=17.954
horizon=0.2*budget         1000=-98.100  1001=-105.500  1002=-113.100  mean=-105.567  std=7.500
horizon=0.4*

# Global RATE Tuning

There are really two defensible designs:

| Design                     | What we do                                                                             | What we can claim                                                                                                  |
| -------------------------- | --------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------- |
| **Subset-global tuning**   | Tune RATE on 2 environments, use the same winner on all 4                               | Stronger evidence that RATE transfers to environments it was not tuned on                                           |
| **All-four global tuning** | Evaluate each $\beta,\kappa$ on all 4, combine normalized scores, choose one winner | RATE needs only **one global setting across these four environments**, but every environment influenced that choice |

Sticking to the subset gives us the stronger experimental claim.

In [35]:
%%writefile src/scripts/tune_rate.py
import json
import os

from pathlib import Path

from src.sweep import run_stage
from src.sweep_configs import (
    TUNING_SEEDS,
    STEP_BUDGET,
    LAMBDA_FIXED,
    RATE_BETA_GRID,
    RATE_KAPPA_GRID,
    RATE_TUNING_ENVS,
    rate_specs,
    build_linear_configs,
)


MAX_WORKERS = 8

SELECTED_ALPHA_PATH = Path(
    "src/results/tuning/selected_alpha.json"
)

OUT_PATH = Path(
    "src/results/tuning/rate_global.pkl"
)


def _load_selected_alpha():
    if not SELECTED_ALPHA_PATH.exists():
        raise FileNotFoundError(
            "Selected-alpha file not found: "
            f"{SELECTED_ALPHA_PATH}"
        )

    with open(
        SELECTED_ALPHA_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        selected = json.load(f)

    if not isinstance(
        selected,
        dict,
    ):
        raise TypeError(
            "selected_alpha.json must "
            "contain a dictionary."
        )

    cleaned = {}

    for env_id in RATE_TUNING_ENVS:
        if env_id not in selected:
            raise KeyError(
                "Missing selected alpha "
                f"for {env_id}."
            )

        alpha_bar = float(
            selected[
                env_id
            ]
        )

        if alpha_bar <= 0:
            raise ValueError(
                "Selected alpha must "
                f"be positive for "
                f"{env_id}: "
                f"{alpha_bar}"
            )

        cleaned[
            env_id
        ] = alpha_bar

    return cleaned


def main():
    selected_alpha = (
        _load_selected_alpha()
    )

    specs = rate_specs()

    expected_specs = (
        len(RATE_BETA_GRID)
        * len(RATE_KAPPA_GRID)
    )

    if len(specs) != expected_specs:
        raise RuntimeError(
            "Unexpected number of RATE "
            "configurations: "
            f"expected {expected_specs}, "
            f"found {len(specs)}."
        )

    configs = []

    for env_id in RATE_TUNING_ENVS:
        env_configs = (
            build_linear_configs(
                env_ids=(
                    env_id,
                ),
                seeds=TUNING_SEEDS,
                explorer_specs=specs,
                step_budget=STEP_BUDGET,
                alpha_bars=(
                    selected_alpha[
                        env_id
                    ],
                ),
                algo="sarsa-lambda",
                gamma=1.0,
                lam=LAMBDA_FIXED,
                q_init=0.0,
                reward_scale=1.0,
                n_bins=100,
                n_eval_points=20,
                n_eval_episodes=10,
            )
        )

        configs.extend(
            env_configs
        )

    expected_total = (
        len(RATE_TUNING_ENVS)
        * len(RATE_BETA_GRID)
        * len(RATE_KAPPA_GRID)
        * len(TUNING_SEEDS)
    )

    if len(configs) != expected_total:
        raise RuntimeError(
            "Unexpected total number "
            "of RATE tuning configs: "
            f"expected {expected_total}, "
            f"found {len(configs)}."
        )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(configs),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "GLOBAL RATE TUNING"
    )

    print()

    print(
        "Tuning environments:"
    )

    for env_id in RATE_TUNING_ENVS:
        print(
            f"  {env_id}"
        )

    print()

    print(
        "Selected alpha values:"
    )

    for env_id in RATE_TUNING_ENVS:
        print(
            f"  {env_id}: "
            f"{selected_alpha[env_id]:g}"
        )

    print()

    print(
        f"Beta values: "
        f"{tuple(RATE_BETA_GRID)}"
    )

    print(
        f"Kappa values: "
        f"{tuple(RATE_KAPPA_GRID)}"
    )

    print(
        f"RATE configurations: "
        f"{expected_specs}"
    )

    print(
        f"Seeds per configuration "
        f"per environment: "
        f"{len(TUNING_SEEDS)}"
    )

    print(
        f"Total runs: "
        f"{len(configs)}"
    )

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=OUT_PATH,
        label="global RATE tuning",
        workers=workers,
        save_every=4,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    successful = [
        result
        for result in results
        if "error" not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED CONFIGURATIONS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            config = result.get(
                "config",
                {}
            )

            kwargs = config.get(
                "explorer_kwargs",
                {}
            )

            print(
                f"{i}. "
                f"env="
                f"{config.get('env_id')} "
                f"beta="
                f"{kwargs.get('beta')} "
                f"kappa="
                f"{kwargs.get('kappa')} "
                f"seed="
                f"{config.get('seed')}"
            )

            print(
                f"   "
                f"{result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} RATE "
            "tuning runs failed."
        )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected stored result "
            f"count: expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected successful "
            f"result count: expected "
            f"{expected_total}, "
            f"found "
            f"{len(successful)}."
        )

    print()

    print(
        "GLOBAL RATE TUNING COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/tune_rate.py


In [36]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.tune_rate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Global RATE tuning failed."
    )

GLOBAL RATE TUNING

Tuning environments:
  MountainCar-v0
  LunarLander-v3

Selected alpha values:
  MountainCar-v0: 0.75
  LunarLander-v3: 0.1

Beta values: (0.01, 0.05)
Kappa values: (1.0, 2.0)
RATE configurations: 4
Seeds per configuration per environment: 3
Total runs: 24
Detected CPUs: 20
Workers: 8
Output: src\results\tuning\rate_global.pkl

[global RATE tuning] loaded=0 pending=24 workers=8
[global RATE tuning] 1/24 OK
[global RATE tuning] 2/24 OK
[global RATE tuning] 3/24 OK
[global RATE tuning] 4/24 OK
[global RATE tuning] 5/24 OK
[global RATE tuning] 6/24 OK
[global RATE tuning] 7/24 OK
[global RATE tuning] 8/24 OK
[global RATE tuning] 9/24 OK
[global RATE tuning] 10/24 OK
[global RATE tuning] 11/24 OK
[global RATE tuning] 12/24 OK
[global RATE tuning] 13/24 OK
[global RATE tuning] 14/24 OK
[global RATE tuning] 15/24 OK
[global RATE tuning] 16/24 OK
[global RATE tuning] 17/24 OK
[global RATE tuning] 18/24 OK
[global RATE tuning] 19/24 OK
[global RATE tuning] 20/24 OK
[global 

Now we choose one global $\beta,\kappa$ from those four candidates.

The important part is that we must not average MountainCar and LunarLander raw returns. The normalization is

$$ S_{\text{norm}} = \frac{S-\text{random}} {\text{reference}-\text{random}}, $$

with MountainCar using \((-200,-105)\) and LunarLander using \((-180,200)\). Only after this transformation are scores from the two environments comparable.

Also, unlike our baseline edge-extension procedure, we do not extend $\beta$ or $\kappa$ just because a winner uses one end of these tiny grids. The guide explicitly defines RATE's tuning grid as the four combinations of $\beta$ in \{0.01,0.05\} and $\kappa$ in\{1,2\}, with one global winner.

In [37]:
%%writefile src/scripts/select_rate.py
import json
import pickle

from pathlib import Path

import numpy as np

from src.sweep_configs import (
    TUNING_SEEDS,
    RATE_BETA_GRID,
    RATE_KAPPA_GRID,
    RATE_TUNING_ENVS,
)


RESULTS_PATH = Path(
    "src/results/tuning/rate_global.pkl"
)

SUMMARY_PATH = Path(
    "src/results/tuning/"
    "rate_global_selection_summary.json"
)

SELECTED_PATH = Path(
    "src/results/tuning/"
    "selected_rate.json"
)

NORMALISATION = {
    "MountainCar-v0": {
        "random": -200.0,
        "reference": -105.0,
    },
    "LunarLander-v3": {
        "random": -180.0,
        "reference": 200.0,
    },
}


def _write_json_atomic(
    obj,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(path)


def _normalise(
    score,
    env_id,
):
    if env_id not in NORMALISATION:
        raise KeyError(
            "No normalisation constants "
            f"defined for {env_id}."
        )

    random_score = float(
        NORMALISATION[
            env_id
        ][
            "random"
        ]
    )

    reference_score = float(
        NORMALISATION[
            env_id
        ][
            "reference"
        ]
    )

    denominator = (
        reference_score
        - random_score
    )

    if denominator <= 0:
        raise ValueError(
            "Reference score must exceed "
            "random score for "
            f"{env_id}."
        )

    return float(
        (
            float(score)
            - random_score
        )
        / denominator
    )


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            "RATE tuning results "
            "not found: "
            f"{RESULTS_PATH}"
        )

    if SELECTED_PATH.exists():
        SELECTED_PATH.unlink()

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(f)

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "RATE tuning results "
            "must be a list."
        )

    expected_total = (
        len(RATE_TUNING_ENVS)
        * len(RATE_BETA_GRID)
        * len(RATE_KAPPA_GRID)
        * len(TUNING_SEEDS)
    )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected RATE result "
            f"count: expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    if errors:
        raise RuntimeError(
            "RATE tuning results "
            f"contain {len(errors)} "
            "error records."
        )

    expected_envs = set(
        RATE_TUNING_ENVS
    )

    expected_seeds = {
        int(seed)
        for seed
        in TUNING_SEEDS
    }

    expected_betas = {
        float(beta)
        for beta
        in RATE_BETA_GRID
    }

    expected_kappas = {
        float(kappa)
        for kappa
        in RATE_KAPPA_GRID
    }

    grouped = {}

    for result in results:
        config = result.get(
            "config",
            {}
        )

        env_id = config.get(
            "env_id",
            result.get(
                "env_id"
            ),
        )

        explorer = config.get(
            "explorer",
            result.get(
                "explorer"
            ),
        )

        explorer_kwargs = config.get(
            "explorer_kwargs",
            result.get(
                "explorer_kwargs"
            ),
        )

        seed = config.get(
            "seed",
            result.get(
                "seed"
            ),
        )

        alpha_bar = config.get(
            "alpha_bar",
            result.get(
                "alpha_bar"
            ),
        )

        if env_id not in expected_envs:
            raise ValueError(
                "Unexpected RATE tuning "
                f"environment: "
                f"{env_id}"
            )

        if explorer != "rate":
            raise ValueError(
                "Unexpected explorer "
                "in RATE tuning file: "
                f"{explorer}"
            )

        if not isinstance(
            explorer_kwargs,
            dict,
        ):
            raise TypeError(
                "explorer_kwargs must "
                "be a dictionary."
            )

        if "beta" not in explorer_kwargs:
            raise KeyError(
                "RATE config missing beta."
            )

        if "kappa" not in explorer_kwargs:
            raise KeyError(
                "RATE config missing kappa."
            )

        if seed is None:
            raise KeyError(
                "RATE result missing seed."
            )

        if alpha_bar is None:
            raise KeyError(
                "RATE result missing "
                "alpha_bar."
            )

        if "final_eval" not in result:
            raise KeyError(
                "RATE result missing "
                "final_eval."
            )

        beta = float(
            explorer_kwargs[
                "beta"
            ]
        )

        kappa = float(
            explorer_kwargs[
                "kappa"
            ]
        )

        seed = int(
            seed
        )

        alpha_bar = float(
            alpha_bar
        )

        final_eval = float(
            result[
                "final_eval"
            ]
        )

        if beta not in expected_betas:
            raise ValueError(
                f"Unexpected beta: "
                f"{beta}"
            )

        if kappa not in expected_kappas:
            raise ValueError(
                f"Unexpected kappa: "
                f"{kappa}"
            )

        if seed not in expected_seeds:
            raise ValueError(
                f"Unexpected tuning seed: "
                f"{seed}"
            )

        if not np.isfinite(
            final_eval
        ):
            raise ValueError(
                "Non-finite final RATE "
                f"evaluation for "
                f"{env_id}, "
                f"beta={beta}, "
                f"kappa={kappa}, "
                f"seed={seed}."
            )

        normalised = _normalise(
            final_eval,
            env_id,
        )

        key = (
            beta,
            kappa,
        )

        grouped.setdefault(
            key,
            {}
        )

        grouped[
            key
        ].setdefault(
            env_id,
            {}
        )

        if (
            seed
            in grouped[
                key
            ][
                env_id
            ]
        ):
            raise RuntimeError(
                "Duplicate RATE result "
                f"for beta={beta}, "
                f"kappa={kappa}, "
                f"env={env_id}, "
                f"seed={seed}."
            )

        grouped[
            key
        ][
            env_id
        ][
            seed
        ] = {
            "raw_score": (
                final_eval
            ),
            "normalised_score": (
                normalised
            ),
            "alpha_bar": (
                alpha_bar
            ),
            "explorer_kwargs": (
                dict(
                    explorer_kwargs
                )
            ),
        }

    candidates = []

    print(
        "GLOBAL RATE SELECTION RESULTS"
    )

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Expected results: "
        f"{expected_total}"
    )

    print()

    print(
        "Selection uses normalised "
        "scores across:"
    )

    for env_id in RATE_TUNING_ENVS:
        constants = (
            NORMALISATION[
                env_id
            ]
        )

        print(
            f"  {env_id}: "
            f"random="
            f"{constants['random']:g}, "
            f"reference="
            f"{constants['reference']:g}"
        )

    print()

    for beta in RATE_BETA_GRID:
        for kappa in RATE_KAPPA_GRID:
            beta = float(
                beta
            )

            kappa = float(
                kappa
            )

            key = (
                beta,
                kappa,
            )

            if key not in grouped:
                raise RuntimeError(
                    "Missing RATE tuning "
                    f"group for "
                    f"beta={beta}, "
                    f"kappa={kappa}."
                )

            env_records = (
                grouped[
                    key
                ]
            )

            if (
                set(env_records)
                != expected_envs
            ):
                raise RuntimeError(
                    "Environment mismatch "
                    f"for beta={beta}, "
                    f"kappa={kappa}."
                )

            per_env = {}

            global_env_means = []

            print(
                "=" * 72
            )

            print(
                f"beta={beta:g}, "
                f"kappa={kappa:g}"
            )

            print(
                "-" * 72
            )

            representative_kwargs = None

            for env_id in RATE_TUNING_ENVS:
                seed_records = (
                    env_records[
                        env_id
                    ]
                )

                if (
                    set(seed_records)
                    != expected_seeds
                ):
                    raise RuntimeError(
                        "Seed mismatch for "
                        f"beta={beta}, "
                        f"kappa={kappa}, "
                        f"env={env_id}."
                    )

                raw_scores = [
                    float(
                        seed_records[
                            int(seed)
                        ][
                            "raw_score"
                        ]
                    )
                    for seed
                    in TUNING_SEEDS
                ]

                normalised_scores = [
                    float(
                        seed_records[
                            int(seed)
                        ][
                            "normalised_score"
                        ]
                    )
                    for seed
                    in TUNING_SEEDS
                ]

                alpha_values = {
                    float(
                        seed_records[
                            int(seed)
                        ][
                            "alpha_bar"
                        ]
                    )
                    for seed
                    in TUNING_SEEDS
                }

                if len(
                    alpha_values
                ) != 1:
                    raise RuntimeError(
                        "Alpha mismatch "
                        "between RATE seeds "
                        f"for {env_id}, "
                        f"beta={beta}, "
                        f"kappa={kappa}."
                    )

                raw_mean = float(
                    np.mean(
                        raw_scores
                    )
                )

                raw_std = float(
                    np.std(
                        raw_scores,
                        ddof=1,
                    )
                )

                norm_mean = float(
                    np.mean(
                        normalised_scores
                    )
                )

                norm_std = float(
                    np.std(
                        normalised_scores,
                        ddof=1,
                    )
                )

                global_env_means.append(
                    norm_mean
                )

                per_env[
                    env_id
                ] = {
                    "raw_scores": (
                        raw_scores
                    ),
                    "raw_mean": (
                        raw_mean
                    ),
                    "raw_std": (
                        raw_std
                    ),
                    "normalised_scores": (
                        normalised_scores
                    ),
                    "normalised_mean": (
                        norm_mean
                    ),
                    "normalised_std": (
                        norm_std
                    ),
                    "alpha_bar": (
                        float(
                            next(
                                iter(
                                    alpha_values
                                )
                            )
                        )
                    ),
                }

                score_text = (
                    "  ".join(
                        f"{int(seed)}="
                        f"{score:.3f}"
                        for seed, score
                        in zip(
                            TUNING_SEEDS,
                            raw_scores,
                        )
                    )
                )

                print(
                    f"{env_id:<18} "
                    f"{score_text}"
                )

                print(
                    f"{'':18} "
                    f"raw mean="
                    f"{raw_mean:.3f}  "
                    f"raw std="
                    f"{raw_std:.3f}  "
                    f"norm mean="
                    f"{norm_mean:.6f}"
                )

                if representative_kwargs is None:
                    representative_kwargs = dict(
                        seed_records[
                            int(
                                TUNING_SEEDS[
                                    0
                                ]
                            )
                        ][
                            "explorer_kwargs"
                        ]
                    )

            global_score = float(
                np.mean(
                    global_env_means
                )
            )

            between_env_std = float(
                np.std(
                    global_env_means,
                    ddof=1,
                )
            )

            candidate = {
                "beta": (
                    beta
                ),
                "kappa": (
                    kappa
                ),
                "explorer_kwargs": (
                    representative_kwargs
                ),
                "per_environment": (
                    per_env
                ),
                "global_normalised_mean": (
                    global_score
                ),
                "between_environment_std": (
                    between_env_std
                ),
            }

            candidates.append(
                candidate
            )

            print()

            print(
                "GLOBAL NORMALISED "
                "MEAN: "
                f"{global_score:.6f}"
            )

            print(
                "Between-environment "
                "std: "
                f"{between_env_std:.6f}"
            )

            print()
            print()

    global_scores = np.asarray(
        [
            candidate[
                "global_normalised_mean"
            ]
            for candidate
            in candidates
        ],
        dtype=np.float64,
    )

    best_score = float(
        np.max(
            global_scores
        )
    )

    winner_indices = (
        np.flatnonzero(
            np.isclose(
                global_scores,
                best_score,
                rtol=1e-12,
                atol=1e-12,
            )
        )
    )

    if (
        len(
            winner_indices
        )
        != 1
    ):
        ties = [
            (
                candidates[i][
                    "beta"
                ],
                candidates[i][
                    "kappa"
                ],
            )
            for i
            in winner_indices
        ]

        raise RuntimeError(
            "RATE global tuning "
            f"produced a tie: "
            f"{ties}"
        )

    winner = candidates[
        int(
            winner_indices[
                0
            ]
        )
    ]

    summary = {
        "tuning_environments": (
            list(
                RATE_TUNING_ENVS
            )
        ),
        "tuning_seeds": [
            int(seed)
            for seed
            in TUNING_SEEDS
        ],
        "normalisation": (
            NORMALISATION
        ),
        "selection_metric": (
            "mean of per-environment "
            "mean normalised final "
            "greedy returns"
        ),
        "candidates": (
            candidates
        ),
        "selected": (
            winner
        ),
    }

    selected = {
        "explorer": "rate",
        "explorer_kwargs": (
            winner[
                "explorer_kwargs"
            ]
        ),
        "beta": (
            winner[
                "beta"
            ]
        ),
        "kappa": (
            winner[
                "kappa"
            ]
        ),
        "global_normalised_mean": (
            winner[
                "global_normalised_mean"
            ]
        ),
        "tuning_environments": (
            list(
                RATE_TUNING_ENVS
            )
        ),
        "tuning_seeds": [
            int(seed)
            for seed
            in TUNING_SEEDS
        ],
    }

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    _write_json_atomic(
        selected,
        SELECTED_PATH,
    )

    print(
        "=" * 72
    )

    print(
        "GLOBAL RATE WINNER"
    )

    print(
        "=" * 72
    )

    print()

    print(
        f"beta: "
        f"{winner['beta']:g}"
    )

    print(
        f"kappa: "
        f"{winner['kappa']:g}"
    )

    print(
        "Global normalised mean: "
        f"{winner['global_normalised_mean']:.6f}"
    )

    print()

    print(
        "Per-environment "
        "normalised means:"
    )

    for env_id in RATE_TUNING_ENVS:
        value = (
            winner[
                "per_environment"
            ][
                env_id
            ][
                "normalised_mean"
            ]
        )

        print(
            f"  {env_id}: "
            f"{value:.6f}"
        )

    print()

    print(
        "Summary written to:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    print(
        "Selected RATE "
        "configuration written to:"
    )

    print(
        SELECTED_PATH
    )

    print()

    print(
        "GLOBAL RATE "
        "SELECTION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/select_rate.py


In [38]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.select_rate",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
GLOBAL RATE SELECTION RESULTS

Stored results: 24
Expected results: 24

Selection uses normalised scores across:
  MountainCar-v0: random=-200, reference=-105
  LunarLander-v3: random=-180, reference=200

beta=0.01, kappa=1
------------------------------------------------------------------------
MountainCar-v0     1000=-128.300  1001=-162.400  1002=-112.600
                   raw mean=-134.433  raw std=25.460  norm mean=0.690175
LunarLander-v3     1000=75.530  1001=21.860  1002=39.880
                   raw mean=45.756  raw std=27.313  norm mean=0.594096

GLOBAL NORMALISED MEAN: 0.642136
Between-environment std: 0.067939


beta=0.01, kappa=2
------------------------------------------------------------------------
MountainCar-v0     1000=-132.800  1001=-101.800  1002=-112.200
                   raw mean=-115.600  raw std=15.777  norm mean=0.888421
LunarLander-v3     1000=243.096  1001=-14.266  1002=20.224
                   raw mean=83.018  raw std=139.700  norm 

**NEXT**

Experiment A — TD-error trajectory: does $|\delta|$ fall during successful learning?
Experiment B — VDBE degeneration sweep: does large $\sigma$ produce the predicted plateau?
Experiment C — reward-scale invariance: RATE should be invariant to reward scaling while VDBE should not.

# Experiment A: TD-error Trajectory

This experiment tests the premise behind VDBE: if TD error is supposed to indicate “how much is left to learn,” then $|\delta|$ should generally fall while greedy performance improves.

We log mean $|\delta|$ in 1,000-training-step blocks alongside greedy evaluation.

We'll run this on all four environments, because later the intended mechanism claim is that the TD-error behavior is not just a MountainCar curiosity.

For the behavior policy, we'll use each environment's already-tuned decaying $\epsilon$-greedy baseline.

What this stage will run

Fresh diagnostic seeds:
```
2000, 2001, ..., 2009
```
so they overlap with neither:
```
tuning: 1000–1002
final:  0–29
```
The selected decay configurations are already frozen as:
```
MountainCar   alpha=0.75   decay horizon=0.2 × budget
CartPole      alpha=1.0    decay horizon=0.4 × budget
Acrobot       alpha=0.5    decay horizon=0.6 × budget
LunarLander   alpha=0.1    decay horizon=0.4 × budget
```
There are:

$$ 4\times10=\boxed{40\text{ diagnostic runs}}. $$

We do not need to modify agents.py or runner.py. Instead, we'll wrap the existing decay explorer so that every call to explorer.update(...) records the absolute TD error. This keeps the actual learning algorithm unchanged.

In [1]:
%%writefile src/scripts/diagnose_td_error.py
import json
import multiprocessing as mp
import os
import pickle
import traceback

from pathlib import Path

import numpy as np

from src.agents import SarsaLambdaAgent
from src.environments import GymEnv
from src.exploration import make_explorer
from src.sweep_configs import (
    ENV_IDS,
    STEP_BUDGET,
    LAMBDA_FIXED,
)
from src.tilecoding import (
    TileCoder,
    default_tiling_config,
)


SELECTED_BASELINES_PATH = Path(
    "src/results/tuning/"
    "selected_linear_baselines.json"
)

OUT_PATH = Path(
    "src/results/diagnostics/"
    "td_error_trajectory.pkl"
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        2000,
        2010,
    )
)

BLOCK_SIZE = 1000

N_EVAL_POINTS = 20

N_EVAL_EPISODES = 10

MAX_WORKERS = 8


class RecordingExplorer:
    def __init__(
        self,
        base,
        block_size,
    ):
        self.base = base
        self.block_size = int(
            block_size
        )

        self.block_sum = 0.0
        self.block_count = 0
        self.recorded_steps = 0

        self.block_steps = []
        self.block_means = []

    def __getattr__(
        self,
        name,
    ):
        return getattr(
            self.base,
            name,
        )

    def reset_episode(
        self,
    ):
        return self.base.reset_episode()

    def select(
        self,
        q_values,
        rng,
        feat_idx=None,
    ):
        return self.base.select(
            q_values,
            rng,
            feat_idx,
        )

    def update(
        self,
        td_error,
        value=0.0,
        feat_idx=None,
    ):
        self.base.update(
            td_error,
            value,
            feat_idx,
        )

        self.block_sum += abs(
            float(
                td_error
            )
        )

        self.block_count += 1
        self.recorded_steps += 1

        if (
            self.block_count
            == self.block_size
        ):
            self.block_steps.append(
                self.recorded_steps
            )

            self.block_means.append(
                self.block_sum
                / self.block_count
            )

            self.block_sum = 0.0
            self.block_count = 0

    def finalize(
        self,
    ):
        if self.block_count:
            self.block_steps.append(
                self.recorded_steps
            )

            self.block_means.append(
                self.block_sum
                / self.block_count
            )

            self.block_sum = 0.0
            self.block_count = 0


def _config_key(
    cfg,
):
    return json.dumps(
        cfg,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def _save_atomic(
    obj,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "wb",
    ) as f:
        pickle.dump(
            obj,
            f,
        )

    tmp.replace(
        path
    )


def _load_selected_decay():
    if not SELECTED_BASELINES_PATH.exists():
        raise FileNotFoundError(
            "Selected linear baselines "
            "not found: "
            f"{SELECTED_BASELINES_PATH}"
        )

    with open(
        SELECTED_BASELINES_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        selected = json.load(f)

    if not isinstance(
        selected,
        dict,
    ):
        raise TypeError(
            "Selected baseline file "
            "must contain a dictionary."
        )

    configs = {}

    for env_id in ENV_IDS:
        if env_id not in selected:
            raise KeyError(
                f"Missing environment "
                f"{env_id}."
            )

        if (
            "decay"
            not in selected[
                env_id
            ]
        ):
            raise KeyError(
                f"Missing decay baseline "
                f"for {env_id}."
            )

        item = (
            selected[
                env_id
            ][
                "decay"
            ]
        )

        if (
            "alpha_bar"
            not in item
        ):
            raise KeyError(
                f"Missing alpha_bar "
                f"for {env_id}."
            )

        if (
            "explorer_kwargs"
            not in item
        ):
            raise KeyError(
                "Missing explorer_kwargs "
                f"for {env_id}."
            )

        configs[
            env_id
        ] = {
            "alpha_bar": float(
                item[
                    "alpha_bar"
                ]
            ),
            "explorer_kwargs": dict(
                item[
                    "explorer_kwargs"
                ]
            ),
        }

    return configs


def _run_one(
    cfg,
):
    env_id = cfg[
        "env_id"
    ]

    seed = int(
        cfg[
            "seed"
        ]
    )

    n_steps = int(
        cfg[
            "n_steps"
        ]
    )

    alpha_bar = float(
        cfg[
            "alpha_bar"
        ]
    )

    explorer_kwargs = dict(
        cfg[
            "explorer_kwargs"
        ]
    )

    if n_steps <= 0:
        raise ValueError(
            "n_steps must be positive."
        )

    if (
        n_steps
        % BLOCK_SIZE
        != 0
    ):
        raise ValueError(
            f"{env_id} step budget "
            f"{n_steps} is not divisible "
            f"by block size "
            f"{BLOCK_SIZE}."
        )

    env = GymEnv(
        env_id,
        seed=seed,
    )

    rng = np.random.default_rng(
        100_000 + seed
    )

    eval_env = GymEnv(
        env_id,
        seed=500_000 + seed,
    )

    eval_rng = np.random.default_rng(
        300_000 + seed
    )

    low, high = (
        env.tile_bounds()
    )

    tiling_cfg = (
        default_tiling_config(
            env_id
        )
    )

    coder = TileCoder(
        low=low,
        high=high,
        **tiling_cfg,
    )

    base_explorer = (
        make_explorer(
            "decay",
            n_actions=(
                env.n_actions
            ),
            n_features=(
                coder.n_features
            ),
            **explorer_kwargs,
        )
    )

    explorer = RecordingExplorer(
        base=base_explorer,
        block_size=BLOCK_SIZE,
    )

    agent = SarsaLambdaAgent(
        coder=coder,
        n_actions=env.n_actions,
        explorer=explorer,
        alpha_bar=alpha_bar,
        gamma=1.0,
        lam=LAMBDA_FIXED,
        q_init=0.0,
    )

    class TrainEnv:
        def __init__(
            self,
            base_env,
            max_steps,
        ):
            self.base_env = base_env
            self.max_steps = int(
                max_steps
            )
            self.steps = 0
            self.last_budget_cut = False

        def reset(
            self,
        ):
            self.last_budget_cut = False

            return (
                self.base_env.reset()
            )

        def step(
            self,
            action,
        ):
            if (
                self.steps
                >= self.max_steps
            ):
                raise RuntimeError(
                    "Training step budget "
                    "exhausted."
                )

            (
                obs,
                reward,
                terminated,
                truncated,
            ) = self.base_env.step(
                action
            )

            self.steps += 1

            budget_cut = (
                self.steps
                >= self.max_steps
                and not terminated
                and not truncated
            )

            self.last_budget_cut = (
                budget_cut
            )

            return (
                obs,
                reward,
                terminated,
                truncated
                or budget_cut,
            )

    train_env = TrainEnv(
        base_env=env,
        max_steps=n_steps,
    )

    eval_every = max(
        1,
        n_steps
        // N_EVAL_POINTS,
    )

    next_eval = (
        eval_every
    )

    eval_steps = []
    eval_returns = []

    episode_steps = []
    episode_returns = []

    while (
        agent.total_steps
        < n_steps
    ):
        rec = agent.run_episode(
            train_env,
            rng,
        )

        episode_steps.append(
            rec.total_steps
        )

        episode_returns.append(
            rec.ret
        )

        if (
            agent.total_steps
            >= next_eval
        ):
            eval_steps.append(
                agent.total_steps
            )

            eval_returns.append(
                agent.evaluate(
                    eval_env,
                    eval_rng,
                    N_EVAL_EPISODES,
                )
            )

            while (
                next_eval
                <= agent.total_steps
            ):
                next_eval += (
                    eval_every
                )

    explorer.finalize()

    if (
        agent.total_steps
        != n_steps
    ):
        raise RuntimeError(
            f"Expected {n_steps} "
            f"training steps, got "
            f"{agent.total_steps}."
        )

    if (
        train_env.steps
        != n_steps
    ):
        raise RuntimeError(
            "TrainEnv step count "
            f"is {train_env.steps}, "
            f"expected {n_steps}."
        )

    if (
        explorer.recorded_steps
        != n_steps
    ):
        raise RuntimeError(
            "TD recorder observed "
            f"{explorer.recorded_steps} "
            f"updates, expected "
            f"{n_steps}."
        )

    td_block_steps = (
        np.asarray(
            explorer.block_steps,
            dtype=np.int64,
        )
    )

    td_block_mean = (
        np.asarray(
            explorer.block_means,
            dtype=np.float64,
        )
    )

    expected_blocks = (
        n_steps
        // BLOCK_SIZE
    )

    if (
        td_block_steps.size
        != expected_blocks
    ):
        raise RuntimeError(
            "Expected "
            f"{expected_blocks} "
            "TD blocks, found "
            f"{td_block_steps.size}."
        )

    if (
        td_block_mean.size
        != expected_blocks
    ):
        raise RuntimeError(
            "TD block-value count "
            "does not match expected "
            f"{expected_blocks}."
        )

    if (
        td_block_steps[-1]
        != n_steps
    ):
        raise RuntimeError(
            "Final TD block ends at "
            f"{td_block_steps[-1]}, "
            f"expected {n_steps}."
        )

    eval_steps = np.asarray(
        eval_steps,
        dtype=np.int64,
    )

    eval_returns = np.asarray(
        eval_returns,
        dtype=np.float64,
    )

    episode_steps = np.asarray(
        episode_steps,
        dtype=np.int64,
    )

    episode_returns = np.asarray(
        episode_returns,
        dtype=np.float64,
    )

    final_eval = (
        float(
            eval_returns[
                -1
            ]
        )
        if eval_returns.size
        else float(
            "nan"
        )
    )

    result = {
        "env_id": env_id,
        "seed": seed,
        "n_steps": n_steps,
        "alpha_bar": (
            alpha_bar
        ),
        "algo": (
            "sarsa-lambda"
        ),
        "explorer": (
            "decay"
        ),
        "explorer_kwargs": (
            explorer_kwargs
        ),
        "gamma": 1.0,
        "lam": float(
            LAMBDA_FIXED
        ),
        "block_size": (
            BLOCK_SIZE
        ),
        "n_eval_points": (
            N_EVAL_POINTS
        ),
        "n_eval_episodes": (
            N_EVAL_EPISODES
        ),
        "td_block_steps": (
            td_block_steps
        ),
        "td_block_mean": (
            td_block_mean
        ),
        "eval_steps": (
            eval_steps
        ),
        "eval_returns": (
            eval_returns
        ),
        "final_eval": (
            final_eval
        ),
        "episode_steps": (
            episode_steps
        ),
        "episode_returns": (
            episode_returns
        ),
        "total_steps": int(
            agent.total_steps
        ),
        "iht_fullness": (
            coder.iht.fullness
        ),
        "iht_overfull_count": (
            coder.iht.overfull_count
        ),
    }

    env.env.close()
    eval_env.env.close()

    return result


def _worker(
    cfg,
):
    key = _config_key(
        cfg
    )

    try:
        result = _run_one(
            cfg
        )

        result[
            "key"
        ] = key

        result[
            "config"
        ] = dict(
            cfg
        )

        return result

    except Exception as exc:
        return {
            "key": key,
            "config": dict(
                cfg
            ),
            "error": (
                f"{type(exc).__name__}: "
                f"{exc}"
            ),
            "traceback": (
                traceback.format_exc()
            ),
        }


def main():
    selected_decay = (
        _load_selected_decay()
    )

    configs = []

    for env_id in ENV_IDS:
        for seed in DIAGNOSTIC_SEEDS:
            configs.append(
                {
                    "env_id": (
                        env_id
                    ),
                    "seed": int(
                        seed
                    ),
                    "n_steps": int(
                        STEP_BUDGET[
                            env_id
                        ]
                    ),
                    "alpha_bar": float(
                        selected_decay[
                            env_id
                        ][
                            "alpha_bar"
                        ]
                    ),
                    "explorer_kwargs": dict(
                        selected_decay[
                            env_id
                        ][
                            "explorer_kwargs"
                        ]
                    ),
                }
            )

    expected_total = (
        len(ENV_IDS)
        * len(
            DIAGNOSTIC_SEEDS
        )
    )

    if (
        len(configs)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected diagnostic "
            f"config count: expected "
            f"{expected_total}, "
            f"found {len(configs)}."
        )

    existing = {}

    if OUT_PATH.exists():
        with open(
            OUT_PATH,
            "rb",
        ) as f:
            loaded = (
                pickle.load(f)
            )

        if not isinstance(
            loaded,
            list,
        ):
            raise TypeError(
                "Existing diagnostic "
                "results must be a list."
            )

        for result in loaded:
            key = result.get(
                "key"
            )

            if key is None:
                raise KeyError(
                    "Existing result "
                    "missing key."
                )

            existing[
                key
            ] = result

    pending = []

    for cfg in configs:
        key = _config_key(
            cfg
        )

        previous = (
            existing.get(
                key
            )
        )

        if (
            previous is None
            or "error" in previous
        ):
            pending.append(
                cfg
            )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(pending)
            if pending
            else 1,
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "TD-ERROR DIAGNOSTIC"
    )

    print()

    print(
        "Environments:"
    )

    for env_id in ENV_IDS:
        item = (
            selected_decay[
                env_id
            ]
        )

        print(
            f"  {env_id}: "
            f"alpha_bar="
            f"{item['alpha_bar']:g}, "
            f"decay="
            f"{item['explorer_kwargs']}"
        )

    print()

    print(
        f"Diagnostic seeds: "
        f"{DIAGNOSTIC_SEEDS}"
    )

    print(
        f"TD block size: "
        f"{BLOCK_SIZE}"
    )

    print(
        f"Greedy evaluation points: "
        f"{N_EVAL_POINTS}"
    )

    print(
        f"Greedy evaluation episodes: "
        f"{N_EVAL_EPISODES}"
    )

    print(
        f"Total configs: "
        f"{expected_total}"
    )

    print(
        f"Loaded: "
        f"{len(existing)}"
    )

    print(
        f"Pending: "
        f"{len(pending)}"
    )

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    if pending:
        ctx = (
            mp.get_context(
                "spawn"
            )
        )

        with ctx.Pool(
            processes=workers
        ) as pool:
            iterator = (
                pool.imap_unordered(
                    _worker,
                    pending,
                )
            )

            for i, result in enumerate(
                iterator,
                start=1,
            ):
                existing[
                    result[
                        "key"
                    ]
                ] = result

                _save_atomic(
                    list(
                        existing.values()
                    ),
                    OUT_PATH,
                )

                status = (
                    "ERROR"
                    if "error" in result
                    else "OK"
                )

                cfg = result[
                    "config"
                ]

                print(
                    f"{i}/{len(pending)} "
                    f"{status}  "
                    f"{cfg['env_id']}  "
                    f"seed="
                    f"{cfg['seed']}"
                )

    results = list(
        existing.values()
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    successful = [
        result
        for result in results
        if "error" not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED DIAGNOSTIC RUNS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            cfg = result[
                "config"
            ]

            print(
                f"{i}. "
                f"{cfg['env_id']} "
                f"seed={cfg['seed']}"
            )

            print(
                result[
                    "error"
                ]
            )

        raise RuntimeError(
            f"{len(errors)} TD-error "
            "diagnostic runs failed."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Expected "
            f"{expected_total} "
            "successful diagnostic "
            f"runs, found "
            f"{len(successful)}."
        )

    print()

    print(
        "TD-ERROR DIAGNOSTIC "
        "COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/diagnose_td_error.py


In [2]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.diagnose_td_error",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "TD-error diagnostic failed."
    )

TD-ERROR DIAGNOSTIC

Environments:
  MountainCar-v0: alpha_bar=0.75, decay={'decay_steps': 30000, 'eps_end': 0.01, 'eps_start': 1.0, 'mode': 'linear'}
  CartPole-v1: alpha_bar=1, decay={'decay_steps': 40000, 'eps_end': 0.01, 'eps_start': 1.0, 'mode': 'linear'}
  Acrobot-v1: alpha_bar=0.5, decay={'decay_steps': 60000, 'eps_end': 0.01, 'eps_start': 1.0, 'mode': 'linear'}
  LunarLander-v3: alpha_bar=0.1, decay={'decay_steps': 120000, 'eps_end': 0.01, 'eps_start': 1.0, 'mode': 'linear'}

Diagnostic seeds: (2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009)
TD block size: 1000
Greedy evaluation points: 20
Greedy evaluation episodes: 10
Total configs: 40
Loaded: 0
Pending: 40
Detected CPUs: 20
Workers: 8
Output: src\results\diagnostics\td_error_trajectory.pkl

1/40 OK  MountainCar-v0  seed=2003
2/40 OK  MountainCar-v0  seed=2001
3/40 OK  MountainCar-v0  seed=2004
4/40 OK  MountainCar-v0  seed=2000
5/40 OK  MountainCar-v0  seed=2002
6/40 OK  MountainCar-v0  seed=2005
7/40 OK  Mountai

In [3]:
%%writefile src/scripts/analyze_td_error.py
import csv
import json
import pickle

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    STEP_BUDGET,
)


RESULTS_PATH = Path(
    "src/results/diagnostics/"
    "td_error_trajectory.pkl"
)

SUMMARY_PATH = Path(
    "src/results/diagnostics/"
    "td_error_summary.json"
)

SEED_TABLE_PATH = Path(
    "src/results/diagnostics/"
    "td_error_seed_summary.csv"
)

TRAJECTORY_PATH = Path(
    "src/results/diagnostics/"
    "td_error_mean_trajectories.npz"
)

PLOT_DIR = Path(
    "src/results/diagnostics/"
    "td_error_plots"
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        2000,
        2010,
    )
)

EXPECTED_BLOCK_SIZE = 1000

EXPECTED_EVAL_POINTS = 20

WINDOW_FRACTION = 0.10


def _write_json_atomic(
    obj,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(
        path
    )


def _mean_std(
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    if values.size == 0:
        return (
            float("nan"),
            float("nan"),
        )

    mean = float(
        np.mean(
            values
        )
    )

    if values.size >= 2:
        std = float(
            np.std(
                values,
                ddof=1,
            )
        )
    else:
        std = 0.0

    return (
        mean,
        std,
    )


def _validate_result(
    result,
):
    required = (
        "env_id",
        "seed",
        "n_steps",
        "alpha_bar",
        "explorer",
        "explorer_kwargs",
        "block_size",
        "td_block_steps",
        "td_block_mean",
        "eval_steps",
        "eval_returns",
        "final_eval",
        "total_steps",
    )

    for field in required:
        if field not in result:
            raise KeyError(
                "Diagnostic result "
                f"missing {field}."
            )

    env_id = result[
        "env_id"
    ]

    seed = int(
        result[
            "seed"
        ]
    )

    n_steps = int(
        result[
            "n_steps"
        ]
    )

    block_size = int(
        result[
            "block_size"
        ]
    )

    if env_id not in ENV_IDS:
        raise ValueError(
            "Unexpected environment: "
            f"{env_id}"
        )

    if seed not in DIAGNOSTIC_SEEDS:
        raise ValueError(
            "Unexpected diagnostic "
            f"seed: {seed}"
        )

    if (
        n_steps
        != STEP_BUDGET[
            env_id
        ]
    ):
        raise ValueError(
            f"Step-budget mismatch for "
            f"{env_id}: "
            f"{n_steps}"
        )

    if (
        block_size
        != EXPECTED_BLOCK_SIZE
    ):
        raise ValueError(
            "Unexpected TD block size "
            f"for {env_id}, "
            f"seed={seed}: "
            f"{block_size}"
        )

    if (
        int(
            result[
                "total_steps"
            ]
        )
        != n_steps
    ):
        raise ValueError(
            "total_steps mismatch "
            f"for {env_id}, "
            f"seed={seed}."
        )

    td_steps = np.asarray(
        result[
            "td_block_steps"
        ],
        dtype=np.int64,
    )

    td_values = np.asarray(
        result[
            "td_block_mean"
        ],
        dtype=np.float64,
    )

    eval_steps = np.asarray(
        result[
            "eval_steps"
        ],
        dtype=np.int64,
    )

    eval_returns = np.asarray(
        result[
            "eval_returns"
        ],
        dtype=np.float64,
    )

    expected_blocks = (
        n_steps
        // block_size
    )

    if (
        td_steps.size
        != expected_blocks
    ):
        raise ValueError(
            "TD-step array length "
            f"mismatch for {env_id}, "
            f"seed={seed}: expected "
            f"{expected_blocks}, "
            f"found {td_steps.size}."
        )

    if (
        td_values.size
        != expected_blocks
    ):
        raise ValueError(
            "TD-value array length "
            f"mismatch for {env_id}, "
            f"seed={seed}."
        )

    if (
        eval_steps.size
        != EXPECTED_EVAL_POINTS
    ):
        raise ValueError(
            "Expected "
            f"{EXPECTED_EVAL_POINTS} "
            "greedy evaluations for "
            f"{env_id}, seed={seed}, "
            f"found "
            f"{eval_steps.size}."
        )

    if (
        eval_returns.size
        != EXPECTED_EVAL_POINTS
    ):
        raise ValueError(
            "Evaluation-return count "
            f"mismatch for {env_id}, "
            f"seed={seed}."
        )

    if (
        td_steps[-1]
        != n_steps
    ):
        raise ValueError(
            "Final TD block does not "
            f"end at budget for "
            f"{env_id}, seed={seed}."
        )

    if not np.all(
        np.isfinite(
            td_values
        )
    ):
        raise ValueError(
            "Non-finite TD values "
            f"for {env_id}, "
            f"seed={seed}."
        )

    if not np.all(
        np.isfinite(
            eval_returns
        )
    ):
        raise ValueError(
            "Non-finite evaluation "
            f"returns for {env_id}, "
            f"seed={seed}."
        )


def _save_seed_csv(
    rows,
):
    SEED_TABLE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fields = (
        "env_id",
        "seed",
        "n_steps",
        "alpha_bar",
        "early_td",
        "late_td",
        "td_ratio",
        "td_percent_change",
        "early_greedy_return",
        "late_greedy_return",
        "greedy_return_change",
        "td_increased",
        "greedy_improved",
        "inversion_observed",
    )

    with open(
        SEED_TABLE_PATH,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fields,
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(
                row
            )


def _save_td_plot(
    env_id,
    progress,
    mean_values,
    std_values,
):
    PLOT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    fig = plt.figure(
        figsize=(
            8,
            5,
        )
    )

    ax = fig.add_subplot(
        111
    )

    ax.plot(
        progress,
        mean_values,
    )

    ax.fill_between(
        progress,
        mean_values
        - std_values,
        mean_values
        + std_values,
        alpha=0.2,
    )

    ax.set_xlabel(
        "Training progress"
    )

    ax.set_ylabel(
        "Mean absolute TD error"
    )

    ax.set_title(
        f"{env_id}: "
        "TD-error trajectory"
    )

    ax.grid(
        alpha=0.2
    )

    fig.tight_layout()

    path = (
        PLOT_DIR
        / (
            env_id
            .replace(
                "-",
                "_"
            )
            + "_td_error.png"
        )
    )

    fig.savefig(
        path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )


def _save_return_plot(
    env_id,
    progress,
    mean_values,
    std_values,
):
    PLOT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    fig = plt.figure(
        figsize=(
            8,
            5,
        )
    )

    ax = fig.add_subplot(
        111
    )

    ax.plot(
        progress,
        mean_values,
    )

    ax.fill_between(
        progress,
        mean_values
        - std_values,
        mean_values
        + std_values,
        alpha=0.2,
    )

    ax.set_xlabel(
        "Training progress"
    )

    ax.set_ylabel(
        "Greedy evaluation return"
    )

    ax.set_title(
        f"{env_id}: "
        "greedy performance"
    )

    ax.grid(
        alpha=0.2
    )

    fig.tight_layout()

    path = (
        PLOT_DIR
        / (
            env_id
            .replace(
                "-",
                "_"
            )
            + "_greedy_return.png"
        )
    )

    fig.savefig(
        path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            "TD-error diagnostic "
            "results not found: "
            f"{RESULTS_PATH}"
        )

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(
            f
        )

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Diagnostic results "
            "must be stored as a list."
        )

    expected_total = (
        len(ENV_IDS)
        * len(
            DIAGNOSTIC_SEEDS
        )
    )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected diagnostic "
            f"result count: expected "
            f"{expected_total}, "
            f"found {len(results)}."
        )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    if errors:
        raise RuntimeError(
            "Diagnostic results "
            f"contain {len(errors)} "
            "error records."
        )

    grouped = {
        env_id: {}
        for env_id
        in ENV_IDS
    }

    for result in results:
        _validate_result(
            result
        )

        env_id = result[
            "env_id"
        ]

        seed = int(
            result[
                "seed"
            ]
        )

        if (
            seed
            in grouped[
                env_id
            ]
        ):
            raise RuntimeError(
                "Duplicate diagnostic "
                f"result for "
                f"{env_id}, "
                f"seed={seed}."
            )

        grouped[
            env_id
        ][
            seed
        ] = result

    expected_seed_set = set(
        DIAGNOSTIC_SEEDS
    )

    for env_id in ENV_IDS:
        actual = set(
            grouped[
                env_id
            ]
        )

        if (
            actual
            != expected_seed_set
        ):
            raise RuntimeError(
                "Diagnostic seed "
                f"mismatch for "
                f"{env_id}: expected "
                f"{sorted(expected_seed_set)}, "
                f"found "
                f"{sorted(actual)}."
            )

    summary = {}
    seed_rows = []
    trajectory_data = {}

    print(
        "TD-ERROR DIAGNOSTIC ANALYSIS"
    )

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Expected results: "
        f"{expected_total}"
    )

    print(
        f"Seeds per environment: "
        f"{len(DIAGNOSTIC_SEEDS)}"
    )

    print(
        f"Early window: first "
        f"{WINDOW_FRACTION * 100:.0f}%"
    )

    print(
        f"Late window: last "
        f"{WINDOW_FRACTION * 100:.0f}%"
    )

    print()

    all_env_support = True

    for env_id in ENV_IDS:
        env_results = [
            grouped[
                env_id
            ][
                seed
            ]
            for seed
            in DIAGNOSTIC_SEEDS
        ]

        n_steps = int(
            STEP_BUDGET[
                env_id
            ]
        )

        n_blocks = (
            n_steps
            // EXPECTED_BLOCK_SIZE
        )

        window_blocks = int(
            round(
                n_blocks
                * WINDOW_FRACTION
            )
        )

        window_blocks = max(
            1,
            window_blocks,
        )

        n_eval = (
            EXPECTED_EVAL_POINTS
        )

        window_evals = int(
            round(
                n_eval
                * WINDOW_FRACTION
            )
        )

        window_evals = max(
            1,
            window_evals,
        )

        td_matrix = np.stack(
            [
                np.asarray(
                    result[
                        "td_block_mean"
                    ],
                    dtype=np.float64,
                )
                for result
                in env_results
            ],
            axis=0,
        )

        td_step_matrix = np.stack(
            [
                np.asarray(
                    result[
                        "td_block_steps"
                    ],
                    dtype=np.float64,
                )
                for result
                in env_results
            ],
            axis=0,
        )

        eval_matrix = np.stack(
            [
                np.asarray(
                    result[
                        "eval_returns"
                    ],
                    dtype=np.float64,
                )
                for result
                in env_results
            ],
            axis=0,
        )

        eval_step_matrix = np.stack(
            [
                np.asarray(
                    result[
                        "eval_steps"
                    ],
                    dtype=np.float64,
                )
                for result
                in env_results
            ],
            axis=0,
        )

        td_curve_mean = np.mean(
            td_matrix,
            axis=0,
        )

        td_curve_std = np.std(
            td_matrix,
            axis=0,
            ddof=1,
        )

        td_steps_mean = np.mean(
            td_step_matrix,
            axis=0,
        )

        eval_curve_mean = np.mean(
            eval_matrix,
            axis=0,
        )

        eval_curve_std = np.std(
            eval_matrix,
            axis=0,
            ddof=1,
        )

        eval_steps_mean = np.mean(
            eval_step_matrix,
            axis=0,
        )

        td_progress = (
            td_steps_mean
            / n_steps
        )

        eval_progress = (
            eval_steps_mean
            / n_steps
        )

        per_seed_early_td = []
        per_seed_late_td = []

        per_seed_early_return = []
        per_seed_late_return = []

        per_seed_td_ratio = []
        per_seed_td_percent = []
        per_seed_return_change = []

        inversion_flags = []
        td_increase_flags = []
        return_improve_flags = []

        alpha_values = set()

        for result in env_results:
            td = np.asarray(
                result[
                    "td_block_mean"
                ],
                dtype=np.float64,
            )

            returns = np.asarray(
                result[
                    "eval_returns"
                ],
                dtype=np.float64,
            )

            early_td = float(
                np.mean(
                    td[
                        :window_blocks
                    ]
                )
            )

            late_td = float(
                np.mean(
                    td[
                        -window_blocks:
                    ]
                )
            )

            early_return = float(
                np.mean(
                    returns[
                        :window_evals
                    ]
                )
            )

            late_return = float(
                np.mean(
                    returns[
                        -window_evals:
                    ]
                )
            )

            if (
                early_td
                <= 0
            ):
                raise ValueError(
                    "Early mean TD error "
                    "must be positive for "
                    f"{env_id}, "
                    f"seed="
                    f"{result['seed']}."
                )

            td_ratio = (
                late_td
                / early_td
            )

            td_percent = (
                100.0
                * (
                    td_ratio
                    - 1.0
                )
            )

            return_change = (
                late_return
                - early_return
            )

            td_increased = (
                late_td
                > early_td
            )

            greedy_improved = (
                late_return
                > early_return
            )

            inversion_observed = (
                td_increased
                and greedy_improved
            )

            per_seed_early_td.append(
                early_td
            )

            per_seed_late_td.append(
                late_td
            )

            per_seed_early_return.append(
                early_return
            )

            per_seed_late_return.append(
                late_return
            )

            per_seed_td_ratio.append(
                td_ratio
            )

            per_seed_td_percent.append(
                td_percent
            )

            per_seed_return_change.append(
                return_change
            )

            td_increase_flags.append(
                td_increased
            )

            return_improve_flags.append(
                greedy_improved
            )

            inversion_flags.append(
                inversion_observed
            )

            alpha_values.add(
                float(
                    result[
                        "alpha_bar"
                    ]
                )
            )

            seed_rows.append(
                {
                    "env_id": (
                        env_id
                    ),
                    "seed": int(
                        result[
                            "seed"
                        ]
                    ),
                    "n_steps": (
                        n_steps
                    ),
                    "alpha_bar": float(
                        result[
                            "alpha_bar"
                        ]
                    ),
                    "early_td": (
                        early_td
                    ),
                    "late_td": (
                        late_td
                    ),
                    "td_ratio": (
                        td_ratio
                    ),
                    "td_percent_change": (
                        td_percent
                    ),
                    "early_greedy_return": (
                        early_return
                    ),
                    "late_greedy_return": (
                        late_return
                    ),
                    "greedy_return_change": (
                        return_change
                    ),
                    "td_increased": (
                        td_increased
                    ),
                    "greedy_improved": (
                        greedy_improved
                    ),
                    "inversion_observed": (
                        inversion_observed
                    ),
                }
            )

        if (
            len(
                alpha_values
            )
            != 1
        ):
            raise RuntimeError(
                "Alpha mismatch between "
                f"diagnostic seeds for "
                f"{env_id}."
            )

        early_td_mean, early_td_std = (
            _mean_std(
                per_seed_early_td
            )
        )

        late_td_mean, late_td_std = (
            _mean_std(
                per_seed_late_td
            )
        )

        early_return_mean, early_return_std = (
            _mean_std(
                per_seed_early_return
            )
        )

        late_return_mean, late_return_std = (
            _mean_std(
                per_seed_late_return
            )
        )

        mean_td_ratio = (
            late_td_mean
            / early_td_mean
        )

        aggregate_td_percent = (
            100.0
            * (
                mean_td_ratio
                - 1.0
            )
        )

        aggregate_return_change = (
            late_return_mean
            - early_return_mean
        )

        aggregate_td_increased = (
            late_td_mean
            > early_td_mean
        )

        aggregate_return_improved = (
            late_return_mean
            > early_return_mean
        )

        aggregate_inversion = (
            aggregate_td_increased
            and aggregate_return_improved
        )

        if not aggregate_inversion:
            all_env_support = False

        td_increase_count = int(
            np.sum(
                td_increase_flags
            )
        )

        return_improve_count = int(
            np.sum(
                return_improve_flags
            )
        )

        inversion_count = int(
            np.sum(
                inversion_flags
            )
        )

        summary[
            env_id
        ] = {
            "n_seeds": (
                len(
                    DIAGNOSTIC_SEEDS
                )
            ),
            "seeds": [
                int(seed)
                for seed
                in DIAGNOSTIC_SEEDS
            ],
            "n_steps": (
                n_steps
            ),
            "alpha_bar": float(
                next(
                    iter(
                        alpha_values
                    )
                )
            ),
            "block_size": (
                EXPECTED_BLOCK_SIZE
            ),
            "n_td_blocks": (
                n_blocks
            ),
            "early_td_blocks": (
                window_blocks
            ),
            "late_td_blocks": (
                window_blocks
            ),
            "early_eval_points": (
                window_evals
            ),
            "late_eval_points": (
                window_evals
            ),
            "early_td_mean": (
                early_td_mean
            ),
            "early_td_std": (
                early_td_std
            ),
            "late_td_mean": (
                late_td_mean
            ),
            "late_td_std": (
                late_td_std
            ),
            "td_ratio_of_means": (
                mean_td_ratio
            ),
            "td_percent_change": (
                aggregate_td_percent
            ),
            "early_greedy_return_mean": (
                early_return_mean
            ),
            "early_greedy_return_std": (
                early_return_std
            ),
            "late_greedy_return_mean": (
                late_return_mean
            ),
            "late_greedy_return_std": (
                late_return_std
            ),
            "greedy_return_change": (
                aggregate_return_change
            ),
            "td_increased": (
                aggregate_td_increased
            ),
            "greedy_improved": (
                aggregate_return_improved
            ),
            "inversion_observed": (
                aggregate_inversion
            ),
            "seeds_td_increased": (
                td_increase_count
            ),
            "seeds_greedy_improved": (
                return_improve_count
            ),
            "seeds_inversion_observed": (
                inversion_count
            ),
        }

        trajectory_data[
            f"{env_id}_td_progress"
        ] = td_progress

        trajectory_data[
            f"{env_id}_td_mean"
        ] = td_curve_mean

        trajectory_data[
            f"{env_id}_td_std"
        ] = td_curve_std

        trajectory_data[
            f"{env_id}_eval_progress"
        ] = eval_progress

        trajectory_data[
            f"{env_id}_eval_mean"
        ] = eval_curve_mean

        trajectory_data[
            f"{env_id}_eval_std"
        ] = eval_curve_std

        _save_td_plot(
            env_id,
            td_progress,
            td_curve_mean,
            td_curve_std,
        )

        _save_return_plot(
            env_id,
            eval_progress,
            eval_curve_mean,
            eval_curve_std,
        )

        print(
            "=" * 78
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 78
        )

        print(
            f"alpha_bar: "
            f"{next(iter(alpha_values)):g}"
        )

        print(
            f"TD blocks: "
            f"{n_blocks}"
        )

        print(
            f"First/last 10% TD blocks: "
            f"{window_blocks}"
        )

        print(
            f"First/last 10% "
            f"evaluation points: "
            f"{window_evals}"
        )

        print()

        print(
            "Mean |delta|:"
        )

        print(
            f"  first 10%: "
            f"{early_td_mean:.6f} "
            f"+/- "
            f"{early_td_std:.6f}"
        )

        print(
            f"  last 10%:  "
            f"{late_td_mean:.6f} "
            f"+/- "
            f"{late_td_std:.6f}"
        )

        print(
            f"  ratio:     "
            f"{mean_td_ratio:.6f}"
        )

        print(
            f"  change:    "
            f"{aggregate_td_percent:+.2f}%"
        )

        print()

        print(
            "Greedy return:"
        )

        print(
            f"  first 10%: "
            f"{early_return_mean:.3f} "
            f"+/- "
            f"{early_return_std:.3f}"
        )

        print(
            f"  last 10%:  "
            f"{late_return_mean:.3f} "
            f"+/- "
            f"{late_return_std:.3f}"
        )

        print(
            f"  change:    "
            f"{aggregate_return_change:+.3f}"
        )

        print()

        print(
            "Seed-level counts:"
        )

        print(
            f"  TD error increased: "
            f"{td_increase_count}/"
            f"{len(DIAGNOSTIC_SEEDS)}"
        )

        print(
            f"  Greedy return improved: "
            f"{return_improve_count}/"
            f"{len(DIAGNOSTIC_SEEDS)}"
        )

        print(
            f"  Both occurred: "
            f"{inversion_count}/"
            f"{len(DIAGNOSTIC_SEEDS)}"
        )

        print()

        if (
            aggregate_inversion
        ):
            print(
                "RESULT: SUPPORTS THE "
                "INVERSION HYPOTHESIS"
            )
        elif (
            aggregate_return_improved
            and not aggregate_td_increased
        ):
            print(
                "RESULT: GREEDY PERFORMANCE "
                "IMPROVED WHILE TD ERROR "
                "DID NOT INCREASE"
            )
        elif (
            aggregate_td_increased
            and not aggregate_return_improved
        ):
            print(
                "RESULT: TD ERROR INCREASED, "
                "BUT GREEDY PERFORMANCE "
                "DID NOT IMPROVE"
            )
        else:
            print(
                "RESULT: NEITHER REQUIRED "
                "CONDITION WAS OBSERVED"
            )

        print()
        print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    _save_seed_csv(
        seed_rows
    )

    TRAJECTORY_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez(
        TRAJECTORY_PATH,
        **trajectory_data,
    )

    print(
        "=" * 78
    )

    print(
        "OVERALL EXPERIMENT-A RESULT"
    )

    print(
        "=" * 78
    )

    print()

    supported = [
        env_id
        for env_id
        in ENV_IDS
        if summary[
            env_id
        ][
            "inversion_observed"
        ]
    ]

    print(
        "Environments showing both "
        "improved greedy return and "
        "increased mean |delta|:"
    )

    for env_id in supported:
        print(
            f"  {env_id}"
        )

    print()

    print(
        f"Count: "
        f"{len(supported)}/"
        f"{len(ENV_IDS)}"
    )

    print()

    if all_env_support:
        print(
            "ALL FOUR ENVIRONMENTS "
            "SUPPORT THE INVERSION "
            "HYPOTHESIS."
        )
    else:
        print(
            "THE INVERSION PATTERN "
            "WAS NOT OBSERVED IN "
            "ALL FOUR ENVIRONMENTS."
        )

    print()

    print(
        "Summary JSON:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    print(
        "Per-seed CSV:"
    )

    print(
        SEED_TABLE_PATH
    )

    print()

    print(
        "Mean trajectories:"
    )

    print(
        TRAJECTORY_PATH
    )

    print()

    print(
        "Plots:"
    )

    print(
        PLOT_DIR
    )

    print()

    print(
        "TD-ERROR DIAGNOSTIC "
        "ANALYSIS COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/analyze_td_error.py


What this is measuring

For MountainCar there are:

$$ 150000/1000=150 $$

TD-error blocks, so the first and last \(10\%\) are each:

$$ 0.10\times150=15 $$

blocks.

Thus for one seed,

$$ \overline{|\delta|}_{\text{early}} = \frac{1}{15} \sum_{b=1}^{15} D_b $$

and

$$ \overline{|\delta|}_{\text{late}} = \frac{1}{15} \sum_{b=136}^{150} D_b. $$

Then we calculate

$$ R_\delta = \frac{ \overline{|\delta|}_{\text{late}} }{ \overline{|\delta|}_{\text{early}} }. $$

Interpretation is simple:
```
Rδ < 1    TD error fell
Rδ = 1    TD error stayed flat
Rδ > 1    TD error grew
```
For example,

$$ R_\delta=1.47 $$

means

$$ (1.47-1)\times100=47\% $$

increase—the exact type of pattern Experiment A is testing. The guide's illustrative MountainCar example is precisely “performance improves while mean absolute TD error rises,” which contradicts the premise that TD magnitude naturally vanishes as learning succeeds under function approximation.

For greedy return, since we used 20 evaluation checkpoints, the first \(10\%\) corresponds to the first two greedy evaluations and the last \(10\%\) to the final two. We average within each seed first, and only then average across the 10 seeds. That way every seed has equal weight.

The script also reports how many individual seeds display both:

$$ |\delta|_{\text{late}}> |\delta|_{\text{early}} $$

and

$$ G_{\text{late}}> G_{\text{early}}. $$

That seed-level count is useful because an aggregate increase driven by one bizarre run is much less persuasive than, say, 8/10 or 10/10 seeds moving in the same direction.

In [6]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.analyze_td_error",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
TD-ERROR DIAGNOSTIC ANALYSIS

Stored results: 40
Expected results: 40
Seeds per environment: 10
Early window: first 10%
Late window: last 10%

Environment: MountainCar-v0
alpha_bar: 0.75
TD blocks: 150
First/last 10% TD blocks: 15
First/last 10% evaluation points: 2

Mean |delta|:
  first 10%: 12.224679 +/- 0.829056
  last 10%:  2.625412 +/- 0.090932
  ratio:     0.214763
  change:    -78.52%

Greedy return:
  first 10%: -200.000 +/- 0.000
  last 10%:  -113.030 +/- 9.776
  change:    +86.970

Seed-level counts:
  TD error increased: 0/10
  Greedy return improved: 10/10
  Both occurred: 0/10

RESULT: GREEDY PERFORMANCE IMPROVED WHILE TD ERROR DID NOT INCREASE


Environment: CartPole-v1
alpha_bar: 1
TD blocks: 100
First/last 10% TD blocks: 10
First/last 10% evaluation points: 2

Mean |delta|:
  first 10%: 6.713149 +/- 0.551767
  last 10%:  81.299093 +/- 34.534513
  ratio:     12.110426
  change:    +1111.04%

Greedy return:
  first 10%: 87.845 +/- 49.321
  last 10

The absolute TD error is not a universally decreasing indicator of learning progress under function approximation: it rises strongly in some tasks while performance improves, but decreases in others.

We need to know whether MountainCar and Acrobot truly decrease throughout training, or whether they have a large initialization transient followed by the later increase that the first-vs-last statistic hides.

For example, MountainCar might actually do:

$$ 12 \rightarrow 3 \rightarrow 1.5 \rightarrow 2.6 $$

In that case:

$$ \text{first 10\%}=12,\quad \text{last 10\%}=2.6 $$

says “TD error decreased,” but after the initial transient it actually bottomed out and then rose again during successful learning. That would be scientifically very different from a monotonic decline.

In [8]:
%%writefile src/scripts/audit_td_error_shape.py
import csv
import json
import pickle

from pathlib import Path

import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    STEP_BUDGET,
)


RESULTS_PATH = Path(
    "src/results/diagnostics/"
    "td_error_trajectory.pkl"
)

SUMMARY_PATH = Path(
    "src/results/diagnostics/"
    "td_error_shape_audit.json"
)

CSV_PATH = Path(
    "src/results/diagnostics/"
    "td_error_deciles.csv"
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        2000,
        2010,
    )
)

N_DECILES = 10


def _write_json_atomic(
    obj,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(
        path
    )


def _decile_means(
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    chunks = np.array_split(
        values,
        N_DECILES,
    )

    return np.asarray(
        [
            np.mean(
                chunk
            )
            for chunk in chunks
        ],
        dtype=np.float64,
    )


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"Missing diagnostic file: "
            f"{RESULTS_PATH}"
        )

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(
            f
        )

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Results must be a list."
        )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    if errors:
        raise RuntimeError(
            f"Found {len(errors)} "
            "error records."
        )

    expected_total = (
        len(ENV_IDS)
        * len(DIAGNOSTIC_SEEDS)
    )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            f"Expected {expected_total} "
            f"results, found "
            f"{len(results)}."
        )

    grouped = {
        env_id: {}
        for env_id in ENV_IDS
    }

    for result in results:
        env_id = result[
            "env_id"
        ]

        seed = int(
            result[
                "seed"
            ]
        )

        if env_id not in grouped:
            raise ValueError(
                f"Unexpected environment: "
                f"{env_id}"
            )

        if seed in grouped[
            env_id
        ]:
            raise RuntimeError(
                f"Duplicate result: "
                f"{env_id}, "
                f"seed={seed}"
            )

        grouped[
            env_id
        ][
            seed
        ] = result

    summary = {}
    csv_rows = []

    print(
        "TD-ERROR TRAJECTORY "
        "SHAPE AUDIT"
    )

    print()

    for env_id in ENV_IDS:
        records = []

        for seed in DIAGNOSTIC_SEEDS:
            if (
                seed
                not in grouped[
                    env_id
                ]
            ):
                raise RuntimeError(
                    f"Missing "
                    f"{env_id}, "
                    f"seed={seed}"
                )

            records.append(
                grouped[
                    env_id
                ][
                    seed
                ]
            )

        td_deciles = np.stack(
            [
                _decile_means(
                    result[
                        "td_block_mean"
                    ]
                )
                for result
                in records
            ],
            axis=0,
        )

        eval_deciles = np.stack(
            [
                _decile_means(
                    result[
                        "eval_returns"
                    ]
                )
                for result
                in records
            ],
            axis=0,
        )

        td_mean = np.mean(
            td_deciles,
            axis=0,
        )

        td_std = np.std(
            td_deciles,
            axis=0,
            ddof=1,
        )

        eval_mean = np.mean(
            eval_deciles,
            axis=0,
        )

        eval_std = np.std(
            eval_deciles,
            axis=0,
            ddof=1,
        )

        min_index = int(
            np.argmin(
                td_mean
            )
        )

        min_td = float(
            td_mean[
                min_index
            ]
        )

        final_td = float(
            td_mean[
                -1
            ]
        )

        rebound_ratio = (
            final_td
            / min_td
            if min_td > 0
            else float(
                "nan"
            )
        )

        rebound_percent = (
            100.0
            * (
                rebound_ratio
                - 1.0
            )
            if np.isfinite(
                rebound_ratio
            )
            else float(
                "nan"
            )
        )

        first_half_x = np.arange(
            5,
            55,
            10,
            dtype=np.float64,
        )

        second_half_x = np.arange(
            55,
            105,
            10,
            dtype=np.float64,
        )

        first_half_slope = float(
            np.polyfit(
                first_half_x,
                td_mean[
                    :5
                ],
                1,
            )[0]
        )

        second_half_slope = float(
            np.polyfit(
                second_half_x,
                td_mean[
                    5:
                ],
                1,
            )[0]
        )

        late_rise = bool(
            second_half_slope
            > 0
        )

        improved_overall = bool(
            eval_mean[
                -1
            ]
            > eval_mean[
                0
            ]
        )

        post_minimum_growth = bool(
            min_index
            < N_DECILES - 1
            and final_td
            > min_td
        )

        summary[
            env_id
        ] = {
            "n_steps": int(
                STEP_BUDGET[
                    env_id
                ]
            ),
            "td_decile_mean": (
                td_mean.tolist()
            ),
            "td_decile_std": (
                td_std.tolist()
            ),
            "eval_decile_mean": (
                eval_mean.tolist()
            ),
            "eval_decile_std": (
                eval_std.tolist()
            ),
            "minimum_td_decile": (
                min_index + 1
            ),
            "minimum_td_progress_percent": (
                int(
                    (
                        min_index
                        + 1
                    )
                    * 10
                )
            ),
            "minimum_td": (
                min_td
            ),
            "final_decile_td": (
                final_td
            ),
            "post_minimum_rebound_ratio": (
                rebound_ratio
            ),
            "post_minimum_rebound_percent": (
                rebound_percent
            ),
            "first_half_td_slope": (
                first_half_slope
            ),
            "second_half_td_slope": (
                second_half_slope
            ),
            "late_td_trend_upward": (
                late_rise
            ),
            "post_minimum_growth": (
                post_minimum_growth
            ),
            "greedy_improved_overall": (
                improved_overall
            ),
        }

        print(
            "=" * 86
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 86
        )

        print()

        print(
            "Progress    "
            "Mean |delta|       "
            "Greedy return"
        )

        print(
            "-" * 58
        )

        for i in range(
            N_DECILES
        ):
            progress = (
                (
                    i + 1
                )
                * 10
            )

            print(
                f"{progress:>3}%        "
                f"{td_mean[i]:>10.6f} "
                f"+/- "
                f"{td_std[i]:<10.6f}   "
                f"{eval_mean[i]:>9.3f} "
                f"+/- "
                f"{eval_std[i]:.3f}"
            )

            csv_rows.append(
                {
                    "env_id": (
                        env_id
                    ),
                    "decile": (
                        i + 1
                    ),
                    "progress_percent": (
                        progress
                    ),
                    "td_mean": float(
                        td_mean[
                            i
                        ]
                    ),
                    "td_std": float(
                        td_std[
                            i
                        ]
                    ),
                    "greedy_mean": float(
                        eval_mean[
                            i
                        ]
                    ),
                    "greedy_std": float(
                        eval_std[
                            i
                        ]
                    ),
                }
            )

        print()

        print(
            f"Minimum TD-error decile: "
            f"{min_index + 1} "
            f"({(min_index + 1) * 10}% "
            f"progress)"
        )

        print(
            f"Minimum mean |delta|: "
            f"{min_td:.6f}"
        )

        print(
            f"Final-decile mean "
            f"|delta|: "
            f"{final_td:.6f}"
        )

        print(
            "Change from minimum "
            "to final decile: "
            f"{rebound_percent:+.2f}%"
        )

        print()

        print(
            "TD trend slope:"
        )

        print(
            f"  first half:  "
            f"{first_half_slope:+.6f} "
            "per percentage point"
        )

        print(
            f"  second half: "
            f"{second_half_slope:+.6f} "
            "per percentage point"
        )

        print()

        print(
            "Late TD trend upward: "
            f"{late_rise}"
        )

        print(
            "TD grows after its "
            "minimum: "
            f"{post_minimum_growth}"
        )

        print(
            "Greedy performance "
            "improved overall: "
            f"{improved_overall}"
        )

        print()
        print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    CSV_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        CSV_PATH,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=(
                "env_id",
                "decile",
                "progress_percent",
                "td_mean",
                "td_std",
                "greedy_mean",
                "greedy_std",
            ),
        )

        writer.writeheader()

        writer.writerows(
            csv_rows
        )

    print(
        "=" * 86
    )

    print(
        "AUDIT SUMMARY"
    )

    print(
        "=" * 86
    )

    print()

    for env_id in ENV_IDS:
        item = summary[
            env_id
        ]

        print(
            f"{env_id}: "
            f"minimum at "
            f"{item['minimum_td_progress_percent']}%, "
            f"minimum->final "
            f"{item['post_minimum_rebound_percent']:+.2f}%, "
            f"late slope "
            f"{item['second_half_td_slope']:+.6f}, "
            f"greedy improved="
            f"{item['greedy_improved_overall']}"
        )

    print()

    print(
        "Summary JSON:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    print(
        "Decile CSV:"
    )

    print(
        CSV_PATH
    )

    print()

    print(
        "TD-ERROR TRAJECTORY "
        "SHAPE AUDIT COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/audit_td_error_shape.py


In [9]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.audit_td_error_shape",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
TD-ERROR TRAJECTORY SHAPE AUDIT

Environment: MountainCar-v0

Progress    Mean |delta|       Greedy return
----------------------------------------------------------
 10%         12.224679 +/- 0.829056      -200.000 +/- 0.000
 20%          6.181104 +/- 0.365576      -166.490 +/- 7.408
 30%          2.336828 +/- 0.438309      -136.545 +/- 11.657
 40%          2.573194 +/- 0.277810      -133.025 +/- 19.149
 50%          2.713664 +/- 0.166454      -126.215 +/- 13.127
 60%          2.746046 +/- 0.180691      -123.635 +/- 14.517
 70%          2.634408 +/- 0.194134      -122.330 +/- 20.310
 80%          2.740572 +/- 0.118156      -114.755 +/- 10.165
 90%          2.653754 +/- 0.151189      -110.290 +/- 6.949
100%          2.625412 +/- 0.090932      -113.030 +/- 9.776

Minimum TD-error decile: 3 (30% progress)
Minimum mean |delta|: 2.336828
Final-decile mean |delta|: 2.625412
Change from minimum to final decile: +12.35%

TD trend slope:
  first half:  -0.226299 per per

| Environment     | What the full trajectory shows                                                                                               | Interpretation                                                                                                                     |
| --------------- | ---------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| **MountainCar** | Huge early TD-error spike, collapse by ~30%, then a small rebound/plateau while return keeps improving                       | **Partial failure of a simple monotonic-progress interpretation**, but not the strong “TD error rises throughout learning” pattern |
| **CartPole**    | TD error rises almost continuously from ~4 to >100 while greedy performance improves massively                               | **Very strong contradiction of the VDBE premise**                                                                                  |
| **Acrobot**     | TD error initially rises, peaks around 20–30%, then steadily falls while performance improves                                | **Mostly consistent with the conventional premise later in learning**                                                              |
| **LunarLander** | TD error falls initially, bottoms around 30%, then rises ~31% from its minimum while performance becomes dramatically better | **Clear late-training contradiction**                                                                                              |


Now we run Experiment A using Double DQN

In [10]:
%%writefile src/scripts/diagnose_td_error_dqn.py
import json
import multiprocessing as mp
import os
import pickle
import traceback

from pathlib import Path

import numpy as np
import torch

from src.dqn import (
    DEVICE,
    QNetwork,
    ReplayBuffer,
    _dqn_update,
    _evaluate_dqn,
)
from src.environments import GymEnv
from src.exploration import make_explorer
from src.sweep_configs import (
    ENV_IDS,
    STEP_BUDGET,
)


SELECTED_BASELINES_PATH = Path(
    "src/results/tuning/"
    "selected_linear_baselines.json"
)

OUT_PATH = Path(
    "src/results/diagnostics/"
    "td_error_trajectory_dqn.pkl"
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        2000,
        2010,
    )
)

BLOCK_SIZE = 1000

N_EVAL_POINTS = 20

N_EVAL_EPISODES = 10

HIDDEN = 128

REPLAY_CAPACITY = 50_000

BATCH_SIZE = 64

LEARNING_RATE = 1e-3

LEARNING_STARTS = 1_000

TRAIN_EVERY = 1

TARGET_UPDATE_EVERY = 1_000

GAMMA = 1.0

REWARD_SCALE = 1.0

MAX_WORKERS = 8


class DiagnosticDQNConfig:
    def __init__(
        self,
        env_id,
        seed,
        n_steps,
        explorer_kwargs,
    ):
        self.env_id = str(
            env_id
        )

        self.explorer = "decay"

        self.explorer_kwargs = dict(
            explorer_kwargs
        )

        self.seed = int(
            seed
        )

        self.n_steps = int(
            n_steps
        )

        self.gamma = float(
            GAMMA
        )

        self.reward_scale = float(
            REWARD_SCALE
        )

        self.hidden = int(
            HIDDEN
        )

        self.replay_capacity = int(
            REPLAY_CAPACITY
        )

        self.batch_size = int(
            BATCH_SIZE
        )

        self.learning_rate = float(
            LEARNING_RATE
        )

        self.learning_starts = int(
            LEARNING_STARTS
        )

        self.train_every = int(
            TRAIN_EVERY
        )

        self.target_update_every = int(
            TARGET_UPDATE_EVERY
        )

        self.double = True

        self.n_bins = 100

        self.n_eval_points = int(
            N_EVAL_POINTS
        )

        self.n_eval_episodes = int(
            N_EVAL_EPISODES
        )


def _config_dict(
    cfg,
):
    return {
        "env_id": (
            cfg.env_id
        ),
        "explorer": (
            cfg.explorer
        ),
        "explorer_kwargs": dict(
            cfg.explorer_kwargs
        ),
        "seed": (
            cfg.seed
        ),
        "n_steps": (
            cfg.n_steps
        ),
        "gamma": (
            cfg.gamma
        ),
        "reward_scale": (
            cfg.reward_scale
        ),
        "hidden": (
            cfg.hidden
        ),
        "replay_capacity": (
            cfg.replay_capacity
        ),
        "batch_size": (
            cfg.batch_size
        ),
        "learning_rate": (
            cfg.learning_rate
        ),
        "learning_starts": (
            cfg.learning_starts
        ),
        "train_every": (
            cfg.train_every
        ),
        "target_update_every": (
            cfg.target_update_every
        ),
        "double": (
            cfg.double
        ),
        "n_eval_points": (
            cfg.n_eval_points
        ),
        "n_eval_episodes": (
            cfg.n_eval_episodes
        ),
        "block_size": (
            BLOCK_SIZE
        ),
    }


def _config_key(
    cfg,
):
    return json.dumps(
        _config_dict(
            cfg
        ),
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def _save_atomic(
    obj,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "wb",
    ) as f:
        pickle.dump(
            obj,
            f,
        )

    tmp.replace(
        path
    )


def _load_decay_configs():
    if not SELECTED_BASELINES_PATH.exists():
        raise FileNotFoundError(
            "Selected linear baseline "
            "file not found: "
            f"{SELECTED_BASELINES_PATH}"
        )

    with open(
        SELECTED_BASELINES_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        selected = json.load(
            f
        )

    if not isinstance(
        selected,
        dict,
    ):
        raise TypeError(
            "Selected baseline file "
            "must contain a dictionary."
        )

    output = {}

    for env_id in ENV_IDS:
        if env_id not in selected:
            raise KeyError(
                "Missing environment "
                f"{env_id}."
            )

        if (
            "decay"
            not in selected[
                env_id
            ]
        ):
            raise KeyError(
                "Missing decay baseline "
                f"for {env_id}."
            )

        item = selected[
            env_id
        ][
            "decay"
        ]

        if (
            "explorer_kwargs"
            not in item
        ):
            raise KeyError(
                "Missing decay "
                "explorer_kwargs "
                f"for {env_id}."
            )

        kwargs = dict(
            item[
                "explorer_kwargs"
            ]
        )

        required = (
            "eps_start",
            "eps_end",
            "decay_steps",
            "mode",
        )

        for name in required:
            if name not in kwargs:
                raise KeyError(
                    f"{env_id} decay "
                    f"config missing "
                    f"{name}."
                )

        output[
            env_id
        ] = kwargs

    return output


def _run_one(
    cfg,
):
    if (
        cfg.n_steps
        % BLOCK_SIZE
        != 0
    ):
        raise ValueError(
            f"{cfg.env_id} budget "
            f"{cfg.n_steps} is not "
            f"divisible by "
            f"{BLOCK_SIZE}."
        )

    if (
        cfg.learning_starts
        < cfg.batch_size
    ):
        raise ValueError(
            "learning_starts must "
            "be at least batch_size."
        )

    if (
        cfg.train_every
        <= 0
    ):
        raise ValueError(
            "train_every must "
            "be positive."
        )

    env = GymEnv(
        cfg.env_id,
        seed=cfg.seed,
    )

    eval_env = GymEnv(
        cfg.env_id,
        seed=(
            500_000
            + cfg.seed
        ),
    )

    action_rng = (
        np.random.default_rng(
            100_000
            + cfg.seed
        )
    )

    replay_rng = (
        np.random.default_rng(
            200_000
            + cfg.seed
        )
    )

    eval_rng = (
        np.random.default_rng(
            300_000
            + cfg.seed
        )
    )

    torch.manual_seed(
        400_000
        + cfg.seed
    )

    obs_shape = (
        env.env
        .observation_space
        .shape
    )

    if (
        obs_shape is None
        or len(
            obs_shape
        )
        != 1
    ):
        raise ValueError(
            "DQN requires a "
            "one-dimensional "
            "observation vector."
        )

    obs_dim = int(
        obs_shape[
            0
        ]
    )

    q = QNetwork(
        obs_dim=obs_dim,
        n_actions=(
            env.n_actions
        ),
        hidden=(
            cfg.hidden
        ),
    ).to(
        DEVICE
    )

    target = QNetwork(
        obs_dim=obs_dim,
        n_actions=(
            env.n_actions
        ),
        hidden=(
            cfg.hidden
        ),
    ).to(
        DEVICE
    )

    target.load_state_dict(
        q.state_dict()
    )

    q.train()
    target.eval()

    optimizer = (
        torch.optim.Adam(
            q.parameters(),
            lr=(
                cfg.learning_rate
            ),
        )
    )

    replay = ReplayBuffer(
        capacity=(
            cfg.replay_capacity
        ),
        obs_dim=(
            obs_dim
        ),
        rng=(
            replay_rng
        ),
    )

    explorer = make_explorer(
        cfg.explorer,
        n_actions=(
            env.n_actions
        ),
        **cfg.explorer_kwargs,
    )

    n_blocks = (
        cfg.n_steps
        // BLOCK_SIZE
    )

    td_block_sum = np.zeros(
        n_blocks,
        dtype=np.float64,
    )

    td_block_count = np.zeros(
        n_blocks,
        dtype=np.int64,
    )

    q_block_sum = np.zeros(
        n_blocks,
        dtype=np.float64,
    )

    loss_block_sum = np.zeros(
        n_blocks,
        dtype=np.float64,
    )

    eval_steps = []
    eval_returns = []

    eval_every = max(
        1,
        cfg.n_steps
        // cfg.n_eval_points,
    )

    next_eval = (
        eval_every
    )

    total_steps = 0
    n_updates = 0

    explorer.reset_episode()

    obs = env.reset()

    while (
        total_steps
        < cfg.n_steps
    ):
        with torch.no_grad():
            x = torch.as_tensor(
                obs,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(
                0
            )

            q_values = (
                q(
                    x
                )
                .squeeze(
                    0
                )
                .cpu()
                .numpy()
            )

        action = explorer.select(
            q_values,
            action_rng,
            None,
        )

        (
            obs2,
            reward,
            terminated,
            truncated,
        ) = env.step(
            action
        )

        replay.add(
            obs,
            action,
            reward
            * cfg.reward_scale,
            obs2,
            terminated,
        )

        total_steps += 1

        if (
            total_steps
            >= cfg.learning_starts
            and len(
                replay
            )
            >= cfg.batch_size
            and (
                total_steps
                % cfg.train_every
                == 0
            )
        ):
            (
                loss_value,
                td_mean,
                q_mean,
            ) = _dqn_update(
                q=q,
                target=target,
                optimizer=optimizer,
                replay=replay,
                explorer=explorer,
                cfg=cfg,
            )

            n_updates += 1

            block_index = min(
                (
                    total_steps
                    - 1
                )
                // BLOCK_SIZE,
                n_blocks
                - 1,
            )

            td_block_sum[
                block_index
            ] += float(
                td_mean
            )

            q_block_sum[
                block_index
            ] += float(
                q_mean
            )

            loss_block_sum[
                block_index
            ] += float(
                loss_value
            )

            td_block_count[
                block_index
            ] += 1

        if (
            total_steps
            % cfg.target_update_every
            == 0
        ):
            target.load_state_dict(
                q.state_dict()
            )

            target.eval()

        if (
            terminated
            or truncated
        ):
            if (
                total_steps
                < cfg.n_steps
            ):
                explorer.reset_episode()
                obs = env.reset()

        else:
            obs = obs2

        if (
            total_steps
            >= next_eval
        ):
            eval_steps.append(
                total_steps
            )

            eval_returns.append(
                _evaluate_dqn(
                    q,
                    eval_env,
                    eval_rng,
                    cfg.n_eval_episodes,
                )
            )

            while (
                next_eval
                <= total_steps
            ):
                next_eval += (
                    eval_every
                )

    if (
        total_steps
        != cfg.n_steps
    ):
        raise RuntimeError(
            f"Expected exactly "
            f"{cfg.n_steps} "
            "environment steps, "
            f"got {total_steps}."
        )

    expected_updates = (
        (
            cfg.n_steps
            - cfg.learning_starts
        )
        // cfg.train_every
        + 1
    )

    if (
        n_updates
        != expected_updates
    ):
        raise RuntimeError(
            "Unexpected DQN update "
            f"count: expected "
            f"{expected_updates}, "
            f"found "
            f"{n_updates}."
        )

    if (
        int(
            td_block_count.sum()
        )
        != n_updates
    ):
        raise RuntimeError(
            "TD block counts do "
            "not sum to the total "
            "number of updates."
        )

    td_block_mean = np.full(
        n_blocks,
        np.nan,
        dtype=np.float64,
    )

    q_block_mean = np.full(
        n_blocks,
        np.nan,
        dtype=np.float64,
    )

    loss_block_mean = np.full(
        n_blocks,
        np.nan,
        dtype=np.float64,
    )

    valid = (
        td_block_count
        > 0
    )

    td_block_mean[
        valid
    ] = (
        td_block_sum[
            valid
        ]
        / td_block_count[
            valid
        ]
    )

    q_block_mean[
        valid
    ] = (
        q_block_sum[
            valid
        ]
        / td_block_count[
            valid
        ]
    )

    loss_block_mean[
        valid
    ] = (
        loss_block_sum[
            valid
        ]
        / td_block_count[
            valid
        ]
    )

    td_block_steps = (
        np.arange(
            1,
            n_blocks + 1,
            dtype=np.int64,
        )
        * BLOCK_SIZE
    )

    eval_steps = np.asarray(
        eval_steps,
        dtype=np.int64,
    )

    eval_returns = np.asarray(
        eval_returns,
        dtype=np.float64,
    )

    if (
        eval_steps.size
        != cfg.n_eval_points
    ):
        raise RuntimeError(
            "Unexpected number of "
            "evaluation checkpoints: "
            f"expected "
            f"{cfg.n_eval_points}, "
            f"found "
            f"{eval_steps.size}."
        )

    if (
        eval_returns.size
        != cfg.n_eval_points
    ):
        raise RuntimeError(
            "Evaluation return "
            "count mismatch."
        )

    final_eval = float(
        eval_returns[
            -1
        ]
    )

    result = {
        "env_id": (
            cfg.env_id
        ),
        "algo": (
            "double-dqn"
        ),
        "explorer": (
            cfg.explorer
        ),
        "explorer_kwargs": dict(
            cfg.explorer_kwargs
        ),
        "seed": (
            cfg.seed
        ),
        "n_steps": (
            cfg.n_steps
        ),
        "gamma": (
            cfg.gamma
        ),
        "reward_scale": (
            cfg.reward_scale
        ),
        "hidden": (
            cfg.hidden
        ),
        "replay_capacity": (
            cfg.replay_capacity
        ),
        "batch_size": (
            cfg.batch_size
        ),
        "learning_rate": (
            cfg.learning_rate
        ),
        "learning_starts": (
            cfg.learning_starts
        ),
        "train_every": (
            cfg.train_every
        ),
        "target_update_every": (
            cfg.target_update_every
        ),
        "double": True,
        "block_size": (
            BLOCK_SIZE
        ),
        "td_block_steps": (
            td_block_steps
        ),
        "td_block_mean": (
            td_block_mean
        ),
        "td_block_count": (
            td_block_count
        ),
        "q_block_mean": (
            q_block_mean
        ),
        "loss_block_mean": (
            loss_block_mean
        ),
        "eval_steps": (
            eval_steps
        ),
        "eval_returns": (
            eval_returns
        ),
        "final_eval": (
            final_eval
        ),
        "total_steps": (
            total_steps
        ),
        "n_updates": (
            n_updates
        ),
        "replay_size": int(
            len(
                replay
            )
        ),
    }

    env.env.close()
    eval_env.env.close()

    return result


def _worker(
    cfg,
):
    key = _config_key(
        cfg
    )

    try:
        result = _run_one(
            cfg
        )

        result[
            "key"
        ] = key

        result[
            "config"
        ] = _config_dict(
            cfg
        )

        return result

    except Exception as exc:
        return {
            "key": (
                key
            ),
            "config": (
                _config_dict(
                    cfg
                )
            ),
            "error": (
                f"{type(exc).__name__}: "
                f"{exc}"
            ),
            "traceback": (
                traceback.format_exc()
            ),
        }


def main():
    decay_configs = (
        _load_decay_configs()
    )

    configs = []

    for env_id in ENV_IDS:
        for seed in DIAGNOSTIC_SEEDS:
            configs.append(
                DiagnosticDQNConfig(
                    env_id=env_id,
                    seed=seed,
                    n_steps=(
                        STEP_BUDGET[
                            env_id
                        ]
                    ),
                    explorer_kwargs=(
                        decay_configs[
                            env_id
                        ]
                    ),
                )
            )

    expected_total = (
        len(
            ENV_IDS
        )
        * len(
            DIAGNOSTIC_SEEDS
        )
    )

    if (
        len(configs)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected config count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(configs)}."
        )

    existing = {}

    if OUT_PATH.exists():
        with open(
            OUT_PATH,
            "rb",
        ) as f:
            loaded = pickle.load(
                f
            )

        if not isinstance(
            loaded,
            list,
        ):
            raise TypeError(
                "Existing DQN "
                "diagnostic results "
                "must be a list."
            )

        for result in loaded:
            key = result.get(
                "key"
            )

            if key is None:
                raise KeyError(
                    "Existing result "
                    "missing key."
                )

            existing[
                key
            ] = result

    pending = []

    for cfg in configs:
        key = _config_key(
            cfg
        )

        previous = (
            existing.get(
                key
            )
        )

        if (
            previous is None
            or "error"
            in previous
        ):
            pending.append(
                cfg
            )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(
                pending
            )
            if pending
            else 1,
            max(
                1,
                cpu_count
                // 2,
            ),
        ),
    )

    print(
        "DOUBLE DQN TD-ERROR "
        "DIAGNOSTIC"
    )

    print()

    print(
        "Environments and "
        "decay schedules:"
    )

    for env_id in ENV_IDS:
        print(
            f"  {env_id}: "
            f"{decay_configs[env_id]}"
        )

    print()

    print(
        f"Diagnostic seeds: "
        f"{DIAGNOSTIC_SEEDS}"
    )

    print(
        f"TD block size: "
        f"{BLOCK_SIZE} "
        "environment steps"
    )

    print(
        f"Evaluation points: "
        f"{N_EVAL_POINTS}"
    )

    print(
        f"Evaluation episodes: "
        f"{N_EVAL_EPISODES}"
    )

    print()

    print(
        "Double DQN:"
    )

    print(
        f"  hidden="
        f"{HIDDEN}"
    )

    print(
        f"  replay_capacity="
        f"{REPLAY_CAPACITY}"
    )

    print(
        f"  batch_size="
        f"{BATCH_SIZE}"
    )

    print(
        f"  learning_rate="
        f"{LEARNING_RATE}"
    )

    print(
        f"  learning_starts="
        f"{LEARNING_STARTS}"
    )

    print(
        f"  train_every="
        f"{TRAIN_EVERY}"
    )

    print(
        f"  target_update_every="
        f"{TARGET_UPDATE_EVERY}"
    )

    print(
        f"  gamma="
        f"{GAMMA}"
    )

    print()

    print(
        f"Total configs: "
        f"{expected_total}"
    )

    print(
        f"Loaded: "
        f"{len(existing)}"
    )

    print(
        f"Pending: "
        f"{len(pending)}"
    )

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    if pending:
        ctx = (
            mp.get_context(
                "spawn"
            )
        )

        with ctx.Pool(
            processes=workers
        ) as pool:
            iterator = (
                pool.imap_unordered(
                    _worker,
                    pending,
                    chunksize=1,
                )
            )

            for i, result in enumerate(
                iterator,
                start=1,
            ):
                existing[
                    result[
                        "key"
                    ]
                ] = result

                _save_atomic(
                    list(
                        existing.values()
                    ),
                    OUT_PATH,
                )

                status = (
                    "ERROR"
                    if "error"
                    in result
                    else "OK"
                )

                config = result[
                    "config"
                ]

                print(
                    f"{i}/"
                    f"{len(pending)} "
                    f"{status}  "
                    f"{config['env_id']}  "
                    f"seed="
                    f"{config['seed']}"
                )

    results = list(
        existing.values()
    )

    errors = [
        result
        for result in results
        if "error"
        in result
    ]

    successful = [
        result
        for result in results
        if "error"
        not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED DOUBLE DQN "
            "DIAGNOSTIC RUNS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            config = result[
                "config"
            ]

            print(
                f"{i}. "
                f"{config['env_id']} "
                f"seed="
                f"{config['seed']}"
            )

            print(
                f"   "
                f"{result['error']}"
            )

        raise RuntimeError(
            f"{len(errors)} "
            "Double DQN diagnostic "
            "runs failed."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Expected "
            f"{expected_total} "
            "successful runs, "
            f"found "
            f"{len(successful)}."
        )

    print()

    print(
        "Per-environment "
        "update counts:"
    )

    for env_id in ENV_IDS:
        env_results = [
            result
            for result
            in successful
            if result[
                "env_id"
            ]
            == env_id
        ]

        counts = {
            int(
                result[
                    "n_updates"
                ]
            )
            for result
            in env_results
        }

        print(
            f"  {env_id}: "
            f"{sorted(counts)}"
        )

    print()

    print(
        "DOUBLE DQN TD-ERROR "
        "DIAGNOSTIC COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/diagnose_td_error_dqn.py


Why this implementation is slightly different from the linear one

For the linear run, at training step \(t\) we effectively had one online TD error:

$$ \delta_t = R_{t+1} +\gamma Q(S_{t+1},A_{t+1}) -Q(S_t,A_t). $$

For Double DQN, one optimizer update samples \(B=64\) old transitions from replay and produces

$$ \delta_1,\delta_2,\ldots,\delta_{64}. $$

The diagnostic value from that optimizer update is

$D_u​=\frac{1}{64}\sum_{i=1}^{64} ​∣δi​∣$

In [11]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.diagnose_td_error_dqn",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Double DQN TD-error "
        "diagnostic failed."
    )

DOUBLE DQN TD-ERROR DIAGNOSTIC

Environments and decay schedules:
  MountainCar-v0: {'decay_steps': 30000, 'eps_end': 0.01, 'eps_start': 1.0, 'mode': 'linear'}
  CartPole-v1: {'decay_steps': 40000, 'eps_end': 0.01, 'eps_start': 1.0, 'mode': 'linear'}
  Acrobot-v1: {'decay_steps': 60000, 'eps_end': 0.01, 'eps_start': 1.0, 'mode': 'linear'}
  LunarLander-v3: {'decay_steps': 120000, 'eps_end': 0.01, 'eps_start': 1.0, 'mode': 'linear'}

Diagnostic seeds: (2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009)
TD block size: 1000 environment steps
Evaluation points: 20
Evaluation episodes: 10

Double DQN:
  hidden=128
  replay_capacity=50000
  batch_size=64
  learning_rate=0.001
  learning_starts=1000
  train_every=1
  target_update_every=1000
  gamma=1.0

Total configs: 40
Loaded: 0
Pending: 40
Detected CPUs: 20
Workers: 8
Output: src\results\diagnostics\td_error_trajectory_dqn.pkl

1/40 OK  MountainCar-v0  seed=2004
2/40 OK  MountainCar-v0  seed=2002
3/40 OK  MountainCar-v0  seed=200

In [12]:
%%writefile src/scripts/analyze_td_error_dqn.py
import csv
import json
import pickle

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    STEP_BUDGET,
)


RESULTS_PATH = Path(
    "src/results/diagnostics/"
    "td_error_trajectory_dqn.pkl"
)

SUMMARY_PATH = Path(
    "src/results/diagnostics/"
    "td_error_dqn_summary.json"
)

SEED_TABLE_PATH = Path(
    "src/results/diagnostics/"
    "td_error_dqn_seed_summary.csv"
)

DECILE_TABLE_PATH = Path(
    "src/results/diagnostics/"
    "td_error_dqn_deciles.csv"
)

TRAJECTORY_PATH = Path(
    "src/results/diagnostics/"
    "td_error_dqn_mean_trajectories.npz"
)

PLOT_DIR = Path(
    "src/results/diagnostics/"
    "td_error_dqn_plots"
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        2000,
        2010,
    )
)

EXPECTED_BLOCK_SIZE = 1000

EXPECTED_EVAL_POINTS = 20

WINDOW_FRACTION = 0.10

N_DECILES = 10


def _write_json_atomic(
    obj,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(
        path
    )


def _mean_std(
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    mean = float(
        np.mean(
            values
        )
    )

    if values.size >= 2:
        std = float(
            np.std(
                values,
                ddof=1,
            )
        )
    else:
        std = 0.0

    return mean, std


def _weighted_mean(
    values,
    counts,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    counts = np.asarray(
        counts,
        dtype=np.float64,
    )

    valid = (
        np.isfinite(
            values
        )
        & (
            counts
            > 0
        )
    )

    if not np.any(
        valid
    ):
        raise ValueError(
            "No valid weighted "
            "TD observations."
        )

    return float(
        np.sum(
            values[
                valid
            ]
            * counts[
                valid
            ]
        )
        / np.sum(
            counts[
                valid
            ]
        )
    )


def _weighted_deciles(
    values,
    counts,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    counts = np.asarray(
        counts,
        dtype=np.float64,
    )

    value_chunks = (
        np.array_split(
            values,
            N_DECILES,
        )
    )

    count_chunks = (
        np.array_split(
            counts,
            N_DECILES,
        )
    )

    output = []

    for value_chunk, count_chunk in zip(
        value_chunks,
        count_chunks,
    ):
        output.append(
            _weighted_mean(
                value_chunk,
                count_chunk,
            )
        )

    return np.asarray(
        output,
        dtype=np.float64,
    )


def _ordinary_deciles(
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    chunks = np.array_split(
        values,
        N_DECILES,
    )

    return np.asarray(
        [
            np.mean(
                chunk
            )
            for chunk
            in chunks
        ],
        dtype=np.float64,
    )


def _validate_result(
    result,
):
    required = (
        "env_id",
        "algo",
        "explorer",
        "seed",
        "n_steps",
        "block_size",
        "td_block_steps",
        "td_block_mean",
        "td_block_count",
        "q_block_mean",
        "loss_block_mean",
        "eval_steps",
        "eval_returns",
        "final_eval",
        "total_steps",
        "n_updates",
    )

    for field in required:
        if field not in result:
            raise KeyError(
                "DQN diagnostic result "
                f"missing {field}."
            )

    env_id = result[
        "env_id"
    ]

    seed = int(
        result[
            "seed"
        ]
    )

    if env_id not in ENV_IDS:
        raise ValueError(
            "Unexpected environment: "
            f"{env_id}"
        )

    if seed not in DIAGNOSTIC_SEEDS:
        raise ValueError(
            "Unexpected diagnostic "
            f"seed: {seed}"
        )

    if (
        result[
            "algo"
        ]
        != "double-dqn"
    ):
        raise ValueError(
            "Expected Double DQN, "
            f"found "
            f"{result['algo']}."
        )

    if (
        result[
            "explorer"
        ]
        != "decay"
    ):
        raise ValueError(
            "Experiment A must use "
            "the decay explorer."
        )

    n_steps = int(
        result[
            "n_steps"
        ]
    )

    if (
        n_steps
        != STEP_BUDGET[
            env_id
        ]
    ):
        raise ValueError(
            f"Step-budget mismatch "
            f"for {env_id}."
        )

    if (
        int(
            result[
                "block_size"
            ]
        )
        != EXPECTED_BLOCK_SIZE
    ):
        raise ValueError(
            "Unexpected TD block "
            f"size for {env_id}, "
            f"seed={seed}."
        )

    if (
        int(
            result[
                "total_steps"
            ]
        )
        != n_steps
    ):
        raise ValueError(
            "total_steps mismatch "
            f"for {env_id}, "
            f"seed={seed}."
        )

    expected_updates = (
        n_steps
        - int(
            result.get(
                "learning_starts",
                1000,
            )
        )
        + 1
    )

    if (
        int(
            result[
                "n_updates"
            ]
        )
        != expected_updates
    ):
        raise ValueError(
            "Unexpected update count "
            f"for {env_id}, "
            f"seed={seed}: "
            f"expected "
            f"{expected_updates}, "
            f"found "
            f"{result['n_updates']}."
        )

    td_steps = np.asarray(
        result[
            "td_block_steps"
        ],
        dtype=np.int64,
    )

    td_values = np.asarray(
        result[
            "td_block_mean"
        ],
        dtype=np.float64,
    )

    td_counts = np.asarray(
        result[
            "td_block_count"
        ],
        dtype=np.int64,
    )

    expected_blocks = (
        n_steps
        // EXPECTED_BLOCK_SIZE
    )

    if (
        td_steps.size
        != expected_blocks
    ):
        raise ValueError(
            "TD step count mismatch "
            f"for {env_id}, "
            f"seed={seed}."
        )

    if (
        td_values.size
        != expected_blocks
    ):
        raise ValueError(
            "TD value count mismatch "
            f"for {env_id}, "
            f"seed={seed}."
        )

    if (
        td_counts.size
        != expected_blocks
    ):
        raise ValueError(
            "TD update-count array "
            f"mismatch for {env_id}, "
            f"seed={seed}."
        )

    if (
        int(
            td_counts.sum()
        )
        != int(
            result[
                "n_updates"
            ]
        )
    ):
        raise ValueError(
            "TD block counts do not "
            "sum to n_updates for "
            f"{env_id}, seed={seed}."
        )

    if (
        td_counts[
            0
        ]
        != 1
    ):
        raise ValueError(
            "Expected exactly one "
            "optimizer update in "
            "the first TD block for "
            f"{env_id}, seed={seed}; "
            f"found "
            f"{td_counts[0]}."
        )

    if not np.all(
        td_counts[
            1:
        ]
        == EXPECTED_BLOCK_SIZE
    ):
        raise ValueError(
            "Expected 1000 updates "
            "in every post-warmup "
            f"block for {env_id}, "
            f"seed={seed}."
        )

    if not np.all(
        np.isfinite(
            td_values
        )
    ):
        raise ValueError(
            "Non-finite TD values "
            f"for {env_id}, "
            f"seed={seed}."
        )

    eval_steps = np.asarray(
        result[
            "eval_steps"
        ],
        dtype=np.int64,
    )

    eval_returns = np.asarray(
        result[
            "eval_returns"
        ],
        dtype=np.float64,
    )

    if (
        eval_steps.size
        != EXPECTED_EVAL_POINTS
    ):
        raise ValueError(
            "Unexpected evaluation "
            f"count for {env_id}, "
            f"seed={seed}."
        )

    if (
        eval_returns.size
        != EXPECTED_EVAL_POINTS
    ):
        raise ValueError(
            "Evaluation-return "
            f"count mismatch for "
            f"{env_id}, seed={seed}."
        )

    if not np.all(
        np.isfinite(
            eval_returns
        )
    ):
        raise ValueError(
            "Non-finite evaluation "
            f"returns for {env_id}, "
            f"seed={seed}."
        )


def _save_seed_csv(
    rows,
):
    SEED_TABLE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fields = (
        "env_id",
        "seed",
        "early_td",
        "late_td",
        "td_ratio",
        "td_percent_change",
        "early_greedy_return",
        "late_greedy_return",
        "greedy_return_change",
        "td_increased",
        "greedy_improved",
        "inversion_observed",
    )

    with open(
        SEED_TABLE_PATH,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fields,
        )

        writer.writeheader()

        writer.writerows(
            rows
        )


def _save_decile_csv(
    rows,
):
    DECILE_TABLE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fields = (
        "env_id",
        "decile",
        "progress_percent",
        "td_mean",
        "td_std",
        "greedy_mean",
        "greedy_std",
    )

    with open(
        DECILE_TABLE_PATH,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fields,
        )

        writer.writeheader()

        writer.writerows(
            rows
        )


def _plot(
    env_id,
    suffix,
    progress,
    mean_values,
    std_values,
    ylabel,
    title,
):
    PLOT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    fig = plt.figure(
        figsize=(
            8,
            5,
        )
    )

    ax = fig.add_subplot(
        111
    )

    ax.plot(
        progress,
        mean_values,
    )

    ax.fill_between(
        progress,
        mean_values
        - std_values,
        mean_values
        + std_values,
        alpha=0.2,
    )

    ax.set_xlabel(
        "Training progress"
    )

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        title
    )

    ax.grid(
        alpha=0.2
    )

    fig.tight_layout()

    filename = (
        env_id
        .replace(
            "-",
            "_"
        )
        + suffix
    )

    fig.savefig(
        PLOT_DIR
        / filename,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            "Double DQN TD-error "
            "diagnostic results "
            "not found: "
            f"{RESULTS_PATH}"
        )

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(
            f
        )

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Results must be "
            "stored as a list."
        )

    expected_total = (
        len(
            ENV_IDS
        )
        * len(
            DIAGNOSTIC_SEEDS
        )
    )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected result count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    errors = [
        result
        for result in results
        if "error"
        in result
    ]

    if errors:
        raise RuntimeError(
            "DQN diagnostic file "
            f"contains "
            f"{len(errors)} "
            "error records."
        )

    grouped = {
        env_id: {}
        for env_id
        in ENV_IDS
    }

    for result in results:
        _validate_result(
            result
        )

        env_id = result[
            "env_id"
        ]

        seed = int(
            result[
                "seed"
            ]
        )

        if (
            seed
            in grouped[
                env_id
            ]
        ):
            raise RuntimeError(
                "Duplicate DQN "
                f"diagnostic result "
                f"for {env_id}, "
                f"seed={seed}."
            )

        grouped[
            env_id
        ][
            seed
        ] = result

    expected_seeds = set(
        DIAGNOSTIC_SEEDS
    )

    for env_id in ENV_IDS:
        if (
            set(
                grouped[
                    env_id
                ]
            )
            != expected_seeds
        ):
            raise RuntimeError(
                "Seed mismatch for "
                f"{env_id}."
            )

    summary = {}
    seed_rows = []
    decile_rows = []
    trajectory_data = {}

    print(
        "DOUBLE DQN TD-ERROR "
        "DIAGNOSTIC ANALYSIS"
    )

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Seeds per environment: "
        f"{len(DIAGNOSTIC_SEEDS)}"
    )

    print(
        "TD summaries are weighted "
        "by optimizer-update count."
    )

    print()

    for env_id in ENV_IDS:
        env_results = [
            grouped[
                env_id
            ][
                seed
            ]
            for seed
            in DIAGNOSTIC_SEEDS
        ]

        n_steps = int(
            STEP_BUDGET[
                env_id
            ]
        )

        n_blocks = (
            n_steps
            // EXPECTED_BLOCK_SIZE
        )

        window_blocks = max(
            1,
            int(
                round(
                    n_blocks
                    * WINDOW_FRACTION
                )
            ),
        )

        window_evals = max(
            1,
            int(
                round(
                    EXPECTED_EVAL_POINTS
                    * WINDOW_FRACTION
                )
            ),
        )

        td_matrix = np.stack(
            [
                np.asarray(
                    result[
                        "td_block_mean"
                    ],
                    dtype=np.float64,
                )
                for result
                in env_results
            ],
            axis=0,
        )

        count_matrix = np.stack(
            [
                np.asarray(
                    result[
                        "td_block_count"
                    ],
                    dtype=np.float64,
                )
                for result
                in env_results
            ],
            axis=0,
        )

        eval_matrix = np.stack(
            [
                np.asarray(
                    result[
                        "eval_returns"
                    ],
                    dtype=np.float64,
                )
                for result
                in env_results
            ],
            axis=0,
        )

        td_steps = np.asarray(
            env_results[
                0
            ][
                "td_block_steps"
            ],
            dtype=np.float64,
        )

        eval_steps = np.asarray(
            env_results[
                0
            ][
                "eval_steps"
            ],
            dtype=np.float64,
        )

        td_progress = (
            td_steps
            / n_steps
        )

        eval_progress = (
            eval_steps
            / n_steps
        )

        td_curve_mean = np.mean(
            td_matrix,
            axis=0,
        )

        td_curve_std = np.std(
            td_matrix,
            axis=0,
            ddof=1,
        )

        eval_curve_mean = np.mean(
            eval_matrix,
            axis=0,
        )

        eval_curve_std = np.std(
            eval_matrix,
            axis=0,
            ddof=1,
        )

        per_seed_early_td = []
        per_seed_late_td = []

        per_seed_early_return = []
        per_seed_late_return = []

        td_increase_flags = []
        return_improve_flags = []
        inversion_flags = []

        seed_td_deciles = []
        seed_eval_deciles = []

        for result in env_results:
            td = np.asarray(
                result[
                    "td_block_mean"
                ],
                dtype=np.float64,
            )

            counts = np.asarray(
                result[
                    "td_block_count"
                ],
                dtype=np.float64,
            )

            returns = np.asarray(
                result[
                    "eval_returns"
                ],
                dtype=np.float64,
            )

            early_td = _weighted_mean(
                td[
                    :window_blocks
                ],
                counts[
                    :window_blocks
                ],
            )

            late_td = _weighted_mean(
                td[
                    -window_blocks:
                ],
                counts[
                    -window_blocks:
                ],
            )

            early_return = float(
                np.mean(
                    returns[
                        :window_evals
                    ]
                )
            )

            late_return = float(
                np.mean(
                    returns[
                        -window_evals:
                    ]
                )
            )

            td_ratio = (
                late_td
                / early_td
            )

            td_percent = (
                100.0
                * (
                    td_ratio
                    - 1.0
                )
            )

            return_change = (
                late_return
                - early_return
            )

            td_increased = bool(
                late_td
                > early_td
            )

            greedy_improved = bool(
                late_return
                > early_return
            )

            inversion = bool(
                td_increased
                and greedy_improved
            )

            per_seed_early_td.append(
                early_td
            )

            per_seed_late_td.append(
                late_td
            )

            per_seed_early_return.append(
                early_return
            )

            per_seed_late_return.append(
                late_return
            )

            td_increase_flags.append(
                td_increased
            )

            return_improve_flags.append(
                greedy_improved
            )

            inversion_flags.append(
                inversion
            )

            seed_td_deciles.append(
                _weighted_deciles(
                    td,
                    counts,
                )
            )

            seed_eval_deciles.append(
                _ordinary_deciles(
                    returns
                )
            )

            seed_rows.append(
                {
                    "env_id": (
                        env_id
                    ),
                    "seed": int(
                        result[
                            "seed"
                        ]
                    ),
                    "early_td": (
                        early_td
                    ),
                    "late_td": (
                        late_td
                    ),
                    "td_ratio": (
                        td_ratio
                    ),
                    "td_percent_change": (
                        td_percent
                    ),
                    "early_greedy_return": (
                        early_return
                    ),
                    "late_greedy_return": (
                        late_return
                    ),
                    "greedy_return_change": (
                        return_change
                    ),
                    "td_increased": (
                        td_increased
                    ),
                    "greedy_improved": (
                        greedy_improved
                    ),
                    "inversion_observed": (
                        inversion
                    ),
                }
            )

        early_td_mean, early_td_std = (
            _mean_std(
                per_seed_early_td
            )
        )

        late_td_mean, late_td_std = (
            _mean_std(
                per_seed_late_td
            )
        )

        early_return_mean, early_return_std = (
            _mean_std(
                per_seed_early_return
            )
        )

        late_return_mean, late_return_std = (
            _mean_std(
                per_seed_late_return
            )
        )

        td_ratio = (
            late_td_mean
            / early_td_mean
        )

        td_percent_change = (
            100.0
            * (
                td_ratio
                - 1.0
            )
        )

        return_change = (
            late_return_mean
            - early_return_mean
        )

        td_decile_matrix = np.stack(
            seed_td_deciles,
            axis=0,
        )

        eval_decile_matrix = np.stack(
            seed_eval_deciles,
            axis=0,
        )

        td_decile_mean = np.mean(
            td_decile_matrix,
            axis=0,
        )

        td_decile_std = np.std(
            td_decile_matrix,
            axis=0,
            ddof=1,
        )

        eval_decile_mean = np.mean(
            eval_decile_matrix,
            axis=0,
        )

        eval_decile_std = np.std(
            eval_decile_matrix,
            axis=0,
            ddof=1,
        )

        min_index = int(
            np.argmin(
                td_decile_mean
            )
        )

        min_td = float(
            td_decile_mean[
                min_index
            ]
        )

        final_td = float(
            td_decile_mean[
                -1
            ]
        )

        rebound_ratio = (
            final_td
            / min_td
        )

        rebound_percent = (
            100.0
            * (
                rebound_ratio
                - 1.0
            )
        )

        first_half_x = np.arange(
            5,
            55,
            10,
            dtype=np.float64,
        )

        second_half_x = np.arange(
            55,
            105,
            10,
            dtype=np.float64,
        )

        first_half_slope = float(
            np.polyfit(
                first_half_x,
                td_decile_mean[
                    :5
                ],
                1,
            )[0]
        )

        second_half_slope = float(
            np.polyfit(
                second_half_x,
                td_decile_mean[
                    5:
                ],
                1,
            )[0]
        )

        aggregate_td_increased = bool(
            late_td_mean
            > early_td_mean
        )

        aggregate_return_improved = bool(
            late_return_mean
            > early_return_mean
        )

        aggregate_inversion = bool(
            aggregate_td_increased
            and aggregate_return_improved
        )

        post_minimum_growth = bool(
            min_index
            < N_DECILES - 1
            and final_td
            > min_td
        )

        late_td_trend_upward = bool(
            second_half_slope
            > 0
        )

        summary[
            env_id
        ] = {
            "n_seeds": (
                len(
                    DIAGNOSTIC_SEEDS
                )
            ),
            "n_steps": (
                n_steps
            ),
            "n_updates": int(
                env_results[
                    0
                ][
                    "n_updates"
                ]
            ),
            "early_td_mean": (
                early_td_mean
            ),
            "early_td_std": (
                early_td_std
            ),
            "late_td_mean": (
                late_td_mean
            ),
            "late_td_std": (
                late_td_std
            ),
            "td_ratio": (
                td_ratio
            ),
            "td_percent_change": (
                td_percent_change
            ),
            "early_greedy_return_mean": (
                early_return_mean
            ),
            "early_greedy_return_std": (
                early_return_std
            ),
            "late_greedy_return_mean": (
                late_return_mean
            ),
            "late_greedy_return_std": (
                late_return_std
            ),
            "greedy_return_change": (
                return_change
            ),
            "td_increased": (
                aggregate_td_increased
            ),
            "greedy_improved": (
                aggregate_return_improved
            ),
            "inversion_observed": (
                aggregate_inversion
            ),
            "seeds_td_increased": int(
                np.sum(
                    td_increase_flags
                )
            ),
            "seeds_greedy_improved": int(
                np.sum(
                    return_improve_flags
                )
            ),
            "seeds_inversion_observed": int(
                np.sum(
                    inversion_flags
                )
            ),
            "td_decile_mean": (
                td_decile_mean.tolist()
            ),
            "td_decile_std": (
                td_decile_std.tolist()
            ),
            "greedy_decile_mean": (
                eval_decile_mean.tolist()
            ),
            "greedy_decile_std": (
                eval_decile_std.tolist()
            ),
            "minimum_td_decile": (
                min_index + 1
            ),
            "minimum_td_progress_percent": (
                (
                    min_index
                    + 1
                )
                * 10
            ),
            "minimum_td": (
                min_td
            ),
            "final_decile_td": (
                final_td
            ),
            "post_minimum_rebound_ratio": (
                rebound_ratio
            ),
            "post_minimum_rebound_percent": (
                rebound_percent
            ),
            "first_half_td_slope": (
                first_half_slope
            ),
            "second_half_td_slope": (
                second_half_slope
            ),
            "late_td_trend_upward": (
                late_td_trend_upward
            ),
            "post_minimum_growth": (
                post_minimum_growth
            ),
        }

        trajectory_data[
            f"{env_id}_td_progress"
        ] = td_progress

        trajectory_data[
            f"{env_id}_td_mean"
        ] = td_curve_mean

        trajectory_data[
            f"{env_id}_td_std"
        ] = td_curve_std

        trajectory_data[
            f"{env_id}_eval_progress"
        ] = eval_progress

        trajectory_data[
            f"{env_id}_eval_mean"
        ] = eval_curve_mean

        trajectory_data[
            f"{env_id}_eval_std"
        ] = eval_curve_std

        for i in range(
            N_DECILES
        ):
            decile_rows.append(
                {
                    "env_id": (
                        env_id
                    ),
                    "decile": (
                        i + 1
                    ),
                    "progress_percent": (
                        (
                            i + 1
                        )
                        * 10
                    ),
                    "td_mean": float(
                        td_decile_mean[
                            i
                        ]
                    ),
                    "td_std": float(
                        td_decile_std[
                            i
                        ]
                    ),
                    "greedy_mean": float(
                        eval_decile_mean[
                            i
                        ]
                    ),
                    "greedy_std": float(
                        eval_decile_std[
                            i
                        ]
                    ),
                }
            )

        _plot(
            env_id,
            "_dqn_td_error.png",
            td_progress,
            td_curve_mean,
            td_curve_std,
            "Mean absolute minibatch TD error",
            (
                f"{env_id}: "
                "Double DQN TD-error trajectory"
            ),
        )

        _plot(
            env_id,
            "_dqn_greedy_return.png",
            eval_progress,
            eval_curve_mean,
            eval_curve_std,
            "Greedy evaluation return",
            (
                f"{env_id}: "
                "Double DQN greedy performance"
            ),
        )

        print(
            "=" * 82
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 82
        )

        print()

        print(
            "Weighted mean |delta|:"
        )

        print(
            f"  first 10%: "
            f"{early_td_mean:.6f} "
            f"+/- "
            f"{early_td_std:.6f}"
        )

        print(
            f"  last 10%:  "
            f"{late_td_mean:.6f} "
            f"+/- "
            f"{late_td_std:.6f}"
        )

        print(
            f"  ratio:     "
            f"{td_ratio:.6f}"
        )

        print(
            f"  change:    "
            f"{td_percent_change:+.2f}%"
        )

        print()

        print(
            "Greedy return:"
        )

        print(
            f"  first 10%: "
            f"{early_return_mean:.3f} "
            f"+/- "
            f"{early_return_std:.3f}"
        )

        print(
            f"  last 10%:  "
            f"{late_return_mean:.3f} "
            f"+/- "
            f"{late_return_std:.3f}"
        )

        print(
            f"  change:    "
            f"{return_change:+.3f}"
        )

        print()

        print(
            "Seed-level counts:"
        )

        print(
            "  TD error increased: "
            f"{np.sum(td_increase_flags)}/"
            f"{len(DIAGNOSTIC_SEEDS)}"
        )

        print(
            "  Greedy return improved: "
            f"{np.sum(return_improve_flags)}/"
            f"{len(DIAGNOSTIC_SEEDS)}"
        )

        print(
            "  Both occurred: "
            f"{np.sum(inversion_flags)}/"
            f"{len(DIAGNOSTIC_SEEDS)}"
        )

        print()

        print(
            "Decile trajectory:"
        )

        print(
            "Progress    "
            "Mean |delta|       "
            "Greedy return"
        )

        print(
            "-" * 60
        )

        for i in range(
            N_DECILES
        ):
            print(
                f"{(i + 1) * 10:>3}%        "
                f"{td_decile_mean[i]:>10.6f} "
                f"+/- "
                f"{td_decile_std[i]:<10.6f}   "
                f"{eval_decile_mean[i]:>9.3f} "
                f"+/- "
                f"{eval_decile_std[i]:.3f}"
            )

        print()

        print(
            "Minimum TD-error "
            f"decile: "
            f"{min_index + 1} "
            f"({(min_index + 1) * 10}% "
            "progress)"
        )

        print(
            "Minimum mean |delta|: "
            f"{min_td:.6f}"
        )

        print(
            "Final-decile mean "
            f"|delta|: "
            f"{final_td:.6f}"
        )

        print(
            "Minimum -> final "
            f"change: "
            f"{rebound_percent:+.2f}%"
        )

        print()

        print(
            "TD trend slope:"
        )

        print(
            f"  first half:  "
            f"{first_half_slope:+.6f}"
        )

        print(
            f"  second half: "
            f"{second_half_slope:+.6f}"
        )

        print()

        print(
            "Late TD trend upward: "
            f"{late_td_trend_upward}"
        )

        print(
            "TD grows after its "
            "minimum: "
            f"{post_minimum_growth}"
        )

        print()

        if aggregate_inversion:
            print(
                "FIRST-VS-LAST RESULT: "
                "SUPPORTS TD-ERROR "
                "INVERSION"
            )

        elif aggregate_return_improved:
            print(
                "FIRST-VS-LAST RESULT: "
                "PERFORMANCE IMPROVED "
                "WITHOUT AN OVERALL "
                "TD-ERROR INCREASE"
            )

        else:
            print(
                "FIRST-VS-LAST RESULT: "
                "GREEDY PERFORMANCE "
                "DID NOT IMPROVE"
            )

        print()
        print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    _save_seed_csv(
        seed_rows
    )

    _save_decile_csv(
        decile_rows
    )

    TRAJECTORY_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez(
        TRAJECTORY_PATH,
        **trajectory_data,
    )

    print(
        "=" * 82
    )

    print(
        "DOUBLE DQN EXPERIMENT-A "
        "SUMMARY"
    )

    print(
        "=" * 82
    )

    print()

    print(
        "Environment          "
        "TD change     "
        "Return change     "
        "Inversion   "
        "Late up"
    )

    print(
        "-" * 82
    )

    for env_id in ENV_IDS:
        item = summary[
            env_id
        ]

        print(
            f"{env_id:<20} "
            f"{item['td_percent_change']:>+9.2f}%   "
            f"{item['greedy_return_change']:>+12.3f}   "
            f"{str(item['inversion_observed']):<10}  "
            f"{item['late_td_trend_upward']}"
        )

    print()

    print(
        "Summary JSON:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    print(
        "Seed-level CSV:"
    )

    print(
        SEED_TABLE_PATH
    )

    print()

    print(
        "Decile CSV:"
    )

    print(
        DECILE_TABLE_PATH
    )

    print()

    print(
        "Trajectory archive:"
    )

    print(
        TRAJECTORY_PATH
    )

    print()

    print(
        "Plots:"
    )

    print(
        PLOT_DIR
    )

    print()

    print(
        "DOUBLE DQN TD-ERROR "
        "ANALYSIS COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/analyze_td_error_dqn.py


In [13]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.analyze_td_error_dqn",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
DOUBLE DQN TD-ERROR DIAGNOSTIC ANALYSIS

Stored results: 40
Seeds per environment: 10
TD summaries are weighted by optimizer-update count.

Environment: MountainCar-v0

Weighted mean |delta|:
  first 10%: 0.018599 +/- 0.002705
  last 10%:  0.034909 +/- 0.049550
  ratio:     1.876930
  change:    +87.69%

Greedy return:
  first 10%: -200.000 +/- 0.000
  last 10%:  -199.255 +/- 2.356
  change:    +0.745

Seed-level counts:
  TD error increased: 5/10
  Greedy return improved: 1/10
  Both occurred: 1/10

Decile trajectory:
Progress    Mean |delta|       Greedy return
------------------------------------------------------------
 10%          0.018599 +/- 0.002705      -200.000 +/- 0.000
 20%          0.013545 +/- 0.000295      -200.000 +/- 0.000
 30%          0.014244 +/- 0.000656      -200.000 +/- 0.000
 40%          0.015398 +/- 0.002371      -200.000 +/- 0.000
 50%          0.016205 +/- 0.003560      -200.000 +/- 0.000
 60%          0.018672 +/- 0.009073      -200

In [14]:
%%writefile src/scripts/tune_dqn_backbone_decay.py
import os

from pathlib import Path

from src.dqn import DQNConfig
from src.sweep import run_stage
from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    STEP_BUDGET,
)


OUT_PATH = Path(
    "src/results/tuning/"
    "dqn_backbone_decay.pkl"
)

LEARNING_RATE_GRID = (
    3e-4,
    1e-3,
    3e-3,
)

DECAY_HORIZON_FRACTIONS = (
    0.2,
    0.4,
    0.6,
)

EPS_START = 1.0
EPS_END = 0.01
DECAY_MODE = "linear"

HIDDEN = 128
REPLAY_CAPACITY = 50_000
BATCH_SIZE = 64

LEARNING_STARTS = 1_000
TRAIN_EVERY = 1
TARGET_UPDATE_EVERY = 1_000

GAMMA = 1.0
REWARD_SCALE = 1.0

N_BINS = 100
N_EVAL_POINTS = 20
N_EVAL_EPISODES = 10

MAX_WORKERS = 8


def main():
    configs = []

    for env_id in ENV_IDS:
        budget = int(
            STEP_BUDGET[
                env_id
            ]
        )

        for learning_rate in LEARNING_RATE_GRID:
            for fraction in DECAY_HORIZON_FRACTIONS:
                decay_steps = int(
                    round(
                        fraction
                        * budget
                    )
                )

                for seed in TUNING_SEEDS:
                    configs.append(
                        DQNConfig(
                            env_id=env_id,
                            explorer="decay",
                            explorer_kwargs={
                                "eps_start": (
                                    EPS_START
                                ),
                                "eps_end": (
                                    EPS_END
                                ),
                                "decay_steps": (
                                    decay_steps
                                ),
                                "mode": (
                                    DECAY_MODE
                                ),
                            },
                            seed=int(
                                seed
                            ),
                            n_steps=budget,
                            gamma=GAMMA,
                            reward_scale=(
                                REWARD_SCALE
                            ),
                            hidden=HIDDEN,
                            replay_capacity=(
                                REPLAY_CAPACITY
                            ),
                            batch_size=(
                                BATCH_SIZE
                            ),
                            learning_rate=float(
                                learning_rate
                            ),
                            learning_starts=(
                                LEARNING_STARTS
                            ),
                            train_every=(
                                TRAIN_EVERY
                            ),
                            target_update_every=(
                                TARGET_UPDATE_EVERY
                            ),
                            double=True,
                            n_bins=N_BINS,
                            n_eval_points=(
                                N_EVAL_POINTS
                            ),
                            n_eval_episodes=(
                                N_EVAL_EPISODES
                            ),
                        )
                    )

    expected_total = (
        len(
            ENV_IDS
        )
        * len(
            LEARNING_RATE_GRID
        )
        * len(
            DECAY_HORIZON_FRACTIONS
        )
        * len(
            TUNING_SEEDS
        )
    )

    if (
        len(configs)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected DQN tuning "
            "configuration count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(configs)}."
        )

    seen = set()

    for cfg in configs:
        fraction = (
            cfg.explorer_kwargs[
                "decay_steps"
            ]
            / cfg.n_steps
        )

        key = (
            cfg.env_id,
            float(
                cfg.learning_rate
            ),
            float(
                fraction
            ),
            int(
                cfg.seed
            ),
        )

        if key in seen:
            raise RuntimeError(
                "Duplicate DQN tuning "
                f"configuration: "
                f"{key}"
            )

        seen.add(
            key
        )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(
                configs
            ),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "DOUBLE DQN BACKBONE "
        "+ DECAY TUNING"
    )

    print()

    print(
        "Environments:"
    )

    for env_id in ENV_IDS:
        print(
            f"  {env_id}: "
            f"budget="
            f"{STEP_BUDGET[env_id]}"
        )

    print()

    print(
        "Adam learning-rate grid:"
    )

    for value in LEARNING_RATE_GRID:
        print(
            f"  {value:g}"
        )

    print()

    print(
        "Decay-horizon grid:"
    )

    for fraction in DECAY_HORIZON_FRACTIONS:
        print(
            f"  {fraction:g} "
            f"x budget"
        )

    print()

    print(
        f"Tuning seeds: "
        f"{TUNING_SEEDS}"
    )

    print(
        f"Learning rates: "
        f"{len(LEARNING_RATE_GRID)}"
    )

    print(
        f"Decay horizons: "
        f"{len(DECAY_HORIZON_FRACTIONS)}"
    )

    print(
        f"Configurations per "
        f"environment: "
        f"{len(LEARNING_RATE_GRID) * len(DECAY_HORIZON_FRACTIONS)}"
    )

    print(
        f"Runs per environment: "
        f"{len(LEARNING_RATE_GRID) * len(DECAY_HORIZON_FRACTIONS) * len(TUNING_SEEDS)}"
    )

    print(
        f"Total runs: "
        f"{len(configs)}"
    )

    print()

    print(
        "Fixed Double DQN "
        "parameters:"
    )

    print(
        f"  hidden="
        f"{HIDDEN}"
    )

    print(
        f"  replay_capacity="
        f"{REPLAY_CAPACITY}"
    )

    print(
        f"  batch_size="
        f"{BATCH_SIZE}"
    )

    print(
        f"  learning_starts="
        f"{LEARNING_STARTS}"
    )

    print(
        f"  train_every="
        f"{TRAIN_EVERY}"
    )

    print(
        f"  target_update_every="
        f"{TARGET_UPDATE_EVERY}"
    )

    print(
        f"  gamma="
        f"{GAMMA}"
    )

    print(
        f"  eps_start="
        f"{EPS_START}"
    )

    print(
        f"  eps_end="
        f"{EPS_END}"
    )

    print(
        f"  decay_mode="
        f"{DECAY_MODE}"
    )

    print(
        f"  evaluation episodes="
        f"{N_EVAL_EPISODES}"
    )

    print()

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=OUT_PATH,
        label=(
            "DQN backbone+decay tuning"
        ),
        workers=workers,
        save_every=4,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error"
        in result
    ]

    successful = [
        result
        for result in results
        if "error"
        not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED DQN TUNING RUNS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            cfg = result.get(
                "config",
                {}
            )

            kwargs = cfg.get(
                "explorer_kwargs",
                {}
            )

            env_id = cfg.get(
                "env_id"
            )

            n_steps = cfg.get(
                "n_steps"
            )

            decay_steps = (
                kwargs.get(
                    "decay_steps"
                )
            )

            if (
                decay_steps is not None
                and n_steps
            ):
                fraction = (
                    float(
                        decay_steps
                    )
                    / float(
                        n_steps
                    )
                )
            else:
                fraction = None

            print(
                f"{i}. "
                f"env="
                f"{env_id} "
                f"lr="
                f"{cfg.get('learning_rate')} "
                f"horizon="
                f"{fraction} "
                f"seed="
                f"{cfg.get('seed')}"
            )

            print(
                f"   "
                f"{result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} "
            "Double DQN tuning "
            "runs failed."
        )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected stored "
            "result count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected successful "
            "result count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(successful)}."
        )

    print()

    print(
        "DOUBLE DQN BACKBONE "
        "+ DECAY TUNING COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/tune_dqn_backbone_decay.py


In [15]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.tune_dqn_backbone_decay",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Double DQN backbone + "
        "decay tuning failed."
    )

DOUBLE DQN BACKBONE + DECAY TUNING

Environments:
  MountainCar-v0: budget=150000
  CartPole-v1: budget=100000
  Acrobot-v1: budget=100000
  LunarLander-v3: budget=300000

Adam learning-rate grid:
  0.0003
  0.001
  0.003

Decay-horizon grid:
  0.2 x budget
  0.4 x budget
  0.6 x budget

Tuning seeds: (1000, 1001, 1002)
Learning rates: 3
Decay horizons: 3
Configurations per environment: 9
Runs per environment: 27
Total runs: 108

Fixed Double DQN parameters:
  hidden=128
  replay_capacity=50000
  batch_size=64
  learning_starts=1000
  train_every=1
  target_update_every=1000
  gamma=1.0
  eps_start=1.0
  eps_end=0.01
  decay_mode=linear
  evaluation episodes=10

Detected CPUs: 20
Workers: 8
Output: src\results\tuning\dqn_backbone_decay.pkl

[DQN backbone+decay tuning] loaded=0 pending=108 workers=8
[DQN backbone+decay tuning] 1/108 OK
[DQN backbone+decay tuning] 2/108 OK
[DQN backbone+decay tuning] 3/108 OK
[DQN backbone+decay tuning] 4/108 OK
[DQN backbone+decay tuning] 5/108 OK
[DQN 

In [16]:
%%writefile src/scripts/select_dqn_backbone_decay.py
import json
import pickle

from collections import defaultdict
from pathlib import Path

import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    STEP_BUDGET,
)


RESULTS_PATH = Path(
    "src/results/tuning/"
    "dqn_backbone_decay.pkl"
)

SUMMARY_PATH = Path(
    "src/results/tuning/"
    "dqn_backbone_decay_selection_summary.json"
)

PROVISIONAL_PATH = Path(
    "src/results/tuning/"
    "provisional_dqn_backbone_decay.json"
)

FINAL_PATH = Path(
    "src/results/tuning/"
    "selected_dqn_backbone_decay.json"
)

LEARNING_RATE_GRID = (
    3e-4,
    1e-3,
    3e-3,
)

DECAY_HORIZON_FRACTIONS = (
    0.2,
    0.4,
    0.6,
)

EXPECTED_EXPLORER = "decay"

EXPECTED_EPS_START = 1.0
EXPECTED_EPS_END = 0.01
EXPECTED_MODE = "linear"

EXPECTED_HIDDEN = 128
EXPECTED_REPLAY_CAPACITY = 50_000
EXPECTED_BATCH_SIZE = 64
EXPECTED_LEARNING_STARTS = 1_000
EXPECTED_TRAIN_EVERY = 1
EXPECTED_TARGET_UPDATE = 1_000

EXPECTED_GAMMA = 1.0
EXPECTED_REWARD_SCALE = 1.0
EXPECTED_DOUBLE = True


def _write_json_atomic(
    obj,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(
        path
    )


def _close(
    a,
    b,
    atol=1e-12,
):
    return bool(
        np.isclose(
            float(a),
            float(b),
            rtol=0.0,
            atol=atol,
        )
    )


def _validate_result(
    result,
):
    required = (
        "env_id",
        "algo",
        "explorer",
        "explorer_kwargs",
        "seed",
        "n_steps",
        "gamma",
        "reward_scale",
        "hidden",
        "replay_capacity",
        "batch_size",
        "learning_rate",
        "learning_starts",
        "train_every",
        "target_update_every",
        "double",
        "final_eval",
    )

    for name in required:
        if name not in result:
            raise KeyError(
                "Tuning result missing "
                f"{name}."
            )

    env_id = result[
        "env_id"
    ]

    if env_id not in ENV_IDS:
        raise ValueError(
            "Unexpected environment: "
            f"{env_id}"
        )

    if (
        result[
            "algo"
        ]
        != "double-dqn"
    ):
        raise ValueError(
            "Expected Double DQN, "
            f"found "
            f"{result['algo']}."
        )

    if (
        result[
            "explorer"
        ]
        != EXPECTED_EXPLORER
    ):
        raise ValueError(
            "Unexpected explorer for "
            f"{env_id}: "
            f"{result['explorer']}"
        )

    seed = int(
        result[
            "seed"
        ]
    )

    if seed not in TUNING_SEEDS:
        raise ValueError(
            "Unexpected tuning seed: "
            f"{seed}"
        )

    n_steps = int(
        result[
            "n_steps"
        ]
    )

    if (
        n_steps
        != int(
            STEP_BUDGET[
                env_id
            ]
        )
    ):
        raise ValueError(
            "Step-budget mismatch "
            f"for {env_id}."
        )

    kwargs = dict(
        result[
            "explorer_kwargs"
        ]
    )

    for name in (
        "eps_start",
        "eps_end",
        "decay_steps",
        "mode",
    ):
        if name not in kwargs:
            raise KeyError(
                "Decay configuration "
                f"missing {name}."
            )

    if not _close(
        kwargs[
            "eps_start"
        ],
        EXPECTED_EPS_START,
    ):
        raise ValueError(
            "Unexpected eps_start."
        )

    if not _close(
        kwargs[
            "eps_end"
        ],
        EXPECTED_EPS_END,
    ):
        raise ValueError(
            "Unexpected eps_end."
        )

    if (
        kwargs[
            "mode"
        ]
        != EXPECTED_MODE
    ):
        raise ValueError(
            "Unexpected decay mode."
        )

    fraction = (
        int(
            kwargs[
                "decay_steps"
            ]
        )
        / n_steps
    )

    if not any(
        _close(
            fraction,
            candidate,
        )
        for candidate
        in DECAY_HORIZON_FRACTIONS
    ):
        raise ValueError(
            "Unexpected decay "
            f"fraction: {fraction}"
        )

    lr = float(
        result[
            "learning_rate"
        ]
    )

    if not any(
        _close(
            lr,
            candidate,
        )
        for candidate
        in LEARNING_RATE_GRID
    ):
        raise ValueError(
            "Unexpected learning "
            f"rate: {lr}"
        )

    if (
        int(
            result[
                "hidden"
            ]
        )
        != EXPECTED_HIDDEN
    ):
        raise ValueError(
            "Unexpected hidden size."
        )

    if (
        int(
            result[
                "replay_capacity"
            ]
        )
        != EXPECTED_REPLAY_CAPACITY
    ):
        raise ValueError(
            "Unexpected replay capacity."
        )

    if (
        int(
            result[
                "batch_size"
            ]
        )
        != EXPECTED_BATCH_SIZE
    ):
        raise ValueError(
            "Unexpected batch size."
        )

    if (
        int(
            result[
                "learning_starts"
            ]
        )
        != EXPECTED_LEARNING_STARTS
    ):
        raise ValueError(
            "Unexpected learning_starts."
        )

    if (
        int(
            result[
                "train_every"
            ]
        )
        != EXPECTED_TRAIN_EVERY
    ):
        raise ValueError(
            "Unexpected train_every."
        )

    if (
        int(
            result[
                "target_update_every"
            ]
        )
        != EXPECTED_TARGET_UPDATE
    ):
        raise ValueError(
            "Unexpected target update."
        )

    if not _close(
        result[
            "gamma"
        ],
        EXPECTED_GAMMA,
    ):
        raise ValueError(
            "Unexpected gamma."
        )

    if not _close(
        result[
            "reward_scale"
        ],
        EXPECTED_REWARD_SCALE,
    ):
        raise ValueError(
            "Unexpected reward scale."
        )

    if (
        bool(
            result[
                "double"
            ]
        )
        != EXPECTED_DOUBLE
    ):
        raise ValueError(
            "Expected Double DQN."
        )

    final_eval = float(
        result[
            "final_eval"
        ]
    )

    if not np.isfinite(
        final_eval
    ):
        raise ValueError(
            "Non-finite final "
            f"evaluation for "
            f"{env_id}, seed={seed}."
        )

    return (
        env_id,
        lr,
        float(
            fraction
        ),
        seed,
        final_eval,
    )


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            "Missing DQN tuning file: "
            f"{RESULTS_PATH}"
        )

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(
            f
        )

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Tuning results must "
            "be a list."
        )

    expected_total = (
        len(
            ENV_IDS
        )
        * len(
            LEARNING_RATE_GRID
        )
        * len(
            DECAY_HORIZON_FRACTIONS
        )
        * len(
            TUNING_SEEDS
        )
    )

    print(
        "DOUBLE DQN BACKBONE "
        "+ DECAY SELECTION"
    )

    print()

    print(
        f"Stored records: "
        f"{len(results)}"
    )

    print(
        f"Expected records: "
        f"{expected_total}"
    )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected number of "
            "tuning records."
        )

    errors = [
        result
        for result in results
        if "error"
        in result
    ]

    if errors:
        raise RuntimeError(
            f"Found {len(errors)} "
            "error records."
        )

    grouped = defaultdict(
        dict
    )

    seen = set()

    for result in results:
        (
            env_id,
            lr,
            fraction,
            seed,
            final_eval,
        ) = _validate_result(
            result
        )

        key = (
            env_id,
            lr,
            fraction,
            seed,
        )

        if key in seen:
            raise RuntimeError(
                "Duplicate result: "
                f"{key}"
            )

        seen.add(
            key
        )

        grouped[
            (
                env_id,
                lr,
                fraction,
            )
        ][
            seed
        ] = final_eval

    if (
        len(seen)
        != expected_total
    ):
        raise RuntimeError(
            "Unique-cell count "
            "does not match expected "
            "run count."
        )

    expected_seed_set = set(
        TUNING_SEEDS
    )

    summary = {}
    provisional = {}

    environments_with_edges = []

    for env_id in ENV_IDS:
        cells = []

        for lr in LEARNING_RATE_GRID:
            for fraction in DECAY_HORIZON_FRACTIONS:
                matching_key = None

                for key in grouped:
                    (
                        key_env,
                        key_lr,
                        key_fraction,
                    ) = key

                    if (
                        key_env
                        == env_id
                        and _close(
                            key_lr,
                            lr,
                        )
                        and _close(
                            key_fraction,
                            fraction,
                        )
                    ):
                        matching_key = key
                        break

                if matching_key is None:
                    raise RuntimeError(
                        "Missing tuning cell: "
                        f"{env_id}, "
                        f"lr={lr}, "
                        f"horizon={fraction}"
                    )

                seed_scores = grouped[
                    matching_key
                ]

                if (
                    set(
                        seed_scores
                    )
                    != expected_seed_set
                ):
                    raise RuntimeError(
                        "Seed mismatch for "
                        f"{env_id}, "
                        f"lr={lr}, "
                        f"horizon={fraction}"
                    )

                scores = np.asarray(
                    [
                        seed_scores[
                            seed
                        ]
                        for seed
                        in TUNING_SEEDS
                    ],
                    dtype=np.float64,
                )

                mean_score = float(
                    np.mean(
                        scores
                    )
                )

                std_score = float(
                    np.std(
                        scores,
                        ddof=1,
                    )
                )

                cells.append(
                    {
                        "learning_rate": float(
                            lr
                        ),
                        "decay_fraction": float(
                            fraction
                        ),
                        "decay_steps": int(
                            round(
                                fraction
                                * STEP_BUDGET[
                                    env_id
                                ]
                            )
                        ),
                        "seed_scores": {
                            str(seed): float(
                                seed_scores[
                                    seed
                                ]
                            )
                            for seed
                            in TUNING_SEEDS
                        },
                        "mean_final_eval": (
                            mean_score
                        ),
                        "std_final_eval": (
                            std_score
                        ),
                    }
                )

        cells.sort(
            key=lambda item: (
                item[
                    "mean_final_eval"
                ],
                -item[
                    "std_final_eval"
                ],
            ),
            reverse=True,
        )

        winner = cells[
            0
        ]

        winner_lr = float(
            winner[
                "learning_rate"
            ]
        )

        winner_fraction = float(
            winner[
                "decay_fraction"
            ]
        )

        lr_low_edge = _close(
            winner_lr,
            min(
                LEARNING_RATE_GRID
            ),
        )

        lr_high_edge = _close(
            winner_lr,
            max(
                LEARNING_RATE_GRID
            ),
        )

        decay_low_edge = _close(
            winner_fraction,
            min(
                DECAY_HORIZON_FRACTIONS
            ),
        )

        decay_high_edge = _close(
            winner_fraction,
            max(
                DECAY_HORIZON_FRACTIONS
            ),
        )

        lr_edge = bool(
            lr_low_edge
            or lr_high_edge
        )

        decay_edge = bool(
            decay_low_edge
            or decay_high_edge
        )

        any_edge = bool(
            lr_edge
            or decay_edge
        )

        if any_edge:
            environments_with_edges.append(
                env_id
            )

        summary[
            env_id
        ] = {
            "cells_ranked": (
                cells
            ),
            "winner": (
                winner
            ),
            "learning_rate_edge": (
                lr_edge
            ),
            "learning_rate_low_edge": (
                lr_low_edge
            ),
            "learning_rate_high_edge": (
                lr_high_edge
            ),
            "decay_edge": (
                decay_edge
            ),
            "decay_low_edge": (
                decay_low_edge
            ),
            "decay_high_edge": (
                decay_high_edge
            ),
            "needs_extension": (
                any_edge
            ),
        }

        provisional[
            env_id
        ] = {
            "learning_rate": (
                winner_lr
            ),
            "decay_fraction": (
                winner_fraction
            ),
            "decay_steps": int(
                winner[
                    "decay_steps"
                ]
            ),
            "eps_start": (
                EXPECTED_EPS_START
            ),
            "eps_end": (
                EXPECTED_EPS_END
            ),
            "mode": (
                EXPECTED_MODE
            ),
            "mean_final_eval": float(
                winner[
                    "mean_final_eval"
                ]
            ),
            "std_final_eval": float(
                winner[
                    "std_final_eval"
                ]
            ),
            "needs_extension": (
                any_edge
            ),
        }

        print(
            "=" * 100
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 100
        )

        print()

        print(
            "Rank   LR         Horizon   "
            "Seed 1000    Seed 1001    "
            "Seed 1002    Mean         Std"
        )

        print(
            "-" * 100
        )

        for rank, cell in enumerate(
            cells,
            start=1,
        ):
            scores = cell[
                "seed_scores"
            ]

            print(
                f"{rank:<6}"
                f"{cell['learning_rate']:<11g}"
                f"{cell['decay_fraction']:<10.1f}"
                f"{scores[str(TUNING_SEEDS[0])]:>12.3f}"
                f"{scores[str(TUNING_SEEDS[1])]:>13.3f}"
                f"{scores[str(TUNING_SEEDS[2])]:>13.3f}"
                f"{cell['mean_final_eval']:>13.3f}"
                f"{cell['std_final_eval']:>12.3f}"
            )

        print()

        print(
            "WINNER:"
        )

        print(
            f"  learning_rate = "
            f"{winner_lr:g}"
        )

        print(
            f"  decay_fraction = "
            f"{winner_fraction:g}"
        )

        print(
            f"  decay_steps = "
            f"{winner['decay_steps']}"
        )

        print(
            f"  mean final greedy = "
            f"{winner['mean_final_eval']:.3f}"
        )

        print(
            f"  std = "
            f"{winner['std_final_eval']:.3f}"
        )

        print()

        print(
            "Boundary check:"
        )

        print(
            f"  learning-rate edge: "
            f"{lr_edge}"
        )

        if lr_low_edge:
            print(
                "    winner is at the "
                "LOW learning-rate edge"
            )

        if lr_high_edge:
            print(
                "    winner is at the "
                "HIGH learning-rate edge"
            )

        print(
            f"  decay-horizon edge: "
            f"{decay_edge}"
        )

        if decay_low_edge:
            print(
                "    winner is at the "
                "SHORT decay edge"
            )

        if decay_high_edge:
            print(
                "    winner is at the "
                "LONG decay edge"
            )

        print()

        if any_edge:
            print(
                "STATUS: PROVISIONAL — "
                "GRID EXTENSION REQUIRED"
            )

        else:
            print(
                "STATUS: INTERIOR WINNER"
            )

        print()
        print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    _write_json_atomic(
        provisional,
        PROVISIONAL_PATH,
    )

    if FINAL_PATH.exists():
        FINAL_PATH.unlink()

    print(
        "=" * 100
    )

    print(
        "OVERALL SELECTION STATUS"
    )

    print(
        "=" * 100
    )

    print()

    if environments_with_edges:
        print(
            "The following environments "
            "have at least one winner "
            "on a grid boundary:"
        )

        for env_id in environments_with_edges:
            item = summary[
                env_id
            ]

            labels = []

            if item[
                "learning_rate_low_edge"
            ]:
                labels.append(
                    "LR-low"
                )

            if item[
                "learning_rate_high_edge"
            ]:
                labels.append(
                    "LR-high"
                )

            if item[
                "decay_low_edge"
            ]:
                labels.append(
                    "decay-short"
                )

            if item[
                "decay_high_edge"
            ]:
                labels.append(
                    "decay-long"
                )

            print(
                f"  {env_id}: "
                f"{', '.join(labels)}"
            )

        print()

        print(
            "NO FINAL DQN BACKBONE "
            "CONFIGURATION HAS BEEN "
            "FROZEN YET."
        )

        print()

        print(
            "Only the affected "
            "environment/dimension(s) "
            "need extension."
        )

    else:
        final_selected = {
            env_id: {
                "learning_rate": float(
                    provisional[
                        env_id
                    ][
                        "learning_rate"
                    ]
                ),
                "decay_fraction": float(
                    provisional[
                        env_id
                    ][
                        "decay_fraction"
                    ]
                ),
                "decay_steps": int(
                    provisional[
                        env_id
                    ][
                        "decay_steps"
                    ]
                ),
                "eps_start": (
                    EXPECTED_EPS_START
                ),
                "eps_end": (
                    EXPECTED_EPS_END
                ),
                "mode": (
                    EXPECTED_MODE
                ),
            }
            for env_id
            in ENV_IDS
        }

        _write_json_atomic(
            final_selected,
            FINAL_PATH,
        )

        print(
            "All four winners are "
            "interior to both grids."
        )

        print()

        print(
            "FINAL DQN BACKBONE + "
            "DECAY CONFIGURATIONS "
            "FROZEN."
        )

        print()

        print(
            "Final selection:"
        )

        print(
            FINAL_PATH
        )

    print()

    print(
        "Selection summary:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    print(
        "Provisional winners:"
    )

    print(
        PROVISIONAL_PATH
    )

    print()

    print(
        "DOUBLE DQN BACKBONE "
        "+ DECAY SELECTION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/select_dqn_backbone_decay.py


In [17]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.select_dqn_backbone_decay",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
DOUBLE DQN BACKBONE + DECAY SELECTION

Stored records: 108
Expected records: 108
Environment: MountainCar-v0

Rank   LR         Horizon   Seed 1000    Seed 1001    Seed 1002    Mean         Std
----------------------------------------------------------------------------------------------------
1     0.0003     0.2           -200.000     -130.100     -200.000     -176.700      40.357
2     0.0003     0.4           -200.000     -200.000     -200.000     -200.000       0.000
3     0.0003     0.6           -200.000     -200.000     -200.000     -200.000       0.000
4     0.001      0.2           -200.000     -200.000     -200.000     -200.000       0.000
5     0.001      0.4           -200.000     -200.000     -200.000     -200.000       0.000
6     0.001      0.6           -200.000     -200.000     -200.000     -200.000       0.000
7     0.003      0.2           -200.000     -200.000     -200.000     -200.000       0.000
8     0.003      0.4           -200.000     

In [18]:
%%writefile src/scripts/extend_dqn_backbone_decay.py
import os

from pathlib import Path

from src.dqn import DQNConfig
from src.sweep import run_stage
from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    STEP_BUDGET,
)


OUT_PATH = Path(
    "src/results/tuning/"
    "dqn_backbone_decay_extension.pkl"
)

BASE_LEARNING_RATES = (
    3e-4,
    1e-3,
    3e-3,
)

NEW_LEARNING_RATE = 1e-4

LOW_LR_EXTENSION_ENVS = (
    "MountainCar-v0",
    "LunarLander-v3",
)

BASE_DECAY_FRACTIONS = (
    0.2,
    0.4,
    0.6,
)

NEW_DECAY_FRACTION = 0.1

EPS_START = 1.0
EPS_END = 0.01
DECAY_MODE = "linear"

HIDDEN = 128
REPLAY_CAPACITY = 50_000
BATCH_SIZE = 64

LEARNING_STARTS = 1_000
TRAIN_EVERY = 1
TARGET_UPDATE_EVERY = 1_000

GAMMA = 1.0
REWARD_SCALE = 1.0

N_BINS = 100
N_EVAL_POINTS = 20
N_EVAL_EPISODES = 10

MAX_WORKERS = 8


def _make_config(
    env_id,
    learning_rate,
    decay_fraction,
    seed,
):
    budget = int(
        STEP_BUDGET[
            env_id
        ]
    )

    decay_steps = int(
        round(
            decay_fraction
            * budget
        )
    )

    return DQNConfig(
        env_id=env_id,
        explorer="decay",
        explorer_kwargs={
            "eps_start": (
                EPS_START
            ),
            "eps_end": (
                EPS_END
            ),
            "decay_steps": (
                decay_steps
            ),
            "mode": (
                DECAY_MODE
            ),
        },
        seed=int(
            seed
        ),
        n_steps=budget,
        gamma=GAMMA,
        reward_scale=(
            REWARD_SCALE
        ),
        hidden=HIDDEN,
        replay_capacity=(
            REPLAY_CAPACITY
        ),
        batch_size=(
            BATCH_SIZE
        ),
        learning_rate=float(
            learning_rate
        ),
        learning_starts=(
            LEARNING_STARTS
        ),
        train_every=(
            TRAIN_EVERY
        ),
        target_update_every=(
            TARGET_UPDATE_EVERY
        ),
        double=True,
        n_bins=N_BINS,
        n_eval_points=(
            N_EVAL_POINTS
        ),
        n_eval_episodes=(
            N_EVAL_EPISODES
        ),
    )


def main():
    cells = set()

    for env_id in ENV_IDS:
        for learning_rate in BASE_LEARNING_RATES:
            cells.add(
                (
                    env_id,
                    float(
                        learning_rate
                    ),
                    float(
                        NEW_DECAY_FRACTION
                    ),
                )
            )

    for env_id in LOW_LR_EXTENSION_ENVS:
        for decay_fraction in (
            NEW_DECAY_FRACTION,
            *BASE_DECAY_FRACTIONS,
        ):
            cells.add(
                (
                    env_id,
                    float(
                        NEW_LEARNING_RATE
                    ),
                    float(
                        decay_fraction
                    ),
                )
            )

    expected_cells = 20

    if (
        len(cells)
        != expected_cells
    ):
        raise RuntimeError(
            "Unexpected number of "
            "extension cells: "
            f"expected "
            f"{expected_cells}, "
            f"found "
            f"{len(cells)}."
        )

    configs = []

    ordered_cells = sorted(
        cells,
        key=lambda item: (
            ENV_IDS.index(
                item[
                    0
                ]
            ),
            item[
                1
            ],
            item[
                2
            ],
        ),
    )

    for (
        env_id,
        learning_rate,
        decay_fraction,
    ) in ordered_cells:
        for seed in TUNING_SEEDS:
            configs.append(
                _make_config(
                    env_id,
                    learning_rate,
                    decay_fraction,
                    seed,
                )
            )

    expected_total = (
        expected_cells
        * len(
            TUNING_SEEDS
        )
    )

    if (
        len(configs)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected extension "
            "run count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(configs)}."
        )

    print(
        "DOUBLE DQN BACKBONE "
        "+ DECAY GRID EXTENSION"
    )

    print()

    print(
        "New search directions:"
    )

    print(
        "  decay fraction 0.1 "
        "for all environments"
    )

    print(
        "  learning rate 0.0001 "
        "for MountainCar-v0 "
        "and LunarLander-v3"
    )

    print()

    print(
        "Extension cells:"
    )

    current_env = None

    for (
        env_id,
        learning_rate,
        decay_fraction,
    ) in ordered_cells:
        if (
            env_id
            != current_env
        ):
            print()

            print(
                f"  {env_id}"
            )

            current_env = (
                env_id
            )

        print(
            f"    lr="
            f"{learning_rate:g}, "
            f"horizon="
            f"{decay_fraction:g}"
        )

    print()

    print(
        f"Unique new cells: "
        f"{len(cells)}"
    )

    print(
        f"Seeds per cell: "
        f"{len(TUNING_SEEDS)}"
    )

    print(
        f"Total new runs: "
        f"{len(configs)}"
    )

    print()

    per_env = {}

    for env_id in ENV_IDS:
        n_cells = sum(
            1
            for cell
            in cells
            if cell[
                0
            ]
            == env_id
        )

        per_env[
            env_id
        ] = n_cells

        print(
            f"  {env_id}: "
            f"{n_cells} new cells, "
            f"{n_cells * len(TUNING_SEEDS)} "
            "runs"
        )

    if (
        per_env[
            "MountainCar-v0"
        ]
        != 7
    ):
        raise RuntimeError(
            "Expected 7 new "
            "MountainCar cells."
        )

    if (
        per_env[
            "CartPole-v1"
        ]
        != 3
    ):
        raise RuntimeError(
            "Expected 3 new "
            "CartPole cells."
        )

    if (
        per_env[
            "Acrobot-v1"
        ]
        != 3
    ):
        raise RuntimeError(
            "Expected 3 new "
            "Acrobot cells."
        )

    if (
        per_env[
            "LunarLander-v3"
        ]
        != 7
    ):
        raise RuntimeError(
            "Expected 7 new "
            "LunarLander cells."
        )

    print()

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(
                configs
            ),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=OUT_PATH,
        label=(
            "DQN backbone+decay "
            "extension"
        ),
        workers=workers,
        save_every=4,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error"
        in result
    ]

    successful = [
        result
        for result in results
        if "error"
        not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED EXTENSION RUNS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            cfg = result.get(
                "config",
                {}
            )

            kwargs = cfg.get(
                "explorer_kwargs",
                {}
            )

            env_id = cfg.get(
                "env_id"
            )

            n_steps = cfg.get(
                "n_steps"
            )

            decay_steps = kwargs.get(
                "decay_steps"
            )

            if (
                n_steps
                and decay_steps
                is not None
            ):
                fraction = (
                    float(
                        decay_steps
                    )
                    / float(
                        n_steps
                    )
                )
            else:
                fraction = None

            print(
                f"{i}. "
                f"env="
                f"{env_id} "
                f"lr="
                f"{cfg.get('learning_rate')} "
                f"horizon="
                f"{fraction} "
                f"seed="
                f"{cfg.get('seed')}"
            )

            print(
                f"   "
                f"{result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} "
            "extension runs failed."
        )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected stored "
            "extension count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected successful "
            "extension count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(successful)}."
        )

    print()

    print(
        "DOUBLE DQN BACKBONE "
        "+ DECAY GRID EXTENSION "
        "COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/extend_dqn_backbone_decay.py


In [19]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.extend_dqn_backbone_decay",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Double DQN grid "
        "extension failed."
    )

DOUBLE DQN BACKBONE + DECAY GRID EXTENSION

New search directions:
  decay fraction 0.1 for all environments
  learning rate 0.0001 for MountainCar-v0 and LunarLander-v3

Extension cells:

  MountainCar-v0
    lr=0.0001, horizon=0.1
    lr=0.0001, horizon=0.2
    lr=0.0001, horizon=0.4
    lr=0.0001, horizon=0.6
    lr=0.0003, horizon=0.1
    lr=0.001, horizon=0.1
    lr=0.003, horizon=0.1

  CartPole-v1
    lr=0.0003, horizon=0.1
    lr=0.001, horizon=0.1
    lr=0.003, horizon=0.1

  Acrobot-v1
    lr=0.0003, horizon=0.1
    lr=0.001, horizon=0.1
    lr=0.003, horizon=0.1

  LunarLander-v3
    lr=0.0001, horizon=0.1
    lr=0.0001, horizon=0.2
    lr=0.0001, horizon=0.4
    lr=0.0001, horizon=0.6
    lr=0.0003, horizon=0.1
    lr=0.001, horizon=0.1
    lr=0.003, horizon=0.1

Unique new cells: 20
Seeds per cell: 3
Total new runs: 60

  MountainCar-v0: 7 new cells, 21 runs
  CartPole-v1: 3 new cells, 9 runs
  Acrobot-v1: 3 new cells, 9 runs
  LunarLander-v3: 7 new cells, 21 runs

Detecte

In [20]:
%%writefile src/scripts/select_dqn_backbone_decay_expanded.py
import json
import pickle

from collections import defaultdict
from pathlib import Path

import numpy as np

from src.sweep_configs import (
    ENV_IDS,
    TUNING_SEEDS,
    STEP_BUDGET,
)


BASE_PATH = Path(
    "src/results/tuning/"
    "dqn_backbone_decay.pkl"
)

EXTENSION_PATH = Path(
    "src/results/tuning/"
    "dqn_backbone_decay_extension.pkl"
)

SUMMARY_PATH = Path(
    "src/results/tuning/"
    "dqn_backbone_decay_expanded_summary.json"
)

PROVISIONAL_PATH = Path(
    "src/results/tuning/"
    "provisional_dqn_backbone_decay_expanded.json"
)

FINAL_PATH = Path(
    "src/results/tuning/"
    "selected_dqn_backbone_decay.json"
)


LR_GRID_BY_ENV = {
    "MountainCar-v0": (
        1e-4,
        3e-4,
        1e-3,
        3e-3,
    ),
    "CartPole-v1": (
        3e-4,
        1e-3,
        3e-3,
    ),
    "Acrobot-v1": (
        3e-4,
        1e-3,
        3e-3,
    ),
    "LunarLander-v3": (
        1e-4,
        3e-4,
        1e-3,
        3e-3,
    ),
}

DECAY_FRACTIONS = (
    0.1,
    0.2,
    0.4,
    0.6,
)

EXPECTED_EXPLORER = "decay"

EXPECTED_EPS_START = 1.0
EXPECTED_EPS_END = 0.01
EXPECTED_MODE = "linear"

EXPECTED_HIDDEN = 128
EXPECTED_REPLAY_CAPACITY = 50_000
EXPECTED_BATCH_SIZE = 64

EXPECTED_LEARNING_STARTS = 1_000
EXPECTED_TRAIN_EVERY = 1
EXPECTED_TARGET_UPDATE_EVERY = 1_000

EXPECTED_GAMMA = 1.0
EXPECTED_REWARD_SCALE = 1.0


def _write_json_atomic(
    obj,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(
        path
    )


def _close(
    a,
    b,
    atol=1e-12,
):
    return bool(
        np.isclose(
            float(a),
            float(b),
            rtol=0.0,
            atol=atol,
        )
    )


def _load_results(
    path,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing results file: "
            f"{path}"
        )

    with open(
        path,
        "rb",
    ) as f:
        results = pickle.load(
            f
        )

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            f"{path} must contain "
            "a list."
        )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    if errors:
        raise RuntimeError(
            f"{path} contains "
            f"{len(errors)} "
            "error records."
        )

    return results


def _validate_and_extract(
    result,
):
    required = (
        "env_id",
        "algo",
        "explorer",
        "explorer_kwargs",
        "seed",
        "n_steps",
        "gamma",
        "reward_scale",
        "hidden",
        "replay_capacity",
        "batch_size",
        "learning_rate",
        "learning_starts",
        "train_every",
        "target_update_every",
        "double",
        "final_eval",
    )

    for name in required:
        if name not in result:
            raise KeyError(
                "Result missing "
                f"{name}."
            )

    env_id = result[
        "env_id"
    ]

    if env_id not in ENV_IDS:
        raise ValueError(
            "Unexpected environment: "
            f"{env_id}"
        )

    if (
        result[
            "algo"
        ]
        != "double-dqn"
    ):
        raise ValueError(
            "Expected double-dqn."
        )

    if (
        result[
            "explorer"
        ]
        != EXPECTED_EXPLORER
    ):
        raise ValueError(
            "Expected decay explorer."
        )

    seed = int(
        result[
            "seed"
        ]
    )

    if seed not in TUNING_SEEDS:
        raise ValueError(
            "Unexpected tuning seed: "
            f"{seed}"
        )

    n_steps = int(
        result[
            "n_steps"
        ]
    )

    if (
        n_steps
        != int(
            STEP_BUDGET[
                env_id
            ]
        )
    ):
        raise ValueError(
            "Step-budget mismatch "
            f"for {env_id}."
        )

    lr = float(
        result[
            "learning_rate"
        ]
    )

    if not any(
        _close(
            lr,
            candidate,
        )
        for candidate
        in LR_GRID_BY_ENV[
            env_id
        ]
    ):
        raise ValueError(
            f"Unexpected LR "
            f"{lr} for "
            f"{env_id}."
        )

    kwargs = dict(
        result[
            "explorer_kwargs"
        ]
    )

    for name in (
        "eps_start",
        "eps_end",
        "decay_steps",
        "mode",
    ):
        if name not in kwargs:
            raise KeyError(
                "Explorer kwargs "
                f"missing {name}."
            )

    if not _close(
        kwargs[
            "eps_start"
        ],
        EXPECTED_EPS_START,
    ):
        raise ValueError(
            "Unexpected eps_start."
        )

    if not _close(
        kwargs[
            "eps_end"
        ],
        EXPECTED_EPS_END,
    ):
        raise ValueError(
            "Unexpected eps_end."
        )

    if (
        kwargs[
            "mode"
        ]
        != EXPECTED_MODE
    ):
        raise ValueError(
            "Unexpected decay mode."
        )

    decay_steps = int(
        kwargs[
            "decay_steps"
        ]
    )

    decay_fraction = (
        decay_steps
        / n_steps
    )

    if not any(
        _close(
            decay_fraction,
            candidate,
        )
        for candidate
        in DECAY_FRACTIONS
    ):
        raise ValueError(
            "Unexpected decay "
            f"fraction "
            f"{decay_fraction}."
        )

    if (
        int(
            result[
                "hidden"
            ]
        )
        != EXPECTED_HIDDEN
    ):
        raise ValueError(
            "Unexpected hidden size."
        )

    if (
        int(
            result[
                "replay_capacity"
            ]
        )
        != EXPECTED_REPLAY_CAPACITY
    ):
        raise ValueError(
            "Unexpected replay capacity."
        )

    if (
        int(
            result[
                "batch_size"
            ]
        )
        != EXPECTED_BATCH_SIZE
    ):
        raise ValueError(
            "Unexpected batch size."
        )

    if (
        int(
            result[
                "learning_starts"
            ]
        )
        != EXPECTED_LEARNING_STARTS
    ):
        raise ValueError(
            "Unexpected learning_starts."
        )

    if (
        int(
            result[
                "train_every"
            ]
        )
        != EXPECTED_TRAIN_EVERY
    ):
        raise ValueError(
            "Unexpected train_every."
        )

    if (
        int(
            result[
                "target_update_every"
            ]
        )
        != EXPECTED_TARGET_UPDATE_EVERY
    ):
        raise ValueError(
            "Unexpected target update."
        )

    if not _close(
        result[
            "gamma"
        ],
        EXPECTED_GAMMA,
    ):
        raise ValueError(
            "Unexpected gamma."
        )

    if not _close(
        result[
            "reward_scale"
        ],
        EXPECTED_REWARD_SCALE,
    ):
        raise ValueError(
            "Unexpected reward scale."
        )

    if not bool(
        result[
            "double"
        ]
    ):
        raise ValueError(
            "Expected Double DQN."
        )

    score = float(
        result[
            "final_eval"
        ]
    )

    if not np.isfinite(
        score
    ):
        raise ValueError(
            "Non-finite final score."
        )

    return (
        env_id,
        lr,
        float(
            decay_fraction
        ),
        seed,
        score,
    )


def main():
    base = _load_results(
        BASE_PATH
    )

    extension = _load_results(
        EXTENSION_PATH
    )

    print(
        "EXPANDED DOUBLE DQN "
        "BACKBONE + DECAY SELECTION"
    )

    print()

    print(
        f"Original records: "
        f"{len(base)}"
    )

    print(
        f"Extension records: "
        f"{len(extension)}"
    )

    combined = (
        base
        + extension
    )

    print(
        f"Combined records: "
        f"{len(combined)}"
    )

    expected_total = 168

    if (
        len(combined)
        != expected_total
    ):
        raise RuntimeError(
            "Expected 168 combined "
            f"records, found "
            f"{len(combined)}."
        )

    grouped = defaultdict(
        dict
    )

    seen = set()

    for result in combined:
        (
            env_id,
            lr,
            fraction,
            seed,
            score,
        ) = _validate_and_extract(
            result
        )

        key = (
            env_id,
            lr,
            fraction,
            seed,
        )

        if key in seen:
            raise RuntimeError(
                "Duplicate result "
                f"detected: {key}"
            )

        seen.add(
            key
        )

        grouped[
            (
                env_id,
                lr,
                fraction,
            )
        ][
            seed
        ] = score

    if (
        len(seen)
        != expected_total
    ):
        raise RuntimeError(
            "Unique record count "
            "does not equal 168."
        )

    expected_cells = (
        16
        + 12
        + 12
        + 16
    )

    if (
        len(grouped)
        != expected_cells
    ):
        raise RuntimeError(
            "Expected 56 unique "
            f"hyperparameter cells, "
            f"found "
            f"{len(grouped)}."
        )

    print(
        f"Unique run keys: "
        f"{len(seen)}"
    )

    print(
        f"Unique hyperparameter "
        f"cells: "
        f"{len(grouped)}"
    )

    print()

    summary = {}
    provisional = {}

    edge_envs = []

    for env_id in ENV_IDS:
        lr_grid = (
            LR_GRID_BY_ENV[
                env_id
            ]
        )

        expected_env_cells = (
            len(
                lr_grid
            )
            * len(
                DECAY_FRACTIONS
            )
        )

        cells = []

        for lr in lr_grid:
            for fraction in DECAY_FRACTIONS:
                matching_key = None

                for key in grouped:
                    (
                        key_env,
                        key_lr,
                        key_fraction,
                    ) = key

                    if (
                        key_env
                        == env_id
                        and _close(
                            key_lr,
                            lr,
                        )
                        and _close(
                            key_fraction,
                            fraction,
                        )
                    ):
                        matching_key = key
                        break

                if matching_key is None:
                    raise RuntimeError(
                        "Missing cell: "
                        f"{env_id}, "
                        f"lr={lr:g}, "
                        f"horizon="
                        f"{fraction:g}"
                    )

                seed_scores = grouped[
                    matching_key
                ]

                if (
                    set(
                        seed_scores
                    )
                    != set(
                        TUNING_SEEDS
                    )
                ):
                    raise RuntimeError(
                        "Seed mismatch for "
                        f"{env_id}, "
                        f"lr={lr:g}, "
                        f"horizon="
                        f"{fraction:g}"
                    )

                scores = np.asarray(
                    [
                        seed_scores[
                            seed
                        ]
                        for seed
                        in TUNING_SEEDS
                    ],
                    dtype=np.float64,
                )

                cells.append(
                    {
                        "learning_rate": float(
                            lr
                        ),
                        "decay_fraction": float(
                            fraction
                        ),
                        "decay_steps": int(
                            round(
                                fraction
                                * STEP_BUDGET[
                                    env_id
                                ]
                            )
                        ),
                        "seed_scores": {
                            str(seed): float(
                                seed_scores[
                                    seed
                                ]
                            )
                            for seed
                            in TUNING_SEEDS
                        },
                        "mean_final_eval": float(
                            np.mean(
                                scores
                            )
                        ),
                        "std_final_eval": float(
                            np.std(
                                scores,
                                ddof=1,
                            )
                        ),
                    }
                )

        if (
            len(cells)
            != expected_env_cells
        ):
            raise RuntimeError(
                "Wrong number of cells "
                f"for {env_id}: "
                f"expected "
                f"{expected_env_cells}, "
                f"found "
                f"{len(cells)}."
            )

        cells.sort(
            key=lambda item: (
                item[
                    "mean_final_eval"
                ],
                -item[
                    "std_final_eval"
                ],
            ),
            reverse=True,
        )

        winner = cells[
            0
        ]

        winner_lr = float(
            winner[
                "learning_rate"
            ]
        )

        winner_fraction = float(
            winner[
                "decay_fraction"
            ]
        )

        lr_low_edge = _close(
            winner_lr,
            min(
                lr_grid
            ),
        )

        lr_high_edge = _close(
            winner_lr,
            max(
                lr_grid
            ),
        )

        decay_short_edge = _close(
            winner_fraction,
            min(
                DECAY_FRACTIONS
            ),
        )

        decay_long_edge = _close(
            winner_fraction,
            max(
                DECAY_FRACTIONS
            ),
        )

        lr_edge = bool(
            lr_low_edge
            or lr_high_edge
        )

        decay_edge = bool(
            decay_short_edge
            or decay_long_edge
        )

        needs_extension = bool(
            lr_edge
            or decay_edge
        )

        if needs_extension:
            edge_envs.append(
                env_id
            )

        summary[
            env_id
        ] = {
            "lr_grid": [
                float(
                    value
                )
                for value
                in lr_grid
            ],
            "decay_grid": [
                float(
                    value
                )
                for value
                in DECAY_FRACTIONS
            ],
            "cells_ranked": (
                cells
            ),
            "winner": (
                winner
            ),
            "learning_rate_low_edge": (
                lr_low_edge
            ),
            "learning_rate_high_edge": (
                lr_high_edge
            ),
            "decay_short_edge": (
                decay_short_edge
            ),
            "decay_long_edge": (
                decay_long_edge
            ),
            "needs_extension": (
                needs_extension
            ),
        }

        provisional[
            env_id
        ] = {
            "learning_rate": (
                winner_lr
            ),
            "decay_fraction": (
                winner_fraction
            ),
            "decay_steps": int(
                winner[
                    "decay_steps"
                ]
            ),
            "eps_start": (
                EXPECTED_EPS_START
            ),
            "eps_end": (
                EXPECTED_EPS_END
            ),
            "mode": (
                EXPECTED_MODE
            ),
            "mean_final_eval": float(
                winner[
                    "mean_final_eval"
                ]
            ),
            "std_final_eval": float(
                winner[
                    "std_final_eval"
                ]
            ),
            "needs_extension": (
                needs_extension
            ),
        }

        print(
            "=" * 105
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 105
        )

        print()

        print(
            "Rank   LR         Horizon   "
            "Seed 1000    Seed 1001    "
            "Seed 1002    Mean         Std"
        )

        print(
            "-" * 105
        )

        for rank, cell in enumerate(
            cells,
            start=1,
        ):
            scores = (
                cell[
                    "seed_scores"
                ]
            )

            print(
                f"{rank:<6}"
                f"{cell['learning_rate']:<11g}"
                f"{cell['decay_fraction']:<10.2f}"
                f"{scores[str(TUNING_SEEDS[0])]:>12.3f}"
                f"{scores[str(TUNING_SEEDS[1])]:>13.3f}"
                f"{scores[str(TUNING_SEEDS[2])]:>13.3f}"
                f"{cell['mean_final_eval']:>13.3f}"
                f"{cell['std_final_eval']:>12.3f}"
            )

        print()

        print(
            "WINNER:"
        )

        print(
            f"  learning_rate = "
            f"{winner_lr:g}"
        )

        print(
            f"  decay_fraction = "
            f"{winner_fraction:g}"
        )

        print(
            f"  decay_steps = "
            f"{winner['decay_steps']}"
        )

        print(
            f"  mean final greedy = "
            f"{winner['mean_final_eval']:.3f}"
        )

        print(
            f"  std = "
            f"{winner['std_final_eval']:.3f}"
        )

        print()

        print(
            "Boundary check:"
        )

        print(
            f"  LR low edge: "
            f"{lr_low_edge}"
        )

        print(
            f"  LR high edge: "
            f"{lr_high_edge}"
        )

        print(
            f"  decay short edge: "
            f"{decay_short_edge}"
        )

        print(
            f"  decay long edge: "
            f"{decay_long_edge}"
        )

        print()

        if needs_extension:
            print(
                "STATUS: STILL ON "
                "A GRID EDGE"
            )
        else:
            print(
                "STATUS: INTERIOR "
                "WINNER"
            )

        print()
        print()

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    _write_json_atomic(
        provisional,
        PROVISIONAL_PATH,
    )

    if FINAL_PATH.exists():
        FINAL_PATH.unlink()

    print(
        "=" * 105
    )

    print(
        "EXPANDED GRID STATUS"
    )

    print(
        "=" * 105
    )

    print()

    if edge_envs:
        print(
            "Further extension is "
            "required for:"
        )

        for env_id in edge_envs:
            item = summary[
                env_id
            ]

            labels = []

            if item[
                "learning_rate_low_edge"
            ]:
                labels.append(
                    "LR-low"
                )

            if item[
                "learning_rate_high_edge"
            ]:
                labels.append(
                    "LR-high"
                )

            if item[
                "decay_short_edge"
            ]:
                labels.append(
                    "decay-short"
                )

            if item[
                "decay_long_edge"
            ]:
                labels.append(
                    "decay-long"
                )

            print(
                f"  {env_id}: "
                f"{', '.join(labels)}"
            )

        print()

        print(
            "DQN configuration remains "
            "PROVISIONAL."
        )

    else:
        final_selected = {}

        for env_id in ENV_IDS:
            winner = summary[
                env_id
            ][
                "winner"
            ]

            final_selected[
                env_id
            ] = {
                "learning_rate": float(
                    winner[
                        "learning_rate"
                    ]
                ),
                "decay_fraction": float(
                    winner[
                        "decay_fraction"
                    ]
                ),
                "decay_steps": int(
                    winner[
                        "decay_steps"
                    ]
                ),
                "eps_start": (
                    EXPECTED_EPS_START
                ),
                "eps_end": (
                    EXPECTED_EPS_END
                ),
                "mode": (
                    EXPECTED_MODE
                ),
            }

        _write_json_atomic(
            final_selected,
            FINAL_PATH,
        )

        print(
            "All winners are interior."
        )

        print()

        print(
            "DQN BACKBONE + DECAY "
            "CONFIGURATIONS FROZEN."
        )

        print()

        print(
            "Final selection file:"
        )

        print(
            FINAL_PATH
        )

    print()

    print(
        "Expanded summary:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    print(
        "Expanded provisional winners:"
    )

    print(
        PROVISIONAL_PATH
    )

    print()

    print(
        "EXPANDED DOUBLE DQN "
        "SELECTION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/select_dqn_backbone_decay_expanded.py


In [21]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.select_dqn_backbone_decay_expanded",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
EXPANDED DOUBLE DQN BACKBONE + DECAY SELECTION

Original records: 108
Extension records: 60
Combined records: 168
Unique run keys: 168
Unique hyperparameter cells: 56

Environment: MountainCar-v0

Rank   LR         Horizon   Seed 1000    Seed 1001    Seed 1002    Mean         Std
---------------------------------------------------------------------------------------------------------
1     0.0003     0.20          -200.000     -130.100     -200.000     -176.700      40.357
2     0.001      0.10          -200.000     -200.000     -148.200     -182.733      29.907
3     0.0001     0.10          -176.200     -174.500     -200.000     -183.567      14.257
4     0.0001     0.60          -200.000     -200.000     -197.700     -199.233       1.328
5     0.0001     0.20          -200.000     -200.000     -200.000     -200.000       0.000
6     0.0001     0.40          -200.000     -200.000     -200.000     -200.000       0.000
7     0.0003     0.10          -200.000    

In [22]:
%%writefile src/scripts/finalize_dqn_backbone_decay.py
import json

from pathlib import Path

import numpy as np


SUMMARY_PATH = Path(
    "src/results/tuning/"
    "dqn_backbone_decay_expanded_summary.json"
)

OUT_PATH = Path(
    "src/results/tuning/"
    "selected_dqn_backbone_decay.json"
)


EXPECTED = {
    "MountainCar-v0": {
        "learning_rate": 3e-4,
        "decay_fraction": 0.2,
        "decay_steps": 30_000,
    },
    "CartPole-v1": {
        "learning_rate": 1e-3,
        "decay_fraction": 0.2,
        "decay_steps": 20_000,
    },
    "Acrobot-v1": {
        "learning_rate": 1e-3,
        "decay_fraction": 0.2,
        "decay_steps": 20_000,
    },
    "LunarLander-v3": {
        "learning_rate": 3e-4,
        "decay_fraction": 0.2,
        "decay_steps": 60_000,
    },
}


def _close(
    a,
    b,
):
    return bool(
        np.isclose(
            float(a),
            float(b),
            rtol=0.0,
            atol=1e-12,
        )
    )


def _write_json_atomic(
    obj,
    path,
):
    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(
        path
    )


def _find_cell(
    cells,
    learning_rate,
    decay_fraction,
):
    matches = [
        cell
        for cell in cells
        if (
            _close(
                cell[
                    "learning_rate"
                ],
                learning_rate,
            )
            and _close(
                cell[
                    "decay_fraction"
                ],
                decay_fraction,
            )
        )
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            "Expected exactly one "
            "matching tuning cell."
        )

    return matches[
        0
    ]


def main():
    if not SUMMARY_PATH.exists():
        raise FileNotFoundError(
            f"Missing expanded summary: "
            f"{SUMMARY_PATH}"
        )

    with open(
        SUMMARY_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        summary = json.load(
            f
        )

    selected = {}

    for (
        env_id,
        expected,
    ) in EXPECTED.items():
        if env_id not in summary:
            raise KeyError(
                f"Missing {env_id}."
            )

        item = summary[
            env_id
        ]

        cells = item[
            "cells_ranked"
        ]

        chosen = _find_cell(
            cells,
            expected[
                "learning_rate"
            ],
            expected[
                "decay_fraction"
            ],
        )

        best_mean = max(
            float(
                cell[
                    "mean_final_eval"
                ]
            )
            for cell in cells
        )

        chosen_mean = float(
            chosen[
                "mean_final_eval"
            ]
        )

        if not _close(
            chosen_mean,
            best_mean,
        ):
            raise RuntimeError(
                f"Selected configuration "
                f"for {env_id} is not "
                "tied for best mean."
            )

        if env_id == "CartPole-v1":
            edge_cell = _find_cell(
                cells,
                1e-3,
                0.1,
            )

            if not _close(
                edge_cell[
                    "mean_final_eval"
                ],
                500.0,
            ):
                raise RuntimeError(
                    "CartPole short-edge "
                    "cell is not at 500."
                )

            if not _close(
                chosen_mean,
                500.0,
            ):
                raise RuntimeError(
                    "CartPole selected "
                    "cell is not at 500."
                )

            if not _close(
                edge_cell[
                    "std_final_eval"
                ],
                0.0,
            ):
                raise RuntimeError(
                    "CartPole edge cell "
                    "does not have zero "
                    "seed spread."
                )

            if not _close(
                chosen[
                    "std_final_eval"
                ],
                0.0,
            ):
                raise RuntimeError(
                    "CartPole selected "
                    "cell does not have "
                    "zero seed spread."
                )

        selected[
            env_id
        ] = {
            "learning_rate": float(
                expected[
                    "learning_rate"
                ]
            ),
            "decay_fraction": float(
                expected[
                    "decay_fraction"
                ]
            ),
            "decay_steps": int(
                expected[
                    "decay_steps"
                ]
            ),
            "eps_start": 1.0,
            "eps_end": 0.01,
            "mode": "linear",
            "tuning_mean_final_eval": (
                chosen_mean
            ),
            "tuning_std_final_eval": float(
                chosen[
                    "std_final_eval"
                ]
            ),
        }

    _write_json_atomic(
        selected,
        OUT_PATH,
    )

    print(
        "FINAL DOUBLE DQN "
        "BACKBONE + DECAY SELECTION"
    )

    print()

    for (
        env_id,
        item,
    ) in selected.items():
        print(
            f"{env_id}: "
            f"lr="
            f"{item['learning_rate']:g}, "
            f"decay="
            f"{item['decay_fraction']:g} "
            f"({item['decay_steps']} steps), "
            f"tuning mean="
            f"{item['tuning_mean_final_eval']:.3f}, "
            f"std="
            f"{item['tuning_std_final_eval']:.3f}"
        )

    print()

    print(
        "CartPole note:"
    )

    print(
        "  horizons 0.1 and 0.2 "
        "at lr=0.001 both achieved "
        "500.000 +/- 0.000."
    )

    print(
        "  The interior 0.2 setting "
        "was selected to resolve "
        "the exact ceiling tie."
    )

    print()

    print(
        "Selection frozen:"
    )

    print(
        OUT_PATH
    )

    print()

    print(
        "DOUBLE DQN BACKBONE "
        "FINALIZATION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/finalize_dqn_backbone_decay.py


In [23]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.finalize_dqn_backbone_decay",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
FINAL DOUBLE DQN BACKBONE + DECAY SELECTION

MountainCar-v0: lr=0.0003, decay=0.2 (30000 steps), tuning mean=-176.700, std=40.357
CartPole-v1: lr=0.001, decay=0.2 (20000 steps), tuning mean=500.000, std=0.000
Acrobot-v1: lr=0.001, decay=0.2 (20000 steps), tuning mean=-82.767, std=2.747
LunarLander-v3: lr=0.0003, decay=0.2 (60000 steps), tuning mean=48.048, std=215.190

CartPole note:
  horizons 0.1 and 0.2 at lr=0.001 both achieved 500.000 +/- 0.000.
  The interior 0.2 setting was selected to resolve the exact ceiling tie.

Selection frozen:
src\results\tuning\selected_dqn_backbone_decay.json

DOUBLE DQN BACKBONE FINALIZATION COMPLETED

STDERR:



In [24]:
%%writefile src/scripts/diagnose_td_error_dqn_tuned.py
import json
import multiprocessing as mp
import os
import pickle
import traceback

from pathlib import Path

import numpy as np
import torch

from src.dqn import (
    DEVICE,
    QNetwork,
    ReplayBuffer,
    _dqn_update,
    _evaluate_dqn,
)
from src.environments import GymEnv
from src.exploration import make_explorer
from src.sweep_configs import (
    ENV_IDS,
    STEP_BUDGET,
)


SELECTED_PATH = Path(
    "src/results/tuning/"
    "selected_dqn_backbone_decay.json"
)

OUT_PATH = Path(
    "src/results/diagnostics/"
    "td_error_trajectory_dqn_tuned.pkl"
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        3000,
        3010,
    )
)

BLOCK_SIZE = 1000

N_EVAL_POINTS = 20
N_EVAL_EPISODES = 10

HIDDEN = 128
REPLAY_CAPACITY = 50_000
BATCH_SIZE = 64

LEARNING_STARTS = 1_000
TRAIN_EVERY = 1
TARGET_UPDATE_EVERY = 1_000

GAMMA = 1.0
REWARD_SCALE = 1.0

MAX_WORKERS = 8


class DiagnosticDQNConfig:
    def __init__(
        self,
        env_id,
        seed,
        n_steps,
        learning_rate,
        explorer_kwargs,
    ):
        self.env_id = str(
            env_id
        )

        self.explorer = "decay"

        self.explorer_kwargs = dict(
            explorer_kwargs
        )

        self.seed = int(
            seed
        )

        self.n_steps = int(
            n_steps
        )

        self.gamma = float(
            GAMMA
        )

        self.reward_scale = float(
            REWARD_SCALE
        )

        self.hidden = int(
            HIDDEN
        )

        self.replay_capacity = int(
            REPLAY_CAPACITY
        )

        self.batch_size = int(
            BATCH_SIZE
        )

        self.learning_rate = float(
            learning_rate
        )

        self.learning_starts = int(
            LEARNING_STARTS
        )

        self.train_every = int(
            TRAIN_EVERY
        )

        self.target_update_every = int(
            TARGET_UPDATE_EVERY
        )

        self.double = True

        self.n_bins = 100

        self.n_eval_points = int(
            N_EVAL_POINTS
        )

        self.n_eval_episodes = int(
            N_EVAL_EPISODES
        )


def _config_dict(
    cfg,
):
    return {
        "env_id": (
            cfg.env_id
        ),
        "algo": (
            "double-dqn"
        ),
        "explorer": (
            cfg.explorer
        ),
        "explorer_kwargs": dict(
            cfg.explorer_kwargs
        ),
        "seed": (
            cfg.seed
        ),
        "n_steps": (
            cfg.n_steps
        ),
        "gamma": (
            cfg.gamma
        ),
        "reward_scale": (
            cfg.reward_scale
        ),
        "hidden": (
            cfg.hidden
        ),
        "replay_capacity": (
            cfg.replay_capacity
        ),
        "batch_size": (
            cfg.batch_size
        ),
        "learning_rate": (
            cfg.learning_rate
        ),
        "learning_starts": (
            cfg.learning_starts
        ),
        "train_every": (
            cfg.train_every
        ),
        "target_update_every": (
            cfg.target_update_every
        ),
        "double": (
            cfg.double
        ),
        "n_eval_points": (
            cfg.n_eval_points
        ),
        "n_eval_episodes": (
            cfg.n_eval_episodes
        ),
        "block_size": (
            BLOCK_SIZE
        ),
    }


def _config_key(
    cfg,
):
    return json.dumps(
        _config_dict(
            cfg
        ),
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def _save_atomic(
    obj,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "wb",
    ) as f:
        pickle.dump(
            obj,
            f,
        )

    tmp.replace(
        path
    )


def _load_selected():
    if not SELECTED_PATH.exists():
        raise FileNotFoundError(
            "Final selected DQN "
            "configuration not found: "
            f"{SELECTED_PATH}"
        )

    with open(
        SELECTED_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        selected = json.load(
            f
        )

    if not isinstance(
        selected,
        dict,
    ):
        raise TypeError(
            "Selected DQN file must "
            "contain a dictionary."
        )

    cleaned = {}

    for env_id in ENV_IDS:
        if env_id not in selected:
            raise KeyError(
                f"Missing {env_id} "
                "from selected DQN "
                "configuration."
            )

        item = dict(
            selected[
                env_id
            ]
        )

        required = (
            "learning_rate",
            "decay_fraction",
            "decay_steps",
            "eps_start",
            "eps_end",
            "mode",
        )

        for field in required:
            if field not in item:
                raise KeyError(
                    f"{env_id} missing "
                    f"{field}."
                )

        learning_rate = float(
            item[
                "learning_rate"
            ]
        )

        decay_fraction = float(
            item[
                "decay_fraction"
            ]
        )

        decay_steps = int(
            item[
                "decay_steps"
            ]
        )

        expected_decay_steps = int(
            round(
                decay_fraction
                * STEP_BUDGET[
                    env_id
                ]
            )
        )

        if (
            decay_steps
            != expected_decay_steps
        ):
            raise ValueError(
                f"{env_id}: "
                "decay_steps does not "
                "match decay_fraction "
                "and step budget."
            )

        if (
            learning_rate
            <= 0.0
        ):
            raise ValueError(
                f"{env_id}: learning "
                "rate must be positive."
            )

        if (
            item[
                "mode"
            ]
            != "linear"
        ):
            raise ValueError(
                f"{env_id}: expected "
                "linear decay."
            )

        cleaned[
            env_id
        ] = {
            "learning_rate": (
                learning_rate
            ),
            "decay_fraction": (
                decay_fraction
            ),
            "explorer_kwargs": {
                "eps_start": float(
                    item[
                        "eps_start"
                    ]
                ),
                "eps_end": float(
                    item[
                        "eps_end"
                    ]
                ),
                "decay_steps": (
                    decay_steps
                ),
                "mode": (
                    item[
                        "mode"
                    ]
                ),
            },
        }

    return cleaned


def _run_one(
    cfg,
):
    if (
        cfg.n_steps
        % BLOCK_SIZE
        != 0
    ):
        raise ValueError(
            f"{cfg.env_id} budget "
            f"{cfg.n_steps} is not "
            f"divisible by "
            f"{BLOCK_SIZE}."
        )

    if (
        cfg.learning_starts
        < cfg.batch_size
    ):
        raise ValueError(
            "learning_starts must "
            "be at least batch_size."
        )

    env = GymEnv(
        cfg.env_id,
        seed=cfg.seed,
    )

    eval_env = GymEnv(
        cfg.env_id,
        seed=(
            500_000
            + cfg.seed
        ),
    )

    action_rng = (
        np.random.default_rng(
            100_000
            + cfg.seed
        )
    )

    replay_rng = (
        np.random.default_rng(
            200_000
            + cfg.seed
        )
    )

    eval_rng = (
        np.random.default_rng(
            300_000
            + cfg.seed
        )
    )

    torch.manual_seed(
        400_000
        + cfg.seed
    )

    obs_shape = (
        env.env
        .observation_space
        .shape
    )

    if (
        obs_shape is None
        or len(
            obs_shape
        )
        != 1
    ):
        raise ValueError(
            "DQN requires a "
            "one-dimensional "
            "observation vector."
        )

    obs_dim = int(
        obs_shape[
            0
        ]
    )

    q = QNetwork(
        obs_dim=obs_dim,
        n_actions=(
            env.n_actions
        ),
        hidden=(
            cfg.hidden
        ),
    ).to(
        DEVICE
    )

    target = QNetwork(
        obs_dim=obs_dim,
        n_actions=(
            env.n_actions
        ),
        hidden=(
            cfg.hidden
        ),
    ).to(
        DEVICE
    )

    target.load_state_dict(
        q.state_dict()
    )

    q.train()
    target.eval()

    optimizer = (
        torch.optim.Adam(
            q.parameters(),
            lr=(
                cfg.learning_rate
            ),
        )
    )

    replay = ReplayBuffer(
        capacity=(
            cfg.replay_capacity
        ),
        obs_dim=(
            obs_dim
        ),
        rng=(
            replay_rng
        ),
    )

    explorer = make_explorer(
        cfg.explorer,
        n_actions=(
            env.n_actions
        ),
        **cfg.explorer_kwargs,
    )

    n_blocks = (
        cfg.n_steps
        // BLOCK_SIZE
    )

    td_block_sum = np.zeros(
        n_blocks,
        dtype=np.float64,
    )

    td_block_count = np.zeros(
        n_blocks,
        dtype=np.int64,
    )

    q_block_sum = np.zeros(
        n_blocks,
        dtype=np.float64,
    )

    loss_block_sum = np.zeros(
        n_blocks,
        dtype=np.float64,
    )

    eval_steps = []
    eval_returns = []

    eval_every = max(
        1,
        cfg.n_steps
        // cfg.n_eval_points,
    )

    next_eval = (
        eval_every
    )

    total_steps = 0
    n_updates = 0

    explorer.reset_episode()

    obs = env.reset()

    while (
        total_steps
        < cfg.n_steps
    ):
        with torch.no_grad():
            x = torch.as_tensor(
                obs,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(
                0
            )

            q_values = (
                q(
                    x
                )
                .squeeze(
                    0
                )
                .cpu()
                .numpy()
            )

        action = explorer.select(
            q_values,
            action_rng,
            None,
        )

        (
            obs2,
            reward,
            terminated,
            truncated,
        ) = env.step(
            action
        )

        replay.add(
            obs,
            action,
            reward
            * cfg.reward_scale,
            obs2,
            terminated,
        )

        total_steps += 1

        if (
            total_steps
            >= cfg.learning_starts
            and len(
                replay
            )
            >= cfg.batch_size
            and (
                total_steps
                % cfg.train_every
                == 0
            )
        ):
            (
                loss_value,
                td_mean,
                q_mean,
            ) = _dqn_update(
                q=q,
                target=target,
                optimizer=optimizer,
                replay=replay,
                explorer=explorer,
                cfg=cfg,
            )

            n_updates += 1

            block_index = min(
                (
                    total_steps
                    - 1
                )
                // BLOCK_SIZE,
                n_blocks
                - 1,
            )

            td_block_sum[
                block_index
            ] += float(
                td_mean
            )

            q_block_sum[
                block_index
            ] += float(
                q_mean
            )

            loss_block_sum[
                block_index
            ] += float(
                loss_value
            )

            td_block_count[
                block_index
            ] += 1

        if (
            total_steps
            % cfg.target_update_every
            == 0
        ):
            target.load_state_dict(
                q.state_dict()
            )

            target.eval()

        if (
            terminated
            or truncated
        ):
            if (
                total_steps
                < cfg.n_steps
            ):
                explorer.reset_episode()

                obs = env.reset()

        else:
            obs = obs2

        if (
            total_steps
            >= next_eval
        ):
            eval_steps.append(
                total_steps
            )

            eval_returns.append(
                _evaluate_dqn(
                    q,
                    eval_env,
                    eval_rng,
                    cfg.n_eval_episodes,
                )
            )

            while (
                next_eval
                <= total_steps
            ):
                next_eval += (
                    eval_every
                )

    if (
        total_steps
        != cfg.n_steps
    ):
        raise RuntimeError(
            f"Expected "
            f"{cfg.n_steps} steps, "
            f"found "
            f"{total_steps}."
        )

    expected_updates = (
        (
            cfg.n_steps
            - cfg.learning_starts
        )
        // cfg.train_every
        + 1
    )

    if (
        n_updates
        != expected_updates
    ):
        raise RuntimeError(
            "Unexpected number of "
            "DQN updates: "
            f"expected "
            f"{expected_updates}, "
            f"found "
            f"{n_updates}."
        )

    if (
        int(
            td_block_count.sum()
        )
        != n_updates
    ):
        raise RuntimeError(
            "TD block update counts "
            "do not sum to n_updates."
        )

    td_block_mean = np.full(
        n_blocks,
        np.nan,
        dtype=np.float64,
    )

    q_block_mean = np.full(
        n_blocks,
        np.nan,
        dtype=np.float64,
    )

    loss_block_mean = np.full(
        n_blocks,
        np.nan,
        dtype=np.float64,
    )

    valid = (
        td_block_count
        > 0
    )

    td_block_mean[
        valid
    ] = (
        td_block_sum[
            valid
        ]
        / td_block_count[
            valid
        ]
    )

    q_block_mean[
        valid
    ] = (
        q_block_sum[
            valid
        ]
        / td_block_count[
            valid
        ]
    )

    loss_block_mean[
        valid
    ] = (
        loss_block_sum[
            valid
        ]
        / td_block_count[
            valid
        ]
    )

    td_block_steps = (
        np.arange(
            1,
            n_blocks + 1,
            dtype=np.int64,
        )
        * BLOCK_SIZE
    )

    eval_steps = np.asarray(
        eval_steps,
        dtype=np.int64,
    )

    eval_returns = np.asarray(
        eval_returns,
        dtype=np.float64,
    )

    if (
        eval_steps.size
        != cfg.n_eval_points
    ):
        raise RuntimeError(
            "Unexpected evaluation "
            "checkpoint count: "
            f"expected "
            f"{cfg.n_eval_points}, "
            f"found "
            f"{eval_steps.size}."
        )

    if (
        eval_returns.size
        != cfg.n_eval_points
    ):
        raise RuntimeError(
            "Evaluation-return "
            "count mismatch."
        )

    result = {
        "env_id": (
            cfg.env_id
        ),
        "algo": (
            "double-dqn"
        ),
        "explorer": (
            cfg.explorer
        ),
        "explorer_kwargs": dict(
            cfg.explorer_kwargs
        ),
        "seed": (
            cfg.seed
        ),
        "n_steps": (
            cfg.n_steps
        ),
        "gamma": (
            cfg.gamma
        ),
        "reward_scale": (
            cfg.reward_scale
        ),
        "hidden": (
            cfg.hidden
        ),
        "replay_capacity": (
            cfg.replay_capacity
        ),
        "batch_size": (
            cfg.batch_size
        ),
        "learning_rate": (
            cfg.learning_rate
        ),
        "learning_starts": (
            cfg.learning_starts
        ),
        "train_every": (
            cfg.train_every
        ),
        "target_update_every": (
            cfg.target_update_every
        ),
        "double": True,
        "block_size": (
            BLOCK_SIZE
        ),
        "td_block_steps": (
            td_block_steps
        ),
        "td_block_mean": (
            td_block_mean
        ),
        "td_block_count": (
            td_block_count
        ),
        "q_block_mean": (
            q_block_mean
        ),
        "loss_block_mean": (
            loss_block_mean
        ),
        "eval_steps": (
            eval_steps
        ),
        "eval_returns": (
            eval_returns
        ),
        "final_eval": float(
            eval_returns[
                -1
            ]
        ),
        "total_steps": (
            total_steps
        ),
        "n_updates": (
            n_updates
        ),
        "replay_size": int(
            len(
                replay
            )
        ),
    }

    env.env.close()
    eval_env.env.close()

    return result


def _worker(
    cfg,
):
    key = _config_key(
        cfg
    )

    try:
        result = _run_one(
            cfg
        )

        result[
            "key"
        ] = key

        result[
            "config"
        ] = _config_dict(
            cfg
        )

        return result

    except Exception as exc:
        return {
            "key": (
                key
            ),
            "config": (
                _config_dict(
                    cfg
                )
            ),
            "error": (
                f"{type(exc).__name__}: "
                f"{exc}"
            ),
            "traceback": (
                traceback.format_exc()
            ),
        }


def main():
    selected = _load_selected()

    configs = []

    for env_id in ENV_IDS:
        item = selected[
            env_id
        ]

        for seed in DIAGNOSTIC_SEEDS:
            configs.append(
                DiagnosticDQNConfig(
                    env_id=env_id,
                    seed=seed,
                    n_steps=(
                        STEP_BUDGET[
                            env_id
                        ]
                    ),
                    learning_rate=(
                        item[
                            "learning_rate"
                        ]
                    ),
                    explorer_kwargs=(
                        item[
                            "explorer_kwargs"
                        ]
                    ),
                )
            )

    expected_total = (
        len(
            ENV_IDS
        )
        * len(
            DIAGNOSTIC_SEEDS
        )
    )

    if (
        len(configs)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected diagnostic "
            "configuration count."
        )

    existing = {}

    if OUT_PATH.exists():
        with open(
            OUT_PATH,
            "rb",
        ) as f:
            loaded = pickle.load(
                f
            )

        if not isinstance(
            loaded,
            list,
        ):
            raise TypeError(
                "Existing results "
                "must be a list."
            )

        for result in loaded:
            key = result.get(
                "key"
            )

            if key is None:
                raise KeyError(
                    "Existing result "
                    "missing key."
                )

            existing[
                key
            ] = result

    pending = []

    for cfg in configs:
        key = _config_key(
            cfg
        )

        previous = (
            existing.get(
                key
            )
        )

        if (
            previous is None
            or "error"
            in previous
        ):
            pending.append(
                cfg
            )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(
                pending
            )
            if pending
            else 1,
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "DEFINITIVE DOUBLE DQN "
        "TD-ERROR DIAGNOSTIC"
    )

    print()

    print(
        "Selected configurations:"
    )

    for env_id in ENV_IDS:
        item = selected[
            env_id
        ]

        print(
            f"  {env_id}: "
            f"lr="
            f"{item['learning_rate']:g}, "
            f"decay="
            f"{item['decay_fraction']:g}, "
            f"kwargs="
            f"{item['explorer_kwargs']}"
        )

    print()

    print(
        f"Diagnostic seeds: "
        f"{DIAGNOSTIC_SEEDS}"
    )

    print(
        f"TD block size: "
        f"{BLOCK_SIZE}"
    )

    print(
        f"Evaluation points: "
        f"{N_EVAL_POINTS}"
    )

    print(
        f"Evaluation episodes: "
        f"{N_EVAL_EPISODES}"
    )

    print()

    print(
        f"Total configs: "
        f"{expected_total}"
    )

    print(
        f"Loaded: "
        f"{len(existing)}"
    )

    print(
        f"Pending: "
        f"{len(pending)}"
    )

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    if pending:
        ctx = (
            mp.get_context(
                "spawn"
            )
        )

        with ctx.Pool(
            processes=workers
        ) as pool:
            iterator = (
                pool.imap_unordered(
                    _worker,
                    pending,
                    chunksize=1,
                )
            )

            for i, result in enumerate(
                iterator,
                start=1,
            ):
                existing[
                    result[
                        "key"
                    ]
                ] = result

                _save_atomic(
                    list(
                        existing.values()
                    ),
                    OUT_PATH,
                )

                status = (
                    "ERROR"
                    if "error"
                    in result
                    else "OK"
                )

                cfg = result[
                    "config"
                ]

                print(
                    f"{i}/"
                    f"{len(pending)} "
                    f"{status}  "
                    f"{cfg['env_id']}  "
                    f"seed="
                    f"{cfg['seed']}  "
                    f"lr="
                    f"{cfg['learning_rate']:g}"
                )

    results = list(
        existing.values()
    )

    errors = [
        result
        for result in results
        if "error"
        in result
    ]

    successful = [
        result
        for result in results
        if "error"
        not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED DEFINITIVE "
            "DQN DIAGNOSTIC RUNS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            cfg = result[
                "config"
            ]

            print(
                f"{i}. "
                f"{cfg['env_id']} "
                f"seed="
                f"{cfg['seed']} "
                f"lr="
                f"{cfg['learning_rate']}"
            )

            print(
                f"   "
                f"{result['error']}"
            )

        raise RuntimeError(
            f"{len(errors)} "
            "definitive DQN "
            "diagnostic runs failed."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Expected "
            f"{expected_total} "
            "successful runs, "
            f"found "
            f"{len(successful)}."
        )

    print()

    print(
        "Per-environment "
        "update counts:"
    )

    for env_id in ENV_IDS:
        env_results = [
            result
            for result
            in successful
            if result[
                "env_id"
            ]
            == env_id
        ]

        counts = {
            int(
                result[
                    "n_updates"
                ]
            )
            for result
            in env_results
        }

        print(
            f"  {env_id}: "
            f"{sorted(counts)}"
        )

    print()

    print(
        "DEFINITIVE DOUBLE DQN "
        "TD-ERROR DIAGNOSTIC "
        "COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/diagnose_td_error_dqn_tuned.py


In [25]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.diagnose_td_error_dqn_tuned",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Definitive Double DQN "
        "TD-error diagnostic failed."
    )

DEFINITIVE DOUBLE DQN TD-ERROR DIAGNOSTIC

Selected configurations:
  MountainCar-v0: lr=0.0003, decay=0.2, kwargs={'eps_start': 1.0, 'eps_end': 0.01, 'decay_steps': 30000, 'mode': 'linear'}
  CartPole-v1: lr=0.001, decay=0.2, kwargs={'eps_start': 1.0, 'eps_end': 0.01, 'decay_steps': 20000, 'mode': 'linear'}
  Acrobot-v1: lr=0.001, decay=0.2, kwargs={'eps_start': 1.0, 'eps_end': 0.01, 'decay_steps': 20000, 'mode': 'linear'}
  LunarLander-v3: lr=0.0003, decay=0.2, kwargs={'eps_start': 1.0, 'eps_end': 0.01, 'decay_steps': 60000, 'mode': 'linear'}

Diagnostic seeds: (3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009)
TD block size: 1000
Evaluation points: 20
Evaluation episodes: 10

Total configs: 40
Loaded: 0
Pending: 40
Detected CPUs: 20
Workers: 8
Output: src\results\diagnostics\td_error_trajectory_dqn_tuned.pkl

1/40 OK  MountainCar-v0  seed=3006  lr=0.0003
2/40 OK  MountainCar-v0  seed=3005  lr=0.0003
3/40 OK  MountainCar-v0  seed=3007  lr=0.0003
4/40 OK  MountainCar-v0  see

In [26]:
%%writefile src/scripts/analyze_td_error_dqn_tuned.py

from pathlib import Path

import src.scripts.analyze_td_error_dqn as analysis


analysis.RESULTS_PATH = Path(
    "src/results/diagnostics/"
    "td_error_trajectory_dqn_tuned.pkl"
)

analysis.SUMMARY_PATH = Path(
    "src/results/diagnostics/"
    "td_error_dqn_tuned_summary.json"
)

analysis.SEED_TABLE_PATH = Path(
    "src/results/diagnostics/"
    "td_error_dqn_tuned_seed_summary.csv"
)

analysis.DECILE_TABLE_PATH = Path(
    "src/results/diagnostics/"
    "td_error_dqn_tuned_deciles.csv"
)

analysis.TRAJECTORY_PATH = Path(
    "src/results/diagnostics/"
    "td_error_dqn_tuned_mean_trajectories.npz"
)

analysis.PLOT_DIR = Path(
    "src/results/diagnostics/"
    "td_error_dqn_tuned_plots"
)

analysis.DIAGNOSTIC_SEEDS = tuple(
    range(
        3000,
        3010,
    )
)


def main():
    print(
        "DEFINITIVE TUNED "
        "DOUBLE DQN EXPERIMENT-A "
        "ANALYSIS"
    )

    print()

    analysis.main()


if __name__ == "__main__":
    main()

Writing src/scripts/analyze_td_error_dqn_tuned.py


In [27]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.analyze_td_error_dqn_tuned",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
DEFINITIVE TUNED DOUBLE DQN EXPERIMENT-A ANALYSIS

DOUBLE DQN TD-ERROR DIAGNOSTIC ANALYSIS

Stored results: 40
Seeds per environment: 10
TD summaries are weighted by optimizer-update count.

Environment: MountainCar-v0

Weighted mean |delta|:
  first 10%: 0.019395 +/- 0.001972
  last 10%:  0.027096 +/- 0.014583
  ratio:     1.397070
  change:    +39.71%

Greedy return:
  first 10%: -200.000 +/- 0.000
  last 10%:  -200.000 +/- 0.000
  change:    +0.000

Seed-level counts:
  TD error increased: 6/10
  Greedy return improved: 0/10
  Both occurred: 0/10

Decile trajectory:
Progress    Mean |delta|       Greedy return
------------------------------------------------------------
 10%          0.019395 +/- 0.001972      -200.000 +/- 0.000
 20%          0.015116 +/- 0.000724      -199.870 +/- 0.411
 30%          0.015902 +/- 0.000911      -199.945 +/- 0.174
 40%          0.016698 +/- 0.001021      -199.905 +/- 0.227
 50%          0.018291 +/- 0.002150      -198.670 +/- 

| Environment | Linear FA                  | Tuned Double DQN          | Reading                                                      |
| ----------- | -------------------------- | ------------------------- | ------------------------------------------------------------ |
| MountainCar | Return +86.97, TD −78.52%  | Return 0, TD +39.71%      | DQN did not learn → inconclusive for inversion               |
| CartPole    | Return +310.52, TD +1111%  | Return +184.63, TD +491%  | **Strong inversion in both**                                 |
| Acrobot     | Return +257.77, TD −27.90% | Return +354.20, TD +2858% | **Deep FA changes the picture dramatically**                 |
| LunarLander | Return +474.33, TD +6.86%  | Return −69.94, TD +50.42% | Linear inversion; DQN becomes unstable rather than improving |


Raw TD-error magnitude is not a universally monotonic learning-progress signal under function approximation. Clear inversion occurs under Double DQN on CartPole and Acrobot, while other tasks exhibit different or unstable relationships between TD magnitude and policy performance.

# Experiment B: VDBE Degenration Test

Now we test the second part of the argument:

$$ \boxed{\text{Does VDBE tuning drive }\sigma\text{ toward its TD-independent limit?}} $$

For VDBE,

$$ f(\delta) = \frac{1-e^{-|\alpha\delta|/\sigma}} {1+e^{-|\alpha\delta|/\sigma}}, $$

and

$$ \epsilon_{t+1} = \frac1{|A|}f(\delta_t) + \left(1-\frac1{|A|}\right)\epsilon_t. $$

As

$$ \sigma\rightarrow\infty, $$ $$ f(\delta)\rightarrow0, $$

so

$$ \boxed{ \epsilon_{t+1} \rightarrow \left(1-\frac1{|A|}\right)\epsilon_t } $$

and VDBE has stopped using TD error. This limit follows algebraically, independently of whether the value approximator is linear or neural.

In [28]:
%%writefile src/scripts/diagnose_vdbe_degeneration_linear.py

import json
import os

from pathlib import Path

from src.runner import RunConfig
from src.sweep import run_stage
from src.sweep_configs import (
    STEP_BUDGET,
)


SELECTED_ALPHA_PATH = Path(
    "src/results/tuning/"
    "selected_alpha.json"
)

OUT_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_degeneration_linear.pkl"
)


DIAGNOSTIC_ENVS = (
    "MountainCar-v0",
    "Acrobot-v1",
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        4000,
        4010,
    )
)

SIGMA_GRID = (
    0.5,
    1.0,
    5.0,
    20.0,
    100.0,
    500.0,
    2000.0,
    10000.0,
)

LAMBDA_FIXED = 0.9

GAMMA = 1.0
Q_INIT = 0.0
REWARD_SCALE = 1.0

N_BINS = 100
N_EVAL_POINTS = 20
N_EVAL_EPISODES = 10

MAX_WORKERS = 8


def _load_selected_alpha():
    if not SELECTED_ALPHA_PATH.exists():
        raise FileNotFoundError(
            "Selected-alpha file "
            "not found: "
            f"{SELECTED_ALPHA_PATH}"
        )

    with open(
        SELECTED_ALPHA_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        selected = json.load(
            f
        )

    if not isinstance(
        selected,
        dict,
    ):
        raise TypeError(
            "selected_alpha.json "
            "must contain a dictionary."
        )

    output = {}

    for env_id in DIAGNOSTIC_ENVS:
        if env_id not in selected:
            raise KeyError(
                f"Missing selected "
                f"alpha for {env_id}."
            )

        alpha_bar = float(
            selected[
                env_id
            ]
        )

        if alpha_bar <= 0.0:
            raise ValueError(
                f"Invalid alpha_bar "
                f"for {env_id}: "
                f"{alpha_bar}"
            )

        output[
            env_id
        ] = alpha_bar

    return output


def main():
    selected_alpha = (
        _load_selected_alpha()
    )

    configs = []

    for env_id in DIAGNOSTIC_ENVS:
        budget = int(
            STEP_BUDGET[
                env_id
            ]
        )

        alpha_bar = float(
            selected_alpha[
                env_id
            ]
        )

        for sigma in SIGMA_GRID:
            for seed in DIAGNOSTIC_SEEDS:
                configs.append(
                    RunConfig(
                        env_id=env_id,
                        algo="sarsa-lambda",
                        explorer="vdbe",
                        explorer_kwargs={
                            "sigma": float(
                                sigma
                            ),
                            "eps_init": 1.0,
                            "alpha_scale": 1.0,
                            "eps_min": 0.0,
                        },
                        seed=int(
                            seed
                        ),
                        n_steps=budget,
                        alpha_bar=(
                            alpha_bar
                        ),
                        gamma=GAMMA,
                        lam=LAMBDA_FIXED,
                        q_init=Q_INIT,
                        reward_scale=(
                            REWARD_SCALE
                        ),
                        n_bins=N_BINS,
                        n_eval_points=(
                            N_EVAL_POINTS
                        ),
                        n_eval_episodes=(
                            N_EVAL_EPISODES
                        ),
                    )
                )

    expected_total = (
        len(
            DIAGNOSTIC_ENVS
        )
        * len(
            SIGMA_GRID
        )
        * len(
            DIAGNOSTIC_SEEDS
        )
    )

    if (
        len(configs)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected VDBE "
            "diagnostic config "
            "count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(configs)}."
        )

    seen = set()

    for cfg in configs:
        sigma = float(
            cfg.explorer_kwargs[
                "sigma"
            ]
        )

        key = (
            cfg.env_id,
            float(
                cfg.alpha_bar
            ),
            sigma,
            int(
                cfg.seed
            ),
        )

        if key in seen:
            raise RuntimeError(
                "Duplicate diagnostic "
                f"configuration: "
                f"{key}"
            )

        seen.add(
            key
        )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(
                configs
            ),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "LINEAR VDBE "
        "DEGENERATION DIAGNOSTIC"
    )

    print()

    print(
        "Environments:"
    )

    for env_id in DIAGNOSTIC_ENVS:
        print(
            f"  {env_id}: "
            f"alpha_bar="
            f"{selected_alpha[env_id]:g}, "
            f"budget="
            f"{STEP_BUDGET[env_id]}"
        )

    print()

    print(
        "Sigma grid:"
    )

    for sigma in SIGMA_GRID:
        print(
            f"  {sigma:g}"
        )

    print()

    print(
        "VDBE fixed parameters:"
    )

    print(
        "  eps_init=1.0"
    )

    print(
        "  alpha_scale=1.0"
    )

    print(
        "  eps_min=0.0"
    )

    print()

    print(
        "Linear learner:"
    )

    print(
        "  Sarsa(lambda)"
    )

    print(
        f"  lambda="
        f"{LAMBDA_FIXED}"
    )

    print(
        f"  gamma="
        f"{GAMMA}"
    )

    print()

    print(
        f"Diagnostic seeds: "
        f"{DIAGNOSTIC_SEEDS}"
    )

    print(
        f"Environments: "
        f"{len(DIAGNOSTIC_ENVS)}"
    )

    print(
        f"Sigma values: "
        f"{len(SIGMA_GRID)}"
    )

    print(
        f"Seeds per cell: "
        f"{len(DIAGNOSTIC_SEEDS)}"
    )

    print(
        f"Runs per environment: "
        f"{len(SIGMA_GRID) * len(DIAGNOSTIC_SEEDS)}"
    )

    print(
        f"Total runs: "
        f"{len(configs)}"
    )

    print()

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=OUT_PATH,
        label=(
            "linear VDBE degeneration"
        ),
        workers=workers,
        save_every=8,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error"
        in result
    ]

    successful = [
        result
        for result in results
        if "error"
        not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED LINEAR VDBE "
            "DIAGNOSTIC RUNS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            cfg = result.get(
                "config",
                {}
            )

            kwargs = cfg.get(
                "explorer_kwargs",
                {}
            )

            print(
                f"{i}. "
                f"env="
                f"{cfg.get('env_id')} "
                f"sigma="
                f"{kwargs.get('sigma')} "
                f"seed="
                f"{cfg.get('seed')} "
                f"alpha_bar="
                f"{cfg.get('alpha_bar')}"
            )

            print(
                f"   "
                f"{result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} "
            "linear VDBE diagnostic "
            "runs failed."
        )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected stored "
            "result count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(results)}."
        )

    if (
        len(successful)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected successful "
            "result count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(successful)}."
        )

    print()

    print(
        "LINEAR VDBE "
        "DEGENERATION DIAGNOSTIC "
        "COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/diagnose_vdbe_degeneration_linear.py


In [29]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.diagnose_vdbe_degeneration_linear",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Linear VDBE degeneration "
        "diagnostic failed."
    )

LINEAR VDBE DEGENERATION DIAGNOSTIC

Environments:
  MountainCar-v0: alpha_bar=0.75, budget=150000
  Acrobot-v1: alpha_bar=0.5, budget=100000

Sigma grid:
  0.5
  1
  5
  20
  100
  500
  2000
  10000

VDBE fixed parameters:
  eps_init=1.0
  alpha_scale=1.0
  eps_min=0.0

Linear learner:
  Sarsa(lambda)
  lambda=0.9
  gamma=1.0

Diagnostic seeds: (4000, 4001, 4002, 4003, 4004, 4005, 4006, 4007, 4008, 4009)
Environments: 2
Sigma values: 8
Seeds per cell: 10
Runs per environment: 80
Total runs: 160

Detected CPUs: 20
Workers: 8
Output: src\results\diagnostics\vdbe_degeneration_linear.pkl

[linear VDBE degeneration] loaded=0 pending=160 workers=8
[linear VDBE degeneration] 1/160 OK
[linear VDBE degeneration] 2/160 OK
[linear VDBE degeneration] 3/160 OK
[linear VDBE degeneration] 4/160 OK
[linear VDBE degeneration] 5/160 OK
[linear VDBE degeneration] 6/160 OK
[linear VDBE degeneration] 7/160 OK
[linear VDBE degeneration] 8/160 OK
[linear VDBE degeneration] 9/160 OK
[linear VDBE degeneratio

In [30]:
%%writefile src/scripts/analyze_vdbe_degeneration_linear.py
import csv
import json
import pickle

from pathlib import Path

import numpy as np


RESULTS_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_degeneration_linear.pkl"
)

SUMMARY_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_degeneration_linear_summary.json"
)

CELL_CSV_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_degeneration_linear_cells.csv"
)

PAIR_CSV_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_degeneration_linear_pairwise.csv"
)


DIAGNOSTIC_ENVS = (
    "MountainCar-v0",
    "Acrobot-v1",
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        4000,
        4010,
    )
)

SIGMA_GRID = (
    0.5,
    1.0,
    5.0,
    20.0,
    100.0,
    500.0,
    2000.0,
    10000.0,
)

HIGH_SIGMAS = (
    500.0,
    2000.0,
    10000.0,
)


def _write_json_atomic(
    obj,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

    tmp.replace(
        path
    )


def _array_equal(
    a,
    b,
):
    a = np.asarray(
        a
    )

    b = np.asarray(
        b
    )

    if (
        a.shape
        != b.shape
    ):
        return False

    try:
        return bool(
            np.array_equal(
                a,
                b,
                equal_nan=True,
            )
        )

    except TypeError:
        if (
            np.issubdtype(
                a.dtype,
                np.floating,
            )
            or np.issubdtype(
                b.dtype,
                np.floating,
            )
        ):
            nan_same = np.array_equal(
                np.isnan(
                    a
                ),
                np.isnan(
                    b
                ),
            )

            if not nan_same:
                return False

            valid = (
                ~np.isnan(
                    a
                )
                & ~np.isnan(
                    b
                )
            )

            return bool(
                np.array_equal(
                    a[
                        valid
                    ],
                    b[
                        valid
                    ],
                )
            )

        return bool(
            np.array_equal(
                a,
                b,
            )
        )


def _max_abs_diff(
    a,
    b,
):
    a = np.asarray(
        a,
        dtype=np.float64,
    )

    b = np.asarray(
        b,
        dtype=np.float64,
    )

    if (
        a.shape
        != b.shape
    ):
        return float(
            "inf"
        )

    valid = (
        np.isfinite(
            a
        )
        & np.isfinite(
            b
        )
    )

    if not np.any(
        valid
    ):
        return 0.0

    return float(
        np.max(
            np.abs(
                a[
                    valid
                ]
                - b[
                    valid
                ]
            )
        )
    )


def _close(
    a,
    b,
):
    return bool(
        np.isclose(
            float(
                a
            ),
            float(
                b
            ),
            rtol=0.0,
            atol=1e-12,
        )
    )


def _extract_sigma(
    result,
):
    kwargs = result[
        "explorer_kwargs"
    ]

    if (
        "sigma"
        not in kwargs
    ):
        raise KeyError(
            "Result is missing "
            "VDBE sigma."
        )

    return float(
        kwargs[
            "sigma"
        ]
    )


def _validate_result(
    result,
):
    required = (
        "env_id",
        "algo",
        "explorer",
        "explorer_kwargs",
        "seed",
        "n_steps",
        "alpha_bar",
        "gamma",
        "lam",
        "final_eval",
        "ep_steps",
        "ep_returns",
        "epsilon_curve",
        "td_curve",
        "return_curve",
        "eval_steps",
        "eval_returns",
    )

    for field in required:
        if field not in result:
            raise KeyError(
                f"Result missing "
                f"{field}."
            )

    env_id = result[
        "env_id"
    ]

    if (
        env_id
        not in DIAGNOSTIC_ENVS
    ):
        raise ValueError(
            "Unexpected environment: "
            f"{env_id}"
        )

    if (
        result[
            "algo"
        ]
        != "sarsa-lambda"
    ):
        raise ValueError(
            "Expected "
            "sarsa-lambda."
        )

    if (
        result[
            "explorer"
        ]
        != "vdbe"
    ):
        raise ValueError(
            "Expected VDBE."
        )

    seed = int(
        result[
            "seed"
        ]
    )

    if (
        seed
        not in DIAGNOSTIC_SEEDS
    ):
        raise ValueError(
            "Unexpected diagnostic "
            f"seed: {seed}"
        )

    sigma = _extract_sigma(
        result
    )

    if not any(
        _close(
            sigma,
            candidate,
        )
        for candidate
        in SIGMA_GRID
    ):
        raise ValueError(
            f"Unexpected sigma: "
            f"{sigma}"
        )

    if not np.isfinite(
        float(
            result[
                "final_eval"
            ]
        )
    ):
        raise ValueError(
            "Non-finite final "
            "evaluation."
        )

    return (
        env_id,
        sigma,
        seed,
    )


def _get_result(
    grouped,
    env_id,
    sigma,
    seed,
):
    matches = [
        result
        for (
            key_env,
            key_sigma,
            key_seed
        ), result
        in grouped.items()
        if (
            key_env
            == env_id
            and _close(
                key_sigma,
                sigma,
            )
            and key_seed
            == seed
        )
    ]

    if (
        len(
            matches
        )
        != 1
    ):
        raise RuntimeError(
            "Expected exactly one "
            f"result for "
            f"{env_id}, "
            f"sigma={sigma:g}, "
            f"seed={seed}; "
            f"found "
            f"{len(matches)}."
        )

    return matches[
        0
    ]


def _behaviour_equal(
    a,
    b,
):
    checks = {
        "ep_steps": (
            _array_equal(
                a[
                    "ep_steps"
                ],
                b[
                    "ep_steps"
                ],
            )
        ),
        "ep_returns": (
            _array_equal(
                a[
                    "ep_returns"
                ],
                b[
                    "ep_returns"
                ],
            )
        ),
        "return_curve": (
            _array_equal(
                a[
                    "return_curve"
                ],
                b[
                    "return_curve"
                ],
            )
        ),
        "td_curve": (
            _array_equal(
                a[
                    "td_curve"
                ],
                b[
                    "td_curve"
                ],
            )
        ),
        "eval_steps": (
            _array_equal(
                a[
                    "eval_steps"
                ],
                b[
                    "eval_steps"
                ],
            )
        ),
        "eval_returns": (
            _array_equal(
                a[
                    "eval_returns"
                ],
                b[
                    "eval_returns"
                ],
            )
        ),
        "final_eval": bool(
            float(
                a[
                    "final_eval"
                ]
            )
            == float(
                b[
                    "final_eval"
                ]
            )
        ),
    }

    exact = bool(
        all(
            checks.values()
        )
    )

    return (
        exact,
        checks,
    )


def main():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"Missing diagnostic "
            f"results: "
            f"{RESULTS_PATH}"
        )

    with open(
        RESULTS_PATH,
        "rb",
    ) as f:
        results = pickle.load(
            f
        )

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            "Results must be "
            "stored as a list."
        )

    expected_total = (
        len(
            DIAGNOSTIC_ENVS
        )
        * len(
            SIGMA_GRID
        )
        * len(
            DIAGNOSTIC_SEEDS
        )
    )

    print(
        "LINEAR VDBE "
        "DEGENERATION ANALYSIS"
    )

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Expected results: "
        f"{expected_total}"
    )

    errors = [
        result
        for result in results
        if "error"
        in result
    ]

    if errors:
        raise RuntimeError(
            f"Found "
            f"{len(errors)} "
            "error records."
        )

    if (
        len(results)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected result "
            "count."
        )

    grouped = {}

    for result in results:
        (
            env_id,
            sigma,
            seed,
        ) = _validate_result(
            result
        )

        key = (
            env_id,
            sigma,
            seed,
        )

        if key in grouped:
            raise RuntimeError(
                "Duplicate result: "
                f"{key}"
            )

        grouped[
            key
        ] = result

    if (
        len(grouped)
        != expected_total
    ):
        raise RuntimeError(
            "Unique result count "
            "does not match "
            "expected count."
        )

    summary = {}
    cell_rows = []
    pair_rows = []

    print()

    for env_id in DIAGNOSTIC_ENVS:
        print(
            "=" * 100
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 100
        )

        print()

        print(
            "Sigma        "
            "Mean final return    "
            "Std            "
            "Seed scores"
        )

        print(
            "-" * 100
        )

        cell_stats = []

        for sigma in SIGMA_GRID:
            scores = []

            for seed in DIAGNOSTIC_SEEDS:
                result = _get_result(
                    grouped,
                    env_id,
                    sigma,
                    seed,
                )

                scores.append(
                    float(
                        result[
                            "final_eval"
                        ]
                    )
                )

            scores = np.asarray(
                scores,
                dtype=np.float64,
            )

            mean_score = float(
                np.mean(
                    scores
                )
            )

            std_score = float(
                np.std(
                    scores,
                    ddof=1,
                )
            )

            cell = {
                "sigma": float(
                    sigma
                ),
                "mean_final_eval": (
                    mean_score
                ),
                "std_final_eval": (
                    std_score
                ),
                "seed_scores": {
                    str(seed): float(
                        scores[
                            i
                        ]
                    )
                    for i, seed
                    in enumerate(
                        DIAGNOSTIC_SEEDS
                    )
                },
            }

            cell_stats.append(
                cell
            )

            cell_rows.append(
                {
                    "env_id": (
                        env_id
                    ),
                    "sigma": float(
                        sigma
                    ),
                    "mean_final_eval": (
                        mean_score
                    ),
                    "std_final_eval": (
                        std_score
                    ),
                }
            )

            score_text = (
                ", ".join(
                    f"{value:.10f}"
                    for value
                    in scores
                )
            )

            print(
                f"{sigma:<12g}"
                f"{mean_score:>18.10f}"
                f"{std_score:>15.10f}    "
                f"{score_text}"
            )

        ranked = sorted(
            cell_stats,
            key=lambda item: (
                item[
                    "mean_final_eval"
                ],
                -item[
                    "std_final_eval"
                ],
            ),
            reverse=True,
        )

        winner = ranked[
            0
        ]

        print()

        print(
            "BEST MEAN FINAL RETURN:"
        )

        print(
            f"  sigma = "
            f"{winner['sigma']:g}"
        )

        print(
            f"  mean = "
            f"{winner['mean_final_eval']:.10f}"
        )

        print(
            f"  std = "
            f"{winner['std_final_eval']:.10f}"
        )

        print()

        print(
            "HIGH-SIGMA "
            "PAIRWISE EXACTNESS"
        )

        print(
            "-" * 100
        )

        high_pairs = (
            (
                500.0,
                2000.0,
            ),
            (
                2000.0,
                10000.0,
            ),
            (
                500.0,
                10000.0,
            ),
        )

        pair_summaries = []

        for (
            sigma_a,
            sigma_b,
        ) in high_pairs:
            exact_behaviour_count = 0
            exact_final_count = 0
            exact_eval_count = 0
            exact_return_curve_count = 0
            exact_td_curve_count = 0
            exact_epsilon_curve_count = 0

            max_eval_diff = 0.0
            max_return_diff = 0.0
            max_td_diff = 0.0
            max_epsilon_diff = 0.0

            seed_details = []

            for seed in DIAGNOSTIC_SEEDS:
                a = _get_result(
                    grouped,
                    env_id,
                    sigma_a,
                    seed,
                )

                b = _get_result(
                    grouped,
                    env_id,
                    sigma_b,
                    seed,
                )

                (
                    behaviour_exact,
                    checks,
                ) = _behaviour_equal(
                    a,
                    b,
                )

                epsilon_exact = (
                    _array_equal(
                        a[
                            "epsilon_curve"
                        ],
                        b[
                            "epsilon_curve"
                        ],
                    )
                )

                eval_diff = (
                    _max_abs_diff(
                        a[
                            "eval_returns"
                        ],
                        b[
                            "eval_returns"
                        ],
                    )
                )

                return_diff = (
                    _max_abs_diff(
                        a[
                            "return_curve"
                        ],
                        b[
                            "return_curve"
                        ],
                    )
                )

                td_diff = (
                    _max_abs_diff(
                        a[
                            "td_curve"
                        ],
                        b[
                            "td_curve"
                        ],
                    )
                )

                epsilon_diff = (
                    _max_abs_diff(
                        a[
                            "epsilon_curve"
                        ],
                        b[
                            "epsilon_curve"
                        ],
                    )
                )

                exact_behaviour_count += int(
                    behaviour_exact
                )

                exact_final_count += int(
                    checks[
                        "final_eval"
                    ]
                )

                exact_eval_count += int(
                    checks[
                        "eval_returns"
                    ]
                )

                exact_return_curve_count += int(
                    checks[
                        "return_curve"
                    ]
                )

                exact_td_curve_count += int(
                    checks[
                        "td_curve"
                    ]
                )

                exact_epsilon_curve_count += int(
                    epsilon_exact
                )

                max_eval_diff = max(
                    max_eval_diff,
                    eval_diff,
                )

                max_return_diff = max(
                    max_return_diff,
                    return_diff,
                )

                max_td_diff = max(
                    max_td_diff,
                    td_diff,
                )

                max_epsilon_diff = max(
                    max_epsilon_diff,
                    epsilon_diff,
                )

                seed_details.append(
                    {
                        "seed": int(
                            seed
                        ),
                        "behaviour_exact": (
                            behaviour_exact
                        ),
                        "final_eval_exact": (
                            checks[
                                "final_eval"
                            ]
                        ),
                        "eval_returns_exact": (
                            checks[
                                "eval_returns"
                            ]
                        ),
                        "return_curve_exact": (
                            checks[
                                "return_curve"
                            ]
                        ),
                        "td_curve_exact": (
                            checks[
                                "td_curve"
                            ]
                        ),
                        "epsilon_curve_exact": (
                            epsilon_exact
                        ),
                        "max_eval_diff": (
                            eval_diff
                        ),
                        "max_return_curve_diff": (
                            return_diff
                        ),
                        "max_td_curve_diff": (
                            td_diff
                        ),
                        "max_epsilon_curve_diff": (
                            epsilon_diff
                        ),
                    }
                )

            pair_summary = {
                "sigma_a": (
                    sigma_a
                ),
                "sigma_b": (
                    sigma_b
                ),
                "exact_behaviour_seeds": (
                    exact_behaviour_count
                ),
                "exact_final_eval_seeds": (
                    exact_final_count
                ),
                "exact_eval_curve_seeds": (
                    exact_eval_count
                ),
                "exact_return_curve_seeds": (
                    exact_return_curve_count
                ),
                "exact_td_curve_seeds": (
                    exact_td_curve_count
                ),
                "exact_epsilon_curve_seeds": (
                    exact_epsilon_curve_count
                ),
                "max_eval_difference": (
                    max_eval_diff
                ),
                "max_return_curve_difference": (
                    max_return_diff
                ),
                "max_td_curve_difference": (
                    max_td_diff
                ),
                "max_epsilon_curve_difference": (
                    max_epsilon_diff
                ),
                "seed_details": (
                    seed_details
                ),
            }

            pair_summaries.append(
                pair_summary
            )

            pair_rows.append(
                {
                    "env_id": (
                        env_id
                    ),
                    "sigma_a": (
                        sigma_a
                    ),
                    "sigma_b": (
                        sigma_b
                    ),
                    "exact_behaviour_seeds": (
                        exact_behaviour_count
                    ),
                    "exact_final_eval_seeds": (
                        exact_final_count
                    ),
                    "exact_eval_curve_seeds": (
                        exact_eval_count
                    ),
                    "exact_return_curve_seeds": (
                        exact_return_curve_count
                    ),
                    "exact_td_curve_seeds": (
                        exact_td_curve_count
                    ),
                    "exact_epsilon_curve_seeds": (
                        exact_epsilon_curve_count
                    ),
                    "max_eval_difference": (
                        max_eval_diff
                    ),
                    "max_return_curve_difference": (
                        max_return_diff
                    ),
                    "max_td_curve_difference": (
                        max_td_diff
                    ),
                    "max_epsilon_curve_difference": (
                        max_epsilon_diff
                    ),
                }
            )

            print(
                f"sigma "
                f"{sigma_a:g} "
                f"vs "
                f"{sigma_b:g}:"
            )

            print(
                "  exact full "
                "behaviour/learning "
                f"trajectory: "
                f"{exact_behaviour_count}/10"
            )

            print(
                "  exact final eval: "
                f"{exact_final_count}/10"
            )

            print(
                "  exact eval curve: "
                f"{exact_eval_count}/10"
            )

            print(
                "  exact return curve: "
                f"{exact_return_curve_count}/10"
            )

            print(
                "  exact TD curve: "
                f"{exact_td_curve_count}/10"
            )

            print(
                "  exact epsilon curve: "
                f"{exact_epsilon_curve_count}/10"
            )

            print(
                "  maximum absolute "
                "differences:"
            )

            print(
                f"    eval = "
                f"{max_eval_diff:.16g}"
            )

            print(
                f"    return curve = "
                f"{max_return_diff:.16g}"
            )

            print(
                f"    TD curve = "
                f"{max_td_diff:.16g}"
            )

            print(
                f"    epsilon curve = "
                f"{max_epsilon_diff:.16g}"
            )

            print()

        high_sigma_means = {
            str(
                int(
                    sigma
                )
            ): float(
                next(
                    cell[
                        "mean_final_eval"
                    ]
                    for cell
                    in cell_stats
                    if _close(
                        cell[
                            "sigma"
                        ],
                        sigma,
                    )
                )
            )
            for sigma
            in HIGH_SIGMAS
        }

        exact_mean_plateau = bool(
            high_sigma_means[
                "500"
            ]
            == high_sigma_means[
                "2000"
            ]
            == high_sigma_means[
                "10000"
            ]
        )

        all_high_pairs_behaviour_exact = bool(
            all(
                item[
                    "exact_behaviour_seeds"
                ]
                == len(
                    DIAGNOSTIC_SEEDS
                )
                for item
                in pair_summaries
            )
        )

        all_high_pairs_final_exact = bool(
            all(
                item[
                    "exact_final_eval_seeds"
                ]
                == len(
                    DIAGNOSTIC_SEEDS
                )
                for item
                in pair_summaries
            )
        )

        print(
            "HIGH-SIGMA PLATEAU "
            "SUMMARY"
        )

        print(
            "-" * 100
        )

        print(
            f"mean at sigma=500:   "
            f"{high_sigma_means['500']:.10f}"
        )

        print(
            f"mean at sigma=2000:  "
            f"{high_sigma_means['2000']:.10f}"
        )

        print(
            f"mean at sigma=10000: "
            f"{high_sigma_means['10000']:.10f}"
        )

        print()

        print(
            "Exact equality of "
            "all three means: "
            f"{exact_mean_plateau}"
        )

        print(
            "All high-sigma pairs "
            "have identical final "
            "eval for all seeds: "
            f"{all_high_pairs_final_exact}"
        )

        print(
            "All high-sigma pairs "
            "have identical complete "
            "behaviour/learning "
            "trajectory for all seeds: "
            f"{all_high_pairs_behaviour_exact}"
        )

        print()

        if (
            all_high_pairs_behaviour_exact
        ):
            interpretation = (
                "EXACT BEHAVIOURAL "
                "PLATEAU"
            )

        elif (
            all_high_pairs_final_exact
            or exact_mean_plateau
        ):
            interpretation = (
                "EXACT PERFORMANCE "
                "PLATEAU, BUT INTERNAL "
                "TRAJECTORIES ARE NOT "
                "FULLY IDENTICAL"
            )

        else:
            interpretation = (
                "NO EXACT HIGH-SIGMA "
                "PLATEAU"
            )

        print(
            f"RESULT: "
            f"{interpretation}"
        )

        print()
        print()

        summary[
            env_id
        ] = {
            "cells": (
                cell_stats
            ),
            "winner": (
                winner
            ),
            "high_sigma_means": (
                high_sigma_means
            ),
            "exact_mean_plateau": (
                exact_mean_plateau
            ),
            "all_high_pairs_final_exact": (
                all_high_pairs_final_exact
            ),
            "all_high_pairs_behaviour_exact": (
                all_high_pairs_behaviour_exact
            ),
            "pairwise_high_sigma": (
                pair_summaries
            ),
            "interpretation": (
                interpretation
            ),
        }

    SUMMARY_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    with open(
        CELL_CSV_PATH,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=(
                "env_id",
                "sigma",
                "mean_final_eval",
                "std_final_eval",
            ),
        )

        writer.writeheader()

        writer.writerows(
            cell_rows
        )

    with open(
        PAIR_CSV_PATH,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=(
                "env_id",
                "sigma_a",
                "sigma_b",
                "exact_behaviour_seeds",
                "exact_final_eval_seeds",
                "exact_eval_curve_seeds",
                "exact_return_curve_seeds",
                "exact_td_curve_seeds",
                "exact_epsilon_curve_seeds",
                "max_eval_difference",
                "max_return_curve_difference",
                "max_td_curve_difference",
                "max_epsilon_curve_difference",
            ),
        )

        writer.writeheader()

        writer.writerows(
            pair_rows
        )

    print(
        "=" * 100
    )

    print(
        "OVERALL EXPERIMENT-B "
        "RESULT"
    )

    print(
        "=" * 100
    )

    print()

    for env_id in DIAGNOSTIC_ENVS:
        item = summary[
            env_id
        ]

        print(
            f"{env_id}: "
            f"{item['interpretation']}"
        )

    print()

    exact_both = bool(
        all(
            summary[
                env_id
            ][
                "all_high_pairs_behaviour_exact"
            ]
            for env_id
            in DIAGNOSTIC_ENVS
        )
    )

    performance_plateau_both = bool(
        all(
            (
                summary[
                    env_id
                ][
                    "all_high_pairs_final_exact"
                ]
                or summary[
                    env_id
                ][
                    "exact_mean_plateau"
                ]
            )
            for env_id
            in DIAGNOSTIC_ENVS
        )
    )

    print(
        "Exact behavioural plateau "
        "in both environments: "
        f"{exact_both}"
    )

    print(
        "Exact performance plateau "
        "in both environments: "
        f"{performance_plateau_both}"
    )

    print()

    print(
        "Summary JSON:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    print(
        "Cell CSV:"
    )

    print(
        CELL_CSV_PATH
    )

    print()

    print(
        "Pairwise CSV:"
    )

    print(
        PAIR_CSV_PATH
    )

    print()

    print(
        "LINEAR VDBE "
        "DEGENERATION ANALYSIS "
        "COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/analyze_vdbe_degeneration_linear.py


In [31]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.analyze_vdbe_degeneration_linear",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 0

STDOUT:
LINEAR VDBE DEGENERATION ANALYSIS

Stored results: 160
Expected results: 160

Environment: MountainCar-v0

Sigma        Mean final return    Std            Seed scores
----------------------------------------------------------------------------------------------------
0.5            -200.0000000000   0.0000000000    -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000
1              -200.0000000000   0.0000000000    -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000, -200.0000000000
5              -189.6400000000  22.6403867262    -200.0000000000, -200.0000000000, -188.9000000000, -200.0000000000, -200.0000000000, -179.1000000000, -200.0000000000, -200.0000000000, -128.4000000000, -200.0000000000
20             -159.0500000000  24.8426716223    -148

In [32]:
%%writefile src/scripts/extend_vdbe_degeneration_linear.py

import json
import os

from pathlib import Path

from src.runner import RunConfig
from src.sweep import run_stage
from src.sweep_configs import STEP_BUDGET


SELECTED_ALPHA_PATH = Path(
    "src/results/tuning/"
    "selected_alpha.json"
)

OUT_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_degeneration_linear_extension.pkl"
)

DIAGNOSTIC_ENVS = (
    "MountainCar-v0",
    "Acrobot-v1",
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        4000,
        4010,
    )
)

EXTENSION_SIGMAS = (
    50_000.0,
    100_000.0,
    500_000.0,
    1_000_000.0,
    10_000_000.0,
    float("inf"),
)

LAMBDA_FIXED = 0.9

GAMMA = 1.0
Q_INIT = 0.0
REWARD_SCALE = 1.0

N_BINS = 100
N_EVAL_POINTS = 20
N_EVAL_EPISODES = 10

MAX_WORKERS = 8


def _sigma_text(
    sigma,
):
    if sigma == float("inf"):
        return "inf"

    return f"{sigma:g}"


def _load_selected_alpha():
    if not SELECTED_ALPHA_PATH.exists():
        raise FileNotFoundError(
            "Selected-alpha file "
            "not found: "
            f"{SELECTED_ALPHA_PATH}"
        )

    with open(
        SELECTED_ALPHA_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        selected = json.load(
            f
        )

    output = {}

    for env_id in DIAGNOSTIC_ENVS:
        if env_id not in selected:
            raise KeyError(
                f"Missing selected "
                f"alpha for {env_id}."
            )

        alpha_bar = float(
            selected[
                env_id
            ]
        )

        if alpha_bar <= 0:
            raise ValueError(
                f"Invalid alpha_bar "
                f"for {env_id}."
            )

        output[
            env_id
        ] = alpha_bar

    return output


def main():
    selected_alpha = (
        _load_selected_alpha()
    )

    configs = []

    for env_id in DIAGNOSTIC_ENVS:
        budget = int(
            STEP_BUDGET[
                env_id
            ]
        )

        for sigma in EXTENSION_SIGMAS:
            for seed in DIAGNOSTIC_SEEDS:
                configs.append(
                    RunConfig(
                        env_id=env_id,
                        algo="sarsa-lambda",
                        explorer="vdbe",
                        explorer_kwargs={
                            "sigma": float(
                                sigma
                            ),
                            "eps_init": 1.0,
                            "alpha_scale": 1.0,
                            "eps_min": 0.0,
                        },
                        seed=int(
                            seed
                        ),
                        n_steps=budget,
                        alpha_bar=float(
                            selected_alpha[
                                env_id
                            ]
                        ),
                        gamma=GAMMA,
                        lam=LAMBDA_FIXED,
                        q_init=Q_INIT,
                        reward_scale=(
                            REWARD_SCALE
                        ),
                        n_bins=N_BINS,
                        n_eval_points=(
                            N_EVAL_POINTS
                        ),
                        n_eval_episodes=(
                            N_EVAL_EPISODES
                        ),
                    )
                )

    expected_total = (
        len(
            DIAGNOSTIC_ENVS
        )
        * len(
            EXTENSION_SIGMAS
        )
        * len(
            DIAGNOSTIC_SEEDS
        )
    )

    if len(
        configs
    ) != expected_total:
        raise RuntimeError(
            "Unexpected extension "
            "configuration count: "
            f"expected "
            f"{expected_total}, "
            f"found "
            f"{len(configs)}."
        )

    seen = set()

    for cfg in configs:
        sigma = float(
            cfg.explorer_kwargs[
                "sigma"
            ]
        )

        key = (
            cfg.env_id,
            sigma,
            int(
                cfg.seed
            ),
        )

        if key in seen:
            raise RuntimeError(
                "Duplicate extension "
                f"configuration: "
                f"{key}"
            )

        seen.add(
            key
        )

    cpu_count = (
        os.cpu_count()
        or 2
    )

    workers = max(
        1,
        min(
            MAX_WORKERS,
            len(
                configs
            ),
            max(
                1,
                cpu_count // 2,
            ),
        ),
    )

    print(
        "LINEAR VDBE "
        "DEGENERATION LIMIT "
        "EXTENSION"
    )

    print()

    print(
        "Environments:"
    )

    for env_id in DIAGNOSTIC_ENVS:
        print(
            f"  {env_id}: "
            f"alpha_bar="
            f"{selected_alpha[env_id]:g}, "
            f"budget="
            f"{STEP_BUDGET[env_id]}"
        )

    print()

    print(
        "Extension sigma values:"
    )

    for sigma in EXTENSION_SIGMAS:
        print(
            f"  "
            f"{_sigma_text(sigma)}"
        )

    print()

    print(
        "The sigma=inf condition "
        "is the exact analytical "
        "VDBE limit."
    )

    print()

    print(
        f"Diagnostic seeds: "
        f"{DIAGNOSTIC_SEEDS}"
    )

    print(
        f"Sigma values: "
        f"{len(EXTENSION_SIGMAS)}"
    )

    print(
        f"Seeds per cell: "
        f"{len(DIAGNOSTIC_SEEDS)}"
    )

    print(
        f"Runs per environment: "
        f"{len(EXTENSION_SIGMAS) * len(DIAGNOSTIC_SEEDS)}"
    )

    print(
        f"Total new runs: "
        f"{len(configs)}"
    )

    print()

    print(
        f"Detected CPUs: "
        f"{cpu_count}"
    )

    print(
        f"Workers: "
        f"{workers}"
    )

    print(
        f"Output: "
        f"{OUT_PATH}"
    )

    print()

    results = run_stage(
        configs=configs,
        out_path=OUT_PATH,
        label=(
            "linear VDBE limit extension"
        ),
        workers=workers,
        save_every=8,
        retry_errors=False,
    )

    errors = [
        result
        for result in results
        if "error" in result
    ]

    successful = [
        result
        for result in results
        if "error" not in result
    ]

    print()

    print(
        f"Stored results: "
        f"{len(results)}"
    )

    print(
        f"Successful: "
        f"{len(successful)}"
    )

    print(
        f"Errors: "
        f"{len(errors)}"
    )

    if errors:
        print()

        print(
            "FAILED LIMIT "
            "EXTENSION RUNS"
        )

        for i, result in enumerate(
            errors,
            start=1,
        ):
            cfg = result.get(
                "config",
                {}
            )

            kwargs = cfg.get(
                "explorer_kwargs",
                {}
            )

            sigma = kwargs.get(
                "sigma"
            )

            print(
                f"{i}. "
                f"env="
                f"{cfg.get('env_id')} "
                f"sigma="
                f"{sigma} "
                f"seed="
                f"{cfg.get('seed')}"
            )

            print(
                f"   "
                f"{result.get('error')}"
            )

        raise RuntimeError(
            f"{len(errors)} "
            "limit-extension "
            "runs failed."
        )

    if len(
        results
    ) != expected_total:
        raise RuntimeError(
            "Unexpected stored "
            "extension result count."
        )

    if len(
        successful
    ) != expected_total:
        raise RuntimeError(
            "Unexpected successful "
            "extension result count."
        )

    print()

    print(
        "LINEAR VDBE "
        "DEGENERATION LIMIT "
        "EXTENSION COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/extend_vdbe_degeneration_linear.py


In [33]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "src.scripts.extend_vdbe_degeneration_linear",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(
        line,
        end=""
    )

return_code = process.wait()

print()

print(
    "RETURN CODE:",
    return_code
)

if return_code != 0:
    raise RuntimeError(
        "Linear VDBE limit "
        "extension failed."
    )

LINEAR VDBE DEGENERATION LIMIT EXTENSION

Environments:
  MountainCar-v0: alpha_bar=0.75, budget=150000
  Acrobot-v1: alpha_bar=0.5, budget=100000

Extension sigma values:
  50000
  100000
  500000
  1e+06
  1e+07
  inf

The sigma=inf condition is the exact analytical VDBE limit.

Diagnostic seeds: (4000, 4001, 4002, 4003, 4004, 4005, 4006, 4007, 4008, 4009)
Sigma values: 6
Seeds per cell: 10
Runs per environment: 60
Total new runs: 120

Detected CPUs: 20
Workers: 8
Output: src\results\diagnostics\vdbe_degeneration_linear_extension.pkl

[linear VDBE limit extension] loaded=0 pending=120 workers=8
[linear VDBE limit extension] 1/120 OK
[linear VDBE limit extension] 2/120 OK
[linear VDBE limit extension] 3/120 OK
[linear VDBE limit extension] 4/120 OK
[linear VDBE limit extension] 5/120 OK
[linear VDBE limit extension] 6/120 OK
[linear VDBE limit extension] 7/120 OK
[linear VDBE limit extension] 8/120 OK
[linear VDBE limit extension] 9/120 OK
[linear VDBE limit extension] 10/120 OK
[line

In [34]:
%%writefile src/scripts/analyze_vdbe_limit_linear.py

import csv
import json
import pickle

from pathlib import Path

import numpy as np


BASE_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_degeneration_linear.pkl"
)

EXTENSION_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_degeneration_linear_extension.pkl"
)

SUMMARY_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_limit_linear_summary.json"
)

CELL_CSV_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_limit_linear_cells.csv"
)

LIMIT_CSV_PATH = Path(
    "src/results/diagnostics/"
    "vdbe_limit_linear_comparison.csv"
)


DIAGNOSTIC_ENVS = (
    "MountainCar-v0",
    "Acrobot-v1",
)

DIAGNOSTIC_SEEDS = tuple(
    range(
        4000,
        4010,
    )
)

BASE_SIGMAS = (
    0.5,
    1.0,
    5.0,
    20.0,
    100.0,
    500.0,
    2000.0,
    10000.0,
)

EXTENSION_FINITE_SIGMAS = (
    50_000.0,
    100_000.0,
    500_000.0,
    1_000_000.0,
    10_000_000.0,
)

ALL_FINITE_SIGMAS = (
    *BASE_SIGMAS,
    *EXTENSION_FINITE_SIGMAS,
)

LIMIT_SIGMA = float("inf")

LIMIT_CHECK_SIGMAS = (
    500.0,
    2000.0,
    10000.0,
    50_000.0,
    100_000.0,
    500_000.0,
    1_000_000.0,
    10_000_000.0,
)


def _write_json_atomic(
    obj,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

    tmp.replace(
        path
    )


def _load(
    path,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing results file: "
            f"{path}"
        )

    with open(
        path,
        "rb",
    ) as f:
        results = pickle.load(
            f
        )

    if not isinstance(
        results,
        list,
    ):
        raise TypeError(
            f"{path} must contain "
            "a list."
        )

    errors = [
        result
        for result in results
        if "error"
        in result
    ]

    if errors:
        raise RuntimeError(
            f"{path} contains "
            f"{len(errors)} "
            "error records."
        )

    return results


def _close(
    a,
    b,
):
    if (
        np.isinf(
            float(a)
        )
        and np.isinf(
            float(b)
        )
    ):
        return True

    return bool(
        np.isclose(
            float(a),
            float(b),
            rtol=0.0,
            atol=1e-12,
        )
    )


def _sigma_label(
    sigma,
):
    if np.isinf(
        sigma
    ):
        return "inf"

    if sigma >= 1e6:
        return f"{sigma:.0e}"

    return f"{sigma:g}"


def _array_equal(
    a,
    b,
):
    a = np.asarray(
        a
    )

    b = np.asarray(
        b
    )

    if (
        a.shape
        != b.shape
    ):
        return False

    try:
        return bool(
            np.array_equal(
                a,
                b,
                equal_nan=True,
            )
        )

    except TypeError:
        pass

    if (
        np.issubdtype(
            a.dtype,
            np.floating,
        )
        or np.issubdtype(
            b.dtype,
            np.floating,
        )
    ):
        if not np.array_equal(
            np.isnan(
                a
            ),
            np.isnan(
                b
            ),
        ):
            return False

        valid = (
            np.isfinite(
                a
            )
            & np.isfinite(
                b
            )
        )

        return bool(
            np.array_equal(
                a[
                    valid
                ],
                b[
                    valid
                ],
            )
        )

    return bool(
        np.array_equal(
            a,
            b,
        )
    )


def _max_abs_diff(
    a,
    b,
):
    a = np.asarray(
        a,
        dtype=np.float64,
    )

    b = np.asarray(
        b,
        dtype=np.float64,
    )

    if (
        a.shape
        != b.shape
    ):
        return float(
            "inf"
        )

    finite = (
        np.isfinite(
            a
        )
        & np.isfinite(
            b
        )
    )

    if not np.any(
        finite
    ):
        return 0.0

    return float(
        np.max(
            np.abs(
                a[
                    finite
                ]
                - b[
                    finite
                ]
            )
        )
    )


def _extract_sigma(
    result,
):
    kwargs = result[
        "explorer_kwargs"
    ]

    if (
        "sigma"
        not in kwargs
    ):
        raise KeyError(
            "Result missing sigma."
        )

    return float(
        kwargs[
            "sigma"
        ]
    )


def _validate(
    result,
):
    required = (
        "env_id",
        "algo",
        "explorer",
        "explorer_kwargs",
        "seed",
        "alpha_bar",
        "final_eval",
        "ep_steps",
        "ep_returns",
        "ep_eps",
        "ep_td",
        "return_curve",
        "epsilon_curve",
        "td_curve",
        "eval_steps",
        "eval_returns",
    )

    for field in required:
        if field not in result:
            raise KeyError(
                f"Result missing "
                f"{field}."
            )

    env_id = result[
        "env_id"
    ]

    if (
        env_id
        not in DIAGNOSTIC_ENVS
    ):
        raise ValueError(
            f"Unexpected environment: "
            f"{env_id}"
        )

    if (
        result[
            "algo"
        ]
        != "sarsa-lambda"
    ):
        raise ValueError(
            "Expected "
            "sarsa-lambda."
        )

    if (
        result[
            "explorer"
        ]
        != "vdbe"
    ):
        raise ValueError(
            "Expected VDBE."
        )

    seed = int(
        result[
            "seed"
        ]
    )

    if (
        seed
        not in DIAGNOSTIC_SEEDS
    ):
        raise ValueError(
            f"Unexpected seed: "
            f"{seed}"
        )

    sigma = _extract_sigma(
        result
    )

    valid_sigma = (
        np.isinf(
            sigma
        )
        or any(
            _close(
                sigma,
                candidate,
            )
            for candidate
            in ALL_FINITE_SIGMAS
        )
    )

    if not valid_sigma:
        raise ValueError(
            f"Unexpected sigma: "
            f"{sigma}"
        )

    return (
        env_id,
        sigma,
        seed,
    )


def _get(
    grouped,
    env_id,
    sigma,
    seed,
):
    matches = [
        result
        for (
            key_env,
            key_sigma,
            key_seed
        ), result
        in grouped.items()
        if (
            key_env
            == env_id
            and _close(
                key_sigma,
                sigma,
            )
            and key_seed
            == seed
        )
    ]

    if (
        len(
            matches
        )
        != 1
    ):
        raise RuntimeError(
            "Expected exactly one "
            f"record for "
            f"{env_id}, "
            f"sigma="
            f"{_sigma_label(sigma)}, "
            f"seed={seed}; "
            f"found "
            f"{len(matches)}."
        )

    return matches[
        0
    ]


def _stored_learning_equal(
    finite,
    limit,
):
    fields = (
        "ep_steps",
        "ep_returns",
        "ep_td",
        "return_curve",
        "td_curve",
        "eval_steps",
        "eval_returns",
    )

    checks = {
        field: _array_equal(
            finite[
                field
            ],
            limit[
                field
            ],
        )
        for field in fields
    }

    checks[
        "final_eval"
    ] = bool(
        float(
            finite[
                "final_eval"
            ]
        )
        == float(
            limit[
                "final_eval"
            ]
        )
    )

    exact = bool(
        all(
            checks.values()
        )
    )

    return (
        exact,
        checks,
    )


def main():
    base = _load(
        BASE_PATH
    )

    extension = _load(
        EXTENSION_PATH
    )

    combined = (
        base
        + extension
    )

    expected_total = (
        2
        * (
            len(
                ALL_FINITE_SIGMAS
            )
            + 1
        )
        * len(
            DIAGNOSTIC_SEEDS
        )
    )

    print(
        "LINEAR VDBE "
        "TRUE-LIMIT ANALYSIS"
    )

    print()

    print(
        f"Original records: "
        f"{len(base)}"
    )

    print(
        f"Extension records: "
        f"{len(extension)}"
    )

    print(
        f"Combined records: "
        f"{len(combined)}"
    )

    print(
        f"Expected records: "
        f"{expected_total}"
    )

    if (
        len(combined)
        != expected_total
    ):
        raise RuntimeError(
            "Unexpected combined "
            "result count."
        )

    grouped = {}

    for result in combined:
        (
            env_id,
            sigma,
            seed,
        ) = _validate(
            result
        )

        key = (
            env_id,
            sigma,
            seed,
        )

        if key in grouped:
            raise RuntimeError(
                f"Duplicate result: "
                f"{key}"
            )

        grouped[
            key
        ] = result

    if (
        len(grouped)
        != expected_total
    ):
        raise RuntimeError(
            "Unique result count "
            "does not match "
            "expected count."
        )

    print(
        f"Unique records: "
        f"{len(grouped)}"
    )

    print()

    summary = {}
    cell_rows = []
    limit_rows = []

    for env_id in DIAGNOSTIC_ENVS:
        print(
            "=" * 116
        )

        print(
            f"Environment: "
            f"{env_id}"
        )

        print(
            "=" * 116
        )

        print()

        all_sigmas = (
            *ALL_FINITE_SIGMAS,
            LIMIT_SIGMA,
        )

        cells = []

        for sigma in all_sigmas:
            scores = np.asarray(
                [
                    float(
                        _get(
                            grouped,
                            env_id,
                            sigma,
                            seed,
                        )[
                            "final_eval"
                        ]
                    )
                    for seed
                    in DIAGNOSTIC_SEEDS
                ],
                dtype=np.float64,
            )

            cell = {
                "sigma": (
                    _sigma_label(
                        sigma
                    )
                ),
                "sigma_is_infinite": bool(
                    np.isinf(
                        sigma
                    )
                ),
                "mean_final_eval": float(
                    np.mean(
                        scores
                    )
                ),
                "std_final_eval": float(
                    np.std(
                        scores,
                        ddof=1,
                    )
                ),
                "seed_scores": {
                    str(seed): float(
                        scores[
                            i
                        ]
                    )
                    for i, seed
                    in enumerate(
                        DIAGNOSTIC_SEEDS
                    )
                },
            }

            cells.append(
                cell
            )

            cell_rows.append(
                {
                    "env_id": (
                        env_id
                    ),
                    "sigma": (
                        _sigma_label(
                            sigma
                        )
                    ),
                    "mean_final_eval": (
                        cell[
                            "mean_final_eval"
                        ]
                    ),
                    "std_final_eval": (
                        cell[
                            "std_final_eval"
                        ]
                    ),
                }
            )

        limit_cell = next(
            cell
            for cell in cells
            if cell[
                "sigma_is_infinite"
            ]
        )

        limit_mean = float(
            limit_cell[
                "mean_final_eval"
            ]
        )

        finite_cells = [
            cell
            for cell in cells
            if not cell[
                "sigma_is_infinite"
            ]
        ]

        best_finite = max(
            finite_cells,
            key=lambda item: (
                item[
                    "mean_final_eval"
                ],
                -item[
                    "std_final_eval"
                ],
            ),
        )

        overall_best = max(
            cells,
            key=lambda item: (
                item[
                    "mean_final_eval"
                ],
                -item[
                    "std_final_eval"
                ],
            ),
        )

        print(
            "FULL SIGMA SWEEP"
        )

        print(
            "-" * 116
        )

        print(
            "Sigma          "
            "Mean final return     "
            "Std             "
            "Difference from inf"
        )

        print(
            "-" * 116
        )

        for cell in cells:
            diff = (
                cell[
                    "mean_final_eval"
                ]
                - limit_mean
            )

            print(
                f"{cell['sigma']:<15}"
                f"{cell['mean_final_eval']:>18.10f}"
                f"{cell['std_final_eval']:>17.10f}"
                f"{diff:>22.10f}"
            )

        print()

        print(
            "Best finite sigma:"
        )

        print(
            f"  sigma = "
            f"{best_finite['sigma']}"
        )

        print(
            f"  mean = "
            f"{best_finite['mean_final_eval']:.10f}"
        )

        print()

        print(
            "Exact analytical limit:"
        )

        print(
            f"  sigma = inf"
        )

        print(
            f"  mean = "
            f"{limit_mean:.10f}"
        )

        print()

        print(
            "Overall best condition:"
        )

        print(
            f"  sigma = "
            f"{overall_best['sigma']}"
        )

        print(
            f"  mean = "
            f"{overall_best['mean_final_eval']:.10f}"
        )

        print()
        print()

        print(
            "FINITE SIGMA "
            "VERSUS TRUE LIMIT"
        )

        print(
            "-" * 116
        )

        print(
            "Sigma       "
            "Mean |seed-score diff|   "
            "Max score diff   "
            "Exact final   "
            "Exact eval curve   "
            "Exact stored trajectory"
        )

        print(
            "-" * 116
        )

        comparisons = []

        for sigma in LIMIT_CHECK_SIGMAS:
            final_exact_count = 0
            eval_exact_count = 0
            stored_exact_count = 0
            epsilon_exact_count = 0

            score_diffs = []
            max_eval_diff = 0.0
            max_return_curve_diff = 0.0
            max_td_curve_diff = 0.0
            max_epsilon_curve_diff = 0.0
            max_episode_epsilon_diff = 0.0

            seed_details = []

            for seed in DIAGNOSTIC_SEEDS:
                finite = _get(
                    grouped,
                    env_id,
                    sigma,
                    seed,
                )

                limit = _get(
                    grouped,
                    env_id,
                    LIMIT_SIGMA,
                    seed,
                )

                (
                    stored_exact,
                    checks,
                ) = _stored_learning_equal(
                    finite,
                    limit,
                )

                epsilon_curve_exact = (
                    _array_equal(
                        finite[
                            "epsilon_curve"
                        ],
                        limit[
                            "epsilon_curve"
                        ],
                    )
                )

                score_diff = abs(
                    float(
                        finite[
                            "final_eval"
                        ]
                    )
                    - float(
                        limit[
                            "final_eval"
                        ]
                    )
                )

                eval_diff = (
                    _max_abs_diff(
                        finite[
                            "eval_returns"
                        ],
                        limit[
                            "eval_returns"
                        ],
                    )
                )

                return_curve_diff = (
                    _max_abs_diff(
                        finite[
                            "return_curve"
                        ],
                        limit[
                            "return_curve"
                        ],
                    )
                )

                td_curve_diff = (
                    _max_abs_diff(
                        finite[
                            "td_curve"
                        ],
                        limit[
                            "td_curve"
                        ],
                    )
                )

                epsilon_curve_diff = (
                    _max_abs_diff(
                        finite[
                            "epsilon_curve"
                        ],
                        limit[
                            "epsilon_curve"
                        ],
                    )
                )

                episode_epsilon_diff = (
                    _max_abs_diff(
                        finite[
                            "ep_eps"
                        ],
                        limit[
                            "ep_eps"
                        ],
                    )
                )

                final_exact_count += int(
                    checks[
                        "final_eval"
                    ]
                )

                eval_exact_count += int(
                    checks[
                        "eval_returns"
                    ]
                )

                stored_exact_count += int(
                    stored_exact
                )

                epsilon_exact_count += int(
                    epsilon_curve_exact
                )

                score_diffs.append(
                    score_diff
                )

                max_eval_diff = max(
                    max_eval_diff,
                    eval_diff,
                )

                max_return_curve_diff = max(
                    max_return_curve_diff,
                    return_curve_diff,
                )

                max_td_curve_diff = max(
                    max_td_curve_diff,
                    td_curve_diff,
                )

                max_epsilon_curve_diff = max(
                    max_epsilon_curve_diff,
                    epsilon_curve_diff,
                )

                max_episode_epsilon_diff = max(
                    max_episode_epsilon_diff,
                    episode_epsilon_diff,
                )

                seed_details.append(
                    {
                        "seed": int(
                            seed
                        ),
                        "final_eval_exact": bool(
                            checks[
                                "final_eval"
                            ]
                        ),
                        "eval_curve_exact": bool(
                            checks[
                                "eval_returns"
                            ]
                        ),
                        "stored_trajectory_exact": bool(
                            stored_exact
                        ),
                        "epsilon_curve_exact": bool(
                            epsilon_curve_exact
                        ),
                        "final_score_abs_diff": float(
                            score_diff
                        ),
                        "max_eval_curve_diff": float(
                            eval_diff
                        ),
                        "max_return_curve_diff": float(
                            return_curve_diff
                        ),
                        "max_td_curve_diff": float(
                            td_curve_diff
                        ),
                        "max_epsilon_curve_diff": float(
                            epsilon_curve_diff
                        ),
                        "max_episode_epsilon_diff": float(
                            episode_epsilon_diff
                        ),
                    }
                )

            score_diffs = np.asarray(
                score_diffs,
                dtype=np.float64,
            )

            item = {
                "sigma": (
                    _sigma_label(
                        sigma
                    )
                ),
                "mean_abs_seed_score_diff": float(
                    np.mean(
                        score_diffs
                    )
                ),
                "max_abs_seed_score_diff": float(
                    np.max(
                        score_diffs
                    )
                ),
                "exact_final_eval_seeds": int(
                    final_exact_count
                ),
                "exact_eval_curve_seeds": int(
                    eval_exact_count
                ),
                "exact_stored_trajectory_seeds": int(
                    stored_exact_count
                ),
                "exact_epsilon_curve_seeds": int(
                    epsilon_exact_count
                ),
                "max_eval_curve_diff": float(
                    max_eval_diff
                ),
                "max_return_curve_diff": float(
                    max_return_curve_diff
                ),
                "max_td_curve_diff": float(
                    max_td_curve_diff
                ),
                "max_epsilon_curve_diff": float(
                    max_epsilon_curve_diff
                ),
                "max_episode_epsilon_diff": float(
                    max_episode_epsilon_diff
                ),
                "seed_details": (
                    seed_details
                ),
            }

            comparisons.append(
                item
            )

            limit_rows.append(
                {
                    "env_id": (
                        env_id
                    ),
                    "sigma": (
                        item[
                            "sigma"
                        ]
                    ),
                    "mean_abs_seed_score_diff": (
                        item[
                            "mean_abs_seed_score_diff"
                        ]
                    ),
                    "max_abs_seed_score_diff": (
                        item[
                            "max_abs_seed_score_diff"
                        ]
                    ),
                    "exact_final_eval_seeds": (
                        final_exact_count
                    ),
                    "exact_eval_curve_seeds": (
                        eval_exact_count
                    ),
                    "exact_stored_trajectory_seeds": (
                        stored_exact_count
                    ),
                    "exact_epsilon_curve_seeds": (
                        epsilon_exact_count
                    ),
                    "max_epsilon_curve_diff": (
                        max_epsilon_curve_diff
                    ),
                }
            )

            print(
                f"{_sigma_label(sigma):<12}"
                f"{item['mean_abs_seed_score_diff']:>20.10f}"
                f"{item['max_abs_seed_score_diff']:>17.10f}"
                f"{final_exact_count:>10}/10"
                f"{eval_exact_count:>16}/10"
                f"{stored_exact_count:>18}/10"
            )

        print()
        print(
            "EPSILON DISTANCE "
            "FROM TRUE LIMIT"
        )

        print(
            "-" * 116
        )

        for item in comparisons:
            print(
                f"sigma="
                f"{item['sigma']:<8} "
                f"max epsilon-curve diff="
                f"{item['max_epsilon_curve_diff']:.16g}, "
                f"exact epsilon curves="
                f"{item['exact_epsilon_curve_seeds']}/10"
            )

        print()

        smallest_exact_final = None
        smallest_exact_eval = None
        smallest_exact_stored = None

        for item in comparisons:
            sigma_value = float(
                item[
                    "sigma"
                ]
            )

            if (
                smallest_exact_final
                is None
                and item[
                    "exact_final_eval_seeds"
                ]
                == len(
                    DIAGNOSTIC_SEEDS
                )
            ):
                smallest_exact_final = (
                    sigma_value
                )

            if (
                smallest_exact_eval
                is None
                and item[
                    "exact_eval_curve_seeds"
                ]
                == len(
                    DIAGNOSTIC_SEEDS
                )
            ):
                smallest_exact_eval = (
                    sigma_value
                )

            if (
                smallest_exact_stored
                is None
                and item[
                    "exact_stored_trajectory_seeds"
                ]
                == len(
                    DIAGNOSTIC_SEEDS
                )
            ):
                smallest_exact_stored = (
                    sigma_value
                )

        print(
            "CONVERGENCE SUMMARY"
        )

        print(
            "-" * 116
        )

        print(
            "Smallest tested finite "
            "sigma with exact final "
            "evaluation on all 10 seeds: "
            f"{smallest_exact_final}"
        )

        print(
            "Smallest tested finite "
            "sigma with exact full "
            "20-point eval curve on "
            "all 10 seeds: "
            f"{smallest_exact_eval}"
        )

        print(
            "Smallest tested finite "
            "sigma with exact stored "
            "learning/performance "
            "summaries on all 10 seeds: "
            f"{smallest_exact_stored}"
        )

        print()

        limit_is_best = bool(
            overall_best[
                "sigma_is_infinite"
            ]
        )

        highest_finite = next(
            cell
            for cell in cells
            if (
                cell[
                    "sigma"
                ]
                == _sigma_label(
                    10_000_000.0
                )
            )
        )

        high_finite_to_limit_mean_gap = (
            highest_finite[
                "mean_final_eval"
            ]
            - limit_mean
        )

        print(
            "True limit is the "
            "best mean-performing "
            "condition: "
            f"{limit_is_best}"
        )

        print(
            "Mean-return difference "
            "at sigma=1e7 versus inf: "
            f"{high_finite_to_limit_mean_gap:+.10f}"
        )

        print()
        print()

        summary[
            env_id
        ] = {
            "cells": (
                cells
            ),
            "best_finite": (
                best_finite
            ),
            "limit": (
                limit_cell
            ),
            "overall_best": (
                overall_best
            ),
            "true_limit_is_best": (
                limit_is_best
            ),
            "sigma_1e7_minus_limit_mean": float(
                high_finite_to_limit_mean_gap
            ),
            "limit_comparisons": (
                comparisons
            ),
            "smallest_exact_final_sigma": (
                None
                if smallest_exact_final
                is None
                else _sigma_label(
                    smallest_exact_final
                )
            ),
            "smallest_exact_eval_curve_sigma": (
                None
                if smallest_exact_eval
                is None
                else _sigma_label(
                    smallest_exact_eval
                )
            ),
            "smallest_exact_stored_trajectory_sigma": (
                None
                if smallest_exact_stored
                is None
                else _sigma_label(
                    smallest_exact_stored
                )
            ),
        }

    _write_json_atomic(
        summary,
        SUMMARY_PATH,
    )

    CELL_CSV_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        CELL_CSV_PATH,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=(
                "env_id",
                "sigma",
                "mean_final_eval",
                "std_final_eval",
            ),
        )

        writer.writeheader()

        writer.writerows(
            cell_rows
        )

    with open(
        LIMIT_CSV_PATH,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=(
                "env_id",
                "sigma",
                "mean_abs_seed_score_diff",
                "max_abs_seed_score_diff",
                "exact_final_eval_seeds",
                "exact_eval_curve_seeds",
                "exact_stored_trajectory_seeds",
                "exact_epsilon_curve_seeds",
                "max_epsilon_curve_diff",
            ),
        )

        writer.writeheader()

        writer.writerows(
            limit_rows
        )

    print(
        "=" * 116
    )

    print(
        "OVERALL EXPERIMENT-B "
        "TRUE-LIMIT RESULT"
    )

    print(
        "=" * 116
    )

    print()

    for env_id in DIAGNOSTIC_ENVS:
        item = summary[
            env_id
        ]

        print(
            f"{env_id}:"
        )

        print(
            f"  best finite sigma = "
            f"{item['best_finite']['sigma']}"
        )

        print(
            f"  best finite mean = "
            f"{item['best_finite']['mean_final_eval']:.10f}"
        )

        print(
            f"  inf-limit mean = "
            f"{item['limit']['mean_final_eval']:.10f}"
        )

        print(
            f"  true limit best = "
            f"{item['true_limit_is_best']}"
        )

        print(
            "  first exact-final "
            "finite sigma = "
            f"{item['smallest_exact_final_sigma']}"
        )

        print(
            "  first exact-eval-curve "
            "finite sigma = "
            f"{item['smallest_exact_eval_curve_sigma']}"
        )

        print(
            "  first exact-stored-"
            "trajectory finite sigma = "
            f"{item['smallest_exact_stored_trajectory_sigma']}"
        )

        print()

    print(
        "Summary JSON:"
    )

    print(
        SUMMARY_PATH
    )

    print()

    print(
        "Cell CSV:"
    )

    print(
        CELL_CSV_PATH
    )

    print()

    print(
        "Limit comparison CSV:"
    )

    print(
        LIMIT_CSV_PATH
    )

    print()

    print(
        "LINEAR VDBE "
        "TRUE-LIMIT ANALYSIS "
        "COMPLETED"
    )


if __name__ == "__main__":
    main()

Writing src/scripts/analyze_vdbe_limit_linear.py


In [35]:
import subprocess
import sys

process = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.scripts.analyze_vdbe_limit_linear",
    ],
    capture_output=True,
    text=True,
)

print(
    "RETURN CODE:",
    process.returncode
)

print()

print(
    "STDOUT:"
)

print(
    process.stdout
)

print(
    "STDERR:"
)

print(
    process.stderr
)

RETURN CODE: 1

STDOUT:
LINEAR VDBE TRUE-LIMIT ANALYSIS

Original records: 160
Extension records: 120
Combined records: 280
Expected records: 280
Unique records: 280

Environment: MountainCar-v0

FULL SIGMA SWEEP
--------------------------------------------------------------------------------------------------------------------
Sigma          Mean final return     Std             Difference from inf
--------------------------------------------------------------------------------------------------------------------
0.5               -200.0000000000     0.0000000000        -99.7000000000
1                 -200.0000000000     0.0000000000        -99.7000000000
5                 -189.6400000000    22.6403867262        -89.3400000000
20                -159.0500000000    24.8426716223        -58.7500000000
100               -118.4600000000    19.5403969026        -18.1600000000
500               -105.6000000000    10.5953868368         -5.3000000000
2000              -105.3800000000    10.61

At sufficiently large finite $\sigma$, the remaining numerical dependence of VDBE on TD error becomes too small to alter any of the stored learning/performance outcomes across all tested seeds, producing exact empirical equivalence to the TD-independent limit in those diagnostics.

Therefore our eventual paper claim must be:

VDBE converges empirically to its degenerate limit